# PIXOR — train KITTI 50 epoch, split 3.712 / 3.769

Notebook độc lập: nhận KITTI raw hoặc RAR/ZIP/TAR như Notebook Robust.
Bật GPU, sửa hai đường dẫn rồi chạy từ trên xuống. Train **từ đầu, tổng 50 epoch**,
giữ nguyên ID train/val đã kiểm tra với OpenPCDet; không chia ngẫu nhiên lại.

Đây là **PIXOR reproduction — Car / BEV**: input occupancy + reflectance 36 kênh,
ResNet bottleneck `[3,6,6,3]`, top-down fusion và dense classification/regression.
Không phải checkpoint chính thức của tác giả; recipe được ghi đầy đủ trong output.
Không có nhánh z/height, không báo AP3D hoặc macro ba lớp.

- Lưu `last.pth` mỗi epoch, `best.pth` theo Car Moderate R40 và checkpoint mỗi 5 epoch.
- Resume giữ optimizer/scheduler/scaler; `EPOCHS=50` là tổng số epoch, không cộng thêm.
- Chấm Car APBEV@0.7 R40/R11 trên toàn bộ 3.769 val bằng reference evaluator đã ghim.
- Đo FP32 batch 1 trên GPU hiện tại: preprocessing, H2D, network, decode/NMS, pipeline.
- Code, split và evaluator được nhúng sẵn; không cần upload project hoặc cài OpenPCDet.

## 1. Cấu hình

KITTI cần đủ bin/calib/labels của 7.481 frame, có thể nằm trong archive riêng.
Metadata kích thước ảnh theo từng ID được nhúng từ Robust; không cần pixel ảnh.
Đặt output trên Drive để giữ checkpoint khi phiên Colab kết thúc.

Mặc định batch 2 × tích lũy 4 = effective batch 8, AMP khi train. Nếu smoke bị OOM,
đổi batch 1 × tích lũy 8 **trước khi train**. Đổi recipe sau khi có checkpoint cần
`RUN_NAME` mới. Giữ `RUN_NAME` cũ và `RESUME='auto'` để tiếp tục một lượt train.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

ON_COLAB = 'COLAB_RELEASE_TAG' in os.environ or Path('/content').is_dir()
MOUNT_DRIVE = ON_COLAB
KITTI_ROOT = '/content/drive/MyDrive/KITTI_DATASET_ZIP'  # Thư mục RAR/ZIP/TAR hoặc KITTI raw
OUTPUT_ROOT = '/content/drive/MyDrive/lidar_artifacts/pixor'  # SỬA nếu cần
WORK_DIR = '/content/pixor_kitti_training' if ON_COLAB else str(Path.cwd() / 'artifacts/pixor_work')
EXTRACT_DIR = '/content/kitti_raw' if ON_COLAB else str(Path(WORK_DIR).parent / 'kitti_raw')
IMAGE_SIZES_JSON = None  # Có thể dùng metadata riêng; None => tự tìm như Robust
USE_BUNDLED_IMAGE_SIZES = True  # Cùng metadata image_2 theo từng ID của Notebook Robust
RUN_NAME = 'pixor_car_bev_50ep_seed42'
GPU_INDEX = '0'
EPOCHS = 50
BATCH_SIZE = 2
ACCUMULATION_STEPS = 4
WORKERS = 2  # Dùng 0 nếu runtime không cho tạo workers
AMP = True
COPY_DATA_TO_LOCAL = True  # Cần khoảng 30 GB cho bin/calib/labels; tự kiểm tra dung lượng
RESUME = 'auto'  # auto => checkpoint last.pth trong RUN_NAME; '' => train mới
EVAL_EVERY = 5
EVAL_CHECKPOINT = 'best'  # Có thể đổi 'last' để đánh giá epoch 50
RUN_TRAIN = True
RUN_EVALUATE = True

WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
SETTINGS_PATH = WORK_DIR / 'pixor_settings.json'
SETTINGS = dict(kitti_root=KITTI_ROOT, output_root=OUTPUT_ROOT, work_dir=str(WORK_DIR),
    run_name=RUN_NAME, epochs=EPOCHS, batch_size=BATCH_SIZE, accumulation_steps=ACCUMULATION_STEPS,
    workers=WORKERS, amp=AMP, copy_data_to_local=COPY_DATA_TO_LOCAL, resume=RESUME,
    eval_every=EVAL_EVERY, eval_checkpoint=EVAL_CHECKPOINT, image_sizes_json=IMAGE_SIZES_JSON)
SETTINGS_PATH.write_text(json.dumps(SETTINGS, ensure_ascii=False, indent=2))
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['CUDA_VISIBLE_DEVICES'] = GPU_INDEX
PROCESS_ENV.pop('CUDA_LAUNCH_BLOCKING', None)
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
RUN_OUTPUT = Path(OUTPUT_ROOT).expanduser().resolve() / RUN_NAME

def execute(command, log=None):
    print('+ ' + shlex.join(map(str, command)), flush=True)
    if log is None:
        return subprocess.run(list(map(str, command)), env=PROCESS_ENV, check=True)
    with Path(log).open('a') as stream:
        process = subprocess.Popen(list(map(str, command)), env=PROCESS_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            stream.write(line)
            stream.flush()
        result = process.wait()
        if result:
            raise subprocess.CalledProcessError(result, command)

print('Run:', RUN_NAME, '| tổng epoch:', EPOCHS)

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    else:
        print('Drive đã mount; dùng kết nối hiện tại.')
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Nguồn KITTI:', KITTI_ROOT)

## 2. Chuẩn bị dữ liệu và cài môi trường

Nhận RAR/ZIP/TAR hoặc thư mục raw, kể cả cấu trúc lồng nhau như Notebook Robust.
Tự cài bộ đọc RAR trên Colab khi cần; giải nén đủ 7.481 ID train/val vào SSD local,
kiểm tra dung lượng và tái sử dụng file đã xác minh khi chạy lại. Archive Drive
được giữ nguyên; không chép thêm một bộ sau khi đã giải nén. Metadata image_2
nhúng từ Robust thay cho pixel ảnh. Có thể đặt `IMAGE_SIZES_JSON` riêng hoặc tắt
`USE_BUNDLED_IMAGE_SIZES` để dùng PNG.

Cần Linux, Python 3.10–3.13 và PyTorch CUDA có sẵn. Notebook tạo virtualenv kế thừa
Torch hiện tại, cài NumPy/OpenCV/Numba riêng. Không build spconv/C++ extensions.
Numba CUDA được dùng cho rotated IoU của evaluator; smoke kiểm tra kernel trước train.
Chạy cell mã thu gọn, cell dữ liệu, rồi cell cài môi trường. Mã nguồn được xuống
dòng bình thường khi mở cell để đọc.

In [ ]:
#@title Chuẩn bị mã PIXOR và split (chạy một lần, có thể thu gọn)
BUNDLE = {
    'kitti_notebook_data.py': r'''
#!/usr/bin/env python3
"""Resolve raw KITTI or component RAR/ZIP/TAR archives into a local training root."""
from __future__ import annotations

import importlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
from zipfile import ZipFile
import zlib

from kitti_training_pipeline.image_metadata import load_image_sizes

COMPONENTS = {'velodyne': '.bin', 'calib': '.txt', 'label_2': '.txt'}
ALIASES = {'velodyne': ('velodyne',), 'calib': ('calib', 'calibration'),
           'label_2': ('label_2', 'labels', 'label'), 'image_2': ('image_2', 'images', 'image')}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')
    temporary.replace(path)


def component_for(name, archive_name=''):
    path = PurePosixPath(name.replace('\\', '/'))
    parts = [part.lower() for part in path.parts]
    if any(part in ('testing', 'test', '__macosx', 'image_0', 'image_1', 'image_3',
                    'image_00', 'image_01', 'image_02', 'image_03') for part in parts):
        return None
    if not re.fullmatch(r'[0-9]{6}', path.stem):
        return None
    if '..' in parts or path.is_absolute() or re.match(r'^[A-Za-z]:', str(path)):
        raise ValueError(f'Unsafe archive path: {name}')
    if path.suffix.lower() == '.bin':
        return 'velodyne'
    choices = [key for key, aliases in ALIASES.items()
               if any(alias in parts[:-1] for alias in aliases)]
    if not choices:
        choices = [key for key, aliases in ALIASES.items()
                   if any(alias in archive_name.lower() for alias in aliases)]
    choices = [key for key in choices if path.suffix.lower() == ('.png' if key == 'image_2' else COMPONENTS[key])]
    if len(choices) > 1:
        raise ValueError(f'Ambiguous component: {name} in {archive_name}')
    return choices[0] if choices else None


def is_archive(path):
    name = path.name.lower()
    part = re.search(r'\.part(\d+)\.rar$', name)
    if part and int(part[1]) > 1:
        return False
    return name.endswith(('.rar', '.zip', '.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tar.xz'))


def walk_files(root, max_depth=5):
    root = Path(root)
    for parent, directories, filenames in os.walk(root):
        depth = len(Path(parent).relative_to(root).parts)
        directories[:] = [name for name in sorted(directories)
                          if name.lower() not in ('testing', 'test', '__macosx', '.git')
                          and not name.startswith('.') and depth < max_depth]
        for name in sorted(filenames):
            yield Path(parent) / name


def rar_support(destination, allow_install):
    support = Path(destination).parent / '.kitti_rar_support'
    if support.is_dir():
        sys.path.insert(0, str(support))
    try:
        module = importlib.import_module('rarfile')
    except ImportError:
        if not allow_install:
            raise RuntimeError('RAR input requires rarfile and unrar. Install rarfile and unrar, or use Colab automatic setup.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--target', str(support),
                        'rarfile>=4.2,<5'], check=True)
        sys.path.insert(0, str(support))
        module = importlib.import_module('rarfile')
    executable = shutil.which('unrar')
    if executable is None and allow_install and hasattr(os, 'geteuid') and os.geteuid() == 0 and shutil.which('apt-get'):
        print('Cài unrar để đọc RAR trên Colab...', flush=True)
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'unrar'], check=True)
        executable = shutil.which('unrar')
    if executable is None:
        raise RuntimeError('Missing unrar. Install the unrar command (e.g. apt-get install unrar), then rerun the data cell.')
    return module, executable


def crc32(path):
    value = 0
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value = zlib.crc32(block, value)
    return value & 0xffffffff


def valid_file(path, component):
    path = Path(path)
    if not path.is_file() or path.is_symlink() or (component != 'label_2' and path.stat().st_size == 0):
        return False
    return component != 'velodyne' or path.stat().st_size % 16 == 0


def prepare_dataset(source, destination, frame_ids, *, metadata_path=None,
                    fallback_metadata=None, allow_install=False):
    """Select exact IDs, exclude hidden test, flatten components and reuse safely.

    Image dimensions may come from the same per-frame JSON as the Robust notebook.
    RAR extraction runs once per archive (including solid RAR), not once per frame.
    """
    source, destination = Path(source).expanduser().resolve(), Path(destination).expanduser().resolve()
    identifiers = list(frame_ids)
    if not identifiers or len(set(identifiers)) != len(identifiers) or any(not re.fullmatch(r'[0-9]{6}', i) for i in identifiers):
        raise ValueError('Expected a nonempty list of unique six-digit KITTI frame IDs')
    wanted = set(identifiers)
    if not source.exists():
        raise FileNotFoundError(f'KITTI source does not exist: {source}. Set KITTI_ROOT to a RAR/ZIP/TAR file or its containing folder.')
    if source.is_dir() and (source == destination or source in destination.parents):
        # A complete source may be reused, but never recursively scan an extraction tree inside an archive source.
        if not all((source / name).is_dir() for name in COMPONENTS):
            raise ValueError('EXTRACT_DIR must be outside the archive source directory')
    files = list(walk_files(source)) if source.is_dir() else [source]
    archives = [path for path in files if is_archive(path)]
    print('KITTI source:', source, flush=True)
    print('Archives:', ', '.join(path.name for path in archives) or '(unpacked files)', flush=True)
    metadata = Path(metadata_path).expanduser().resolve() if metadata_path else None
    if metadata is None:
        candidates = [path for path in files if path.name == 'kitti_image_sizes.json']
        if len(candidates) > 1:
            raise ValueError('Multiple image metadata files found; set IMAGE_SIZES_JSON explicitly')
        metadata = candidates[0] if candidates else None
    # Explicit/provided metadata is authoritative. The bundled metadata is only fallback.
    if metadata is None and fallback_metadata:
        metadata = Path(fallback_metadata).expanduser().resolve()

    records = {}
    def add(component, identifier, record):
        if identifier not in wanted:
            return
        key = component + '/' + identifier
        if key in records:
            existing = records[key]
            if existing['kind'] == 'raw' and record['kind'] != 'raw':
                return
            raise ValueError(f'Duplicate KITTI input for {key}: {existing["source"]} and {record["source"]}')
        records[key] = record

    # Prefer existing unpacked components over their matching archives.
    for path in files:
        if is_archive(path):
            continue
        component = component_for(str(path.relative_to(source)) if source.is_dir() else path.name)
        if component:
            add(component, path.stem, dict(kind='raw', source=str(path), size=path.stat().st_size, member=None, crc=None))
    # Only parse archive members if a required file is not available unpacked.
    required_components = dict(COMPONENTS)
    if metadata is None:
        required_components['image_2'] = '.png'
    needs_archives = any(component + '/' + identifier not in records
                         for component in required_components for identifier in identifiers)
    unrar = None
    if needs_archives:
        for path in archives:
            if path.suffix.lower() == '.zip':
                with ZipFile(path) as archive:
                    for info in archive.infolist():
                        if info.is_dir():
                            continue
                        component = component_for(info.filename, path.name)
                        if component in required_components:
                            add(component, PurePosixPath(info.filename.replace('\\', '/')).stem,
                                dict(kind='zip', source=str(path), member=info.filename, size=info.file_size, crc=info.CRC))
            elif path.suffix.lower() == '.rar':
                module, unrar = rar_support(destination, allow_install)
                try:
                    with module.RarFile(path, errors='strict') as archive:
                        if archive.needs_password():
                            raise ValueError(f'Password-protected RAR is not supported: {path}')
                        for info in archive.infolist():
                            if info.is_dir():
                                continue
                            component = component_for(info.filename, path.name)
                            if component in required_components:
                                if info.is_symlink() or getattr(info, 'file_redir', None):
                                    raise ValueError(f'Archive links are not supported: {info.filename}')
                                add(component, PurePosixPath(info.filename.replace('\\', '/')).stem,
                                    dict(kind='rar', source=str(path), member=info.filename, size=info.file_size, crc=info.CRC))
                except module.Error as error:
                    raise RuntimeError(f'Cannot read RAR {path}; keep all multipart volumes together and include part1. {error}') from error
            else:
                with tarfile.open(path, 'r:*') as archive:
                    for info in archive:
                        if info.isdir():
                            continue
                        component = component_for(info.name, path.name)
                        if component in required_components:
                            if not info.isfile():
                                raise ValueError(f'Archive links/special files are not supported: {info.name}')
                            add(component, PurePosixPath(info.name).stem,
                                dict(kind='tar', source=str(path), member=info.name, size=info.size, crc=None))
    missing = {component: [i for i in identifiers if component + '/' + i not in records] for component in required_components}
    missing = {component: ids for component, ids in missing.items() if ids}
    if missing:
        details = '; '.join(f'{name}: missing {len(ids)} IDs, e.g. {ids[:5]}' for name, ids in missing.items())
        raise FileNotFoundError(f'Incomplete KITTI input ({details}). For images, supply image_2 or kitti_image_sizes.json. '
                                f'Searched {source}; found archives: {[p.name for p in archives]}')
    # Validate metadata coverage before extracting GBs of LiDAR.
    if metadata is not None:
        load_image_sizes(source if source.is_dir() else source.parent, identifiers, metadata_path=metadata)

    selected = {key: value for key, value in records.items() if key.split('/')[0] in required_components}
    raw_parents = {Path(record['source']).parent.parent for record in selected.values() if record['kind'] == 'raw'}
    direct = (all(record['kind'] == 'raw' and Path(record['source']).parent.name == key.split('/')[0]
                  for key, record in selected.items()) and len(raw_parents) == 1)
    root = next(iter(raw_parents)) if direct else destination / 'training'
    if direct:
        for key, record in selected.items():
            if not valid_file(record['source'], key.split('/')[0]):
                raise ValueError('Invalid raw KITTI file: ' + record['source'])
        load_image_sizes(root, identifiers, metadata_path=metadata)
        return dict(training_root=str(root), image_sizes_json=str(metadata) if metadata else None,
                    staged=False, counts={name: len(identifiers) for name in COMPONENTS}, reused_files=len(selected))

    destination.mkdir(parents=True, exist_ok=True)
    manifest_path = destination / 'archive_preparation.json'
    old = json.loads(manifest_path.read_text()).get('files', {}) if manifest_path.is_file() else {}
    pending, evidence, reused = [], {}, 0
    for key, record in selected.items():
        component, identifier = key.split('/')
        suffix = '.png' if component == 'image_2' else COMPONENTS[component]
        target = root / component / (identifier + suffix)
        source_stat = Path(record['source']).stat()
        identity = dict(**record, source_size=source_stat.st_size, source_mtime_ns=source_stat.st_mtime_ns)
        cached = old.get(key, {})
        matches = valid_file(target, component) and target.stat().st_size == record['size']
        if matches and cached.get('identity') == identity and cached.get('mtime_ns') == target.stat().st_mtime_ns:
            reused += 1
        elif matches and record['crc'] is not None and crc32(target) == record['crc']:
            reused += 1
        else:
            pending.append((key, record, target))
        evidence[key] = dict(identity=identity)
    required = sum(record['size'] for _, record, _ in pending)
    if pending and shutil.disk_usage(destination).free < required + 512 * 1024 * 1024:
        raise RuntimeError(f'Insufficient local disk space: need {required / 1e9:.2f} GB + 0.5 GB reserve. '
                           'Change EXTRACT_DIR or free local disk; archives stay on Drive.')
    print(f'Local KITTI: {root}; extract/copy {len(pending)} files, reuse {reused}; selected size {required / 1e9:.2f} GB', flush=True)

    def finish(key, record, target, temporary):
        if not valid_file(temporary, key.split('/')[0]) or temporary.stat().st_size != record['size']:
            raise ValueError('Invalid extracted KITTI file: ' + key)
        if record['crc'] is not None and crc32(temporary) != record['crc']:
            raise ValueError('Archive CRC mismatch: ' + key)
        temporary.replace(target)

    groups = {}
    for item in pending:
        groups.setdefault((item[1]['kind'], item[1]['source']), []).append(item)
    copied_raw = 0
    for (kind, original), items in groups.items():
        if kind != 'raw':
            print(f'Preparing {Path(original).name}: {len(items)} files', flush=True)
        if kind == 'rar':
            # Batch decompression is essential for solid/multipart RAR performance.
            with tempfile.TemporaryDirectory(prefix='.kitti_rar_', dir=destination) as temporary_dir:
                staging = Path(temporary_dir)
                members = staging / 'members.txt'
                members.write_text('\n'.join(record['member'] for _, record, _ in items) + '\n')
                subprocess.run([unrar, 'x', '-idq', '-o+', '-p-', original, '@' + str(members), str(staging) + os.sep], check=True)
                for key, record, target in items:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    extracted = staging / record['member'].replace('\\', '/')
                    if staging not in extracted.resolve().parents:
                        raise ValueError('Unsafe extracted archive path')
                    finish(key, record, target, extracted)
        else:
            archive = ZipFile(original) if kind == 'zip' else (tarfile.open(original, 'r:*') if kind == 'tar' else None)
            try:
                for key, record, target in items:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_name(target.name + '.extracting')
                    if kind == 'raw':
                        shutil.copy2(original, temporary)
                        copied_raw += 1
                        if copied_raw % 500 == 0:
                            print(f'Copied {copied_raw} raw files to local SSD', flush=True)
                    else:
                        reader = archive.open(record['member']) if kind == 'zip' else archive.extractfile(record['member'])
                        with reader as stream, temporary.open('wb') as writer:
                            shutil.copyfileobj(stream, writer, length=1024 * 1024)
                    finish(key, record, target, temporary)
            finally:
                if archive is not None:
                    archive.close()
    for key, record in evidence.items():
        component, identifier = key.split('/')
        target = root / component / (identifier + ('.png' if component == 'image_2' else COMPONENTS[component]))
        record['mtime_ns'] = target.stat().st_mtime_ns
    # Also checks PNG headers when JSON is absent; never fabricate image sizes.
    load_image_sizes(root, identifiers, metadata_path=metadata)
    result = dict(training_root=str(root), image_sizes_json=str(metadata) if metadata else None,
                  staged=True, counts={name: len(identifiers) for name in COMPONENTS}, reused_files=reused,
                  source=str(source), selected_ids=identifiers, files=evidence)
    save_json(manifest_path, result)
    print('KITTI ready:', result['counts'], '| image dimensions:', metadata or root / 'image_2', flush=True)
    return {key: value for key, value in result.items() if key != 'files'}

'''[1:-1],
    'kitti_training_pipeline/__init__.py': r'''

'''[1:-1],
    'kitti_training_pipeline/image_metadata.py': r'''
"""Resolve KITTI left-camera dimensions from per-frame JSON or PNG headers."""
from __future__ import annotations

import json
import struct
from pathlib import Path


def training_root(path):
    root = Path(path).expanduser().resolve()
    return root / 'training' if (root / 'training').is_dir() else root


def _unique_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate image metadata key: {key}')
        result[key] = value
    return result


def read_image_sizes(path):
    """Validate the versioned training/image_2 metadata and return (width, height)."""
    path = Path(path).expanduser().resolve()
    with path.open(encoding='utf-8') as stream:
        data = json.load(stream, object_pairs_hook=_unique_keys)
    if not isinstance(data, dict) or any(data.get(key) != value for key, value in {
            'format': 'kitti_image_sizes_v1', 'split': 'training', 'camera': 'image_2'}.items()):
        raise ValueError(f'Expected kitti_image_sizes_v1 for training/image_2: {path}')
    frames = data.get('frames')
    if (not isinstance(frames, dict) or not frames or type(data.get('image_count')) is not int
            or data['image_count'] != len(frames)):
        raise ValueError(f'Image metadata count must match nonempty frames: {path}')
    sizes = {}
    for identifier, frame in frames.items():
        if len(identifier) != 6 or not identifier.isascii() or not identifier.isdigit():
            raise ValueError(f'Invalid KITTI frame ID {identifier!r}: {path}')
        if not isinstance(frame, dict) or any(
                type(frame.get(key)) is not int or frame[key] <= 0 for key in ('width', 'height')):
            raise ValueError(f'Invalid width/height for frame {identifier}: {path}')
        sizes[identifier] = (frame['width'], frame['height'])
    return sizes


def png_image_size(path):
    with Path(path).open('rb') as stream:
        header = stream.read(24)
    if len(header) != 24 or header[:8] != b'\x89PNG\r\n\x1a\n' or header[12:16] != b'IHDR':
        raise ValueError(f'Expected KITTI PNG image with IHDR: {path}')
    dimensions = struct.unpack('>II', header[16:24])
    if min(dimensions) <= 0:
        raise ValueError(f'Invalid evaluation image dimensions: {path}')
    return dimensions


def load_image_sizes(kitti_root, frame_ids, *, metadata_path=None):
    """Return per-frame sizes and the actual input paths to hash for provenance.

    Explicit JSON takes precedence; otherwise look in the raw training directory
    (or its dataset parent), then retain the historical PNG-header workflow.
    A present but incomplete/invalid JSON is an error rather than a fallback.
    """
    root = training_root(kitti_root)
    ids = list(frame_ids)
    if metadata_path is not None:
        path = Path(metadata_path).expanduser().resolve()
    else:
        candidates = [root / 'kitti_image_sizes.json']
        if root.name == 'training':
            candidates.append(root.parent / 'kitti_image_sizes.json')
        path = next((candidate for candidate in candidates if candidate.is_file()), None)
    if path is not None:
        sizes = read_image_sizes(path)
        missing = [identifier for identifier in ids if identifier not in sizes]
        if missing:
            raise ValueError(f'Image metadata missing {len(missing)} requested frame(s): '
                             f'{", ".join(missing[:10])}; source: {path}')
        return {identifier: sizes[identifier] for identifier in ids}, [path]
    assets = [root / 'image_2' / f'{identifier}.png' for identifier in ids]
    return {identifier: png_image_size(path) for identifier, path in zip(ids, assets)}, assets

'''[1:-1],
    'metadata/kitti_image_sizes.json': r'''
{
  "camera": "image_2",
  "format": "kitti_image_sizes_v1",
  "frames": {
    "000000": {
      "height": 370,
      "width": 1224
    },
    "000001": {
      "height": 375,
      "width": 1242
    },
    "000002": {
      "height": 375,
      "width": 1242
    },
    "000003": {
      "height": 375,
      "width": 1242
    },
    "000004": {
      "height": 375,
      "width": 1242
    },
    "000005": {
      "height": 375,
      "width": 1242
    },
    "000006": {
      "height": 374,
      "width": 1238
    },
    "000007": {
      "height": 375,
      "width": 1242
    },
    "000008": {
      "height": 375,
      "width": 1242
    },
    "000009": {
      "height": 375,
      "width": 1242
    },
    "000010": {
      "height": 375,
      "width": 1242
    },
    "000011": {
      "height": 375,
      "width": 1242
    },
    "000012": {
      "height": 375,
      "width": 1242
    },
    "000013": {
      "height": 375,
      "width": 1242
    },
    "000014": {
      "height": 375,
      "width": 1242
    },
    "000015": {
      "height": 374,
      "width": 1238
    },
    "000016": {
      "height": 375,
      "width": 1242
    },
    "000017": {
      "height": 375,
      "width": 1242
    },
    "000018": {
      "height": 375,
      "width": 1242
    },
    "000019": {
      "height": 375,
      "width": 1242
    },
    "000020": {
      "height": 375,
      "width": 1242
    },
    "000021": {
      "height": 375,
      "width": 1242
    },
    "000022": {
      "height": 375,
      "width": 1242
    },
    "000023": {
      "height": 375,
      "width": 1242
    },
    "000024": {
      "height": 376,
      "width": 1241
    },
    "000025": {
      "height": 375,
      "width": 1242
    },
    "000026": {
      "height": 375,
      "width": 1242
    },
    "000027": {
      "height": 375,
      "width": 1242
    },
    "000028": {
      "height": 370,
      "width": 1224
    },
    "000029": {
      "height": 375,
      "width": 1242
    },
    "000030": {
      "height": 375,
      "width": 1242
    },
    "000031": {
      "height": 375,
      "width": 1242
    },
    "000032": {
      "height": 375,
      "width": 1242
    },
    "000033": {
      "height": 375,
      "width": 1242
    },
    "000034": {
      "height": 375,
      "width": 1242
    },
    "000035": {
      "height": 375,
      "width": 1242
    },
    "000036": {
      "height": 375,
      "width": 1242
    },
    "000037": {
      "height": 376,
      "width": 1241
    },
    "000038": {
      "height": 375,
      "width": 1242
    },
    "000039": {
      "height": 375,
      "width": 1242
    },
    "000040": {
      "height": 375,
      "width": 1242
    },
    "000041": {
      "height": 375,
      "width": 1242
    },
    "000042": {
      "height": 375,
      "width": 1242
    },
    "000043": {
      "height": 375,
      "width": 1242
    },
    "000044": {
      "height": 375,
      "width": 1242
    },
    "000045": {
      "height": 375,
      "width": 1242
    },
    "000046": {
      "height": 375,
      "width": 1242
    },
    "000047": {
      "height": 376,
      "width": 1241
    },
    "000048": {
      "height": 374,
      "width": 1238
    },
    "000049": {
      "height": 375,
      "width": 1242
    },
    "000050": {
      "height": 375,
      "width": 1242
    },
    "000051": {
      "height": 375,
      "width": 1242
    },
    "000052": {
      "height": 375,
      "width": 1242
    },
    "000053": {
      "height": 375,
      "width": 1242
    },
    "000054": {
      "height": 370,
      "width": 1224
    },
    "000055": {
      "height": 375,
      "width": 1242
    },
    "000056": {
      "height": 375,
      "width": 1242
    },
    "000057": {
      "height": 375,
      "width": 1242
    },
    "000058": {
      "height": 370,
      "width": 1224
    },
    "000059": {
      "height": 375,
      "width": 1242
    },
    "000060": {
      "height": 375,
      "width": 1242
    },
    "000061": {
      "height": 375,
      "width": 1242
    },
    "000062": {
      "height": 375,
      "width": 1242
    },
    "000063": {
      "height": 375,
      "width": 1242
    },
    "000064": {
      "height": 375,
      "width": 1242
    },
    "000065": {
      "height": 374,
      "width": 1238
    },
    "000066": {
      "height": 375,
      "width": 1242
    },
    "000067": {
      "height": 375,
      "width": 1242
    },
    "000068": {
      "height": 375,
      "width": 1242
    },
    "000069": {
      "height": 375,
      "width": 1242
    },
    "000070": {
      "height": 370,
      "width": 1224
    },
    "000071": {
      "height": 375,
      "width": 1242
    },
    "000072": {
      "height": 375,
      "width": 1242
    },
    "000073": {
      "height": 375,
      "width": 1242
    },
    "000074": {
      "height": 370,
      "width": 1224
    },
    "000075": {
      "height": 375,
      "width": 1242
    },
    "000076": {
      "height": 374,
      "width": 1238
    },
    "000077": {
      "height": 375,
      "width": 1242
    },
    "000078": {
      "height": 375,
      "width": 1242
    },
    "000079": {
      "height": 375,
      "width": 1242
    },
    "000080": {
      "height": 375,
      "width": 1242
    },
    "000081": {
      "height": 375,
      "width": 1242
    },
    "000082": {
      "height": 375,
      "width": 1242
    },
    "000083": {
      "height": 374,
      "width": 1238
    },
    "000084": {
      "height": 375,
      "width": 1242
    },
    "000085": {
      "height": 375,
      "width": 1242
    },
    "000086": {
      "height": 375,
      "width": 1242
    },
    "000087": {
      "height": 375,
      "width": 1242
    },
    "000088": {
      "height": 375,
      "width": 1242
    },
    "000089": {
      "height": 375,
      "width": 1242
    },
    "000090": {
      "height": 375,
      "width": 1242
    },
    "000091": {
      "height": 375,
      "width": 1242
    },
    "000092": {
      "height": 375,
      "width": 1242
    },
    "000093": {
      "height": 375,
      "width": 1242
    },
    "000094": {
      "height": 375,
      "width": 1242
    },
    "000095": {
      "height": 375,
      "width": 1242
    },
    "000096": {
      "height": 375,
      "width": 1242
    },
    "000097": {
      "height": 375,
      "width": 1242
    },
    "000098": {
      "height": 376,
      "width": 1241
    },
    "000099": {
      "height": 375,
      "width": 1242
    },
    "000100": {
      "height": 375,
      "width": 1242
    },
    "000101": {
      "height": 375,
      "width": 1242
    },
    "000102": {
      "height": 375,
      "width": 1242
    },
    "000103": {
      "height": 370,
      "width": 1224
    },
    "000104": {
      "height": 375,
      "width": 1242
    },
    "000105": {
      "height": 375,
      "width": 1242
    },
    "000106": {
      "height": 375,
      "width": 1242
    },
    "000107": {
      "height": 375,
      "width": 1242
    },
    "000108": {
      "height": 375,
      "width": 1242
    },
    "000109": {
      "height": 370,
      "width": 1224
    },
    "000110": {
      "height": 375,
      "width": 1242
    },
    "000111": {
      "height": 375,
      "width": 1242
    },
    "000112": {
      "height": 375,
      "width": 1242
    },
    "000113": {
      "height": 375,
      "width": 1242
    },
    "000114": {
      "height": 375,
      "width": 1242
    },
    "000115": {
      "height": 375,
      "width": 1242
    },
    "000116": {
      "height": 375,
      "width": 1242
    },
    "000117": {
      "height": 375,
      "width": 1242
    },
    "000118": {
      "height": 374,
      "width": 1238
    },
    "000119": {
      "height": 375,
      "width": 1242
    },
    "000120": {
      "height": 375,
      "width": 1242
    },
    "000121": {
      "height": 375,
      "width": 1242
    },
    "000122": {
      "height": 375,
      "width": 1242
    },
    "000123": {
      "height": 375,
      "width": 1242
    },
    "000124": {
      "height": 370,
      "width": 1224
    },
    "000125": {
      "height": 375,
      "width": 1242
    },
    "000126": {
      "height": 375,
      "width": 1242
    },
    "000127": {
      "height": 375,
      "width": 1242
    },
    "000128": {
      "height": 375,
      "width": 1242
    },
    "000129": {
      "height": 375,
      "width": 1242
    },
    "000130": {
      "height": 370,
      "width": 1224
    },
    "000131": {
      "height": 375,
      "width": 1242
    },
    "000132": {
      "height": 376,
      "width": 1241
    },
    "000133": {
      "height": 375,
      "width": 1242
    },
    "000134": {
      "height": 370,
      "width": 1224
    },
    "000135": {
      "height": 375,
      "width": 1242
    },
    "000136": {
      "height": 375,
      "width": 1242
    },
    "000137": {
      "height": 375,
      "width": 1242
    },
    "000138": {
      "height": 375,
      "width": 1242
    },
    "000139": {
      "height": 375,
      "width": 1242
    },
    "000140": {
      "height": 375,
      "width": 1242
    },
    "000141": {
      "height": 375,
      "width": 1242
    },
    "000142": {
      "height": 375,
      "width": 1242
    },
    "000143": {
      "height": 375,
      "width": 1242
    },
    "000144": {
      "height": 375,
      "width": 1242
    },
    "000145": {
      "height": 370,
      "width": 1224
    },
    "000146": {
      "height": 375,
      "width": 1242
    },
    "000147": {
      "height": 374,
      "width": 1238
    },
    "000148": {
      "height": 375,
      "width": 1242
    },
    "000149": {
      "height": 375,
      "width": 1242
    },
    "000150": {
      "height": 375,
      "width": 1242
    },
    "000151": {
      "height": 370,
      "width": 1224
    },
    "000152": {
      "height": 375,
      "width": 1242
    },
    "000153": {
      "height": 370,
      "width": 1224
    },
    "000154": {
      "height": 375,
      "width": 1242
    },
    "000155": {
      "height": 375,
      "width": 1242
    },
    "000156": {
      "height": 370,
      "width": 1224
    },
    "000157": {
      "height": 375,
      "width": 1242
    },
    "000158": {
      "height": 375,
      "width": 1242
    },
    "000159": {
      "height": 375,
      "width": 1242
    },
    "000160": {
      "height": 375,
      "width": 1242
    },
    "000161": {
      "height": 375,
      "width": 1242
    },
    "000162": {
      "height": 375,
      "width": 1242
    },
    "000163": {
      "height": 375,
      "width": 1242
    },
    "000164": {
      "height": 375,
      "width": 1242
    },
    "000165": {
      "height": 375,
      "width": 1242
    },
    "000166": {
      "height": 375,
      "width": 1242
    },
    "000167": {
      "height": 375,
      "width": 1242
    },
    "000168": {
      "height": 375,
      "width": 1242
    },
    "000169": {
      "height": 375,
      "width": 1242
    },
    "000170": {
      "height": 376,
      "width": 1241
    },
    "000171": {
      "height": 375,
      "width": 1242
    },
    "000172": {
      "height": 375,
      "width": 1242
    },
    "000173": {
      "height": 375,
      "width": 1242
    },
    "000174": {
      "height": 375,
      "width": 1242
    },
    "000175": {
      "height": 375,
      "width": 1242
    },
    "000176": {
      "height": 375,
      "width": 1242
    },
    "000177": {
      "height": 375,
      "width": 1242
    },
    "000178": {
      "height": 375,
      "width": 1242
    },
    "000179": {
      "height": 375,
      "width": 1242
    },
    "000180": {
      "height": 375,
      "width": 1242
    },
    "000181": {
      "height": 375,
      "width": 1242
    },
    "000182": {
      "height": 375,
      "width": 1242
    },
    "000183": {
      "height": 375,
      "width": 1242
    },
    "000184": {
      "height": 375,
      "width": 1242
    },
    "000185": {
      "height": 375,
      "width": 1242
    },
    "000186": {
      "height": 375,
      "width": 1242
    },
    "000187": {
      "height": 375,
      "width": 1242
    },
    "000188": {
      "height": 375,
      "width": 1242
    },
    "000189": {
      "height": 370,
      "width": 1224
    },
    "000190": {
      "height": 374,
      "width": 1238
    },
    "000191": {
      "height": 376,
      "width": 1241
    },
    "000192": {
      "height": 376,
      "width": 1241
    },
    "000193": {
      "height": 370,
      "width": 1224
    },
    "000194": {
      "height": 375,
      "width": 1242
    },
    "000195": {
      "height": 375,
      "width": 1242
    },
    "000196": {
      "height": 375,
      "width": 1242
    },
    "000197": {
      "height": 375,
      "width": 1242
    },
    "000198": {
      "height": 375,
      "width": 1242
    },
    "000199": {
      "height": 370,
      "width": 1224
    },
    "000200": {
      "height": 375,
      "width": 1242
    },
    "000201": {
      "height": 375,
      "width": 1242
    },
    "000202": {
      "height": 375,
      "width": 1242
    },
    "000203": {
      "height": 375,
      "width": 1242
    },
    "000204": {
      "height": 375,
      "width": 1242
    },
    "000205": {
      "height": 375,
      "width": 1242
    },
    "000206": {
      "height": 370,
      "width": 1224
    },
    "000207": {
      "height": 375,
      "width": 1242
    },
    "000208": {
      "height": 375,
      "width": 1242
    },
    "000209": {
      "height": 370,
      "width": 1224
    },
    "000210": {
      "height": 375,
      "width": 1242
    },
    "000211": {
      "height": 375,
      "width": 1242
    },
    "000212": {
      "height": 375,
      "width": 1242
    },
    "000213": {
      "height": 375,
      "width": 1242
    },
    "000214": {
      "height": 375,
      "width": 1242
    },
    "000215": {
      "height": 375,
      "width": 1242
    },
    "000216": {
      "height": 375,
      "width": 1242
    },
    "000217": {
      "height": 375,
      "width": 1242
    },
    "000218": {
      "height": 375,
      "width": 1242
    },
    "000219": {
      "height": 375,
      "width": 1242
    },
    "000220": {
      "height": 375,
      "width": 1242
    },
    "000221": {
      "height": 375,
      "width": 1242
    },
    "000222": {
      "height": 375,
      "width": 1242
    },
    "000223": {
      "height": 375,
      "width": 1242
    },
    "000224": {
      "height": 375,
      "width": 1242
    },
    "000225": {
      "height": 375,
      "width": 1242
    },
    "000226": {
      "height": 375,
      "width": 1242
    },
    "000227": {
      "height": 375,
      "width": 1242
    },
    "000228": {
      "height": 375,
      "width": 1242
    },
    "000229": {
      "height": 375,
      "width": 1242
    },
    "000230": {
      "height": 375,
      "width": 1242
    },
    "000231": {
      "height": 375,
      "width": 1242
    },
    "000232": {
      "height": 375,
      "width": 1242
    },
    "000233": {
      "height": 375,
      "width": 1242
    },
    "000234": {
      "height": 375,
      "width": 1242
    },
    "000235": {
      "height": 375,
      "width": 1242
    },
    "000236": {
      "height": 376,
      "width": 1241
    },
    "000237": {
      "height": 375,
      "width": 1242
    },
    "000238": {
      "height": 375,
      "width": 1242
    },
    "000239": {
      "height": 375,
      "width": 1242
    },
    "000240": {
      "height": 375,
      "width": 1242
    },
    "000241": {
      "height": 375,
      "width": 1242
    },
    "000242": {
      "height": 375,
      "width": 1242
    },
    "000243": {
      "height": 375,
      "width": 1242
    },
    "000244": {
      "height": 375,
      "width": 1242
    },
    "000245": {
      "height": 375,
      "width": 1242
    },
    "000246": {
      "height": 376,
      "width": 1241
    },
    "000247": {
      "height": 375,
      "width": 1242
    },
    "000248": {
      "height": 376,
      "width": 1241
    },
    "000249": {
      "height": 376,
      "width": 1241
    },
    "000250": {
      "height": 375,
      "width": 1242
    },
    "000251": {
      "height": 374,
      "width": 1238
    },
    "000252": {
      "height": 375,
      "width": 1242
    },
    "000253": {
      "height": 375,
      "width": 1242
    },
    "000254": {
      "height": 375,
      "width": 1242
    },
    "000255": {
      "height": 375,
      "width": 1242
    },
    "000256": {
      "height": 375,
      "width": 1242
    },
    "000257": {
      "height": 375,
      "width": 1242
    },
    "000258": {
      "height": 375,
      "width": 1242
    },
    "000259": {
      "height": 374,
      "width": 1238
    },
    "000260": {
      "height": 375,
      "width": 1242
    },
    "000261": {
      "height": 375,
      "width": 1242
    },
    "000262": {
      "height": 375,
      "width": 1242
    },
    "000263": {
      "height": 370,
      "width": 1224
    },
    "000264": {
      "height": 375,
      "width": 1242
    },
    "000265": {
      "height": 375,
      "width": 1242
    },
    "000266": {
      "height": 375,
      "width": 1242
    },
    "000267": {
      "height": 375,
      "width": 1242
    },
    "000268": {
      "height": 375,
      "width": 1242
    },
    "000269": {
      "height": 375,
      "width": 1242
    },
    "000270": {
      "height": 375,
      "width": 1242
    },
    "000271": {
      "height": 375,
      "width": 1242
    },
    "000272": {
      "height": 375,
      "width": 1242
    },
    "000273": {
      "height": 375,
      "width": 1242
    },
    "000274": {
      "height": 375,
      "width": 1242
    },
    "000275": {
      "height": 375,
      "width": 1242
    },
    "000276": {
      "height": 375,
      "width": 1242
    },
    "000277": {
      "height": 375,
      "width": 1242
    },
    "000278": {
      "height": 375,
      "width": 1242
    },
    "000279": {
      "height": 375,
      "width": 1242
    },
    "000280": {
      "height": 375,
      "width": 1242
    },
    "000281": {
      "height": 375,
      "width": 1242
    },
    "000282": {
      "height": 375,
      "width": 1242
    },
    "000283": {
      "height": 375,
      "width": 1242
    },
    "000284": {
      "height": 375,
      "width": 1242
    },
    "000285": {
      "height": 375,
      "width": 1242
    },
    "000286": {
      "height": 375,
      "width": 1242
    },
    "000287": {
      "height": 375,
      "width": 1242
    },
    "000288": {
      "height": 370,
      "width": 1224
    },
    "000289": {
      "height": 376,
      "width": 1241
    },
    "000290": {
      "height": 370,
      "width": 1224
    },
    "000291": {
      "height": 375,
      "width": 1242
    },
    "000292": {
      "height": 375,
      "width": 1242
    },
    "000293": {
      "height": 375,
      "width": 1242
    },
    "000294": {
      "height": 370,
      "width": 1224
    },
    "000295": {
      "height": 375,
      "width": 1242
    },
    "000296": {
      "height": 375,
      "width": 1242
    },
    "000297": {
      "height": 375,
      "width": 1242
    },
    "000298": {
      "height": 370,
      "width": 1224
    },
    "000299": {
      "height": 375,
      "width": 1242
    },
    "000300": {
      "height": 375,
      "width": 1242
    },
    "000301": {
      "height": 376,
      "width": 1241
    },
    "000302": {
      "height": 374,
      "width": 1238
    },
    "000303": {
      "height": 375,
      "width": 1242
    },
    "000304": {
      "height": 375,
      "width": 1242
    },
    "000305": {
      "height": 374,
      "width": 1238
    },
    "000306": {
      "height": 375,
      "width": 1242
    },
    "000307": {
      "height": 375,
      "width": 1242
    },
    "000308": {
      "height": 370,
      "width": 1224
    },
    "000309": {
      "height": 375,
      "width": 1242
    },
    "000310": {
      "height": 375,
      "width": 1242
    },
    "000311": {
      "height": 375,
      "width": 1242
    },
    "000312": {
      "height": 375,
      "width": 1242
    },
    "000313": {
      "height": 375,
      "width": 1242
    },
    "000314": {
      "height": 375,
      "width": 1242
    },
    "000315": {
      "height": 375,
      "width": 1242
    },
    "000316": {
      "height": 375,
      "width": 1242
    },
    "000317": {
      "height": 370,
      "width": 1224
    },
    "000318": {
      "height": 375,
      "width": 1242
    },
    "000319": {
      "height": 375,
      "width": 1242
    },
    "000320": {
      "height": 375,
      "width": 1242
    },
    "000321": {
      "height": 375,
      "width": 1242
    },
    "000322": {
      "height": 375,
      "width": 1242
    },
    "000323": {
      "height": 375,
      "width": 1242
    },
    "000324": {
      "height": 370,
      "width": 1224
    },
    "000325": {
      "height": 375,
      "width": 1242
    },
    "000326": {
      "height": 375,
      "width": 1242
    },
    "000327": {
      "height": 370,
      "width": 1224
    },
    "000328": {
      "height": 375,
      "width": 1242
    },
    "000329": {
      "height": 375,
      "width": 1242
    },
    "000330": {
      "height": 375,
      "width": 1242
    },
    "000331": {
      "height": 375,
      "width": 1242
    },
    "000332": {
      "height": 374,
      "width": 1238
    },
    "000333": {
      "height": 370,
      "width": 1224
    },
    "000334": {
      "height": 375,
      "width": 1242
    },
    "000335": {
      "height": 375,
      "width": 1242
    },
    "000336": {
      "height": 374,
      "width": 1238
    },
    "000337": {
      "height": 370,
      "width": 1224
    },
    "000338": {
      "height": 370,
      "width": 1224
    },
    "000339": {
      "height": 375,
      "width": 1242
    },
    "000340": {
      "height": 375,
      "width": 1242
    },
    "000341": {
      "height": 374,
      "width": 1238
    },
    "000342": {
      "height": 375,
      "width": 1242
    },
    "000343": {
      "height": 375,
      "width": 1242
    },
    "000344": {
      "height": 375,
      "width": 1242
    },
    "000345": {
      "height": 376,
      "width": 1241
    },
    "000346": {
      "height": 375,
      "width": 1242
    },
    "000347": {
      "height": 375,
      "width": 1242
    },
    "000348": {
      "height": 375,
      "width": 1242
    },
    "000349": {
      "height": 375,
      "width": 1242
    },
    "000350": {
      "height": 375,
      "width": 1242
    },
    "000351": {
      "height": 376,
      "width": 1241
    },
    "000352": {
      "height": 375,
      "width": 1242
    },
    "000353": {
      "height": 375,
      "width": 1242
    },
    "000354": {
      "height": 375,
      "width": 1242
    },
    "000355": {
      "height": 375,
      "width": 1242
    },
    "000356": {
      "height": 375,
      "width": 1242
    },
    "000357": {
      "height": 376,
      "width": 1241
    },
    "000358": {
      "height": 375,
      "width": 1242
    },
    "000359": {
      "height": 375,
      "width": 1242
    },
    "000360": {
      "height": 375,
      "width": 1242
    },
    "000361": {
      "height": 375,
      "width": 1242
    },
    "000362": {
      "height": 375,
      "width": 1242
    },
    "000363": {
      "height": 375,
      "width": 1242
    },
    "000364": {
      "height": 375,
      "width": 1242
    },
    "000365": {
      "height": 370,
      "width": 1224
    },
    "000366": {
      "height": 370,
      "width": 1224
    },
    "000367": {
      "height": 375,
      "width": 1242
    },
    "000368": {
      "height": 370,
      "width": 1224
    },
    "000369": {
      "height": 376,
      "width": 1241
    },
    "000370": {
      "height": 375,
      "width": 1242
    },
    "000371": {
      "height": 375,
      "width": 1242
    },
    "000372": {
      "height": 375,
      "width": 1242
    },
    "000373": {
      "height": 375,
      "width": 1242
    },
    "000374": {
      "height": 375,
      "width": 1242
    },
    "000375": {
      "height": 374,
      "width": 1238
    },
    "000376": {
      "height": 375,
      "width": 1242
    },
    "000377": {
      "height": 375,
      "width": 1242
    },
    "000378": {
      "height": 375,
      "width": 1242
    },
    "000379": {
      "height": 375,
      "width": 1242
    },
    "000380": {
      "height": 375,
      "width": 1242
    },
    "000381": {
      "height": 370,
      "width": 1224
    },
    "000382": {
      "height": 375,
      "width": 1242
    },
    "000383": {
      "height": 370,
      "width": 1224
    },
    "000384": {
      "height": 370,
      "width": 1224
    },
    "000385": {
      "height": 375,
      "width": 1242
    },
    "000386": {
      "height": 376,
      "width": 1241
    },
    "000387": {
      "height": 375,
      "width": 1242
    },
    "000388": {
      "height": 375,
      "width": 1242
    },
    "000389": {
      "height": 375,
      "width": 1242
    },
    "000390": {
      "height": 375,
      "width": 1242
    },
    "000391": {
      "height": 376,
      "width": 1241
    },
    "000392": {
      "height": 375,
      "width": 1242
    },
    "000393": {
      "height": 375,
      "width": 1242
    },
    "000394": {
      "height": 375,
      "width": 1242
    },
    "000395": {
      "height": 375,
      "width": 1242
    },
    "000396": {
      "height": 375,
      "width": 1242
    },
    "000397": {
      "height": 375,
      "width": 1242
    },
    "000398": {
      "height": 370,
      "width": 1224
    },
    "000399": {
      "height": 375,
      "width": 1242
    },
    "000400": {
      "height": 375,
      "width": 1242
    },
    "000401": {
      "height": 375,
      "width": 1242
    },
    "000402": {
      "height": 375,
      "width": 1242
    },
    "000403": {
      "height": 375,
      "width": 1242
    },
    "000404": {
      "height": 376,
      "width": 1241
    },
    "000405": {
      "height": 375,
      "width": 1242
    },
    "000406": {
      "height": 375,
      "width": 1242
    },
    "000407": {
      "height": 370,
      "width": 1224
    },
    "000408": {
      "height": 375,
      "width": 1242
    },
    "000409": {
      "height": 375,
      "width": 1242
    },
    "000410": {
      "height": 375,
      "width": 1242
    },
    "000411": {
      "height": 375,
      "width": 1242
    },
    "000412": {
      "height": 375,
      "width": 1242
    },
    "000413": {
      "height": 375,
      "width": 1242
    },
    "000414": {
      "height": 375,
      "width": 1242
    },
    "000415": {
      "height": 375,
      "width": 1242
    },
    "000416": {
      "height": 375,
      "width": 1242
    },
    "000417": {
      "height": 375,
      "width": 1242
    },
    "000418": {
      "height": 375,
      "width": 1242
    },
    "000419": {
      "height": 376,
      "width": 1241
    },
    "000420": {
      "height": 375,
      "width": 1242
    },
    "000421": {
      "height": 375,
      "width": 1242
    },
    "000422": {
      "height": 374,
      "width": 1238
    },
    "000423": {
      "height": 375,
      "width": 1242
    },
    "000424": {
      "height": 375,
      "width": 1242
    },
    "000425": {
      "height": 375,
      "width": 1242
    },
    "000426": {
      "height": 375,
      "width": 1242
    },
    "000427": {
      "height": 376,
      "width": 1241
    },
    "000428": {
      "height": 370,
      "width": 1224
    },
    "000429": {
      "height": 375,
      "width": 1242
    },
    "000430": {
      "height": 375,
      "width": 1242
    },
    "000431": {
      "height": 375,
      "width": 1242
    },
    "000432": {
      "height": 375,
      "width": 1242
    },
    "000433": {
      "height": 375,
      "width": 1242
    },
    "000434": {
      "height": 375,
      "width": 1242
    },
    "000435": {
      "height": 375,
      "width": 1242
    },
    "000436": {
      "height": 375,
      "width": 1242
    },
    "000437": {
      "height": 375,
      "width": 1242
    },
    "000438": {
      "height": 375,
      "width": 1242
    },
    "000439": {
      "height": 370,
      "width": 1224
    },
    "000440": {
      "height": 375,
      "width": 1242
    },
    "000441": {
      "height": 375,
      "width": 1242
    },
    "000442": {
      "height": 375,
      "width": 1242
    },
    "000443": {
      "height": 375,
      "width": 1242
    },
    "000444": {
      "height": 375,
      "width": 1242
    },
    "000445": {
      "height": 375,
      "width": 1242
    },
    "000446": {
      "height": 375,
      "width": 1242
    },
    "000447": {
      "height": 375,
      "width": 1242
    },
    "000448": {
      "height": 370,
      "width": 1224
    },
    "000449": {
      "height": 375,
      "width": 1242
    },
    "000450": {
      "height": 375,
      "width": 1242
    },
    "000451": {
      "height": 375,
      "width": 1242
    },
    "000452": {
      "height": 375,
      "width": 1242
    },
    "000453": {
      "height": 370,
      "width": 1224
    },
    "000454": {
      "height": 375,
      "width": 1242
    },
    "000455": {
      "height": 375,
      "width": 1242
    },
    "000456": {
      "height": 375,
      "width": 1242
    },
    "000457": {
      "height": 375,
      "width": 1242
    },
    "000458": {
      "height": 375,
      "width": 1242
    },
    "000459": {
      "height": 375,
      "width": 1242
    },
    "000460": {
      "height": 370,
      "width": 1224
    },
    "000461": {
      "height": 375,
      "width": 1242
    },
    "000462": {
      "height": 375,
      "width": 1242
    },
    "000463": {
      "height": 375,
      "width": 1242
    },
    "000464": {
      "height": 375,
      "width": 1242
    },
    "000465": {
      "height": 370,
      "width": 1224
    },
    "000466": {
      "height": 375,
      "width": 1242
    },
    "000467": {
      "height": 375,
      "width": 1242
    },
    "000468": {
      "height": 375,
      "width": 1242
    },
    "000469": {
      "height": 370,
      "width": 1224
    },
    "000470": {
      "height": 375,
      "width": 1242
    },
    "000471": {
      "height": 375,
      "width": 1242
    },
    "000472": {
      "height": 375,
      "width": 1242
    },
    "000473": {
      "height": 375,
      "width": 1242
    },
    "000474": {
      "height": 375,
      "width": 1242
    },
    "000475": {
      "height": 375,
      "width": 1242
    },
    "000476": {
      "height": 375,
      "width": 1242
    },
    "000477": {
      "height": 370,
      "width": 1224
    },
    "000478": {
      "height": 374,
      "width": 1238
    },
    "000479": {
      "height": 375,
      "width": 1242
    },
    "000480": {
      "height": 375,
      "width": 1242
    },
    "000481": {
      "height": 375,
      "width": 1242
    },
    "000482": {
      "height": 375,
      "width": 1242
    },
    "000483": {
      "height": 375,
      "width": 1242
    },
    "000484": {
      "height": 375,
      "width": 1242
    },
    "000485": {
      "height": 375,
      "width": 1242
    },
    "000486": {
      "height": 375,
      "width": 1242
    },
    "000487": {
      "height": 375,
      "width": 1242
    },
    "000488": {
      "height": 375,
      "width": 1242
    },
    "000489": {
      "height": 375,
      "width": 1242
    },
    "000490": {
      "height": 375,
      "width": 1242
    },
    "000491": {
      "height": 375,
      "width": 1242
    },
    "000492": {
      "height": 376,
      "width": 1241
    },
    "000493": {
      "height": 375,
      "width": 1242
    },
    "000494": {
      "height": 375,
      "width": 1242
    },
    "000495": {
      "height": 370,
      "width": 1224
    },
    "000496": {
      "height": 374,
      "width": 1238
    },
    "000497": {
      "height": 375,
      "width": 1242
    },
    "000498": {
      "height": 370,
      "width": 1224
    },
    "000499": {
      "height": 375,
      "width": 1242
    },
    "000500": {
      "height": 375,
      "width": 1242
    },
    "000501": {
      "height": 375,
      "width": 1242
    },
    "000502": {
      "height": 375,
      "width": 1242
    },
    "000503": {
      "height": 375,
      "width": 1242
    },
    "000504": {
      "height": 375,
      "width": 1242
    },
    "000505": {
      "height": 370,
      "width": 1224
    },
    "000506": {
      "height": 375,
      "width": 1242
    },
    "000507": {
      "height": 375,
      "width": 1242
    },
    "000508": {
      "height": 375,
      "width": 1242
    },
    "000509": {
      "height": 370,
      "width": 1224
    },
    "000510": {
      "height": 375,
      "width": 1242
    },
    "000511": {
      "height": 375,
      "width": 1242
    },
    "000512": {
      "height": 375,
      "width": 1242
    },
    "000513": {
      "height": 375,
      "width": 1242
    },
    "000514": {
      "height": 375,
      "width": 1242
    },
    "000515": {
      "height": 376,
      "width": 1241
    },
    "000516": {
      "height": 375,
      "width": 1242
    },
    "000517": {
      "height": 374,
      "width": 1238
    },
    "000518": {
      "height": 375,
      "width": 1242
    },
    "000519": {
      "height": 375,
      "width": 1242
    },
    "000520": {
      "height": 370,
      "width": 1224
    },
    "000521": {
      "height": 375,
      "width": 1242
    },
    "000522": {
      "height": 375,
      "width": 1242
    },
    "000523": {
      "height": 375,
      "width": 1242
    },
    "000524": {
      "height": 375,
      "width": 1242
    },
    "000525": {
      "height": 375,
      "width": 1242
    },
    "000526": {
      "height": 374,
      "width": 1238
    },
    "000527": {
      "height": 376,
      "width": 1241
    },
    "000528": {
      "height": 375,
      "width": 1242
    },
    "000529": {
      "height": 375,
      "width": 1242
    },
    "000530": {
      "height": 370,
      "width": 1224
    },
    "000531": {
      "height": 375,
      "width": 1242
    },
    "000532": {
      "height": 370,
      "width": 1224
    },
    "000533": {
      "height": 375,
      "width": 1242
    },
    "000534": {
      "height": 375,
      "width": 1242
    },
    "000535": {
      "height": 370,
      "width": 1224
    },
    "000536": {
      "height": 375,
      "width": 1242
    },
    "000537": {
      "height": 375,
      "width": 1242
    },
    "000538": {
      "height": 375,
      "width": 1242
    },
    "000539": {
      "height": 375,
      "width": 1242
    },
    "000540": {
      "height": 375,
      "width": 1242
    },
    "000541": {
      "height": 375,
      "width": 1242
    },
    "000542": {
      "height": 375,
      "width": 1242
    },
    "000543": {
      "height": 375,
      "width": 1242
    },
    "000544": {
      "height": 375,
      "width": 1242
    },
    "000545": {
      "height": 375,
      "width": 1242
    },
    "000546": {
      "height": 375,
      "width": 1242
    },
    "000547": {
      "height": 375,
      "width": 1242
    },
    "000548": {
      "height": 375,
      "width": 1242
    },
    "000549": {
      "height": 375,
      "width": 1242
    },
    "000550": {
      "height": 375,
      "width": 1242
    },
    "000551": {
      "height": 376,
      "width": 1241
    },
    "000552": {
      "height": 375,
      "width": 1242
    },
    "000553": {
      "height": 375,
      "width": 1242
    },
    "000554": {
      "height": 375,
      "width": 1242
    },
    "000555": {
      "height": 375,
      "width": 1242
    },
    "000556": {
      "height": 375,
      "width": 1242
    },
    "000557": {
      "height": 374,
      "width": 1238
    },
    "000558": {
      "height": 376,
      "width": 1241
    },
    "000559": {
      "height": 374,
      "width": 1238
    },
    "000560": {
      "height": 375,
      "width": 1242
    },
    "000561": {
      "height": 375,
      "width": 1242
    },
    "000562": {
      "height": 375,
      "width": 1242
    },
    "000563": {
      "height": 375,
      "width": 1242
    },
    "000564": {
      "height": 375,
      "width": 1242
    },
    "000565": {
      "height": 375,
      "width": 1242
    },
    "000566": {
      "height": 375,
      "width": 1242
    },
    "000567": {
      "height": 375,
      "width": 1242
    },
    "000568": {
      "height": 375,
      "width": 1242
    },
    "000569": {
      "height": 375,
      "width": 1242
    },
    "000570": {
      "height": 375,
      "width": 1242
    },
    "000571": {
      "height": 374,
      "width": 1238
    },
    "000572": {
      "height": 376,
      "width": 1241
    },
    "000573": {
      "height": 375,
      "width": 1242
    },
    "000574": {
      "height": 375,
      "width": 1242
    },
    "000575": {
      "height": 375,
      "width": 1242
    },
    "000576": {
      "height": 375,
      "width": 1242
    },
    "000577": {
      "height": 375,
      "width": 1242
    },
    "000578": {
      "height": 375,
      "width": 1242
    },
    "000579": {
      "height": 375,
      "width": 1242
    },
    "000580": {
      "height": 375,
      "width": 1242
    },
    "000581": {
      "height": 375,
      "width": 1242
    },
    "000582": {
      "height": 375,
      "width": 1242
    },
    "000583": {
      "height": 375,
      "width": 1242
    },
    "000584": {
      "height": 375,
      "width": 1242
    },
    "000585": {
      "height": 375,
      "width": 1242
    },
    "000586": {
      "height": 375,
      "width": 1242
    },
    "000587": {
      "height": 375,
      "width": 1242
    },
    "000588": {
      "height": 375,
      "width": 1242
    },
    "000589": {
      "height": 375,
      "width": 1242
    },
    "000590": {
      "height": 375,
      "width": 1242
    },
    "000591": {
      "height": 375,
      "width": 1242
    },
    "000592": {
      "height": 375,
      "width": 1242
    },
    "000593": {
      "height": 375,
      "width": 1242
    },
    "000594": {
      "height": 375,
      "width": 1242
    },
    "000595": {
      "height": 374,
      "width": 1238
    },
    "000596": {
      "height": 375,
      "width": 1242
    },
    "000597": {
      "height": 375,
      "width": 1242
    },
    "000598": {
      "height": 370,
      "width": 1224
    },
    "000599": {
      "height": 375,
      "width": 1242
    },
    "000600": {
      "height": 376,
      "width": 1241
    },
    "000601": {
      "height": 375,
      "width": 1242
    },
    "000602": {
      "height": 375,
      "width": 1242
    },
    "000603": {
      "height": 370,
      "width": 1224
    },
    "000604": {
      "height": 370,
      "width": 1224
    },
    "000605": {
      "height": 375,
      "width": 1242
    },
    "000606": {
      "height": 375,
      "width": 1242
    },
    "000607": {
      "height": 375,
      "width": 1242
    },
    "000608": {
      "height": 375,
      "width": 1242
    },
    "000609": {
      "height": 375,
      "width": 1242
    },
    "000610": {
      "height": 375,
      "width": 1242
    },
    "000611": {
      "height": 375,
      "width": 1242
    },
    "000612": {
      "height": 375,
      "width": 1242
    },
    "000613": {
      "height": 375,
      "width": 1242
    },
    "000614": {
      "height": 375,
      "width": 1242
    },
    "000615": {
      "height": 375,
      "width": 1242
    },
    "000616": {
      "height": 375,
      "width": 1242
    },
    "000617": {
      "height": 375,
      "width": 1242
    },
    "000618": {
      "height": 375,
      "width": 1242
    },
    "000619": {
      "height": 375,
      "width": 1242
    },
    "000620": {
      "height": 375,
      "width": 1242
    },
    "000621": {
      "height": 375,
      "width": 1242
    },
    "000622": {
      "height": 375,
      "width": 1242
    },
    "000623": {
      "height": 375,
      "width": 1242
    },
    "000624": {
      "height": 375,
      "width": 1242
    },
    "000625": {
      "height": 370,
      "width": 1224
    },
    "000626": {
      "height": 375,
      "width": 1242
    },
    "000627": {
      "height": 375,
      "width": 1242
    },
    "000628": {
      "height": 375,
      "width": 1242
    },
    "000629": {
      "height": 375,
      "width": 1242
    },
    "000630": {
      "height": 375,
      "width": 1242
    },
    "000631": {
      "height": 375,
      "width": 1242
    },
    "000632": {
      "height": 375,
      "width": 1242
    },
    "000633": {
      "height": 370,
      "width": 1224
    },
    "000634": {
      "height": 375,
      "width": 1242
    },
    "000635": {
      "height": 375,
      "width": 1242
    },
    "000636": {
      "height": 375,
      "width": 1242
    },
    "000637": {
      "height": 370,
      "width": 1224
    },
    "000638": {
      "height": 375,
      "width": 1242
    },
    "000639": {
      "height": 375,
      "width": 1242
    },
    "000640": {
      "height": 374,
      "width": 1238
    },
    "000641": {
      "height": 375,
      "width": 1242
    },
    "000642": {
      "height": 375,
      "width": 1242
    },
    "000643": {
      "height": 375,
      "width": 1242
    },
    "000644": {
      "height": 375,
      "width": 1242
    },
    "000645": {
      "height": 374,
      "width": 1238
    },
    "000646": {
      "height": 370,
      "width": 1224
    },
    "000647": {
      "height": 376,
      "width": 1241
    },
    "000648": {
      "height": 375,
      "width": 1242
    },
    "000649": {
      "height": 375,
      "width": 1242
    },
    "000650": {
      "height": 375,
      "width": 1242
    },
    "000651": {
      "height": 375,
      "width": 1242
    },
    "000652": {
      "height": 375,
      "width": 1242
    },
    "000653": {
      "height": 375,
      "width": 1242
    },
    "000654": {
      "height": 375,
      "width": 1242
    },
    "000655": {
      "height": 375,
      "width": 1242
    },
    "000656": {
      "height": 374,
      "width": 1238
    },
    "000657": {
      "height": 375,
      "width": 1242
    },
    "000658": {
      "height": 375,
      "width": 1242
    },
    "000659": {
      "height": 375,
      "width": 1242
    },
    "000660": {
      "height": 375,
      "width": 1242
    },
    "000661": {
      "height": 375,
      "width": 1242
    },
    "000662": {
      "height": 375,
      "width": 1242
    },
    "000663": {
      "height": 375,
      "width": 1242
    },
    "000664": {
      "height": 374,
      "width": 1238
    },
    "000665": {
      "height": 375,
      "width": 1242
    },
    "000666": {
      "height": 375,
      "width": 1242
    },
    "000667": {
      "height": 375,
      "width": 1242
    },
    "000668": {
      "height": 375,
      "width": 1242
    },
    "000669": {
      "height": 376,
      "width": 1241
    },
    "000670": {
      "height": 375,
      "width": 1242
    },
    "000671": {
      "height": 370,
      "width": 1224
    },
    "000672": {
      "height": 375,
      "width": 1242
    },
    "000673": {
      "height": 375,
      "width": 1242
    },
    "000674": {
      "height": 375,
      "width": 1242
    },
    "000675": {
      "height": 375,
      "width": 1242
    },
    "000676": {
      "height": 374,
      "width": 1238
    },
    "000677": {
      "height": 375,
      "width": 1242
    },
    "000678": {
      "height": 375,
      "width": 1242
    },
    "000679": {
      "height": 374,
      "width": 1238
    },
    "000680": {
      "height": 375,
      "width": 1242
    },
    "000681": {
      "height": 375,
      "width": 1242
    },
    "000682": {
      "height": 375,
      "width": 1242
    },
    "000683": {
      "height": 375,
      "width": 1242
    },
    "000684": {
      "height": 375,
      "width": 1242
    },
    "000685": {
      "height": 375,
      "width": 1242
    },
    "000686": {
      "height": 375,
      "width": 1242
    },
    "000687": {
      "height": 375,
      "width": 1242
    },
    "000688": {
      "height": 375,
      "width": 1242
    },
    "000689": {
      "height": 375,
      "width": 1242
    },
    "000690": {
      "height": 375,
      "width": 1242
    },
    "000691": {
      "height": 375,
      "width": 1242
    },
    "000692": {
      "height": 375,
      "width": 1242
    },
    "000693": {
      "height": 375,
      "width": 1242
    },
    "000694": {
      "height": 375,
      "width": 1242
    },
    "000695": {
      "height": 370,
      "width": 1224
    },
    "000696": {
      "height": 375,
      "width": 1242
    },
    "000697": {
      "height": 375,
      "width": 1242
    },
    "000698": {
      "height": 375,
      "width": 1242
    },
    "000699": {
      "height": 375,
      "width": 1242
    },
    "000700": {
      "height": 375,
      "width": 1242
    },
    "000701": {
      "height": 370,
      "width": 1224
    },
    "000702": {
      "height": 375,
      "width": 1242
    },
    "000703": {
      "height": 370,
      "width": 1224
    },
    "000704": {
      "height": 375,
      "width": 1242
    },
    "000705": {
      "height": 375,
      "width": 1242
    },
    "000706": {
      "height": 375,
      "width": 1242
    },
    "000707": {
      "height": 375,
      "width": 1242
    },
    "000708": {
      "height": 375,
      "width": 1242
    },
    "000709": {
      "height": 375,
      "width": 1242
    },
    "000710": {
      "height": 375,
      "width": 1242
    },
    "000711": {
      "height": 375,
      "width": 1242
    },
    "000712": {
      "height": 375,
      "width": 1242
    },
    "000713": {
      "height": 375,
      "width": 1242
    },
    "000714": {
      "height": 375,
      "width": 1242
    },
    "000715": {
      "height": 375,
      "width": 1242
    },
    "000716": {
      "height": 375,
      "width": 1242
    },
    "000717": {
      "height": 375,
      "width": 1242
    },
    "000718": {
      "height": 374,
      "width": 1238
    },
    "000719": {
      "height": 375,
      "width": 1242
    },
    "000720": {
      "height": 375,
      "width": 1242
    },
    "000721": {
      "height": 370,
      "width": 1224
    },
    "000722": {
      "height": 375,
      "width": 1242
    },
    "000723": {
      "height": 375,
      "width": 1242
    },
    "000724": {
      "height": 375,
      "width": 1242
    },
    "000725": {
      "height": 376,
      "width": 1241
    },
    "000726": {
      "height": 370,
      "width": 1224
    },
    "000727": {
      "height": 374,
      "width": 1238
    },
    "000728": {
      "height": 375,
      "width": 1242
    },
    "000729": {
      "height": 375,
      "width": 1242
    },
    "000730": {
      "height": 375,
      "width": 1242
    },
    "000731": {
      "height": 374,
      "width": 1238
    },
    "000732": {
      "height": 375,
      "width": 1242
    },
    "000733": {
      "height": 375,
      "width": 1242
    },
    "000734": {
      "height": 375,
      "width": 1242
    },
    "000735": {
      "height": 375,
      "width": 1242
    },
    "000736": {
      "height": 375,
      "width": 1242
    },
    "000737": {
      "height": 375,
      "width": 1242
    },
    "000738": {
      "height": 375,
      "width": 1242
    },
    "000739": {
      "height": 375,
      "width": 1242
    },
    "000740": {
      "height": 375,
      "width": 1242
    },
    "000741": {
      "height": 376,
      "width": 1241
    },
    "000742": {
      "height": 375,
      "width": 1242
    },
    "000743": {
      "height": 375,
      "width": 1242
    },
    "000744": {
      "height": 370,
      "width": 1224
    },
    "000745": {
      "height": 376,
      "width": 1241
    },
    "000746": {
      "height": 370,
      "width": 1224
    },
    "000747": {
      "height": 375,
      "width": 1242
    },
    "000748": {
      "height": 375,
      "width": 1242
    },
    "000749": {
      "height": 375,
      "width": 1242
    },
    "000750": {
      "height": 375,
      "width": 1242
    },
    "000751": {
      "height": 375,
      "width": 1242
    },
    "000752": {
      "height": 376,
      "width": 1241
    },
    "000753": {
      "height": 375,
      "width": 1242
    },
    "000754": {
      "height": 375,
      "width": 1242
    },
    "000755": {
      "height": 375,
      "width": 1242
    },
    "000756": {
      "height": 376,
      "width": 1241
    },
    "000757": {
      "height": 375,
      "width": 1242
    },
    "000758": {
      "height": 375,
      "width": 1242
    },
    "000759": {
      "height": 375,
      "width": 1242
    },
    "000760": {
      "height": 375,
      "width": 1242
    },
    "000761": {
      "height": 375,
      "width": 1242
    },
    "000762": {
      "height": 375,
      "width": 1242
    },
    "000763": {
      "height": 375,
      "width": 1242
    },
    "000764": {
      "height": 370,
      "width": 1224
    },
    "000765": {
      "height": 370,
      "width": 1224
    },
    "000766": {
      "height": 375,
      "width": 1242
    },
    "000767": {
      "height": 375,
      "width": 1242
    },
    "000768": {
      "height": 376,
      "width": 1241
    },
    "000769": {
      "height": 375,
      "width": 1242
    },
    "000770": {
      "height": 370,
      "width": 1224
    },
    "000771": {
      "height": 370,
      "width": 1224
    },
    "000772": {
      "height": 375,
      "width": 1242
    },
    "000773": {
      "height": 375,
      "width": 1242
    },
    "000774": {
      "height": 375,
      "width": 1242
    },
    "000775": {
      "height": 375,
      "width": 1242
    },
    "000776": {
      "height": 375,
      "width": 1242
    },
    "000777": {
      "height": 375,
      "width": 1242
    },
    "000778": {
      "height": 375,
      "width": 1242
    },
    "000779": {
      "height": 375,
      "width": 1242
    },
    "000780": {
      "height": 375,
      "width": 1242
    },
    "000781": {
      "height": 375,
      "width": 1242
    },
    "000782": {
      "height": 375,
      "width": 1242
    },
    "000783": {
      "height": 375,
      "width": 1242
    },
    "000784": {
      "height": 370,
      "width": 1224
    },
    "000785": {
      "height": 375,
      "width": 1242
    },
    "000786": {
      "height": 375,
      "width": 1242
    },
    "000787": {
      "height": 375,
      "width": 1242
    },
    "000788": {
      "height": 375,
      "width": 1242
    },
    "000789": {
      "height": 370,
      "width": 1224
    },
    "000790": {
      "height": 375,
      "width": 1242
    },
    "000791": {
      "height": 375,
      "width": 1242
    },
    "000792": {
      "height": 375,
      "width": 1242
    },
    "000793": {
      "height": 374,
      "width": 1238
    },
    "000794": {
      "height": 375,
      "width": 1242
    },
    "000795": {
      "height": 375,
      "width": 1242
    },
    "000796": {
      "height": 375,
      "width": 1242
    },
    "000797": {
      "height": 375,
      "width": 1242
    },
    "000798": {
      "height": 375,
      "width": 1242
    },
    "000799": {
      "height": 375,
      "width": 1242
    },
    "000800": {
      "height": 376,
      "width": 1241
    },
    "000801": {
      "height": 375,
      "width": 1242
    },
    "000802": {
      "height": 375,
      "width": 1242
    },
    "000803": {
      "height": 375,
      "width": 1242
    },
    "000804": {
      "height": 375,
      "width": 1242
    },
    "000805": {
      "height": 376,
      "width": 1241
    },
    "000806": {
      "height": 375,
      "width": 1242
    },
    "000807": {
      "height": 375,
      "width": 1242
    },
    "000808": {
      "height": 375,
      "width": 1242
    },
    "000809": {
      "height": 375,
      "width": 1242
    },
    "000810": {
      "height": 375,
      "width": 1242
    },
    "000811": {
      "height": 375,
      "width": 1242
    },
    "000812": {
      "height": 375,
      "width": 1242
    },
    "000813": {
      "height": 375,
      "width": 1242
    },
    "000814": {
      "height": 375,
      "width": 1242
    },
    "000815": {
      "height": 375,
      "width": 1242
    },
    "000816": {
      "height": 374,
      "width": 1238
    },
    "000817": {
      "height": 375,
      "width": 1242
    },
    "000818": {
      "height": 375,
      "width": 1242
    },
    "000819": {
      "height": 376,
      "width": 1241
    },
    "000820": {
      "height": 375,
      "width": 1242
    },
    "000821": {
      "height": 375,
      "width": 1242
    },
    "000822": {
      "height": 374,
      "width": 1238
    },
    "000823": {
      "height": 375,
      "width": 1242
    },
    "000824": {
      "height": 375,
      "width": 1242
    },
    "000825": {
      "height": 370,
      "width": 1224
    },
    "000826": {
      "height": 374,
      "width": 1238
    },
    "000827": {
      "height": 370,
      "width": 1224
    },
    "000828": {
      "height": 375,
      "width": 1242
    },
    "000829": {
      "height": 375,
      "width": 1242
    },
    "000830": {
      "height": 375,
      "width": 1242
    },
    "000831": {
      "height": 375,
      "width": 1242
    },
    "000832": {
      "height": 375,
      "width": 1242
    },
    "000833": {
      "height": 375,
      "width": 1242
    },
    "000834": {
      "height": 375,
      "width": 1242
    },
    "000835": {
      "height": 375,
      "width": 1242
    },
    "000836": {
      "height": 375,
      "width": 1242
    },
    "000837": {
      "height": 376,
      "width": 1241
    },
    "000838": {
      "height": 375,
      "width": 1242
    },
    "000839": {
      "height": 375,
      "width": 1242
    },
    "000840": {
      "height": 374,
      "width": 1238
    },
    "000841": {
      "height": 370,
      "width": 1224
    },
    "000842": {
      "height": 375,
      "width": 1242
    },
    "000843": {
      "height": 375,
      "width": 1242
    },
    "000844": {
      "height": 375,
      "width": 1242
    },
    "000845": {
      "height": 375,
      "width": 1242
    },
    "000846": {
      "height": 375,
      "width": 1242
    },
    "000847": {
      "height": 375,
      "width": 1242
    },
    "000848": {
      "height": 375,
      "width": 1242
    },
    "000849": {
      "height": 375,
      "width": 1242
    },
    "000850": {
      "height": 376,
      "width": 1241
    },
    "000851": {
      "height": 375,
      "width": 1242
    },
    "000852": {
      "height": 375,
      "width": 1242
    },
    "000853": {
      "height": 375,
      "width": 1242
    },
    "000854": {
      "height": 375,
      "width": 1242
    },
    "000855": {
      "height": 375,
      "width": 1242
    },
    "000856": {
      "height": 375,
      "width": 1242
    },
    "000857": {
      "height": 375,
      "width": 1242
    },
    "000858": {
      "height": 375,
      "width": 1242
    },
    "000859": {
      "height": 375,
      "width": 1242
    },
    "000860": {
      "height": 375,
      "width": 1242
    },
    "000861": {
      "height": 375,
      "width": 1242
    },
    "000862": {
      "height": 370,
      "width": 1224
    },
    "000863": {
      "height": 375,
      "width": 1242
    },
    "000864": {
      "height": 375,
      "width": 1242
    },
    "000865": {
      "height": 375,
      "width": 1242
    },
    "000866": {
      "height": 375,
      "width": 1242
    },
    "000867": {
      "height": 375,
      "width": 1242
    },
    "000868": {
      "height": 375,
      "width": 1242
    },
    "000869": {
      "height": 376,
      "width": 1241
    },
    "000870": {
      "height": 375,
      "width": 1242
    },
    "000871": {
      "height": 375,
      "width": 1242
    },
    "000872": {
      "height": 375,
      "width": 1242
    },
    "000873": {
      "height": 375,
      "width": 1242
    },
    "000874": {
      "height": 370,
      "width": 1224
    },
    "000875": {
      "height": 375,
      "width": 1242
    },
    "000876": {
      "height": 376,
      "width": 1241
    },
    "000877": {
      "height": 375,
      "width": 1242
    },
    "000878": {
      "height": 375,
      "width": 1242
    },
    "000879": {
      "height": 375,
      "width": 1242
    },
    "000880": {
      "height": 375,
      "width": 1242
    },
    "000881": {
      "height": 375,
      "width": 1242
    },
    "000882": {
      "height": 375,
      "width": 1242
    },
    "000883": {
      "height": 375,
      "width": 1242
    },
    "000884": {
      "height": 375,
      "width": 1242
    },
    "000885": {
      "height": 375,
      "width": 1242
    },
    "000886": {
      "height": 375,
      "width": 1242
    },
    "000887": {
      "height": 375,
      "width": 1242
    },
    "000888": {
      "height": 375,
      "width": 1242
    },
    "000889": {
      "height": 370,
      "width": 1224
    },
    "000890": {
      "height": 375,
      "width": 1242
    },
    "000891": {
      "height": 375,
      "width": 1242
    },
    "000892": {
      "height": 375,
      "width": 1242
    },
    "000893": {
      "height": 375,
      "width": 1242
    },
    "000894": {
      "height": 375,
      "width": 1242
    },
    "000895": {
      "height": 375,
      "width": 1242
    },
    "000896": {
      "height": 375,
      "width": 1242
    },
    "000897": {
      "height": 375,
      "width": 1242
    },
    "000898": {
      "height": 375,
      "width": 1242
    },
    "000899": {
      "height": 375,
      "width": 1242
    },
    "000900": {
      "height": 375,
      "width": 1242
    },
    "000901": {
      "height": 375,
      "width": 1242
    },
    "000902": {
      "height": 375,
      "width": 1242
    },
    "000903": {
      "height": 375,
      "width": 1242
    },
    "000904": {
      "height": 375,
      "width": 1242
    },
    "000905": {
      "height": 375,
      "width": 1242
    },
    "000906": {
      "height": 375,
      "width": 1242
    },
    "000907": {
      "height": 375,
      "width": 1242
    },
    "000908": {
      "height": 375,
      "width": 1242
    },
    "000909": {
      "height": 375,
      "width": 1242
    },
    "000910": {
      "height": 375,
      "width": 1242
    },
    "000911": {
      "height": 374,
      "width": 1238
    },
    "000912": {
      "height": 375,
      "width": 1242
    },
    "000913": {
      "height": 375,
      "width": 1242
    },
    "000914": {
      "height": 375,
      "width": 1242
    },
    "000915": {
      "height": 375,
      "width": 1242
    },
    "000916": {
      "height": 375,
      "width": 1242
    },
    "000917": {
      "height": 375,
      "width": 1242
    },
    "000918": {
      "height": 375,
      "width": 1242
    },
    "000919": {
      "height": 375,
      "width": 1242
    },
    "000920": {
      "height": 375,
      "width": 1242
    },
    "000921": {
      "height": 375,
      "width": 1242
    },
    "000922": {
      "height": 375,
      "width": 1242
    },
    "000923": {
      "height": 374,
      "width": 1238
    },
    "000924": {
      "height": 375,
      "width": 1242
    },
    "000925": {
      "height": 370,
      "width": 1224
    },
    "000926": {
      "height": 375,
      "width": 1242
    },
    "000927": {
      "height": 375,
      "width": 1242
    },
    "000928": {
      "height": 370,
      "width": 1224
    },
    "000929": {
      "height": 375,
      "width": 1242
    },
    "000930": {
      "height": 375,
      "width": 1242
    },
    "000931": {
      "height": 375,
      "width": 1242
    },
    "000932": {
      "height": 375,
      "width": 1242
    },
    "000933": {
      "height": 375,
      "width": 1242
    },
    "000934": {
      "height": 375,
      "width": 1242
    },
    "000935": {
      "height": 375,
      "width": 1242
    },
    "000936": {
      "height": 375,
      "width": 1242
    },
    "000937": {
      "height": 375,
      "width": 1242
    },
    "000938": {
      "height": 375,
      "width": 1242
    },
    "000939": {
      "height": 375,
      "width": 1242
    },
    "000940": {
      "height": 375,
      "width": 1242
    },
    "000941": {
      "height": 375,
      "width": 1242
    },
    "000942": {
      "height": 375,
      "width": 1242
    },
    "000943": {
      "height": 375,
      "width": 1242
    },
    "000944": {
      "height": 375,
      "width": 1242
    },
    "000945": {
      "height": 375,
      "width": 1242
    },
    "000946": {
      "height": 375,
      "width": 1242
    },
    "000947": {
      "height": 370,
      "width": 1224
    },
    "000948": {
      "height": 375,
      "width": 1242
    },
    "000949": {
      "height": 375,
      "width": 1242
    },
    "000950": {
      "height": 375,
      "width": 1242
    },
    "000951": {
      "height": 375,
      "width": 1242
    },
    "000952": {
      "height": 375,
      "width": 1242
    },
    "000953": {
      "height": 375,
      "width": 1242
    },
    "000954": {
      "height": 375,
      "width": 1242
    },
    "000955": {
      "height": 375,
      "width": 1242
    },
    "000956": {
      "height": 376,
      "width": 1241
    },
    "000957": {
      "height": 370,
      "width": 1224
    },
    "000958": {
      "height": 375,
      "width": 1242
    },
    "000959": {
      "height": 375,
      "width": 1242
    },
    "000960": {
      "height": 375,
      "width": 1242
    },
    "000961": {
      "height": 374,
      "width": 1238
    },
    "000962": {
      "height": 375,
      "width": 1242
    },
    "000963": {
      "height": 375,
      "width": 1242
    },
    "000964": {
      "height": 374,
      "width": 1238
    },
    "000965": {
      "height": 375,
      "width": 1242
    },
    "000966": {
      "height": 370,
      "width": 1224
    },
    "000967": {
      "height": 376,
      "width": 1241
    },
    "000968": {
      "height": 375,
      "width": 1242
    },
    "000969": {
      "height": 375,
      "width": 1242
    },
    "000970": {
      "height": 375,
      "width": 1242
    },
    "000971": {
      "height": 374,
      "width": 1238
    },
    "000972": {
      "height": 375,
      "width": 1242
    },
    "000973": {
      "height": 375,
      "width": 1242
    },
    "000974": {
      "height": 375,
      "width": 1242
    },
    "000975": {
      "height": 375,
      "width": 1242
    },
    "000976": {
      "height": 375,
      "width": 1242
    },
    "000977": {
      "height": 375,
      "width": 1242
    },
    "000978": {
      "height": 375,
      "width": 1242
    },
    "000979": {
      "height": 375,
      "width": 1242
    },
    "000980": {
      "height": 375,
      "width": 1242
    },
    "000981": {
      "height": 375,
      "width": 1242
    },
    "000982": {
      "height": 375,
      "width": 1242
    },
    "000983": {
      "height": 375,
      "width": 1242
    },
    "000984": {
      "height": 375,
      "width": 1242
    },
    "000985": {
      "height": 376,
      "width": 1241
    },
    "000986": {
      "height": 375,
      "width": 1242
    },
    "000987": {
      "height": 375,
      "width": 1242
    },
    "000988": {
      "height": 375,
      "width": 1242
    },
    "000989": {
      "height": 374,
      "width": 1238
    },
    "000990": {
      "height": 375,
      "width": 1242
    },
    "000991": {
      "height": 375,
      "width": 1242
    },
    "000992": {
      "height": 375,
      "width": 1242
    },
    "000993": {
      "height": 375,
      "width": 1242
    },
    "000994": {
      "height": 375,
      "width": 1242
    },
    "000995": {
      "height": 374,
      "width": 1238
    },
    "000996": {
      "height": 375,
      "width": 1242
    },
    "000997": {
      "height": 375,
      "width": 1242
    },
    "000998": {
      "height": 375,
      "width": 1242
    },
    "000999": {
      "height": 374,
      "width": 1238
    },
    "001000": {
      "height": 375,
      "width": 1242
    },
    "001001": {
      "height": 375,
      "width": 1242
    },
    "001002": {
      "height": 375,
      "width": 1242
    },
    "001003": {
      "height": 375,
      "width": 1242
    },
    "001004": {
      "height": 375,
      "width": 1242
    },
    "001005": {
      "height": 375,
      "width": 1242
    },
    "001006": {
      "height": 375,
      "width": 1242
    },
    "001007": {
      "height": 375,
      "width": 1242
    },
    "001008": {
      "height": 374,
      "width": 1238
    },
    "001009": {
      "height": 374,
      "width": 1238
    },
    "001010": {
      "height": 375,
      "width": 1242
    },
    "001011": {
      "height": 375,
      "width": 1242
    },
    "001012": {
      "height": 375,
      "width": 1242
    },
    "001013": {
      "height": 376,
      "width": 1241
    },
    "001014": {
      "height": 375,
      "width": 1242
    },
    "001015": {
      "height": 375,
      "width": 1242
    },
    "001016": {
      "height": 375,
      "width": 1242
    },
    "001017": {
      "height": 375,
      "width": 1242
    },
    "001018": {
      "height": 375,
      "width": 1242
    },
    "001019": {
      "height": 375,
      "width": 1242
    },
    "001020": {
      "height": 374,
      "width": 1238
    },
    "001021": {
      "height": 375,
      "width": 1242
    },
    "001022": {
      "height": 370,
      "width": 1224
    },
    "001023": {
      "height": 375,
      "width": 1242
    },
    "001024": {
      "height": 375,
      "width": 1242
    },
    "001025": {
      "height": 375,
      "width": 1242
    },
    "001026": {
      "height": 375,
      "width": 1242
    },
    "001027": {
      "height": 375,
      "width": 1242
    },
    "001028": {
      "height": 370,
      "width": 1224
    },
    "001029": {
      "height": 370,
      "width": 1224
    },
    "001030": {
      "height": 375,
      "width": 1242
    },
    "001031": {
      "height": 370,
      "width": 1224
    },
    "001032": {
      "height": 375,
      "width": 1242
    },
    "001033": {
      "height": 370,
      "width": 1224
    },
    "001034": {
      "height": 375,
      "width": 1242
    },
    "001035": {
      "height": 375,
      "width": 1242
    },
    "001036": {
      "height": 370,
      "width": 1224
    },
    "001037": {
      "height": 370,
      "width": 1224
    },
    "001038": {
      "height": 375,
      "width": 1242
    },
    "001039": {
      "height": 375,
      "width": 1242
    },
    "001040": {
      "height": 375,
      "width": 1242
    },
    "001041": {
      "height": 375,
      "width": 1242
    },
    "001042": {
      "height": 375,
      "width": 1242
    },
    "001043": {
      "height": 375,
      "width": 1242
    },
    "001044": {
      "height": 375,
      "width": 1242
    },
    "001045": {
      "height": 375,
      "width": 1242
    },
    "001046": {
      "height": 374,
      "width": 1238
    },
    "001047": {
      "height": 375,
      "width": 1242
    },
    "001048": {
      "height": 375,
      "width": 1242
    },
    "001049": {
      "height": 370,
      "width": 1224
    },
    "001050": {
      "height": 375,
      "width": 1242
    },
    "001051": {
      "height": 375,
      "width": 1242
    },
    "001052": {
      "height": 375,
      "width": 1242
    },
    "001053": {
      "height": 375,
      "width": 1242
    },
    "001054": {
      "height": 376,
      "width": 1241
    },
    "001055": {
      "height": 375,
      "width": 1242
    },
    "001056": {
      "height": 375,
      "width": 1242
    },
    "001057": {
      "height": 370,
      "width": 1224
    },
    "001058": {
      "height": 375,
      "width": 1242
    },
    "001059": {
      "height": 375,
      "width": 1242
    },
    "001060": {
      "height": 375,
      "width": 1242
    },
    "001061": {
      "height": 375,
      "width": 1242
    },
    "001062": {
      "height": 375,
      "width": 1242
    },
    "001063": {
      "height": 375,
      "width": 1242
    },
    "001064": {
      "height": 375,
      "width": 1242
    },
    "001065": {
      "height": 375,
      "width": 1242
    },
    "001066": {
      "height": 375,
      "width": 1242
    },
    "001067": {
      "height": 375,
      "width": 1242
    },
    "001068": {
      "height": 374,
      "width": 1238
    },
    "001069": {
      "height": 370,
      "width": 1224
    },
    "001070": {
      "height": 374,
      "width": 1238
    },
    "001071": {
      "height": 375,
      "width": 1242
    },
    "001072": {
      "height": 375,
      "width": 1242
    },
    "001073": {
      "height": 375,
      "width": 1242
    },
    "001074": {
      "height": 375,
      "width": 1242
    },
    "001075": {
      "height": 375,
      "width": 1242
    },
    "001076": {
      "height": 375,
      "width": 1242
    },
    "001077": {
      "height": 375,
      "width": 1242
    },
    "001078": {
      "height": 375,
      "width": 1242
    },
    "001079": {
      "height": 375,
      "width": 1242
    },
    "001080": {
      "height": 375,
      "width": 1242
    },
    "001081": {
      "height": 375,
      "width": 1242
    },
    "001082": {
      "height": 375,
      "width": 1242
    },
    "001083": {
      "height": 375,
      "width": 1242
    },
    "001084": {
      "height": 375,
      "width": 1242
    },
    "001085": {
      "height": 374,
      "width": 1238
    },
    "001086": {
      "height": 375,
      "width": 1242
    },
    "001087": {
      "height": 375,
      "width": 1242
    },
    "001088": {
      "height": 370,
      "width": 1224
    },
    "001089": {
      "height": 375,
      "width": 1242
    },
    "001090": {
      "height": 375,
      "width": 1242
    },
    "001091": {
      "height": 370,
      "width": 1224
    },
    "001092": {
      "height": 375,
      "width": 1242
    },
    "001093": {
      "height": 375,
      "width": 1242
    },
    "001094": {
      "height": 375,
      "width": 1242
    },
    "001095": {
      "height": 375,
      "width": 1242
    },
    "001096": {
      "height": 375,
      "width": 1242
    },
    "001097": {
      "height": 375,
      "width": 1242
    },
    "001098": {
      "height": 375,
      "width": 1242
    },
    "001099": {
      "height": 375,
      "width": 1242
    },
    "001100": {
      "height": 375,
      "width": 1242
    },
    "001101": {
      "height": 374,
      "width": 1238
    },
    "001102": {
      "height": 375,
      "width": 1242
    },
    "001103": {
      "height": 375,
      "width": 1242
    },
    "001104": {
      "height": 376,
      "width": 1241
    },
    "001105": {
      "height": 375,
      "width": 1242
    },
    "001106": {
      "height": 374,
      "width": 1238
    },
    "001107": {
      "height": 375,
      "width": 1242
    },
    "001108": {
      "height": 375,
      "width": 1242
    },
    "001109": {
      "height": 375,
      "width": 1242
    },
    "001110": {
      "height": 375,
      "width": 1242
    },
    "001111": {
      "height": 375,
      "width": 1242
    },
    "001112": {
      "height": 375,
      "width": 1242
    },
    "001113": {
      "height": 370,
      "width": 1224
    },
    "001114": {
      "height": 375,
      "width": 1242
    },
    "001115": {
      "height": 375,
      "width": 1242
    },
    "001116": {
      "height": 375,
      "width": 1242
    },
    "001117": {
      "height": 375,
      "width": 1242
    },
    "001118": {
      "height": 375,
      "width": 1242
    },
    "001119": {
      "height": 375,
      "width": 1242
    },
    "001120": {
      "height": 375,
      "width": 1242
    },
    "001121": {
      "height": 375,
      "width": 1242
    },
    "001122": {
      "height": 375,
      "width": 1242
    },
    "001123": {
      "height": 375,
      "width": 1242
    },
    "001124": {
      "height": 370,
      "width": 1224
    },
    "001125": {
      "height": 375,
      "width": 1242
    },
    "001126": {
      "height": 375,
      "width": 1242
    },
    "001127": {
      "height": 375,
      "width": 1242
    },
    "001128": {
      "height": 375,
      "width": 1242
    },
    "001129": {
      "height": 374,
      "width": 1238
    },
    "001130": {
      "height": 375,
      "width": 1242
    },
    "001131": {
      "height": 375,
      "width": 1242
    },
    "001132": {
      "height": 374,
      "width": 1238
    },
    "001133": {
      "height": 375,
      "width": 1242
    },
    "001134": {
      "height": 375,
      "width": 1242
    },
    "001135": {
      "height": 375,
      "width": 1242
    },
    "001136": {
      "height": 375,
      "width": 1242
    },
    "001137": {
      "height": 375,
      "width": 1242
    },
    "001138": {
      "height": 376,
      "width": 1241
    },
    "001139": {
      "height": 375,
      "width": 1242
    },
    "001140": {
      "height": 375,
      "width": 1242
    },
    "001141": {
      "height": 375,
      "width": 1242
    },
    "001142": {
      "height": 375,
      "width": 1242
    },
    "001143": {
      "height": 375,
      "width": 1242
    },
    "001144": {
      "height": 375,
      "width": 1242
    },
    "001145": {
      "height": 375,
      "width": 1242
    },
    "001146": {
      "height": 375,
      "width": 1242
    },
    "001147": {
      "height": 375,
      "width": 1242
    },
    "001148": {
      "height": 375,
      "width": 1242
    },
    "001149": {
      "height": 375,
      "width": 1242
    },
    "001150": {
      "height": 375,
      "width": 1242
    },
    "001151": {
      "height": 374,
      "width": 1238
    },
    "001152": {
      "height": 375,
      "width": 1242
    },
    "001153": {
      "height": 375,
      "width": 1242
    },
    "001154": {
      "height": 375,
      "width": 1242
    },
    "001155": {
      "height": 375,
      "width": 1242
    },
    "001156": {
      "height": 375,
      "width": 1242
    },
    "001157": {
      "height": 375,
      "width": 1242
    },
    "001158": {
      "height": 376,
      "width": 1241
    },
    "001159": {
      "height": 375,
      "width": 1242
    },
    "001160": {
      "height": 370,
      "width": 1224
    },
    "001161": {
      "height": 370,
      "width": 1224
    },
    "001162": {
      "height": 375,
      "width": 1242
    },
    "001163": {
      "height": 375,
      "width": 1242
    },
    "001164": {
      "height": 374,
      "width": 1238
    },
    "001165": {
      "height": 375,
      "width": 1242
    },
    "001166": {
      "height": 375,
      "width": 1242
    },
    "001167": {
      "height": 375,
      "width": 1242
    },
    "001168": {
      "height": 375,
      "width": 1242
    },
    "001169": {
      "height": 370,
      "width": 1224
    },
    "001170": {
      "height": 375,
      "width": 1242
    },
    "001171": {
      "height": 375,
      "width": 1242
    },
    "001172": {
      "height": 375,
      "width": 1242
    },
    "001173": {
      "height": 375,
      "width": 1242
    },
    "001174": {
      "height": 370,
      "width": 1224
    },
    "001175": {
      "height": 375,
      "width": 1242
    },
    "001176": {
      "height": 375,
      "width": 1242
    },
    "001177": {
      "height": 375,
      "width": 1242
    },
    "001178": {
      "height": 375,
      "width": 1242
    },
    "001179": {
      "height": 375,
      "width": 1242
    },
    "001180": {
      "height": 375,
      "width": 1242
    },
    "001181": {
      "height": 370,
      "width": 1224
    },
    "001182": {
      "height": 375,
      "width": 1242
    },
    "001183": {
      "height": 375,
      "width": 1242
    },
    "001184": {
      "height": 375,
      "width": 1242
    },
    "001185": {
      "height": 370,
      "width": 1224
    },
    "001186": {
      "height": 375,
      "width": 1242
    },
    "001187": {
      "height": 375,
      "width": 1242
    },
    "001188": {
      "height": 370,
      "width": 1224
    },
    "001189": {
      "height": 370,
      "width": 1224
    },
    "001190": {
      "height": 375,
      "width": 1242
    },
    "001191": {
      "height": 370,
      "width": 1224
    },
    "001192": {
      "height": 375,
      "width": 1242
    },
    "001193": {
      "height": 374,
      "width": 1238
    },
    "001194": {
      "height": 375,
      "width": 1242
    },
    "001195": {
      "height": 376,
      "width": 1241
    },
    "001196": {
      "height": 375,
      "width": 1242
    },
    "001197": {
      "height": 375,
      "width": 1242
    },
    "001198": {
      "height": 374,
      "width": 1238
    },
    "001199": {
      "height": 375,
      "width": 1242
    },
    "001200": {
      "height": 375,
      "width": 1242
    },
    "001201": {
      "height": 375,
      "width": 1242
    },
    "001202": {
      "height": 375,
      "width": 1242
    },
    "001203": {
      "height": 375,
      "width": 1242
    },
    "001204": {
      "height": 375,
      "width": 1242
    },
    "001205": {
      "height": 370,
      "width": 1224
    },
    "001206": {
      "height": 375,
      "width": 1242
    },
    "001207": {
      "height": 374,
      "width": 1238
    },
    "001208": {
      "height": 370,
      "width": 1224
    },
    "001209": {
      "height": 375,
      "width": 1242
    },
    "001210": {
      "height": 375,
      "width": 1242
    },
    "001211": {
      "height": 375,
      "width": 1242
    },
    "001212": {
      "height": 375,
      "width": 1242
    },
    "001213": {
      "height": 375,
      "width": 1242
    },
    "001214": {
      "height": 375,
      "width": 1242
    },
    "001215": {
      "height": 375,
      "width": 1242
    },
    "001216": {
      "height": 375,
      "width": 1242
    },
    "001217": {
      "height": 375,
      "width": 1242
    },
    "001218": {
      "height": 375,
      "width": 1242
    },
    "001219": {
      "height": 375,
      "width": 1242
    },
    "001220": {
      "height": 370,
      "width": 1224
    },
    "001221": {
      "height": 375,
      "width": 1242
    },
    "001222": {
      "height": 375,
      "width": 1242
    },
    "001223": {
      "height": 375,
      "width": 1242
    },
    "001224": {
      "height": 375,
      "width": 1242
    },
    "001225": {
      "height": 375,
      "width": 1242
    },
    "001226": {
      "height": 375,
      "width": 1242
    },
    "001227": {
      "height": 375,
      "width": 1242
    },
    "001228": {
      "height": 370,
      "width": 1224
    },
    "001229": {
      "height": 375,
      "width": 1242
    },
    "001230": {
      "height": 375,
      "width": 1242
    },
    "001231": {
      "height": 375,
      "width": 1242
    },
    "001232": {
      "height": 375,
      "width": 1242
    },
    "001233": {
      "height": 370,
      "width": 1224
    },
    "001234": {
      "height": 375,
      "width": 1242
    },
    "001235": {
      "height": 375,
      "width": 1242
    },
    "001236": {
      "height": 375,
      "width": 1242
    },
    "001237": {
      "height": 375,
      "width": 1242
    },
    "001238": {
      "height": 375,
      "width": 1242
    },
    "001239": {
      "height": 375,
      "width": 1242
    },
    "001240": {
      "height": 374,
      "width": 1238
    },
    "001241": {
      "height": 375,
      "width": 1242
    },
    "001242": {
      "height": 375,
      "width": 1242
    },
    "001243": {
      "height": 375,
      "width": 1242
    },
    "001244": {
      "height": 376,
      "width": 1241
    },
    "001245": {
      "height": 375,
      "width": 1242
    },
    "001246": {
      "height": 375,
      "width": 1242
    },
    "001247": {
      "height": 375,
      "width": 1242
    },
    "001248": {
      "height": 375,
      "width": 1242
    },
    "001249": {
      "height": 370,
      "width": 1224
    },
    "001250": {
      "height": 375,
      "width": 1242
    },
    "001251": {
      "height": 375,
      "width": 1242
    },
    "001252": {
      "height": 374,
      "width": 1238
    },
    "001253": {
      "height": 374,
      "width": 1238
    },
    "001254": {
      "height": 375,
      "width": 1242
    },
    "001255": {
      "height": 375,
      "width": 1242
    },
    "001256": {
      "height": 370,
      "width": 1224
    },
    "001257": {
      "height": 375,
      "width": 1242
    },
    "001258": {
      "height": 375,
      "width": 1242
    },
    "001259": {
      "height": 375,
      "width": 1242
    },
    "001260": {
      "height": 375,
      "width": 1242
    },
    "001261": {
      "height": 375,
      "width": 1242
    },
    "001262": {
      "height": 375,
      "width": 1242
    },
    "001263": {
      "height": 375,
      "width": 1242
    },
    "001264": {
      "height": 375,
      "width": 1242
    },
    "001265": {
      "height": 375,
      "width": 1242
    },
    "001266": {
      "height": 370,
      "width": 1224
    },
    "001267": {
      "height": 374,
      "width": 1238
    },
    "001268": {
      "height": 375,
      "width": 1242
    },
    "001269": {
      "height": 375,
      "width": 1242
    },
    "001270": {
      "height": 375,
      "width": 1242
    },
    "001271": {
      "height": 375,
      "width": 1242
    },
    "001272": {
      "height": 370,
      "width": 1224
    },
    "001273": {
      "height": 375,
      "width": 1242
    },
    "001274": {
      "height": 375,
      "width": 1242
    },
    "001275": {
      "height": 375,
      "width": 1242
    },
    "001276": {
      "height": 375,
      "width": 1242
    },
    "001277": {
      "height": 375,
      "width": 1242
    },
    "001278": {
      "height": 375,
      "width": 1242
    },
    "001279": {
      "height": 370,
      "width": 1224
    },
    "001280": {
      "height": 375,
      "width": 1242
    },
    "001281": {
      "height": 375,
      "width": 1242
    },
    "001282": {
      "height": 375,
      "width": 1242
    },
    "001283": {
      "height": 375,
      "width": 1242
    },
    "001284": {
      "height": 375,
      "width": 1242
    },
    "001285": {
      "height": 375,
      "width": 1242
    },
    "001286": {
      "height": 370,
      "width": 1224
    },
    "001287": {
      "height": 376,
      "width": 1241
    },
    "001288": {
      "height": 375,
      "width": 1242
    },
    "001289": {
      "height": 370,
      "width": 1224
    },
    "001290": {
      "height": 375,
      "width": 1242
    },
    "001291": {
      "height": 376,
      "width": 1241
    },
    "001292": {
      "height": 375,
      "width": 1242
    },
    "001293": {
      "height": 375,
      "width": 1242
    },
    "001294": {
      "height": 375,
      "width": 1242
    },
    "001295": {
      "height": 370,
      "width": 1224
    },
    "001296": {
      "height": 375,
      "width": 1242
    },
    "001297": {
      "height": 375,
      "width": 1242
    },
    "001298": {
      "height": 375,
      "width": 1242
    },
    "001299": {
      "height": 375,
      "width": 1242
    },
    "001300": {
      "height": 370,
      "width": 1224
    },
    "001301": {
      "height": 375,
      "width": 1242
    },
    "001302": {
      "height": 375,
      "width": 1242
    },
    "001303": {
      "height": 375,
      "width": 1242
    },
    "001304": {
      "height": 375,
      "width": 1242
    },
    "001305": {
      "height": 375,
      "width": 1242
    },
    "001306": {
      "height": 376,
      "width": 1241
    },
    "001307": {
      "height": 374,
      "width": 1238
    },
    "001308": {
      "height": 370,
      "width": 1224
    },
    "001309": {
      "height": 374,
      "width": 1238
    },
    "001310": {
      "height": 375,
      "width": 1242
    },
    "001311": {
      "height": 375,
      "width": 1242
    },
    "001312": {
      "height": 375,
      "width": 1242
    },
    "001313": {
      "height": 375,
      "width": 1242
    },
    "001314": {
      "height": 370,
      "width": 1224
    },
    "001315": {
      "height": 375,
      "width": 1242
    },
    "001316": {
      "height": 375,
      "width": 1242
    },
    "001317": {
      "height": 375,
      "width": 1242
    },
    "001318": {
      "height": 375,
      "width": 1242
    },
    "001319": {
      "height": 375,
      "width": 1242
    },
    "001320": {
      "height": 375,
      "width": 1242
    },
    "001321": {
      "height": 375,
      "width": 1242
    },
    "001322": {
      "height": 375,
      "width": 1242
    },
    "001323": {
      "height": 370,
      "width": 1224
    },
    "001324": {
      "height": 375,
      "width": 1242
    },
    "001325": {
      "height": 375,
      "width": 1242
    },
    "001326": {
      "height": 375,
      "width": 1242
    },
    "001327": {
      "height": 375,
      "width": 1242
    },
    "001328": {
      "height": 375,
      "width": 1242
    },
    "001329": {
      "height": 375,
      "width": 1242
    },
    "001330": {
      "height": 375,
      "width": 1242
    },
    "001331": {
      "height": 375,
      "width": 1242
    },
    "001332": {
      "height": 374,
      "width": 1238
    },
    "001333": {
      "height": 375,
      "width": 1242
    },
    "001334": {
      "height": 375,
      "width": 1242
    },
    "001335": {
      "height": 375,
      "width": 1242
    },
    "001336": {
      "height": 375,
      "width": 1242
    },
    "001337": {
      "height": 375,
      "width": 1242
    },
    "001338": {
      "height": 375,
      "width": 1242
    },
    "001339": {
      "height": 375,
      "width": 1242
    },
    "001340": {
      "height": 375,
      "width": 1242
    },
    "001341": {
      "height": 375,
      "width": 1242
    },
    "001342": {
      "height": 375,
      "width": 1242
    },
    "001343": {
      "height": 375,
      "width": 1242
    },
    "001344": {
      "height": 375,
      "width": 1242
    },
    "001345": {
      "height": 375,
      "width": 1242
    },
    "001346": {
      "height": 375,
      "width": 1242
    },
    "001347": {
      "height": 376,
      "width": 1241
    },
    "001348": {
      "height": 370,
      "width": 1224
    },
    "001349": {
      "height": 375,
      "width": 1242
    },
    "001350": {
      "height": 375,
      "width": 1242
    },
    "001351": {
      "height": 375,
      "width": 1242
    },
    "001352": {
      "height": 375,
      "width": 1242
    },
    "001353": {
      "height": 375,
      "width": 1242
    },
    "001354": {
      "height": 370,
      "width": 1224
    },
    "001355": {
      "height": 374,
      "width": 1238
    },
    "001356": {
      "height": 375,
      "width": 1242
    },
    "001357": {
      "height": 370,
      "width": 1224
    },
    "001358": {
      "height": 370,
      "width": 1224
    },
    "001359": {
      "height": 375,
      "width": 1242
    },
    "001360": {
      "height": 375,
      "width": 1242
    },
    "001361": {
      "height": 374,
      "width": 1238
    },
    "001362": {
      "height": 375,
      "width": 1242
    },
    "001363": {
      "height": 375,
      "width": 1242
    },
    "001364": {
      "height": 375,
      "width": 1242
    },
    "001365": {
      "height": 375,
      "width": 1242
    },
    "001366": {
      "height": 370,
      "width": 1224
    },
    "001367": {
      "height": 375,
      "width": 1242
    },
    "001368": {
      "height": 375,
      "width": 1242
    },
    "001369": {
      "height": 375,
      "width": 1242
    },
    "001370": {
      "height": 375,
      "width": 1242
    },
    "001371": {
      "height": 375,
      "width": 1242
    },
    "001372": {
      "height": 375,
      "width": 1242
    },
    "001373": {
      "height": 375,
      "width": 1242
    },
    "001374": {
      "height": 370,
      "width": 1224
    },
    "001375": {
      "height": 375,
      "width": 1242
    },
    "001376": {
      "height": 375,
      "width": 1242
    },
    "001377": {
      "height": 375,
      "width": 1242
    },
    "001378": {
      "height": 375,
      "width": 1242
    },
    "001379": {
      "height": 375,
      "width": 1242
    },
    "001380": {
      "height": 375,
      "width": 1242
    },
    "001381": {
      "height": 375,
      "width": 1242
    },
    "001382": {
      "height": 375,
      "width": 1242
    },
    "001383": {
      "height": 375,
      "width": 1242
    },
    "001384": {
      "height": 370,
      "width": 1224
    },
    "001385": {
      "height": 375,
      "width": 1242
    },
    "001386": {
      "height": 375,
      "width": 1242
    },
    "001387": {
      "height": 375,
      "width": 1242
    },
    "001388": {
      "height": 375,
      "width": 1242
    },
    "001389": {
      "height": 375,
      "width": 1242
    },
    "001390": {
      "height": 375,
      "width": 1242
    },
    "001391": {
      "height": 375,
      "width": 1242
    },
    "001392": {
      "height": 375,
      "width": 1242
    },
    "001393": {
      "height": 375,
      "width": 1242
    },
    "001394": {
      "height": 370,
      "width": 1224
    },
    "001395": {
      "height": 375,
      "width": 1242
    },
    "001396": {
      "height": 375,
      "width": 1242
    },
    "001397": {
      "height": 370,
      "width": 1224
    },
    "001398": {
      "height": 375,
      "width": 1242
    },
    "001399": {
      "height": 375,
      "width": 1242
    },
    "001400": {
      "height": 375,
      "width": 1242
    },
    "001401": {
      "height": 375,
      "width": 1242
    },
    "001402": {
      "height": 375,
      "width": 1242
    },
    "001403": {
      "height": 375,
      "width": 1242
    },
    "001404": {
      "height": 375,
      "width": 1242
    },
    "001405": {
      "height": 375,
      "width": 1242
    },
    "001406": {
      "height": 370,
      "width": 1224
    },
    "001407": {
      "height": 375,
      "width": 1242
    },
    "001408": {
      "height": 375,
      "width": 1242
    },
    "001409": {
      "height": 375,
      "width": 1242
    },
    "001410": {
      "height": 375,
      "width": 1242
    },
    "001411": {
      "height": 375,
      "width": 1242
    },
    "001412": {
      "height": 375,
      "width": 1242
    },
    "001413": {
      "height": 375,
      "width": 1242
    },
    "001414": {
      "height": 375,
      "width": 1242
    },
    "001415": {
      "height": 375,
      "width": 1242
    },
    "001416": {
      "height": 375,
      "width": 1242
    },
    "001417": {
      "height": 375,
      "width": 1242
    },
    "001418": {
      "height": 375,
      "width": 1242
    },
    "001419": {
      "height": 374,
      "width": 1238
    },
    "001420": {
      "height": 375,
      "width": 1242
    },
    "001421": {
      "height": 370,
      "width": 1224
    },
    "001422": {
      "height": 370,
      "width": 1224
    },
    "001423": {
      "height": 370,
      "width": 1224
    },
    "001424": {
      "height": 375,
      "width": 1242
    },
    "001425": {
      "height": 375,
      "width": 1242
    },
    "001426": {
      "height": 375,
      "width": 1242
    },
    "001427": {
      "height": 375,
      "width": 1242
    },
    "001428": {
      "height": 370,
      "width": 1224
    },
    "001429": {
      "height": 375,
      "width": 1242
    },
    "001430": {
      "height": 375,
      "width": 1242
    },
    "001431": {
      "height": 370,
      "width": 1224
    },
    "001432": {
      "height": 375,
      "width": 1242
    },
    "001433": {
      "height": 370,
      "width": 1224
    },
    "001434": {
      "height": 375,
      "width": 1242
    },
    "001435": {
      "height": 375,
      "width": 1242
    },
    "001436": {
      "height": 375,
      "width": 1242
    },
    "001437": {
      "height": 375,
      "width": 1242
    },
    "001438": {
      "height": 375,
      "width": 1242
    },
    "001439": {
      "height": 375,
      "width": 1242
    },
    "001440": {
      "height": 375,
      "width": 1242
    },
    "001441": {
      "height": 370,
      "width": 1224
    },
    "001442": {
      "height": 375,
      "width": 1242
    },
    "001443": {
      "height": 375,
      "width": 1242
    },
    "001444": {
      "height": 370,
      "width": 1224
    },
    "001445": {
      "height": 375,
      "width": 1242
    },
    "001446": {
      "height": 376,
      "width": 1241
    },
    "001447": {
      "height": 375,
      "width": 1242
    },
    "001448": {
      "height": 374,
      "width": 1238
    },
    "001449": {
      "height": 370,
      "width": 1224
    },
    "001450": {
      "height": 375,
      "width": 1242
    },
    "001451": {
      "height": 375,
      "width": 1242
    },
    "001452": {
      "height": 375,
      "width": 1242
    },
    "001453": {
      "height": 375,
      "width": 1242
    },
    "001454": {
      "height": 375,
      "width": 1242
    },
    "001455": {
      "height": 375,
      "width": 1242
    },
    "001456": {
      "height": 375,
      "width": 1242
    },
    "001457": {
      "height": 375,
      "width": 1242
    },
    "001458": {
      "height": 375,
      "width": 1242
    },
    "001459": {
      "height": 375,
      "width": 1242
    },
    "001460": {
      "height": 375,
      "width": 1242
    },
    "001461": {
      "height": 375,
      "width": 1242
    },
    "001462": {
      "height": 375,
      "width": 1242
    },
    "001463": {
      "height": 375,
      "width": 1242
    },
    "001464": {
      "height": 375,
      "width": 1242
    },
    "001465": {
      "height": 375,
      "width": 1242
    },
    "001466": {
      "height": 370,
      "width": 1224
    },
    "001467": {
      "height": 375,
      "width": 1242
    },
    "001468": {
      "height": 375,
      "width": 1242
    },
    "001469": {
      "height": 376,
      "width": 1241
    },
    "001470": {
      "height": 375,
      "width": 1242
    },
    "001471": {
      "height": 375,
      "width": 1242
    },
    "001472": {
      "height": 375,
      "width": 1242
    },
    "001473": {
      "height": 375,
      "width": 1242
    },
    "001474": {
      "height": 375,
      "width": 1242
    },
    "001475": {
      "height": 375,
      "width": 1242
    },
    "001476": {
      "height": 375,
      "width": 1242
    },
    "001477": {
      "height": 370,
      "width": 1224
    },
    "001478": {
      "height": 375,
      "width": 1242
    },
    "001479": {
      "height": 375,
      "width": 1242
    },
    "001480": {
      "height": 375,
      "width": 1242
    },
    "001481": {
      "height": 376,
      "width": 1241
    },
    "001482": {
      "height": 375,
      "width": 1242
    },
    "001483": {
      "height": 375,
      "width": 1242
    },
    "001484": {
      "height": 375,
      "width": 1242
    },
    "001485": {
      "height": 375,
      "width": 1242
    },
    "001486": {
      "height": 374,
      "width": 1238
    },
    "001487": {
      "height": 370,
      "width": 1224
    },
    "001488": {
      "height": 374,
      "width": 1238
    },
    "001489": {
      "height": 375,
      "width": 1242
    },
    "001490": {
      "height": 375,
      "width": 1242
    },
    "001491": {
      "height": 375,
      "width": 1242
    },
    "001492": {
      "height": 375,
      "width": 1242
    },
    "001493": {
      "height": 375,
      "width": 1242
    },
    "001494": {
      "height": 375,
      "width": 1242
    },
    "001495": {
      "height": 375,
      "width": 1242
    },
    "001496": {
      "height": 370,
      "width": 1224
    },
    "001497": {
      "height": 375,
      "width": 1242
    },
    "001498": {
      "height": 375,
      "width": 1242
    },
    "001499": {
      "height": 375,
      "width": 1242
    },
    "001500": {
      "height": 374,
      "width": 1238
    },
    "001501": {
      "height": 375,
      "width": 1242
    },
    "001502": {
      "height": 375,
      "width": 1242
    },
    "001503": {
      "height": 375,
      "width": 1242
    },
    "001504": {
      "height": 375,
      "width": 1242
    },
    "001505": {
      "height": 375,
      "width": 1242
    },
    "001506": {
      "height": 375,
      "width": 1242
    },
    "001507": {
      "height": 375,
      "width": 1242
    },
    "001508": {
      "height": 375,
      "width": 1242
    },
    "001509": {
      "height": 370,
      "width": 1224
    },
    "001510": {
      "height": 375,
      "width": 1242
    },
    "001511": {
      "height": 375,
      "width": 1242
    },
    "001512": {
      "height": 375,
      "width": 1242
    },
    "001513": {
      "height": 375,
      "width": 1242
    },
    "001514": {
      "height": 376,
      "width": 1241
    },
    "001515": {
      "height": 375,
      "width": 1242
    },
    "001516": {
      "height": 375,
      "width": 1242
    },
    "001517": {
      "height": 375,
      "width": 1242
    },
    "001518": {
      "height": 375,
      "width": 1242
    },
    "001519": {
      "height": 370,
      "width": 1224
    },
    "001520": {
      "height": 375,
      "width": 1242
    },
    "001521": {
      "height": 375,
      "width": 1242
    },
    "001522": {
      "height": 375,
      "width": 1242
    },
    "001523": {
      "height": 375,
      "width": 1242
    },
    "001524": {
      "height": 375,
      "width": 1242
    },
    "001525": {
      "height": 374,
      "width": 1238
    },
    "001526": {
      "height": 375,
      "width": 1242
    },
    "001527": {
      "height": 375,
      "width": 1242
    },
    "001528": {
      "height": 375,
      "width": 1242
    },
    "001529": {
      "height": 370,
      "width": 1224
    },
    "001530": {
      "height": 375,
      "width": 1242
    },
    "001531": {
      "height": 375,
      "width": 1242
    },
    "001532": {
      "height": 375,
      "width": 1242
    },
    "001533": {
      "height": 376,
      "width": 1241
    },
    "001534": {
      "height": 375,
      "width": 1242
    },
    "001535": {
      "height": 375,
      "width": 1242
    },
    "001536": {
      "height": 375,
      "width": 1242
    },
    "001537": {
      "height": 375,
      "width": 1242
    },
    "001538": {
      "height": 375,
      "width": 1242
    },
    "001539": {
      "height": 375,
      "width": 1242
    },
    "001540": {
      "height": 370,
      "width": 1224
    },
    "001541": {
      "height": 375,
      "width": 1242
    },
    "001542": {
      "height": 375,
      "width": 1242
    },
    "001543": {
      "height": 375,
      "width": 1242
    },
    "001544": {
      "height": 370,
      "width": 1224
    },
    "001545": {
      "height": 375,
      "width": 1242
    },
    "001546": {
      "height": 375,
      "width": 1242
    },
    "001547": {
      "height": 370,
      "width": 1224
    },
    "001548": {
      "height": 370,
      "width": 1224
    },
    "001549": {
      "height": 374,
      "width": 1238
    },
    "001550": {
      "height": 370,
      "width": 1224
    },
    "001551": {
      "height": 375,
      "width": 1242
    },
    "001552": {
      "height": 375,
      "width": 1242
    },
    "001553": {
      "height": 375,
      "width": 1242
    },
    "001554": {
      "height": 375,
      "width": 1242
    },
    "001555": {
      "height": 375,
      "width": 1242
    },
    "001556": {
      "height": 375,
      "width": 1242
    },
    "001557": {
      "height": 375,
      "width": 1242
    },
    "001558": {
      "height": 370,
      "width": 1224
    },
    "001559": {
      "height": 375,
      "width": 1242
    },
    "001560": {
      "height": 375,
      "width": 1242
    },
    "001561": {
      "height": 375,
      "width": 1242
    },
    "001562": {
      "height": 375,
      "width": 1242
    },
    "001563": {
      "height": 375,
      "width": 1242
    },
    "001564": {
      "height": 376,
      "width": 1241
    },
    "001565": {
      "height": 376,
      "width": 1241
    },
    "001566": {
      "height": 375,
      "width": 1242
    },
    "001567": {
      "height": 375,
      "width": 1242
    },
    "001568": {
      "height": 370,
      "width": 1224
    },
    "001569": {
      "height": 375,
      "width": 1242
    },
    "001570": {
      "height": 370,
      "width": 1224
    },
    "001571": {
      "height": 370,
      "width": 1224
    },
    "001572": {
      "height": 375,
      "width": 1242
    },
    "001573": {
      "height": 375,
      "width": 1242
    },
    "001574": {
      "height": 370,
      "width": 1224
    },
    "001575": {
      "height": 375,
      "width": 1242
    },
    "001576": {
      "height": 376,
      "width": 1241
    },
    "001577": {
      "height": 375,
      "width": 1242
    },
    "001578": {
      "height": 375,
      "width": 1242
    },
    "001579": {
      "height": 375,
      "width": 1242
    },
    "001580": {
      "height": 375,
      "width": 1242
    },
    "001581": {
      "height": 375,
      "width": 1242
    },
    "001582": {
      "height": 375,
      "width": 1242
    },
    "001583": {
      "height": 375,
      "width": 1242
    },
    "001584": {
      "height": 375,
      "width": 1242
    },
    "001585": {
      "height": 375,
      "width": 1242
    },
    "001586": {
      "height": 375,
      "width": 1242
    },
    "001587": {
      "height": 375,
      "width": 1242
    },
    "001588": {
      "height": 375,
      "width": 1242
    },
    "001589": {
      "height": 370,
      "width": 1224
    },
    "001590": {
      "height": 375,
      "width": 1242
    },
    "001591": {
      "height": 375,
      "width": 1242
    },
    "001592": {
      "height": 375,
      "width": 1242
    },
    "001593": {
      "height": 375,
      "width": 1242
    },
    "001594": {
      "height": 375,
      "width": 1242
    },
    "001595": {
      "height": 375,
      "width": 1242
    },
    "001596": {
      "height": 375,
      "width": 1242
    },
    "001597": {
      "height": 374,
      "width": 1238
    },
    "001598": {
      "height": 375,
      "width": 1242
    },
    "001599": {
      "height": 375,
      "width": 1242
    },
    "001600": {
      "height": 370,
      "width": 1224
    },
    "001601": {
      "height": 375,
      "width": 1242
    },
    "001602": {
      "height": 375,
      "width": 1242
    },
    "001603": {
      "height": 376,
      "width": 1241
    },
    "001604": {
      "height": 375,
      "width": 1242
    },
    "001605": {
      "height": 375,
      "width": 1242
    },
    "001606": {
      "height": 375,
      "width": 1242
    },
    "001607": {
      "height": 375,
      "width": 1242
    },
    "001608": {
      "height": 375,
      "width": 1242
    },
    "001609": {
      "height": 370,
      "width": 1224
    },
    "001610": {
      "height": 375,
      "width": 1242
    },
    "001611": {
      "height": 370,
      "width": 1224
    },
    "001612": {
      "height": 375,
      "width": 1242
    },
    "001613": {
      "height": 375,
      "width": 1242
    },
    "001614": {
      "height": 375,
      "width": 1242
    },
    "001615": {
      "height": 375,
      "width": 1242
    },
    "001616": {
      "height": 376,
      "width": 1241
    },
    "001617": {
      "height": 375,
      "width": 1242
    },
    "001618": {
      "height": 375,
      "width": 1242
    },
    "001619": {
      "height": 375,
      "width": 1242
    },
    "001620": {
      "height": 375,
      "width": 1242
    },
    "001621": {
      "height": 375,
      "width": 1242
    },
    "001622": {
      "height": 375,
      "width": 1242
    },
    "001623": {
      "height": 375,
      "width": 1242
    },
    "001624": {
      "height": 375,
      "width": 1242
    },
    "001625": {
      "height": 376,
      "width": 1241
    },
    "001626": {
      "height": 375,
      "width": 1242
    },
    "001627": {
      "height": 375,
      "width": 1242
    },
    "001628": {
      "height": 375,
      "width": 1242
    },
    "001629": {
      "height": 375,
      "width": 1242
    },
    "001630": {
      "height": 375,
      "width": 1242
    },
    "001631": {
      "height": 375,
      "width": 1242
    },
    "001632": {
      "height": 370,
      "width": 1224
    },
    "001633": {
      "height": 375,
      "width": 1242
    },
    "001634": {
      "height": 375,
      "width": 1242
    },
    "001635": {
      "height": 375,
      "width": 1242
    },
    "001636": {
      "height": 375,
      "width": 1242
    },
    "001637": {
      "height": 375,
      "width": 1242
    },
    "001638": {
      "height": 375,
      "width": 1242
    },
    "001639": {
      "height": 375,
      "width": 1242
    },
    "001640": {
      "height": 370,
      "width": 1224
    },
    "001641": {
      "height": 375,
      "width": 1242
    },
    "001642": {
      "height": 375,
      "width": 1242
    },
    "001643": {
      "height": 375,
      "width": 1242
    },
    "001644": {
      "height": 375,
      "width": 1242
    },
    "001645": {
      "height": 375,
      "width": 1242
    },
    "001646": {
      "height": 375,
      "width": 1242
    },
    "001647": {
      "height": 375,
      "width": 1242
    },
    "001648": {
      "height": 370,
      "width": 1224
    },
    "001649": {
      "height": 375,
      "width": 1242
    },
    "001650": {
      "height": 375,
      "width": 1242
    },
    "001651": {
      "height": 375,
      "width": 1242
    },
    "001652": {
      "height": 375,
      "width": 1242
    },
    "001653": {
      "height": 375,
      "width": 1242
    },
    "001654": {
      "height": 370,
      "width": 1224
    },
    "001655": {
      "height": 370,
      "width": 1224
    },
    "001656": {
      "height": 375,
      "width": 1242
    },
    "001657": {
      "height": 375,
      "width": 1242
    },
    "001658": {
      "height": 375,
      "width": 1242
    },
    "001659": {
      "height": 375,
      "width": 1242
    },
    "001660": {
      "height": 375,
      "width": 1242
    },
    "001661": {
      "height": 375,
      "width": 1242
    },
    "001662": {
      "height": 375,
      "width": 1242
    },
    "001663": {
      "height": 375,
      "width": 1242
    },
    "001664": {
      "height": 375,
      "width": 1242
    },
    "001665": {
      "height": 375,
      "width": 1242
    },
    "001666": {
      "height": 375,
      "width": 1242
    },
    "001667": {
      "height": 374,
      "width": 1238
    },
    "001668": {
      "height": 374,
      "width": 1238
    },
    "001669": {
      "height": 375,
      "width": 1242
    },
    "001670": {
      "height": 375,
      "width": 1242
    },
    "001671": {
      "height": 375,
      "width": 1242
    },
    "001672": {
      "height": 370,
      "width": 1224
    },
    "001673": {
      "height": 375,
      "width": 1242
    },
    "001674": {
      "height": 375,
      "width": 1242
    },
    "001675": {
      "height": 375,
      "width": 1242
    },
    "001676": {
      "height": 375,
      "width": 1242
    },
    "001677": {
      "height": 375,
      "width": 1242
    },
    "001678": {
      "height": 375,
      "width": 1242
    },
    "001679": {
      "height": 374,
      "width": 1238
    },
    "001680": {
      "height": 375,
      "width": 1242
    },
    "001681": {
      "height": 375,
      "width": 1242
    },
    "001682": {
      "height": 375,
      "width": 1242
    },
    "001683": {
      "height": 375,
      "width": 1242
    },
    "001684": {
      "height": 375,
      "width": 1242
    },
    "001685": {
      "height": 375,
      "width": 1242
    },
    "001686": {
      "height": 375,
      "width": 1242
    },
    "001687": {
      "height": 375,
      "width": 1242
    },
    "001688": {
      "height": 375,
      "width": 1242
    },
    "001689": {
      "height": 370,
      "width": 1224
    },
    "001690": {
      "height": 375,
      "width": 1242
    },
    "001691": {
      "height": 375,
      "width": 1242
    },
    "001692": {
      "height": 370,
      "width": 1224
    },
    "001693": {
      "height": 374,
      "width": 1238
    },
    "001694": {
      "height": 375,
      "width": 1242
    },
    "001695": {
      "height": 375,
      "width": 1242
    },
    "001696": {
      "height": 375,
      "width": 1242
    },
    "001697": {
      "height": 376,
      "width": 1241
    },
    "001698": {
      "height": 375,
      "width": 1242
    },
    "001699": {
      "height": 375,
      "width": 1242
    },
    "001700": {
      "height": 370,
      "width": 1224
    },
    "001701": {
      "height": 375,
      "width": 1242
    },
    "001702": {
      "height": 375,
      "width": 1242
    },
    "001703": {
      "height": 375,
      "width": 1242
    },
    "001704": {
      "height": 375,
      "width": 1242
    },
    "001705": {
      "height": 376,
      "width": 1241
    },
    "001706": {
      "height": 375,
      "width": 1242
    },
    "001707": {
      "height": 375,
      "width": 1242
    },
    "001708": {
      "height": 375,
      "width": 1242
    },
    "001709": {
      "height": 375,
      "width": 1242
    },
    "001710": {
      "height": 375,
      "width": 1242
    },
    "001711": {
      "height": 375,
      "width": 1242
    },
    "001712": {
      "height": 375,
      "width": 1242
    },
    "001713": {
      "height": 374,
      "width": 1238
    },
    "001714": {
      "height": 370,
      "width": 1224
    },
    "001715": {
      "height": 375,
      "width": 1242
    },
    "001716": {
      "height": 370,
      "width": 1224
    },
    "001717": {
      "height": 375,
      "width": 1242
    },
    "001718": {
      "height": 375,
      "width": 1242
    },
    "001719": {
      "height": 375,
      "width": 1242
    },
    "001720": {
      "height": 375,
      "width": 1242
    },
    "001721": {
      "height": 375,
      "width": 1242
    },
    "001722": {
      "height": 375,
      "width": 1242
    },
    "001723": {
      "height": 375,
      "width": 1242
    },
    "001724": {
      "height": 375,
      "width": 1242
    },
    "001725": {
      "height": 370,
      "width": 1224
    },
    "001726": {
      "height": 375,
      "width": 1242
    },
    "001727": {
      "height": 375,
      "width": 1242
    },
    "001728": {
      "height": 375,
      "width": 1242
    },
    "001729": {
      "height": 375,
      "width": 1242
    },
    "001730": {
      "height": 375,
      "width": 1242
    },
    "001731": {
      "height": 375,
      "width": 1242
    },
    "001732": {
      "height": 375,
      "width": 1242
    },
    "001733": {
      "height": 375,
      "width": 1242
    },
    "001734": {
      "height": 370,
      "width": 1224
    },
    "001735": {
      "height": 375,
      "width": 1242
    },
    "001736": {
      "height": 375,
      "width": 1242
    },
    "001737": {
      "height": 375,
      "width": 1242
    },
    "001738": {
      "height": 375,
      "width": 1242
    },
    "001739": {
      "height": 375,
      "width": 1242
    },
    "001740": {
      "height": 375,
      "width": 1242
    },
    "001741": {
      "height": 376,
      "width": 1241
    },
    "001742": {
      "height": 376,
      "width": 1241
    },
    "001743": {
      "height": 375,
      "width": 1242
    },
    "001744": {
      "height": 375,
      "width": 1242
    },
    "001745": {
      "height": 375,
      "width": 1242
    },
    "001746": {
      "height": 375,
      "width": 1242
    },
    "001747": {
      "height": 375,
      "width": 1242
    },
    "001748": {
      "height": 370,
      "width": 1224
    },
    "001749": {
      "height": 375,
      "width": 1242
    },
    "001750": {
      "height": 375,
      "width": 1242
    },
    "001751": {
      "height": 375,
      "width": 1242
    },
    "001752": {
      "height": 375,
      "width": 1242
    },
    "001753": {
      "height": 375,
      "width": 1242
    },
    "001754": {
      "height": 375,
      "width": 1242
    },
    "001755": {
      "height": 375,
      "width": 1242
    },
    "001756": {
      "height": 375,
      "width": 1242
    },
    "001757": {
      "height": 375,
      "width": 1242
    },
    "001758": {
      "height": 370,
      "width": 1224
    },
    "001759": {
      "height": 375,
      "width": 1242
    },
    "001760": {
      "height": 375,
      "width": 1242
    },
    "001761": {
      "height": 375,
      "width": 1242
    },
    "001762": {
      "height": 375,
      "width": 1242
    },
    "001763": {
      "height": 375,
      "width": 1242
    },
    "001764": {
      "height": 376,
      "width": 1241
    },
    "001765": {
      "height": 375,
      "width": 1242
    },
    "001766": {
      "height": 375,
      "width": 1242
    },
    "001767": {
      "height": 375,
      "width": 1242
    },
    "001768": {
      "height": 375,
      "width": 1242
    },
    "001769": {
      "height": 375,
      "width": 1242
    },
    "001770": {
      "height": 375,
      "width": 1242
    },
    "001771": {
      "height": 375,
      "width": 1242
    },
    "001772": {
      "height": 375,
      "width": 1242
    },
    "001773": {
      "height": 375,
      "width": 1242
    },
    "001774": {
      "height": 375,
      "width": 1242
    },
    "001775": {
      "height": 374,
      "width": 1238
    },
    "001776": {
      "height": 374,
      "width": 1238
    },
    "001777": {
      "height": 370,
      "width": 1224
    },
    "001778": {
      "height": 375,
      "width": 1242
    },
    "001779": {
      "height": 375,
      "width": 1242
    },
    "001780": {
      "height": 375,
      "width": 1242
    },
    "001781": {
      "height": 375,
      "width": 1242
    },
    "001782": {
      "height": 370,
      "width": 1224
    },
    "001783": {
      "height": 375,
      "width": 1242
    },
    "001784": {
      "height": 375,
      "width": 1242
    },
    "001785": {
      "height": 370,
      "width": 1224
    },
    "001786": {
      "height": 375,
      "width": 1242
    },
    "001787": {
      "height": 375,
      "width": 1242
    },
    "001788": {
      "height": 375,
      "width": 1242
    },
    "001789": {
      "height": 375,
      "width": 1242
    },
    "001790": {
      "height": 375,
      "width": 1242
    },
    "001791": {
      "height": 375,
      "width": 1242
    },
    "001792": {
      "height": 375,
      "width": 1242
    },
    "001793": {
      "height": 375,
      "width": 1242
    },
    "001794": {
      "height": 376,
      "width": 1241
    },
    "001795": {
      "height": 374,
      "width": 1238
    },
    "001796": {
      "height": 375,
      "width": 1242
    },
    "001797": {
      "height": 375,
      "width": 1242
    },
    "001798": {
      "height": 375,
      "width": 1242
    },
    "001799": {
      "height": 375,
      "width": 1242
    },
    "001800": {
      "height": 375,
      "width": 1242
    },
    "001801": {
      "height": 375,
      "width": 1242
    },
    "001802": {
      "height": 375,
      "width": 1242
    },
    "001803": {
      "height": 375,
      "width": 1242
    },
    "001804": {
      "height": 370,
      "width": 1224
    },
    "001805": {
      "height": 375,
      "width": 1242
    },
    "001806": {
      "height": 375,
      "width": 1242
    },
    "001807": {
      "height": 375,
      "width": 1242
    },
    "001808": {
      "height": 375,
      "width": 1242
    },
    "001809": {
      "height": 375,
      "width": 1242
    },
    "001810": {
      "height": 375,
      "width": 1242
    },
    "001811": {
      "height": 375,
      "width": 1242
    },
    "001812": {
      "height": 375,
      "width": 1242
    },
    "001813": {
      "height": 375,
      "width": 1242
    },
    "001814": {
      "height": 370,
      "width": 1224
    },
    "001815": {
      "height": 375,
      "width": 1242
    },
    "001816": {
      "height": 375,
      "width": 1242
    },
    "001817": {
      "height": 370,
      "width": 1224
    },
    "001818": {
      "height": 375,
      "width": 1242
    },
    "001819": {
      "height": 370,
      "width": 1224
    },
    "001820": {
      "height": 375,
      "width": 1242
    },
    "001821": {
      "height": 375,
      "width": 1242
    },
    "001822": {
      "height": 374,
      "width": 1238
    },
    "001823": {
      "height": 375,
      "width": 1242
    },
    "001824": {
      "height": 375,
      "width": 1242
    },
    "001825": {
      "height": 375,
      "width": 1242
    },
    "001826": {
      "height": 375,
      "width": 1242
    },
    "001827": {
      "height": 375,
      "width": 1242
    },
    "001828": {
      "height": 376,
      "width": 1241
    },
    "001829": {
      "height": 375,
      "width": 1242
    },
    "001830": {
      "height": 375,
      "width": 1242
    },
    "001831": {
      "height": 374,
      "width": 1238
    },
    "001832": {
      "height": 375,
      "width": 1242
    },
    "001833": {
      "height": 375,
      "width": 1242
    },
    "001834": {
      "height": 375,
      "width": 1242
    },
    "001835": {
      "height": 370,
      "width": 1224
    },
    "001836": {
      "height": 375,
      "width": 1242
    },
    "001837": {
      "height": 375,
      "width": 1242
    },
    "001838": {
      "height": 375,
      "width": 1242
    },
    "001839": {
      "height": 375,
      "width": 1242
    },
    "001840": {
      "height": 375,
      "width": 1242
    },
    "001841": {
      "height": 375,
      "width": 1242
    },
    "001842": {
      "height": 375,
      "width": 1242
    },
    "001843": {
      "height": 375,
      "width": 1242
    },
    "001844": {
      "height": 375,
      "width": 1242
    },
    "001845": {
      "height": 375,
      "width": 1242
    },
    "001846": {
      "height": 370,
      "width": 1224
    },
    "001847": {
      "height": 375,
      "width": 1242
    },
    "001848": {
      "height": 375,
      "width": 1242
    },
    "001849": {
      "height": 375,
      "width": 1242
    },
    "001850": {
      "height": 375,
      "width": 1242
    },
    "001851": {
      "height": 375,
      "width": 1242
    },
    "001852": {
      "height": 375,
      "width": 1242
    },
    "001853": {
      "height": 374,
      "width": 1238
    },
    "001854": {
      "height": 375,
      "width": 1242
    },
    "001855": {
      "height": 376,
      "width": 1241
    },
    "001856": {
      "height": 375,
      "width": 1242
    },
    "001857": {
      "height": 375,
      "width": 1242
    },
    "001858": {
      "height": 376,
      "width": 1241
    },
    "001859": {
      "height": 375,
      "width": 1242
    },
    "001860": {
      "height": 375,
      "width": 1242
    },
    "001861": {
      "height": 375,
      "width": 1242
    },
    "001862": {
      "height": 375,
      "width": 1242
    },
    "001863": {
      "height": 375,
      "width": 1242
    },
    "001864": {
      "height": 370,
      "width": 1224
    },
    "001865": {
      "height": 374,
      "width": 1238
    },
    "001866": {
      "height": 375,
      "width": 1242
    },
    "001867": {
      "height": 375,
      "width": 1242
    },
    "001868": {
      "height": 375,
      "width": 1242
    },
    "001869": {
      "height": 375,
      "width": 1242
    },
    "001870": {
      "height": 375,
      "width": 1242
    },
    "001871": {
      "height": 375,
      "width": 1242
    },
    "001872": {
      "height": 375,
      "width": 1242
    },
    "001873": {
      "height": 375,
      "width": 1242
    },
    "001874": {
      "height": 375,
      "width": 1242
    },
    "001875": {
      "height": 375,
      "width": 1242
    },
    "001876": {
      "height": 375,
      "width": 1242
    },
    "001877": {
      "height": 370,
      "width": 1224
    },
    "001878": {
      "height": 375,
      "width": 1242
    },
    "001879": {
      "height": 370,
      "width": 1224
    },
    "001880": {
      "height": 375,
      "width": 1242
    },
    "001881": {
      "height": 375,
      "width": 1242
    },
    "001882": {
      "height": 375,
      "width": 1242
    },
    "001883": {
      "height": 375,
      "width": 1242
    },
    "001884": {
      "height": 375,
      "width": 1242
    },
    "001885": {
      "height": 375,
      "width": 1242
    },
    "001886": {
      "height": 375,
      "width": 1242
    },
    "001887": {
      "height": 375,
      "width": 1242
    },
    "001888": {
      "height": 370,
      "width": 1224
    },
    "001889": {
      "height": 375,
      "width": 1242
    },
    "001890": {
      "height": 375,
      "width": 1242
    },
    "001891": {
      "height": 375,
      "width": 1242
    },
    "001892": {
      "height": 375,
      "width": 1242
    },
    "001893": {
      "height": 376,
      "width": 1241
    },
    "001894": {
      "height": 375,
      "width": 1242
    },
    "001895": {
      "height": 375,
      "width": 1242
    },
    "001896": {
      "height": 375,
      "width": 1242
    },
    "001897": {
      "height": 375,
      "width": 1242
    },
    "001898": {
      "height": 375,
      "width": 1242
    },
    "001899": {
      "height": 375,
      "width": 1242
    },
    "001900": {
      "height": 375,
      "width": 1242
    },
    "001901": {
      "height": 375,
      "width": 1242
    },
    "001902": {
      "height": 370,
      "width": 1224
    },
    "001903": {
      "height": 375,
      "width": 1242
    },
    "001904": {
      "height": 375,
      "width": 1242
    },
    "001905": {
      "height": 375,
      "width": 1242
    },
    "001906": {
      "height": 375,
      "width": 1242
    },
    "001907": {
      "height": 375,
      "width": 1242
    },
    "001908": {
      "height": 375,
      "width": 1242
    },
    "001909": {
      "height": 375,
      "width": 1242
    },
    "001910": {
      "height": 375,
      "width": 1242
    },
    "001911": {
      "height": 370,
      "width": 1224
    },
    "001912": {
      "height": 370,
      "width": 1224
    },
    "001913": {
      "height": 375,
      "width": 1242
    },
    "001914": {
      "height": 375,
      "width": 1242
    },
    "001915": {
      "height": 375,
      "width": 1242
    },
    "001916": {
      "height": 375,
      "width": 1242
    },
    "001917": {
      "height": 375,
      "width": 1242
    },
    "001918": {
      "height": 375,
      "width": 1242
    },
    "001919": {
      "height": 375,
      "width": 1242
    },
    "001920": {
      "height": 375,
      "width": 1242
    },
    "001921": {
      "height": 375,
      "width": 1242
    },
    "001922": {
      "height": 375,
      "width": 1242
    },
    "001923": {
      "height": 370,
      "width": 1224
    },
    "001924": {
      "height": 375,
      "width": 1242
    },
    "001925": {
      "height": 376,
      "width": 1241
    },
    "001926": {
      "height": 375,
      "width": 1242
    },
    "001927": {
      "height": 375,
      "width": 1242
    },
    "001928": {
      "height": 376,
      "width": 1241
    },
    "001929": {
      "height": 375,
      "width": 1242
    },
    "001930": {
      "height": 375,
      "width": 1242
    },
    "001931": {
      "height": 374,
      "width": 1238
    },
    "001932": {
      "height": 375,
      "width": 1242
    },
    "001933": {
      "height": 375,
      "width": 1242
    },
    "001934": {
      "height": 375,
      "width": 1242
    },
    "001935": {
      "height": 375,
      "width": 1242
    },
    "001936": {
      "height": 375,
      "width": 1242
    },
    "001937": {
      "height": 376,
      "width": 1241
    },
    "001938": {
      "height": 375,
      "width": 1242
    },
    "001939": {
      "height": 375,
      "width": 1242
    },
    "001940": {
      "height": 375,
      "width": 1242
    },
    "001941": {
      "height": 375,
      "width": 1242
    },
    "001942": {
      "height": 376,
      "width": 1241
    },
    "001943": {
      "height": 375,
      "width": 1242
    },
    "001944": {
      "height": 375,
      "width": 1242
    },
    "001945": {
      "height": 375,
      "width": 1242
    },
    "001946": {
      "height": 375,
      "width": 1242
    },
    "001947": {
      "height": 370,
      "width": 1224
    },
    "001948": {
      "height": 375,
      "width": 1242
    },
    "001949": {
      "height": 375,
      "width": 1242
    },
    "001950": {
      "height": 375,
      "width": 1242
    },
    "001951": {
      "height": 375,
      "width": 1242
    },
    "001952": {
      "height": 375,
      "width": 1242
    },
    "001953": {
      "height": 375,
      "width": 1242
    },
    "001954": {
      "height": 375,
      "width": 1242
    },
    "001955": {
      "height": 375,
      "width": 1242
    },
    "001956": {
      "height": 375,
      "width": 1242
    },
    "001957": {
      "height": 375,
      "width": 1242
    },
    "001958": {
      "height": 370,
      "width": 1224
    },
    "001959": {
      "height": 375,
      "width": 1242
    },
    "001960": {
      "height": 375,
      "width": 1242
    },
    "001961": {
      "height": 375,
      "width": 1242
    },
    "001962": {
      "height": 375,
      "width": 1242
    },
    "001963": {
      "height": 375,
      "width": 1242
    },
    "001964": {
      "height": 375,
      "width": 1242
    },
    "001965": {
      "height": 375,
      "width": 1242
    },
    "001966": {
      "height": 375,
      "width": 1242
    },
    "001967": {
      "height": 375,
      "width": 1242
    },
    "001968": {
      "height": 375,
      "width": 1242
    },
    "001969": {
      "height": 375,
      "width": 1242
    },
    "001970": {
      "height": 370,
      "width": 1224
    },
    "001971": {
      "height": 370,
      "width": 1224
    },
    "001972": {
      "height": 374,
      "width": 1238
    },
    "001973": {
      "height": 375,
      "width": 1242
    },
    "001974": {
      "height": 375,
      "width": 1242
    },
    "001975": {
      "height": 375,
      "width": 1242
    },
    "001976": {
      "height": 375,
      "width": 1242
    },
    "001977": {
      "height": 375,
      "width": 1242
    },
    "001978": {
      "height": 375,
      "width": 1242
    },
    "001979": {
      "height": 375,
      "width": 1242
    },
    "001980": {
      "height": 375,
      "width": 1242
    },
    "001981": {
      "height": 375,
      "width": 1242
    },
    "001982": {
      "height": 370,
      "width": 1224
    },
    "001983": {
      "height": 375,
      "width": 1242
    },
    "001984": {
      "height": 370,
      "width": 1224
    },
    "001985": {
      "height": 375,
      "width": 1242
    },
    "001986": {
      "height": 376,
      "width": 1241
    },
    "001987": {
      "height": 375,
      "width": 1242
    },
    "001988": {
      "height": 375,
      "width": 1242
    },
    "001989": {
      "height": 375,
      "width": 1242
    },
    "001990": {
      "height": 375,
      "width": 1242
    },
    "001991": {
      "height": 376,
      "width": 1241
    },
    "001992": {
      "height": 375,
      "width": 1242
    },
    "001993": {
      "height": 375,
      "width": 1242
    },
    "001994": {
      "height": 370,
      "width": 1224
    },
    "001995": {
      "height": 375,
      "width": 1242
    },
    "001996": {
      "height": 375,
      "width": 1242
    },
    "001997": {
      "height": 375,
      "width": 1242
    },
    "001998": {
      "height": 370,
      "width": 1224
    },
    "001999": {
      "height": 375,
      "width": 1242
    },
    "002000": {
      "height": 370,
      "width": 1224
    },
    "002001": {
      "height": 375,
      "width": 1242
    },
    "002002": {
      "height": 375,
      "width": 1242
    },
    "002003": {
      "height": 375,
      "width": 1242
    },
    "002004": {
      "height": 375,
      "width": 1242
    },
    "002005": {
      "height": 375,
      "width": 1242
    },
    "002006": {
      "height": 375,
      "width": 1242
    },
    "002007": {
      "height": 375,
      "width": 1242
    },
    "002008": {
      "height": 375,
      "width": 1242
    },
    "002009": {
      "height": 375,
      "width": 1242
    },
    "002010": {
      "height": 375,
      "width": 1242
    },
    "002011": {
      "height": 375,
      "width": 1242
    },
    "002012": {
      "height": 375,
      "width": 1242
    },
    "002013": {
      "height": 375,
      "width": 1242
    },
    "002014": {
      "height": 370,
      "width": 1224
    },
    "002015": {
      "height": 375,
      "width": 1242
    },
    "002016": {
      "height": 375,
      "width": 1242
    },
    "002017": {
      "height": 375,
      "width": 1242
    },
    "002018": {
      "height": 375,
      "width": 1242
    },
    "002019": {
      "height": 375,
      "width": 1242
    },
    "002020": {
      "height": 375,
      "width": 1242
    },
    "002021": {
      "height": 375,
      "width": 1242
    },
    "002022": {
      "height": 375,
      "width": 1242
    },
    "002023": {
      "height": 370,
      "width": 1224
    },
    "002024": {
      "height": 375,
      "width": 1242
    },
    "002025": {
      "height": 375,
      "width": 1242
    },
    "002026": {
      "height": 375,
      "width": 1242
    },
    "002027": {
      "height": 375,
      "width": 1242
    },
    "002028": {
      "height": 375,
      "width": 1242
    },
    "002029": {
      "height": 375,
      "width": 1242
    },
    "002030": {
      "height": 370,
      "width": 1224
    },
    "002031": {
      "height": 375,
      "width": 1242
    },
    "002032": {
      "height": 375,
      "width": 1242
    },
    "002033": {
      "height": 375,
      "width": 1242
    },
    "002034": {
      "height": 375,
      "width": 1242
    },
    "002035": {
      "height": 375,
      "width": 1242
    },
    "002036": {
      "height": 375,
      "width": 1242
    },
    "002037": {
      "height": 375,
      "width": 1242
    },
    "002038": {
      "height": 375,
      "width": 1242
    },
    "002039": {
      "height": 375,
      "width": 1242
    },
    "002040": {
      "height": 375,
      "width": 1242
    },
    "002041": {
      "height": 375,
      "width": 1242
    },
    "002042": {
      "height": 374,
      "width": 1238
    },
    "002043": {
      "height": 375,
      "width": 1242
    },
    "002044": {
      "height": 376,
      "width": 1241
    },
    "002045": {
      "height": 374,
      "width": 1238
    },
    "002046": {
      "height": 375,
      "width": 1242
    },
    "002047": {
      "height": 375,
      "width": 1242
    },
    "002048": {
      "height": 375,
      "width": 1242
    },
    "002049": {
      "height": 375,
      "width": 1242
    },
    "002050": {
      "height": 375,
      "width": 1242
    },
    "002051": {
      "height": 375,
      "width": 1242
    },
    "002052": {
      "height": 376,
      "width": 1241
    },
    "002053": {
      "height": 375,
      "width": 1242
    },
    "002054": {
      "height": 375,
      "width": 1242
    },
    "002055": {
      "height": 375,
      "width": 1242
    },
    "002056": {
      "height": 375,
      "width": 1242
    },
    "002057": {
      "height": 375,
      "width": 1242
    },
    "002058": {
      "height": 370,
      "width": 1224
    },
    "002059": {
      "height": 370,
      "width": 1224
    },
    "002060": {
      "height": 375,
      "width": 1242
    },
    "002061": {
      "height": 375,
      "width": 1242
    },
    "002062": {
      "height": 375,
      "width": 1242
    },
    "002063": {
      "height": 370,
      "width": 1224
    },
    "002064": {
      "height": 375,
      "width": 1242
    },
    "002065": {
      "height": 375,
      "width": 1242
    },
    "002066": {
      "height": 375,
      "width": 1242
    },
    "002067": {
      "height": 375,
      "width": 1242
    },
    "002068": {
      "height": 375,
      "width": 1242
    },
    "002069": {
      "height": 375,
      "width": 1242
    },
    "002070": {
      "height": 375,
      "width": 1242
    },
    "002071": {
      "height": 374,
      "width": 1238
    },
    "002072": {
      "height": 374,
      "width": 1238
    },
    "002073": {
      "height": 374,
      "width": 1238
    },
    "002074": {
      "height": 375,
      "width": 1242
    },
    "002075": {
      "height": 375,
      "width": 1242
    },
    "002076": {
      "height": 370,
      "width": 1224
    },
    "002077": {
      "height": 375,
      "width": 1242
    },
    "002078": {
      "height": 375,
      "width": 1242
    },
    "002079": {
      "height": 375,
      "width": 1242
    },
    "002080": {
      "height": 375,
      "width": 1242
    },
    "002081": {
      "height": 375,
      "width": 1242
    },
    "002082": {
      "height": 375,
      "width": 1242
    },
    "002083": {
      "height": 375,
      "width": 1242
    },
    "002084": {
      "height": 375,
      "width": 1242
    },
    "002085": {
      "height": 375,
      "width": 1242
    },
    "002086": {
      "height": 375,
      "width": 1242
    },
    "002087": {
      "height": 375,
      "width": 1242
    },
    "002088": {
      "height": 375,
      "width": 1242
    },
    "002089": {
      "height": 375,
      "width": 1242
    },
    "002090": {
      "height": 375,
      "width": 1242
    },
    "002091": {
      "height": 374,
      "width": 1238
    },
    "002092": {
      "height": 375,
      "width": 1242
    },
    "002093": {
      "height": 375,
      "width": 1242
    },
    "002094": {
      "height": 375,
      "width": 1242
    },
    "002095": {
      "height": 370,
      "width": 1224
    },
    "002096": {
      "height": 375,
      "width": 1242
    },
    "002097": {
      "height": 375,
      "width": 1242
    },
    "002098": {
      "height": 374,
      "width": 1238
    },
    "002099": {
      "height": 375,
      "width": 1242
    },
    "002100": {
      "height": 375,
      "width": 1242
    },
    "002101": {
      "height": 375,
      "width": 1242
    },
    "002102": {
      "height": 375,
      "width": 1242
    },
    "002103": {
      "height": 375,
      "width": 1242
    },
    "002104": {
      "height": 375,
      "width": 1242
    },
    "002105": {
      "height": 375,
      "width": 1242
    },
    "002106": {
      "height": 375,
      "width": 1242
    },
    "002107": {
      "height": 375,
      "width": 1242
    },
    "002108": {
      "height": 370,
      "width": 1224
    },
    "002109": {
      "height": 375,
      "width": 1242
    },
    "002110": {
      "height": 375,
      "width": 1242
    },
    "002111": {
      "height": 375,
      "width": 1242
    },
    "002112": {
      "height": 375,
      "width": 1242
    },
    "002113": {
      "height": 375,
      "width": 1242
    },
    "002114": {
      "height": 375,
      "width": 1242
    },
    "002115": {
      "height": 375,
      "width": 1242
    },
    "002116": {
      "height": 375,
      "width": 1242
    },
    "002117": {
      "height": 370,
      "width": 1224
    },
    "002118": {
      "height": 375,
      "width": 1242
    },
    "002119": {
      "height": 375,
      "width": 1242
    },
    "002120": {
      "height": 375,
      "width": 1242
    },
    "002121": {
      "height": 375,
      "width": 1242
    },
    "002122": {
      "height": 375,
      "width": 1242
    },
    "002123": {
      "height": 376,
      "width": 1241
    },
    "002124": {
      "height": 370,
      "width": 1224
    },
    "002125": {
      "height": 375,
      "width": 1242
    },
    "002126": {
      "height": 375,
      "width": 1242
    },
    "002127": {
      "height": 375,
      "width": 1242
    },
    "002128": {
      "height": 375,
      "width": 1242
    },
    "002129": {
      "height": 375,
      "width": 1242
    },
    "002130": {
      "height": 375,
      "width": 1242
    },
    "002131": {
      "height": 375,
      "width": 1242
    },
    "002132": {
      "height": 375,
      "width": 1242
    },
    "002133": {
      "height": 375,
      "width": 1242
    },
    "002134": {
      "height": 375,
      "width": 1242
    },
    "002135": {
      "height": 375,
      "width": 1242
    },
    "002136": {
      "height": 375,
      "width": 1242
    },
    "002137": {
      "height": 375,
      "width": 1242
    },
    "002138": {
      "height": 375,
      "width": 1242
    },
    "002139": {
      "height": 376,
      "width": 1241
    },
    "002140": {
      "height": 376,
      "width": 1241
    },
    "002141": {
      "height": 375,
      "width": 1242
    },
    "002142": {
      "height": 376,
      "width": 1241
    },
    "002143": {
      "height": 375,
      "width": 1242
    },
    "002144": {
      "height": 370,
      "width": 1224
    },
    "002145": {
      "height": 374,
      "width": 1238
    },
    "002146": {
      "height": 375,
      "width": 1242
    },
    "002147": {
      "height": 375,
      "width": 1242
    },
    "002148": {
      "height": 374,
      "width": 1238
    },
    "002149": {
      "height": 375,
      "width": 1242
    },
    "002150": {
      "height": 375,
      "width": 1242
    },
    "002151": {
      "height": 376,
      "width": 1241
    },
    "002152": {
      "height": 375,
      "width": 1242
    },
    "002153": {
      "height": 375,
      "width": 1242
    },
    "002154": {
      "height": 375,
      "width": 1242
    },
    "002155": {
      "height": 375,
      "width": 1242
    },
    "002156": {
      "height": 370,
      "width": 1224
    },
    "002157": {
      "height": 375,
      "width": 1242
    },
    "002158": {
      "height": 375,
      "width": 1242
    },
    "002159": {
      "height": 370,
      "width": 1224
    },
    "002160": {
      "height": 374,
      "width": 1238
    },
    "002161": {
      "height": 375,
      "width": 1242
    },
    "002162": {
      "height": 375,
      "width": 1242
    },
    "002163": {
      "height": 375,
      "width": 1242
    },
    "002164": {
      "height": 375,
      "width": 1242
    },
    "002165": {
      "height": 375,
      "width": 1242
    },
    "002166": {
      "height": 375,
      "width": 1242
    },
    "002167": {
      "height": 375,
      "width": 1242
    },
    "002168": {
      "height": 375,
      "width": 1242
    },
    "002169": {
      "height": 375,
      "width": 1242
    },
    "002170": {
      "height": 375,
      "width": 1242
    },
    "002171": {
      "height": 370,
      "width": 1224
    },
    "002172": {
      "height": 375,
      "width": 1242
    },
    "002173": {
      "height": 376,
      "width": 1241
    },
    "002174": {
      "height": 375,
      "width": 1242
    },
    "002175": {
      "height": 370,
      "width": 1224
    },
    "002176": {
      "height": 375,
      "width": 1242
    },
    "002177": {
      "height": 375,
      "width": 1242
    },
    "002178": {
      "height": 370,
      "width": 1224
    },
    "002179": {
      "height": 375,
      "width": 1242
    },
    "002180": {
      "height": 375,
      "width": 1242
    },
    "002181": {
      "height": 375,
      "width": 1242
    },
    "002182": {
      "height": 376,
      "width": 1241
    },
    "002183": {
      "height": 375,
      "width": 1242
    },
    "002184": {
      "height": 375,
      "width": 1242
    },
    "002185": {
      "height": 375,
      "width": 1242
    },
    "002186": {
      "height": 375,
      "width": 1242
    },
    "002187": {
      "height": 375,
      "width": 1242
    },
    "002188": {
      "height": 375,
      "width": 1242
    },
    "002189": {
      "height": 375,
      "width": 1242
    },
    "002190": {
      "height": 375,
      "width": 1242
    },
    "002191": {
      "height": 375,
      "width": 1242
    },
    "002192": {
      "height": 375,
      "width": 1242
    },
    "002193": {
      "height": 375,
      "width": 1242
    },
    "002194": {
      "height": 375,
      "width": 1242
    },
    "002195": {
      "height": 374,
      "width": 1238
    },
    "002196": {
      "height": 375,
      "width": 1242
    },
    "002197": {
      "height": 375,
      "width": 1242
    },
    "002198": {
      "height": 375,
      "width": 1242
    },
    "002199": {
      "height": 375,
      "width": 1242
    },
    "002200": {
      "height": 375,
      "width": 1242
    },
    "002201": {
      "height": 376,
      "width": 1241
    },
    "002202": {
      "height": 375,
      "width": 1242
    },
    "002203": {
      "height": 375,
      "width": 1242
    },
    "002204": {
      "height": 375,
      "width": 1242
    },
    "002205": {
      "height": 375,
      "width": 1242
    },
    "002206": {
      "height": 374,
      "width": 1238
    },
    "002207": {
      "height": 375,
      "width": 1242
    },
    "002208": {
      "height": 375,
      "width": 1242
    },
    "002209": {
      "height": 375,
      "width": 1242
    },
    "002210": {
      "height": 375,
      "width": 1242
    },
    "002211": {
      "height": 375,
      "width": 1242
    },
    "002212": {
      "height": 375,
      "width": 1242
    },
    "002213": {
      "height": 375,
      "width": 1242
    },
    "002214": {
      "height": 374,
      "width": 1238
    },
    "002215": {
      "height": 375,
      "width": 1242
    },
    "002216": {
      "height": 375,
      "width": 1242
    },
    "002217": {
      "height": 375,
      "width": 1242
    },
    "002218": {
      "height": 375,
      "width": 1242
    },
    "002219": {
      "height": 375,
      "width": 1242
    },
    "002220": {
      "height": 375,
      "width": 1242
    },
    "002221": {
      "height": 375,
      "width": 1242
    },
    "002222": {
      "height": 375,
      "width": 1242
    },
    "002223": {
      "height": 375,
      "width": 1242
    },
    "002224": {
      "height": 375,
      "width": 1242
    },
    "002225": {
      "height": 375,
      "width": 1242
    },
    "002226": {
      "height": 370,
      "width": 1224
    },
    "002227": {
      "height": 375,
      "width": 1242
    },
    "002228": {
      "height": 375,
      "width": 1242
    },
    "002229": {
      "height": 375,
      "width": 1242
    },
    "002230": {
      "height": 375,
      "width": 1242
    },
    "002231": {
      "height": 375,
      "width": 1242
    },
    "002232": {
      "height": 375,
      "width": 1242
    },
    "002233": {
      "height": 375,
      "width": 1242
    },
    "002234": {
      "height": 375,
      "width": 1242
    },
    "002235": {
      "height": 375,
      "width": 1242
    },
    "002236": {
      "height": 375,
      "width": 1242
    },
    "002237": {
      "height": 375,
      "width": 1242
    },
    "002238": {
      "height": 375,
      "width": 1242
    },
    "002239": {
      "height": 375,
      "width": 1242
    },
    "002240": {
      "height": 375,
      "width": 1242
    },
    "002241": {
      "height": 375,
      "width": 1242
    },
    "002242": {
      "height": 375,
      "width": 1242
    },
    "002243": {
      "height": 374,
      "width": 1238
    },
    "002244": {
      "height": 375,
      "width": 1242
    },
    "002245": {
      "height": 370,
      "width": 1224
    },
    "002246": {
      "height": 375,
      "width": 1242
    },
    "002247": {
      "height": 370,
      "width": 1224
    },
    "002248": {
      "height": 375,
      "width": 1242
    },
    "002249": {
      "height": 370,
      "width": 1224
    },
    "002250": {
      "height": 375,
      "width": 1242
    },
    "002251": {
      "height": 375,
      "width": 1242
    },
    "002252": {
      "height": 375,
      "width": 1242
    },
    "002253": {
      "height": 375,
      "width": 1242
    },
    "002254": {
      "height": 375,
      "width": 1242
    },
    "002255": {
      "height": 375,
      "width": 1242
    },
    "002256": {
      "height": 375,
      "width": 1242
    },
    "002257": {
      "height": 375,
      "width": 1242
    },
    "002258": {
      "height": 375,
      "width": 1242
    },
    "002259": {
      "height": 375,
      "width": 1242
    },
    "002260": {
      "height": 376,
      "width": 1241
    },
    "002261": {
      "height": 375,
      "width": 1242
    },
    "002262": {
      "height": 376,
      "width": 1241
    },
    "002263": {
      "height": 375,
      "width": 1242
    },
    "002264": {
      "height": 375,
      "width": 1242
    },
    "002265": {
      "height": 375,
      "width": 1242
    },
    "002266": {
      "height": 375,
      "width": 1242
    },
    "002267": {
      "height": 370,
      "width": 1224
    },
    "002268": {
      "height": 375,
      "width": 1242
    },
    "002269": {
      "height": 375,
      "width": 1242
    },
    "002270": {
      "height": 370,
      "width": 1224
    },
    "002271": {
      "height": 370,
      "width": 1224
    },
    "002272": {
      "height": 375,
      "width": 1242
    },
    "002273": {
      "height": 375,
      "width": 1242
    },
    "002274": {
      "height": 374,
      "width": 1238
    },
    "002275": {
      "height": 375,
      "width": 1242
    },
    "002276": {
      "height": 375,
      "width": 1242
    },
    "002277": {
      "height": 375,
      "width": 1242
    },
    "002278": {
      "height": 375,
      "width": 1242
    },
    "002279": {
      "height": 375,
      "width": 1242
    },
    "002280": {
      "height": 375,
      "width": 1242
    },
    "002281": {
      "height": 375,
      "width": 1242
    },
    "002282": {
      "height": 374,
      "width": 1238
    },
    "002283": {
      "height": 375,
      "width": 1242
    },
    "002284": {
      "height": 375,
      "width": 1242
    },
    "002285": {
      "height": 375,
      "width": 1242
    },
    "002286": {
      "height": 375,
      "width": 1242
    },
    "002287": {
      "height": 370,
      "width": 1224
    },
    "002288": {
      "height": 375,
      "width": 1242
    },
    "002289": {
      "height": 370,
      "width": 1224
    },
    "002290": {
      "height": 376,
      "width": 1241
    },
    "002291": {
      "height": 376,
      "width": 1241
    },
    "002292": {
      "height": 375,
      "width": 1242
    },
    "002293": {
      "height": 376,
      "width": 1241
    },
    "002294": {
      "height": 375,
      "width": 1242
    },
    "002295": {
      "height": 376,
      "width": 1241
    },
    "002296": {
      "height": 375,
      "width": 1242
    },
    "002297": {
      "height": 375,
      "width": 1242
    },
    "002298": {
      "height": 375,
      "width": 1242
    },
    "002299": {
      "height": 370,
      "width": 1224
    },
    "002300": {
      "height": 375,
      "width": 1242
    },
    "002301": {
      "height": 375,
      "width": 1242
    },
    "002302": {
      "height": 375,
      "width": 1242
    },
    "002303": {
      "height": 375,
      "width": 1242
    },
    "002304": {
      "height": 375,
      "width": 1242
    },
    "002305": {
      "height": 375,
      "width": 1242
    },
    "002306": {
      "height": 375,
      "width": 1242
    },
    "002307": {
      "height": 375,
      "width": 1242
    },
    "002308": {
      "height": 375,
      "width": 1242
    },
    "002309": {
      "height": 375,
      "width": 1242
    },
    "002310": {
      "height": 370,
      "width": 1224
    },
    "002311": {
      "height": 375,
      "width": 1242
    },
    "002312": {
      "height": 375,
      "width": 1242
    },
    "002313": {
      "height": 375,
      "width": 1242
    },
    "002314": {
      "height": 376,
      "width": 1241
    },
    "002315": {
      "height": 374,
      "width": 1238
    },
    "002316": {
      "height": 375,
      "width": 1242
    },
    "002317": {
      "height": 375,
      "width": 1242
    },
    "002318": {
      "height": 375,
      "width": 1242
    },
    "002319": {
      "height": 375,
      "width": 1242
    },
    "002320": {
      "height": 375,
      "width": 1242
    },
    "002321": {
      "height": 375,
      "width": 1242
    },
    "002322": {
      "height": 375,
      "width": 1242
    },
    "002323": {
      "height": 375,
      "width": 1242
    },
    "002324": {
      "height": 370,
      "width": 1224
    },
    "002325": {
      "height": 375,
      "width": 1242
    },
    "002326": {
      "height": 375,
      "width": 1242
    },
    "002327": {
      "height": 370,
      "width": 1224
    },
    "002328": {
      "height": 375,
      "width": 1242
    },
    "002329": {
      "height": 375,
      "width": 1242
    },
    "002330": {
      "height": 375,
      "width": 1242
    },
    "002331": {
      "height": 375,
      "width": 1242
    },
    "002332": {
      "height": 375,
      "width": 1242
    },
    "002333": {
      "height": 375,
      "width": 1242
    },
    "002334": {
      "height": 370,
      "width": 1224
    },
    "002335": {
      "height": 375,
      "width": 1242
    },
    "002336": {
      "height": 375,
      "width": 1242
    },
    "002337": {
      "height": 375,
      "width": 1242
    },
    "002338": {
      "height": 375,
      "width": 1242
    },
    "002339": {
      "height": 375,
      "width": 1242
    },
    "002340": {
      "height": 375,
      "width": 1242
    },
    "002341": {
      "height": 376,
      "width": 1241
    },
    "002342": {
      "height": 370,
      "width": 1224
    },
    "002343": {
      "height": 375,
      "width": 1242
    },
    "002344": {
      "height": 375,
      "width": 1242
    },
    "002345": {
      "height": 375,
      "width": 1242
    },
    "002346": {
      "height": 375,
      "width": 1242
    },
    "002347": {
      "height": 376,
      "width": 1241
    },
    "002348": {
      "height": 374,
      "width": 1238
    },
    "002349": {
      "height": 375,
      "width": 1242
    },
    "002350": {
      "height": 375,
      "width": 1242
    },
    "002351": {
      "height": 375,
      "width": 1242
    },
    "002352": {
      "height": 375,
      "width": 1242
    },
    "002353": {
      "height": 375,
      "width": 1242
    },
    "002354": {
      "height": 375,
      "width": 1242
    },
    "002355": {
      "height": 375,
      "width": 1242
    },
    "002356": {
      "height": 375,
      "width": 1242
    },
    "002357": {
      "height": 375,
      "width": 1242
    },
    "002358": {
      "height": 375,
      "width": 1242
    },
    "002359": {
      "height": 375,
      "width": 1242
    },
    "002360": {
      "height": 370,
      "width": 1224
    },
    "002361": {
      "height": 375,
      "width": 1242
    },
    "002362": {
      "height": 374,
      "width": 1238
    },
    "002363": {
      "height": 375,
      "width": 1242
    },
    "002364": {
      "height": 375,
      "width": 1242
    },
    "002365": {
      "height": 375,
      "width": 1242
    },
    "002366": {
      "height": 375,
      "width": 1242
    },
    "002367": {
      "height": 375,
      "width": 1242
    },
    "002368": {
      "height": 375,
      "width": 1242
    },
    "002369": {
      "height": 374,
      "width": 1238
    },
    "002370": {
      "height": 375,
      "width": 1242
    },
    "002371": {
      "height": 375,
      "width": 1242
    },
    "002372": {
      "height": 375,
      "width": 1242
    },
    "002373": {
      "height": 375,
      "width": 1242
    },
    "002374": {
      "height": 375,
      "width": 1242
    },
    "002375": {
      "height": 375,
      "width": 1242
    },
    "002376": {
      "height": 375,
      "width": 1242
    },
    "002377": {
      "height": 375,
      "width": 1242
    },
    "002378": {
      "height": 375,
      "width": 1242
    },
    "002379": {
      "height": 375,
      "width": 1242
    },
    "002380": {
      "height": 375,
      "width": 1242
    },
    "002381": {
      "height": 375,
      "width": 1242
    },
    "002382": {
      "height": 375,
      "width": 1242
    },
    "002383": {
      "height": 375,
      "width": 1242
    },
    "002384": {
      "height": 375,
      "width": 1242
    },
    "002385": {
      "height": 375,
      "width": 1242
    },
    "002386": {
      "height": 374,
      "width": 1238
    },
    "002387": {
      "height": 376,
      "width": 1241
    },
    "002388": {
      "height": 375,
      "width": 1242
    },
    "002389": {
      "height": 375,
      "width": 1242
    },
    "002390": {
      "height": 375,
      "width": 1242
    },
    "002391": {
      "height": 375,
      "width": 1242
    },
    "002392": {
      "height": 375,
      "width": 1242
    },
    "002393": {
      "height": 374,
      "width": 1238
    },
    "002394": {
      "height": 375,
      "width": 1242
    },
    "002395": {
      "height": 375,
      "width": 1242
    },
    "002396": {
      "height": 375,
      "width": 1242
    },
    "002397": {
      "height": 376,
      "width": 1241
    },
    "002398": {
      "height": 375,
      "width": 1242
    },
    "002399": {
      "height": 375,
      "width": 1242
    },
    "002400": {
      "height": 375,
      "width": 1242
    },
    "002401": {
      "height": 375,
      "width": 1242
    },
    "002402": {
      "height": 370,
      "width": 1224
    },
    "002403": {
      "height": 370,
      "width": 1224
    },
    "002404": {
      "height": 375,
      "width": 1242
    },
    "002405": {
      "height": 375,
      "width": 1242
    },
    "002406": {
      "height": 370,
      "width": 1224
    },
    "002407": {
      "height": 375,
      "width": 1242
    },
    "002408": {
      "height": 370,
      "width": 1224
    },
    "002409": {
      "height": 375,
      "width": 1242
    },
    "002410": {
      "height": 370,
      "width": 1224
    },
    "002411": {
      "height": 375,
      "width": 1242
    },
    "002412": {
      "height": 375,
      "width": 1242
    },
    "002413": {
      "height": 375,
      "width": 1242
    },
    "002414": {
      "height": 375,
      "width": 1242
    },
    "002415": {
      "height": 375,
      "width": 1242
    },
    "002416": {
      "height": 375,
      "width": 1242
    },
    "002417": {
      "height": 375,
      "width": 1242
    },
    "002418": {
      "height": 375,
      "width": 1242
    },
    "002419": {
      "height": 375,
      "width": 1242
    },
    "002420": {
      "height": 375,
      "width": 1242
    },
    "002421": {
      "height": 375,
      "width": 1242
    },
    "002422": {
      "height": 375,
      "width": 1242
    },
    "002423": {
      "height": 375,
      "width": 1242
    },
    "002424": {
      "height": 370,
      "width": 1224
    },
    "002425": {
      "height": 375,
      "width": 1242
    },
    "002426": {
      "height": 375,
      "width": 1242
    },
    "002427": {
      "height": 375,
      "width": 1242
    },
    "002428": {
      "height": 375,
      "width": 1242
    },
    "002429": {
      "height": 375,
      "width": 1242
    },
    "002430": {
      "height": 375,
      "width": 1242
    },
    "002431": {
      "height": 375,
      "width": 1242
    },
    "002432": {
      "height": 375,
      "width": 1242
    },
    "002433": {
      "height": 375,
      "width": 1242
    },
    "002434": {
      "height": 375,
      "width": 1242
    },
    "002435": {
      "height": 375,
      "width": 1242
    },
    "002436": {
      "height": 375,
      "width": 1242
    },
    "002437": {
      "height": 374,
      "width": 1238
    },
    "002438": {
      "height": 375,
      "width": 1242
    },
    "002439": {
      "height": 375,
      "width": 1242
    },
    "002440": {
      "height": 374,
      "width": 1238
    },
    "002441": {
      "height": 375,
      "width": 1242
    },
    "002442": {
      "height": 375,
      "width": 1242
    },
    "002443": {
      "height": 375,
      "width": 1242
    },
    "002444": {
      "height": 375,
      "width": 1242
    },
    "002445": {
      "height": 375,
      "width": 1242
    },
    "002446": {
      "height": 375,
      "width": 1242
    },
    "002447": {
      "height": 375,
      "width": 1242
    },
    "002448": {
      "height": 375,
      "width": 1242
    },
    "002449": {
      "height": 375,
      "width": 1242
    },
    "002450": {
      "height": 375,
      "width": 1242
    },
    "002451": {
      "height": 375,
      "width": 1242
    },
    "002452": {
      "height": 375,
      "width": 1242
    },
    "002453": {
      "height": 375,
      "width": 1242
    },
    "002454": {
      "height": 375,
      "width": 1242
    },
    "002455": {
      "height": 375,
      "width": 1242
    },
    "002456": {
      "height": 370,
      "width": 1224
    },
    "002457": {
      "height": 375,
      "width": 1242
    },
    "002458": {
      "height": 375,
      "width": 1242
    },
    "002459": {
      "height": 375,
      "width": 1242
    },
    "002460": {
      "height": 375,
      "width": 1242
    },
    "002461": {
      "height": 376,
      "width": 1241
    },
    "002462": {
      "height": 375,
      "width": 1242
    },
    "002463": {
      "height": 375,
      "width": 1242
    },
    "002464": {
      "height": 375,
      "width": 1242
    },
    "002465": {
      "height": 375,
      "width": 1242
    },
    "002466": {
      "height": 375,
      "width": 1242
    },
    "002467": {
      "height": 375,
      "width": 1242
    },
    "002468": {
      "height": 375,
      "width": 1242
    },
    "002469": {
      "height": 375,
      "width": 1242
    },
    "002470": {
      "height": 375,
      "width": 1242
    },
    "002471": {
      "height": 375,
      "width": 1242
    },
    "002472": {
      "height": 375,
      "width": 1242
    },
    "002473": {
      "height": 375,
      "width": 1242
    },
    "002474": {
      "height": 375,
      "width": 1242
    },
    "002475": {
      "height": 375,
      "width": 1242
    },
    "002476": {
      "height": 375,
      "width": 1242
    },
    "002477": {
      "height": 374,
      "width": 1238
    },
    "002478": {
      "height": 370,
      "width": 1224
    },
    "002479": {
      "height": 375,
      "width": 1242
    },
    "002480": {
      "height": 375,
      "width": 1242
    },
    "002481": {
      "height": 375,
      "width": 1242
    },
    "002482": {
      "height": 375,
      "width": 1242
    },
    "002483": {
      "height": 375,
      "width": 1242
    },
    "002484": {
      "height": 374,
      "width": 1238
    },
    "002485": {
      "height": 370,
      "width": 1224
    },
    "002486": {
      "height": 376,
      "width": 1241
    },
    "002487": {
      "height": 370,
      "width": 1224
    },
    "002488": {
      "height": 375,
      "width": 1242
    },
    "002489": {
      "height": 375,
      "width": 1242
    },
    "002490": {
      "height": 375,
      "width": 1242
    },
    "002491": {
      "height": 375,
      "width": 1242
    },
    "002492": {
      "height": 376,
      "width": 1241
    },
    "002493": {
      "height": 370,
      "width": 1224
    },
    "002494": {
      "height": 375,
      "width": 1242
    },
    "002495": {
      "height": 375,
      "width": 1242
    },
    "002496": {
      "height": 375,
      "width": 1242
    },
    "002497": {
      "height": 375,
      "width": 1242
    },
    "002498": {
      "height": 375,
      "width": 1242
    },
    "002499": {
      "height": 375,
      "width": 1242
    },
    "002500": {
      "height": 375,
      "width": 1242
    },
    "002501": {
      "height": 375,
      "width": 1242
    },
    "002502": {
      "height": 374,
      "width": 1238
    },
    "002503": {
      "height": 375,
      "width": 1242
    },
    "002504": {
      "height": 375,
      "width": 1242
    },
    "002505": {
      "height": 375,
      "width": 1242
    },
    "002506": {
      "height": 375,
      "width": 1242
    },
    "002507": {
      "height": 370,
      "width": 1224
    },
    "002508": {
      "height": 375,
      "width": 1242
    },
    "002509": {
      "height": 375,
      "width": 1242
    },
    "002510": {
      "height": 375,
      "width": 1242
    },
    "002511": {
      "height": 375,
      "width": 1242
    },
    "002512": {
      "height": 375,
      "width": 1242
    },
    "002513": {
      "height": 375,
      "width": 1242
    },
    "002514": {
      "height": 375,
      "width": 1242
    },
    "002515": {
      "height": 375,
      "width": 1242
    },
    "002516": {
      "height": 375,
      "width": 1242
    },
    "002517": {
      "height": 375,
      "width": 1242
    },
    "002518": {
      "height": 375,
      "width": 1242
    },
    "002519": {
      "height": 376,
      "width": 1241
    },
    "002520": {
      "height": 376,
      "width": 1241
    },
    "002521": {
      "height": 375,
      "width": 1242
    },
    "002522": {
      "height": 375,
      "width": 1242
    },
    "002523": {
      "height": 375,
      "width": 1242
    },
    "002524": {
      "height": 370,
      "width": 1224
    },
    "002525": {
      "height": 375,
      "width": 1242
    },
    "002526": {
      "height": 374,
      "width": 1238
    },
    "002527": {
      "height": 375,
      "width": 1242
    },
    "002528": {
      "height": 375,
      "width": 1242
    },
    "002529": {
      "height": 370,
      "width": 1224
    },
    "002530": {
      "height": 375,
      "width": 1242
    },
    "002531": {
      "height": 370,
      "width": 1224
    },
    "002532": {
      "height": 375,
      "width": 1242
    },
    "002533": {
      "height": 375,
      "width": 1242
    },
    "002534": {
      "height": 375,
      "width": 1242
    },
    "002535": {
      "height": 375,
      "width": 1242
    },
    "002536": {
      "height": 375,
      "width": 1242
    },
    "002537": {
      "height": 375,
      "width": 1242
    },
    "002538": {
      "height": 375,
      "width": 1242
    },
    "002539": {
      "height": 375,
      "width": 1242
    },
    "002540": {
      "height": 376,
      "width": 1241
    },
    "002541": {
      "height": 375,
      "width": 1242
    },
    "002542": {
      "height": 375,
      "width": 1242
    },
    "002543": {
      "height": 375,
      "width": 1242
    },
    "002544": {
      "height": 375,
      "width": 1242
    },
    "002545": {
      "height": 375,
      "width": 1242
    },
    "002546": {
      "height": 375,
      "width": 1242
    },
    "002547": {
      "height": 375,
      "width": 1242
    },
    "002548": {
      "height": 374,
      "width": 1238
    },
    "002549": {
      "height": 375,
      "width": 1242
    },
    "002550": {
      "height": 375,
      "width": 1242
    },
    "002551": {
      "height": 375,
      "width": 1242
    },
    "002552": {
      "height": 376,
      "width": 1241
    },
    "002553": {
      "height": 375,
      "width": 1242
    },
    "002554": {
      "height": 370,
      "width": 1224
    },
    "002555": {
      "height": 375,
      "width": 1242
    },
    "002556": {
      "height": 375,
      "width": 1242
    },
    "002557": {
      "height": 375,
      "width": 1242
    },
    "002558": {
      "height": 375,
      "width": 1242
    },
    "002559": {
      "height": 375,
      "width": 1242
    },
    "002560": {
      "height": 375,
      "width": 1242
    },
    "002561": {
      "height": 375,
      "width": 1242
    },
    "002562": {
      "height": 375,
      "width": 1242
    },
    "002563": {
      "height": 375,
      "width": 1242
    },
    "002564": {
      "height": 375,
      "width": 1242
    },
    "002565": {
      "height": 370,
      "width": 1224
    },
    "002566": {
      "height": 374,
      "width": 1238
    },
    "002567": {
      "height": 375,
      "width": 1242
    },
    "002568": {
      "height": 375,
      "width": 1242
    },
    "002569": {
      "height": 375,
      "width": 1242
    },
    "002570": {
      "height": 375,
      "width": 1242
    },
    "002571": {
      "height": 375,
      "width": 1242
    },
    "002572": {
      "height": 376,
      "width": 1241
    },
    "002573": {
      "height": 375,
      "width": 1242
    },
    "002574": {
      "height": 370,
      "width": 1224
    },
    "002575": {
      "height": 375,
      "width": 1242
    },
    "002576": {
      "height": 375,
      "width": 1242
    },
    "002577": {
      "height": 375,
      "width": 1242
    },
    "002578": {
      "height": 375,
      "width": 1242
    },
    "002579": {
      "height": 375,
      "width": 1242
    },
    "002580": {
      "height": 375,
      "width": 1242
    },
    "002581": {
      "height": 376,
      "width": 1241
    },
    "002582": {
      "height": 375,
      "width": 1242
    },
    "002583": {
      "height": 374,
      "width": 1238
    },
    "002584": {
      "height": 376,
      "width": 1241
    },
    "002585": {
      "height": 375,
      "width": 1242
    },
    "002586": {
      "height": 375,
      "width": 1242
    },
    "002587": {
      "height": 375,
      "width": 1242
    },
    "002588": {
      "height": 375,
      "width": 1242
    },
    "002589": {
      "height": 375,
      "width": 1242
    },
    "002590": {
      "height": 375,
      "width": 1242
    },
    "002591": {
      "height": 375,
      "width": 1242
    },
    "002592": {
      "height": 374,
      "width": 1238
    },
    "002593": {
      "height": 375,
      "width": 1242
    },
    "002594": {
      "height": 375,
      "width": 1242
    },
    "002595": {
      "height": 375,
      "width": 1242
    },
    "002596": {
      "height": 375,
      "width": 1242
    },
    "002597": {
      "height": 374,
      "width": 1238
    },
    "002598": {
      "height": 375,
      "width": 1242
    },
    "002599": {
      "height": 376,
      "width": 1241
    },
    "002600": {
      "height": 376,
      "width": 1241
    },
    "002601": {
      "height": 375,
      "width": 1242
    },
    "002602": {
      "height": 375,
      "width": 1242
    },
    "002603": {
      "height": 375,
      "width": 1242
    },
    "002604": {
      "height": 376,
      "width": 1241
    },
    "002605": {
      "height": 370,
      "width": 1224
    },
    "002606": {
      "height": 375,
      "width": 1242
    },
    "002607": {
      "height": 375,
      "width": 1242
    },
    "002608": {
      "height": 375,
      "width": 1242
    },
    "002609": {
      "height": 375,
      "width": 1242
    },
    "002610": {
      "height": 375,
      "width": 1242
    },
    "002611": {
      "height": 375,
      "width": 1242
    },
    "002612": {
      "height": 375,
      "width": 1242
    },
    "002613": {
      "height": 374,
      "width": 1238
    },
    "002614": {
      "height": 375,
      "width": 1242
    },
    "002615": {
      "height": 375,
      "width": 1242
    },
    "002616": {
      "height": 375,
      "width": 1242
    },
    "002617": {
      "height": 370,
      "width": 1224
    },
    "002618": {
      "height": 370,
      "width": 1224
    },
    "002619": {
      "height": 375,
      "width": 1242
    },
    "002620": {
      "height": 375,
      "width": 1242
    },
    "002621": {
      "height": 375,
      "width": 1242
    },
    "002622": {
      "height": 375,
      "width": 1242
    },
    "002623": {
      "height": 375,
      "width": 1242
    },
    "002624": {
      "height": 375,
      "width": 1242
    },
    "002625": {
      "height": 375,
      "width": 1242
    },
    "002626": {
      "height": 370,
      "width": 1224
    },
    "002627": {
      "height": 375,
      "width": 1242
    },
    "002628": {
      "height": 375,
      "width": 1242
    },
    "002629": {
      "height": 375,
      "width": 1242
    },
    "002630": {
      "height": 375,
      "width": 1242
    },
    "002631": {
      "height": 375,
      "width": 1242
    },
    "002632": {
      "height": 375,
      "width": 1242
    },
    "002633": {
      "height": 374,
      "width": 1238
    },
    "002634": {
      "height": 375,
      "width": 1242
    },
    "002635": {
      "height": 375,
      "width": 1242
    },
    "002636": {
      "height": 375,
      "width": 1242
    },
    "002637": {
      "height": 375,
      "width": 1242
    },
    "002638": {
      "height": 375,
      "width": 1242
    },
    "002639": {
      "height": 375,
      "width": 1242
    },
    "002640": {
      "height": 374,
      "width": 1238
    },
    "002641": {
      "height": 375,
      "width": 1242
    },
    "002642": {
      "height": 375,
      "width": 1242
    },
    "002643": {
      "height": 370,
      "width": 1224
    },
    "002644": {
      "height": 376,
      "width": 1241
    },
    "002645": {
      "height": 375,
      "width": 1242
    },
    "002646": {
      "height": 375,
      "width": 1242
    },
    "002647": {
      "height": 375,
      "width": 1242
    },
    "002648": {
      "height": 375,
      "width": 1242
    },
    "002649": {
      "height": 375,
      "width": 1242
    },
    "002650": {
      "height": 375,
      "width": 1242
    },
    "002651": {
      "height": 370,
      "width": 1224
    },
    "002652": {
      "height": 375,
      "width": 1242
    },
    "002653": {
      "height": 375,
      "width": 1242
    },
    "002654": {
      "height": 375,
      "width": 1242
    },
    "002655": {
      "height": 375,
      "width": 1242
    },
    "002656": {
      "height": 375,
      "width": 1242
    },
    "002657": {
      "height": 375,
      "width": 1242
    },
    "002658": {
      "height": 375,
      "width": 1242
    },
    "002659": {
      "height": 375,
      "width": 1242
    },
    "002660": {
      "height": 375,
      "width": 1242
    },
    "002661": {
      "height": 375,
      "width": 1242
    },
    "002662": {
      "height": 375,
      "width": 1242
    },
    "002663": {
      "height": 370,
      "width": 1224
    },
    "002664": {
      "height": 375,
      "width": 1242
    },
    "002665": {
      "height": 375,
      "width": 1242
    },
    "002666": {
      "height": 370,
      "width": 1224
    },
    "002667": {
      "height": 370,
      "width": 1224
    },
    "002668": {
      "height": 375,
      "width": 1242
    },
    "002669": {
      "height": 375,
      "width": 1242
    },
    "002670": {
      "height": 375,
      "width": 1242
    },
    "002671": {
      "height": 370,
      "width": 1224
    },
    "002672": {
      "height": 375,
      "width": 1242
    },
    "002673": {
      "height": 375,
      "width": 1242
    },
    "002674": {
      "height": 375,
      "width": 1242
    },
    "002675": {
      "height": 375,
      "width": 1242
    },
    "002676": {
      "height": 375,
      "width": 1242
    },
    "002677": {
      "height": 375,
      "width": 1242
    },
    "002678": {
      "height": 375,
      "width": 1242
    },
    "002679": {
      "height": 375,
      "width": 1242
    },
    "002680": {
      "height": 370,
      "width": 1224
    },
    "002681": {
      "height": 375,
      "width": 1242
    },
    "002682": {
      "height": 375,
      "width": 1242
    },
    "002683": {
      "height": 375,
      "width": 1242
    },
    "002684": {
      "height": 375,
      "width": 1242
    },
    "002685": {
      "height": 375,
      "width": 1242
    },
    "002686": {
      "height": 375,
      "width": 1242
    },
    "002687": {
      "height": 375,
      "width": 1242
    },
    "002688": {
      "height": 375,
      "width": 1242
    },
    "002689": {
      "height": 370,
      "width": 1224
    },
    "002690": {
      "height": 375,
      "width": 1242
    },
    "002691": {
      "height": 375,
      "width": 1242
    },
    "002692": {
      "height": 375,
      "width": 1242
    },
    "002693": {
      "height": 370,
      "width": 1224
    },
    "002694": {
      "height": 375,
      "width": 1242
    },
    "002695": {
      "height": 375,
      "width": 1242
    },
    "002696": {
      "height": 375,
      "width": 1242
    },
    "002697": {
      "height": 375,
      "width": 1242
    },
    "002698": {
      "height": 375,
      "width": 1242
    },
    "002699": {
      "height": 374,
      "width": 1238
    },
    "002700": {
      "height": 375,
      "width": 1242
    },
    "002701": {
      "height": 375,
      "width": 1242
    },
    "002702": {
      "height": 375,
      "width": 1242
    },
    "002703": {
      "height": 375,
      "width": 1242
    },
    "002704": {
      "height": 375,
      "width": 1242
    },
    "002705": {
      "height": 375,
      "width": 1242
    },
    "002706": {
      "height": 376,
      "width": 1241
    },
    "002707": {
      "height": 370,
      "width": 1224
    },
    "002708": {
      "height": 375,
      "width": 1242
    },
    "002709": {
      "height": 374,
      "width": 1238
    },
    "002710": {
      "height": 375,
      "width": 1242
    },
    "002711": {
      "height": 376,
      "width": 1241
    },
    "002712": {
      "height": 375,
      "width": 1242
    },
    "002713": {
      "height": 375,
      "width": 1242
    },
    "002714": {
      "height": 375,
      "width": 1242
    },
    "002715": {
      "height": 375,
      "width": 1242
    },
    "002716": {
      "height": 375,
      "width": 1242
    },
    "002717": {
      "height": 375,
      "width": 1242
    },
    "002718": {
      "height": 370,
      "width": 1224
    },
    "002719": {
      "height": 374,
      "width": 1238
    },
    "002720": {
      "height": 375,
      "width": 1242
    },
    "002721": {
      "height": 375,
      "width": 1242
    },
    "002722": {
      "height": 375,
      "width": 1242
    },
    "002723": {
      "height": 375,
      "width": 1242
    },
    "002724": {
      "height": 375,
      "width": 1242
    },
    "002725": {
      "height": 374,
      "width": 1238
    },
    "002726": {
      "height": 375,
      "width": 1242
    },
    "002727": {
      "height": 375,
      "width": 1242
    },
    "002728": {
      "height": 375,
      "width": 1242
    },
    "002729": {
      "height": 375,
      "width": 1242
    },
    "002730": {
      "height": 375,
      "width": 1242
    },
    "002731": {
      "height": 375,
      "width": 1242
    },
    "002732": {
      "height": 375,
      "width": 1242
    },
    "002733": {
      "height": 375,
      "width": 1242
    },
    "002734": {
      "height": 375,
      "width": 1242
    },
    "002735": {
      "height": 370,
      "width": 1224
    },
    "002736": {
      "height": 375,
      "width": 1242
    },
    "002737": {
      "height": 375,
      "width": 1242
    },
    "002738": {
      "height": 375,
      "width": 1242
    },
    "002739": {
      "height": 370,
      "width": 1224
    },
    "002740": {
      "height": 375,
      "width": 1242
    },
    "002741": {
      "height": 375,
      "width": 1242
    },
    "002742": {
      "height": 370,
      "width": 1224
    },
    "002743": {
      "height": 375,
      "width": 1242
    },
    "002744": {
      "height": 375,
      "width": 1242
    },
    "002745": {
      "height": 375,
      "width": 1242
    },
    "002746": {
      "height": 375,
      "width": 1242
    },
    "002747": {
      "height": 375,
      "width": 1242
    },
    "002748": {
      "height": 375,
      "width": 1242
    },
    "002749": {
      "height": 375,
      "width": 1242
    },
    "002750": {
      "height": 375,
      "width": 1242
    },
    "002751": {
      "height": 375,
      "width": 1242
    },
    "002752": {
      "height": 376,
      "width": 1241
    },
    "002753": {
      "height": 375,
      "width": 1242
    },
    "002754": {
      "height": 375,
      "width": 1242
    },
    "002755": {
      "height": 375,
      "width": 1242
    },
    "002756": {
      "height": 374,
      "width": 1238
    },
    "002757": {
      "height": 375,
      "width": 1242
    },
    "002758": {
      "height": 375,
      "width": 1242
    },
    "002759": {
      "height": 375,
      "width": 1242
    },
    "002760": {
      "height": 375,
      "width": 1242
    },
    "002761": {
      "height": 375,
      "width": 1242
    },
    "002762": {
      "height": 375,
      "width": 1242
    },
    "002763": {
      "height": 370,
      "width": 1224
    },
    "002764": {
      "height": 375,
      "width": 1242
    },
    "002765": {
      "height": 375,
      "width": 1242
    },
    "002766": {
      "height": 375,
      "width": 1242
    },
    "002767": {
      "height": 370,
      "width": 1224
    },
    "002768": {
      "height": 375,
      "width": 1242
    },
    "002769": {
      "height": 370,
      "width": 1224
    },
    "002770": {
      "height": 375,
      "width": 1242
    },
    "002771": {
      "height": 375,
      "width": 1242
    },
    "002772": {
      "height": 375,
      "width": 1242
    },
    "002773": {
      "height": 375,
      "width": 1242
    },
    "002774": {
      "height": 370,
      "width": 1224
    },
    "002775": {
      "height": 374,
      "width": 1238
    },
    "002776": {
      "height": 375,
      "width": 1242
    },
    "002777": {
      "height": 374,
      "width": 1238
    },
    "002778": {
      "height": 375,
      "width": 1242
    },
    "002779": {
      "height": 375,
      "width": 1242
    },
    "002780": {
      "height": 370,
      "width": 1224
    },
    "002781": {
      "height": 375,
      "width": 1242
    },
    "002782": {
      "height": 375,
      "width": 1242
    },
    "002783": {
      "height": 375,
      "width": 1242
    },
    "002784": {
      "height": 375,
      "width": 1242
    },
    "002785": {
      "height": 375,
      "width": 1242
    },
    "002786": {
      "height": 375,
      "width": 1242
    },
    "002787": {
      "height": 375,
      "width": 1242
    },
    "002788": {
      "height": 375,
      "width": 1242
    },
    "002789": {
      "height": 375,
      "width": 1242
    },
    "002790": {
      "height": 375,
      "width": 1242
    },
    "002791": {
      "height": 375,
      "width": 1242
    },
    "002792": {
      "height": 375,
      "width": 1242
    },
    "002793": {
      "height": 375,
      "width": 1242
    },
    "002794": {
      "height": 376,
      "width": 1241
    },
    "002795": {
      "height": 374,
      "width": 1238
    },
    "002796": {
      "height": 376,
      "width": 1241
    },
    "002797": {
      "height": 375,
      "width": 1242
    },
    "002798": {
      "height": 375,
      "width": 1242
    },
    "002799": {
      "height": 370,
      "width": 1224
    },
    "002800": {
      "height": 374,
      "width": 1238
    },
    "002801": {
      "height": 375,
      "width": 1242
    },
    "002802": {
      "height": 375,
      "width": 1242
    },
    "002803": {
      "height": 375,
      "width": 1242
    },
    "002804": {
      "height": 375,
      "width": 1242
    },
    "002805": {
      "height": 375,
      "width": 1242
    },
    "002806": {
      "height": 374,
      "width": 1238
    },
    "002807": {
      "height": 375,
      "width": 1242
    },
    "002808": {
      "height": 375,
      "width": 1242
    },
    "002809": {
      "height": 375,
      "width": 1242
    },
    "002810": {
      "height": 375,
      "width": 1242
    },
    "002811": {
      "height": 370,
      "width": 1224
    },
    "002812": {
      "height": 375,
      "width": 1242
    },
    "002813": {
      "height": 375,
      "width": 1242
    },
    "002814": {
      "height": 375,
      "width": 1242
    },
    "002815": {
      "height": 375,
      "width": 1242
    },
    "002816": {
      "height": 375,
      "width": 1242
    },
    "002817": {
      "height": 375,
      "width": 1242
    },
    "002818": {
      "height": 375,
      "width": 1242
    },
    "002819": {
      "height": 375,
      "width": 1242
    },
    "002820": {
      "height": 375,
      "width": 1242
    },
    "002821": {
      "height": 375,
      "width": 1242
    },
    "002822": {
      "height": 375,
      "width": 1242
    },
    "002823": {
      "height": 375,
      "width": 1242
    },
    "002824": {
      "height": 375,
      "width": 1242
    },
    "002825": {
      "height": 375,
      "width": 1242
    },
    "002826": {
      "height": 375,
      "width": 1242
    },
    "002827": {
      "height": 375,
      "width": 1242
    },
    "002828": {
      "height": 375,
      "width": 1242
    },
    "002829": {
      "height": 375,
      "width": 1242
    },
    "002830": {
      "height": 375,
      "width": 1242
    },
    "002831": {
      "height": 375,
      "width": 1242
    },
    "002832": {
      "height": 375,
      "width": 1242
    },
    "002833": {
      "height": 375,
      "width": 1242
    },
    "002834": {
      "height": 375,
      "width": 1242
    },
    "002835": {
      "height": 375,
      "width": 1242
    },
    "002836": {
      "height": 375,
      "width": 1242
    },
    "002837": {
      "height": 370,
      "width": 1224
    },
    "002838": {
      "height": 375,
      "width": 1242
    },
    "002839": {
      "height": 375,
      "width": 1242
    },
    "002840": {
      "height": 375,
      "width": 1242
    },
    "002841": {
      "height": 375,
      "width": 1242
    },
    "002842": {
      "height": 375,
      "width": 1242
    },
    "002843": {
      "height": 375,
      "width": 1242
    },
    "002844": {
      "height": 375,
      "width": 1242
    },
    "002845": {
      "height": 375,
      "width": 1242
    },
    "002846": {
      "height": 375,
      "width": 1242
    },
    "002847": {
      "height": 376,
      "width": 1241
    },
    "002848": {
      "height": 375,
      "width": 1242
    },
    "002849": {
      "height": 375,
      "width": 1242
    },
    "002850": {
      "height": 375,
      "width": 1242
    },
    "002851": {
      "height": 375,
      "width": 1242
    },
    "002852": {
      "height": 375,
      "width": 1242
    },
    "002853": {
      "height": 375,
      "width": 1242
    },
    "002854": {
      "height": 375,
      "width": 1242
    },
    "002855": {
      "height": 375,
      "width": 1242
    },
    "002856": {
      "height": 375,
      "width": 1242
    },
    "002857": {
      "height": 370,
      "width": 1224
    },
    "002858": {
      "height": 375,
      "width": 1242
    },
    "002859": {
      "height": 375,
      "width": 1242
    },
    "002860": {
      "height": 370,
      "width": 1224
    },
    "002861": {
      "height": 375,
      "width": 1242
    },
    "002862": {
      "height": 375,
      "width": 1242
    },
    "002863": {
      "height": 375,
      "width": 1242
    },
    "002864": {
      "height": 375,
      "width": 1242
    },
    "002865": {
      "height": 375,
      "width": 1242
    },
    "002866": {
      "height": 375,
      "width": 1242
    },
    "002867": {
      "height": 375,
      "width": 1242
    },
    "002868": {
      "height": 370,
      "width": 1224
    },
    "002869": {
      "height": 375,
      "width": 1242
    },
    "002870": {
      "height": 375,
      "width": 1242
    },
    "002871": {
      "height": 375,
      "width": 1242
    },
    "002872": {
      "height": 375,
      "width": 1242
    },
    "002873": {
      "height": 375,
      "width": 1242
    },
    "002874": {
      "height": 375,
      "width": 1242
    },
    "002875": {
      "height": 370,
      "width": 1224
    },
    "002876": {
      "height": 375,
      "width": 1242
    },
    "002877": {
      "height": 375,
      "width": 1242
    },
    "002878": {
      "height": 375,
      "width": 1242
    },
    "002879": {
      "height": 375,
      "width": 1242
    },
    "002880": {
      "height": 375,
      "width": 1242
    },
    "002881": {
      "height": 370,
      "width": 1224
    },
    "002882": {
      "height": 375,
      "width": 1242
    },
    "002883": {
      "height": 375,
      "width": 1242
    },
    "002884": {
      "height": 375,
      "width": 1242
    },
    "002885": {
      "height": 375,
      "width": 1242
    },
    "002886": {
      "height": 375,
      "width": 1242
    },
    "002887": {
      "height": 375,
      "width": 1242
    },
    "002888": {
      "height": 370,
      "width": 1224
    },
    "002889": {
      "height": 374,
      "width": 1238
    },
    "002890": {
      "height": 375,
      "width": 1242
    },
    "002891": {
      "height": 375,
      "width": 1242
    },
    "002892": {
      "height": 374,
      "width": 1238
    },
    "002893": {
      "height": 375,
      "width": 1242
    },
    "002894": {
      "height": 376,
      "width": 1241
    },
    "002895": {
      "height": 375,
      "width": 1242
    },
    "002896": {
      "height": 375,
      "width": 1242
    },
    "002897": {
      "height": 375,
      "width": 1242
    },
    "002898": {
      "height": 370,
      "width": 1224
    },
    "002899": {
      "height": 375,
      "width": 1242
    },
    "002900": {
      "height": 375,
      "width": 1242
    },
    "002901": {
      "height": 375,
      "width": 1242
    },
    "002902": {
      "height": 375,
      "width": 1242
    },
    "002903": {
      "height": 375,
      "width": 1242
    },
    "002904": {
      "height": 370,
      "width": 1224
    },
    "002905": {
      "height": 375,
      "width": 1242
    },
    "002906": {
      "height": 370,
      "width": 1224
    },
    "002907": {
      "height": 375,
      "width": 1242
    },
    "002908": {
      "height": 375,
      "width": 1242
    },
    "002909": {
      "height": 375,
      "width": 1242
    },
    "002910": {
      "height": 375,
      "width": 1242
    },
    "002911": {
      "height": 375,
      "width": 1242
    },
    "002912": {
      "height": 375,
      "width": 1242
    },
    "002913": {
      "height": 375,
      "width": 1242
    },
    "002914": {
      "height": 375,
      "width": 1242
    },
    "002915": {
      "height": 375,
      "width": 1242
    },
    "002916": {
      "height": 375,
      "width": 1242
    },
    "002917": {
      "height": 375,
      "width": 1242
    },
    "002918": {
      "height": 370,
      "width": 1224
    },
    "002919": {
      "height": 375,
      "width": 1242
    },
    "002920": {
      "height": 370,
      "width": 1224
    },
    "002921": {
      "height": 375,
      "width": 1242
    },
    "002922": {
      "height": 375,
      "width": 1242
    },
    "002923": {
      "height": 375,
      "width": 1242
    },
    "002924": {
      "height": 375,
      "width": 1242
    },
    "002925": {
      "height": 375,
      "width": 1242
    },
    "002926": {
      "height": 375,
      "width": 1242
    },
    "002927": {
      "height": 370,
      "width": 1224
    },
    "002928": {
      "height": 375,
      "width": 1242
    },
    "002929": {
      "height": 375,
      "width": 1242
    },
    "002930": {
      "height": 376,
      "width": 1241
    },
    "002931": {
      "height": 375,
      "width": 1242
    },
    "002932": {
      "height": 375,
      "width": 1242
    },
    "002933": {
      "height": 375,
      "width": 1242
    },
    "002934": {
      "height": 375,
      "width": 1242
    },
    "002935": {
      "height": 375,
      "width": 1242
    },
    "002936": {
      "height": 375,
      "width": 1242
    },
    "002937": {
      "height": 375,
      "width": 1242
    },
    "002938": {
      "height": 375,
      "width": 1242
    },
    "002939": {
      "height": 375,
      "width": 1242
    },
    "002940": {
      "height": 375,
      "width": 1242
    },
    "002941": {
      "height": 375,
      "width": 1242
    },
    "002942": {
      "height": 375,
      "width": 1242
    },
    "002943": {
      "height": 375,
      "width": 1242
    },
    "002944": {
      "height": 374,
      "width": 1238
    },
    "002945": {
      "height": 375,
      "width": 1242
    },
    "002946": {
      "height": 375,
      "width": 1242
    },
    "002947": {
      "height": 376,
      "width": 1241
    },
    "002948": {
      "height": 375,
      "width": 1242
    },
    "002949": {
      "height": 375,
      "width": 1242
    },
    "002950": {
      "height": 375,
      "width": 1242
    },
    "002951": {
      "height": 375,
      "width": 1242
    },
    "002952": {
      "height": 370,
      "width": 1224
    },
    "002953": {
      "height": 375,
      "width": 1242
    },
    "002954": {
      "height": 375,
      "width": 1242
    },
    "002955": {
      "height": 375,
      "width": 1242
    },
    "002956": {
      "height": 370,
      "width": 1224
    },
    "002957": {
      "height": 375,
      "width": 1242
    },
    "002958": {
      "height": 375,
      "width": 1242
    },
    "002959": {
      "height": 375,
      "width": 1242
    },
    "002960": {
      "height": 376,
      "width": 1241
    },
    "002961": {
      "height": 375,
      "width": 1242
    },
    "002962": {
      "height": 375,
      "width": 1242
    },
    "002963": {
      "height": 376,
      "width": 1241
    },
    "002964": {
      "height": 375,
      "width": 1242
    },
    "002965": {
      "height": 375,
      "width": 1242
    },
    "002966": {
      "height": 374,
      "width": 1238
    },
    "002967": {
      "height": 375,
      "width": 1242
    },
    "002968": {
      "height": 375,
      "width": 1242
    },
    "002969": {
      "height": 375,
      "width": 1242
    },
    "002970": {
      "height": 375,
      "width": 1242
    },
    "002971": {
      "height": 375,
      "width": 1242
    },
    "002972": {
      "height": 375,
      "width": 1242
    },
    "002973": {
      "height": 375,
      "width": 1242
    },
    "002974": {
      "height": 375,
      "width": 1242
    },
    "002975": {
      "height": 375,
      "width": 1242
    },
    "002976": {
      "height": 375,
      "width": 1242
    },
    "002977": {
      "height": 375,
      "width": 1242
    },
    "002978": {
      "height": 375,
      "width": 1242
    },
    "002979": {
      "height": 370,
      "width": 1224
    },
    "002980": {
      "height": 375,
      "width": 1242
    },
    "002981": {
      "height": 375,
      "width": 1242
    },
    "002982": {
      "height": 375,
      "width": 1242
    },
    "002983": {
      "height": 375,
      "width": 1242
    },
    "002984": {
      "height": 375,
      "width": 1242
    },
    "002985": {
      "height": 375,
      "width": 1242
    },
    "002986": {
      "height": 375,
      "width": 1242
    },
    "002987": {
      "height": 375,
      "width": 1242
    },
    "002988": {
      "height": 375,
      "width": 1242
    },
    "002989": {
      "height": 375,
      "width": 1242
    },
    "002990": {
      "height": 375,
      "width": 1242
    },
    "002991": {
      "height": 375,
      "width": 1242
    },
    "002992": {
      "height": 375,
      "width": 1242
    },
    "002993": {
      "height": 375,
      "width": 1242
    },
    "002994": {
      "height": 370,
      "width": 1224
    },
    "002995": {
      "height": 375,
      "width": 1242
    },
    "002996": {
      "height": 375,
      "width": 1242
    },
    "002997": {
      "height": 375,
      "width": 1242
    },
    "002998": {
      "height": 375,
      "width": 1242
    },
    "002999": {
      "height": 370,
      "width": 1224
    },
    "003000": {
      "height": 375,
      "width": 1242
    },
    "003001": {
      "height": 375,
      "width": 1242
    },
    "003002": {
      "height": 370,
      "width": 1224
    },
    "003003": {
      "height": 375,
      "width": 1242
    },
    "003004": {
      "height": 375,
      "width": 1242
    },
    "003005": {
      "height": 375,
      "width": 1242
    },
    "003006": {
      "height": 375,
      "width": 1242
    },
    "003007": {
      "height": 375,
      "width": 1242
    },
    "003008": {
      "height": 375,
      "width": 1242
    },
    "003009": {
      "height": 375,
      "width": 1242
    },
    "003010": {
      "height": 375,
      "width": 1242
    },
    "003011": {
      "height": 375,
      "width": 1242
    },
    "003012": {
      "height": 375,
      "width": 1242
    },
    "003013": {
      "height": 375,
      "width": 1242
    },
    "003014": {
      "height": 375,
      "width": 1242
    },
    "003015": {
      "height": 375,
      "width": 1242
    },
    "003016": {
      "height": 375,
      "width": 1242
    },
    "003017": {
      "height": 375,
      "width": 1242
    },
    "003018": {
      "height": 375,
      "width": 1242
    },
    "003019": {
      "height": 375,
      "width": 1242
    },
    "003020": {
      "height": 375,
      "width": 1242
    },
    "003021": {
      "height": 375,
      "width": 1242
    },
    "003022": {
      "height": 375,
      "width": 1242
    },
    "003023": {
      "height": 375,
      "width": 1242
    },
    "003024": {
      "height": 370,
      "width": 1224
    },
    "003025": {
      "height": 375,
      "width": 1242
    },
    "003026": {
      "height": 375,
      "width": 1242
    },
    "003027": {
      "height": 375,
      "width": 1242
    },
    "003028": {
      "height": 370,
      "width": 1224
    },
    "003029": {
      "height": 375,
      "width": 1242
    },
    "003030": {
      "height": 375,
      "width": 1242
    },
    "003031": {
      "height": 375,
      "width": 1242
    },
    "003032": {
      "height": 375,
      "width": 1242
    },
    "003033": {
      "height": 375,
      "width": 1242
    },
    "003034": {
      "height": 375,
      "width": 1242
    },
    "003035": {
      "height": 376,
      "width": 1241
    },
    "003036": {
      "height": 375,
      "width": 1242
    },
    "003037": {
      "height": 375,
      "width": 1242
    },
    "003038": {
      "height": 375,
      "width": 1242
    },
    "003039": {
      "height": 375,
      "width": 1242
    },
    "003040": {
      "height": 375,
      "width": 1242
    },
    "003041": {
      "height": 375,
      "width": 1242
    },
    "003042": {
      "height": 375,
      "width": 1242
    },
    "003043": {
      "height": 375,
      "width": 1242
    },
    "003044": {
      "height": 375,
      "width": 1242
    },
    "003045": {
      "height": 375,
      "width": 1242
    },
    "003046": {
      "height": 375,
      "width": 1242
    },
    "003047": {
      "height": 374,
      "width": 1238
    },
    "003048": {
      "height": 376,
      "width": 1241
    },
    "003049": {
      "height": 375,
      "width": 1242
    },
    "003050": {
      "height": 370,
      "width": 1224
    },
    "003051": {
      "height": 375,
      "width": 1242
    },
    "003052": {
      "height": 374,
      "width": 1238
    },
    "003053": {
      "height": 375,
      "width": 1242
    },
    "003054": {
      "height": 375,
      "width": 1242
    },
    "003055": {
      "height": 376,
      "width": 1241
    },
    "003056": {
      "height": 375,
      "width": 1242
    },
    "003057": {
      "height": 375,
      "width": 1242
    },
    "003058": {
      "height": 375,
      "width": 1242
    },
    "003059": {
      "height": 370,
      "width": 1224
    },
    "003060": {
      "height": 375,
      "width": 1242
    },
    "003061": {
      "height": 375,
      "width": 1242
    },
    "003062": {
      "height": 375,
      "width": 1242
    },
    "003063": {
      "height": 375,
      "width": 1242
    },
    "003064": {
      "height": 375,
      "width": 1242
    },
    "003065": {
      "height": 370,
      "width": 1224
    },
    "003066": {
      "height": 375,
      "width": 1242
    },
    "003067": {
      "height": 375,
      "width": 1242
    },
    "003068": {
      "height": 375,
      "width": 1242
    },
    "003069": {
      "height": 374,
      "width": 1238
    },
    "003070": {
      "height": 375,
      "width": 1242
    },
    "003071": {
      "height": 370,
      "width": 1224
    },
    "003072": {
      "height": 375,
      "width": 1242
    },
    "003073": {
      "height": 375,
      "width": 1242
    },
    "003074": {
      "height": 374,
      "width": 1238
    },
    "003075": {
      "height": 375,
      "width": 1242
    },
    "003076": {
      "height": 375,
      "width": 1242
    },
    "003077": {
      "height": 375,
      "width": 1242
    },
    "003078": {
      "height": 375,
      "width": 1242
    },
    "003079": {
      "height": 374,
      "width": 1238
    },
    "003080": {
      "height": 374,
      "width": 1238
    },
    "003081": {
      "height": 375,
      "width": 1242
    },
    "003082": {
      "height": 375,
      "width": 1242
    },
    "003083": {
      "height": 375,
      "width": 1242
    },
    "003084": {
      "height": 375,
      "width": 1242
    },
    "003085": {
      "height": 370,
      "width": 1224
    },
    "003086": {
      "height": 375,
      "width": 1242
    },
    "003087": {
      "height": 375,
      "width": 1242
    },
    "003088": {
      "height": 375,
      "width": 1242
    },
    "003089": {
      "height": 375,
      "width": 1242
    },
    "003090": {
      "height": 375,
      "width": 1242
    },
    "003091": {
      "height": 375,
      "width": 1242
    },
    "003092": {
      "height": 375,
      "width": 1242
    },
    "003093": {
      "height": 375,
      "width": 1242
    },
    "003094": {
      "height": 375,
      "width": 1242
    },
    "003095": {
      "height": 375,
      "width": 1242
    },
    "003096": {
      "height": 375,
      "width": 1242
    },
    "003097": {
      "height": 375,
      "width": 1242
    },
    "003098": {
      "height": 374,
      "width": 1238
    },
    "003099": {
      "height": 375,
      "width": 1242
    },
    "003100": {
      "height": 375,
      "width": 1242
    },
    "003101": {
      "height": 375,
      "width": 1242
    },
    "003102": {
      "height": 375,
      "width": 1242
    },
    "003103": {
      "height": 375,
      "width": 1242
    },
    "003104": {
      "height": 375,
      "width": 1242
    },
    "003105": {
      "height": 375,
      "width": 1242
    },
    "003106": {
      "height": 375,
      "width": 1242
    },
    "003107": {
      "height": 375,
      "width": 1242
    },
    "003108": {
      "height": 375,
      "width": 1242
    },
    "003109": {
      "height": 375,
      "width": 1242
    },
    "003110": {
      "height": 375,
      "width": 1242
    },
    "003111": {
      "height": 375,
      "width": 1242
    },
    "003112": {
      "height": 375,
      "width": 1242
    },
    "003113": {
      "height": 375,
      "width": 1242
    },
    "003114": {
      "height": 370,
      "width": 1224
    },
    "003115": {
      "height": 375,
      "width": 1242
    },
    "003116": {
      "height": 376,
      "width": 1241
    },
    "003117": {
      "height": 375,
      "width": 1242
    },
    "003118": {
      "height": 375,
      "width": 1242
    },
    "003119": {
      "height": 375,
      "width": 1242
    },
    "003120": {
      "height": 375,
      "width": 1242
    },
    "003121": {
      "height": 370,
      "width": 1224
    },
    "003122": {
      "height": 370,
      "width": 1224
    },
    "003123": {
      "height": 375,
      "width": 1242
    },
    "003124": {
      "height": 375,
      "width": 1242
    },
    "003125": {
      "height": 375,
      "width": 1242
    },
    "003126": {
      "height": 375,
      "width": 1242
    },
    "003127": {
      "height": 374,
      "width": 1238
    },
    "003128": {
      "height": 375,
      "width": 1242
    },
    "003129": {
      "height": 375,
      "width": 1242
    },
    "003130": {
      "height": 370,
      "width": 1224
    },
    "003131": {
      "height": 375,
      "width": 1242
    },
    "003132": {
      "height": 375,
      "width": 1242
    },
    "003133": {
      "height": 374,
      "width": 1238
    },
    "003134": {
      "height": 370,
      "width": 1224
    },
    "003135": {
      "height": 375,
      "width": 1242
    },
    "003136": {
      "height": 375,
      "width": 1242
    },
    "003137": {
      "height": 376,
      "width": 1241
    },
    "003138": {
      "height": 375,
      "width": 1242
    },
    "003139": {
      "height": 375,
      "width": 1242
    },
    "003140": {
      "height": 375,
      "width": 1242
    },
    "003141": {
      "height": 375,
      "width": 1242
    },
    "003142": {
      "height": 375,
      "width": 1242
    },
    "003143": {
      "height": 375,
      "width": 1242
    },
    "003144": {
      "height": 375,
      "width": 1242
    },
    "003145": {
      "height": 375,
      "width": 1242
    },
    "003146": {
      "height": 375,
      "width": 1242
    },
    "003147": {
      "height": 375,
      "width": 1242
    },
    "003148": {
      "height": 370,
      "width": 1224
    },
    "003149": {
      "height": 375,
      "width": 1242
    },
    "003150": {
      "height": 375,
      "width": 1242
    },
    "003151": {
      "height": 375,
      "width": 1242
    },
    "003152": {
      "height": 375,
      "width": 1242
    },
    "003153": {
      "height": 375,
      "width": 1242
    },
    "003154": {
      "height": 375,
      "width": 1242
    },
    "003155": {
      "height": 375,
      "width": 1242
    },
    "003156": {
      "height": 370,
      "width": 1224
    },
    "003157": {
      "height": 375,
      "width": 1242
    },
    "003158": {
      "height": 375,
      "width": 1242
    },
    "003159": {
      "height": 375,
      "width": 1242
    },
    "003160": {
      "height": 375,
      "width": 1242
    },
    "003161": {
      "height": 375,
      "width": 1242
    },
    "003162": {
      "height": 370,
      "width": 1224
    },
    "003163": {
      "height": 375,
      "width": 1242
    },
    "003164": {
      "height": 370,
      "width": 1224
    },
    "003165": {
      "height": 375,
      "width": 1242
    },
    "003166": {
      "height": 375,
      "width": 1242
    },
    "003167": {
      "height": 375,
      "width": 1242
    },
    "003168": {
      "height": 375,
      "width": 1242
    },
    "003169": {
      "height": 375,
      "width": 1242
    },
    "003170": {
      "height": 375,
      "width": 1242
    },
    "003171": {
      "height": 370,
      "width": 1224
    },
    "003172": {
      "height": 375,
      "width": 1242
    },
    "003173": {
      "height": 375,
      "width": 1242
    },
    "003174": {
      "height": 375,
      "width": 1242
    },
    "003175": {
      "height": 375,
      "width": 1242
    },
    "003176": {
      "height": 375,
      "width": 1242
    },
    "003177": {
      "height": 370,
      "width": 1224
    },
    "003178": {
      "height": 375,
      "width": 1242
    },
    "003179": {
      "height": 375,
      "width": 1242
    },
    "003180": {
      "height": 375,
      "width": 1242
    },
    "003181": {
      "height": 375,
      "width": 1242
    },
    "003182": {
      "height": 375,
      "width": 1242
    },
    "003183": {
      "height": 370,
      "width": 1224
    },
    "003184": {
      "height": 375,
      "width": 1242
    },
    "003185": {
      "height": 370,
      "width": 1224
    },
    "003186": {
      "height": 375,
      "width": 1242
    },
    "003187": {
      "height": 376,
      "width": 1241
    },
    "003188": {
      "height": 375,
      "width": 1242
    },
    "003189": {
      "height": 375,
      "width": 1242
    },
    "003190": {
      "height": 375,
      "width": 1242
    },
    "003191": {
      "height": 375,
      "width": 1242
    },
    "003192": {
      "height": 375,
      "width": 1242
    },
    "003193": {
      "height": 370,
      "width": 1224
    },
    "003194": {
      "height": 375,
      "width": 1242
    },
    "003195": {
      "height": 375,
      "width": 1242
    },
    "003196": {
      "height": 375,
      "width": 1242
    },
    "003197": {
      "height": 375,
      "width": 1242
    },
    "003198": {
      "height": 375,
      "width": 1242
    },
    "003199": {
      "height": 370,
      "width": 1224
    },
    "003200": {
      "height": 375,
      "width": 1242
    },
    "003201": {
      "height": 375,
      "width": 1242
    },
    "003202": {
      "height": 375,
      "width": 1242
    },
    "003203": {
      "height": 375,
      "width": 1242
    },
    "003204": {
      "height": 375,
      "width": 1242
    },
    "003205": {
      "height": 375,
      "width": 1242
    },
    "003206": {
      "height": 375,
      "width": 1242
    },
    "003207": {
      "height": 374,
      "width": 1238
    },
    "003208": {
      "height": 375,
      "width": 1242
    },
    "003209": {
      "height": 375,
      "width": 1242
    },
    "003210": {
      "height": 375,
      "width": 1242
    },
    "003211": {
      "height": 375,
      "width": 1242
    },
    "003212": {
      "height": 375,
      "width": 1242
    },
    "003213": {
      "height": 375,
      "width": 1242
    },
    "003214": {
      "height": 376,
      "width": 1241
    },
    "003215": {
      "height": 375,
      "width": 1242
    },
    "003216": {
      "height": 376,
      "width": 1241
    },
    "003217": {
      "height": 375,
      "width": 1242
    },
    "003218": {
      "height": 375,
      "width": 1242
    },
    "003219": {
      "height": 375,
      "width": 1242
    },
    "003220": {
      "height": 375,
      "width": 1242
    },
    "003221": {
      "height": 370,
      "width": 1224
    },
    "003222": {
      "height": 375,
      "width": 1242
    },
    "003223": {
      "height": 375,
      "width": 1242
    },
    "003224": {
      "height": 375,
      "width": 1242
    },
    "003225": {
      "height": 375,
      "width": 1242
    },
    "003226": {
      "height": 375,
      "width": 1242
    },
    "003227": {
      "height": 375,
      "width": 1242
    },
    "003228": {
      "height": 375,
      "width": 1242
    },
    "003229": {
      "height": 374,
      "width": 1238
    },
    "003230": {
      "height": 375,
      "width": 1242
    },
    "003231": {
      "height": 375,
      "width": 1242
    },
    "003232": {
      "height": 370,
      "width": 1224
    },
    "003233": {
      "height": 374,
      "width": 1238
    },
    "003234": {
      "height": 370,
      "width": 1224
    },
    "003235": {
      "height": 375,
      "width": 1242
    },
    "003236": {
      "height": 375,
      "width": 1242
    },
    "003237": {
      "height": 375,
      "width": 1242
    },
    "003238": {
      "height": 375,
      "width": 1242
    },
    "003239": {
      "height": 376,
      "width": 1241
    },
    "003240": {
      "height": 375,
      "width": 1242
    },
    "003241": {
      "height": 374,
      "width": 1238
    },
    "003242": {
      "height": 375,
      "width": 1242
    },
    "003243": {
      "height": 375,
      "width": 1242
    },
    "003244": {
      "height": 375,
      "width": 1242
    },
    "003245": {
      "height": 375,
      "width": 1242
    },
    "003246": {
      "height": 375,
      "width": 1242
    },
    "003247": {
      "height": 370,
      "width": 1224
    },
    "003248": {
      "height": 375,
      "width": 1242
    },
    "003249": {
      "height": 375,
      "width": 1242
    },
    "003250": {
      "height": 375,
      "width": 1242
    },
    "003251": {
      "height": 375,
      "width": 1242
    },
    "003252": {
      "height": 375,
      "width": 1242
    },
    "003253": {
      "height": 375,
      "width": 1242
    },
    "003254": {
      "height": 375,
      "width": 1242
    },
    "003255": {
      "height": 375,
      "width": 1242
    },
    "003256": {
      "height": 375,
      "width": 1242
    },
    "003257": {
      "height": 375,
      "width": 1242
    },
    "003258": {
      "height": 375,
      "width": 1242
    },
    "003259": {
      "height": 375,
      "width": 1242
    },
    "003260": {
      "height": 375,
      "width": 1242
    },
    "003261": {
      "height": 375,
      "width": 1242
    },
    "003262": {
      "height": 375,
      "width": 1242
    },
    "003263": {
      "height": 375,
      "width": 1242
    },
    "003264": {
      "height": 375,
      "width": 1242
    },
    "003265": {
      "height": 375,
      "width": 1242
    },
    "003266": {
      "height": 375,
      "width": 1242
    },
    "003267": {
      "height": 375,
      "width": 1242
    },
    "003268": {
      "height": 374,
      "width": 1238
    },
    "003269": {
      "height": 370,
      "width": 1224
    },
    "003270": {
      "height": 375,
      "width": 1242
    },
    "003271": {
      "height": 375,
      "width": 1242
    },
    "003272": {
      "height": 375,
      "width": 1242
    },
    "003273": {
      "height": 375,
      "width": 1242
    },
    "003274": {
      "height": 375,
      "width": 1242
    },
    "003275": {
      "height": 375,
      "width": 1242
    },
    "003276": {
      "height": 375,
      "width": 1242
    },
    "003277": {
      "height": 370,
      "width": 1224
    },
    "003278": {
      "height": 375,
      "width": 1242
    },
    "003279": {
      "height": 375,
      "width": 1242
    },
    "003280": {
      "height": 375,
      "width": 1242
    },
    "003281": {
      "height": 375,
      "width": 1242
    },
    "003282": {
      "height": 375,
      "width": 1242
    },
    "003283": {
      "height": 375,
      "width": 1242
    },
    "003284": {
      "height": 375,
      "width": 1242
    },
    "003285": {
      "height": 375,
      "width": 1242
    },
    "003286": {
      "height": 375,
      "width": 1242
    },
    "003287": {
      "height": 370,
      "width": 1224
    },
    "003288": {
      "height": 375,
      "width": 1242
    },
    "003289": {
      "height": 375,
      "width": 1242
    },
    "003290": {
      "height": 375,
      "width": 1242
    },
    "003291": {
      "height": 370,
      "width": 1224
    },
    "003292": {
      "height": 375,
      "width": 1242
    },
    "003293": {
      "height": 375,
      "width": 1242
    },
    "003294": {
      "height": 375,
      "width": 1242
    },
    "003295": {
      "height": 370,
      "width": 1224
    },
    "003296": {
      "height": 370,
      "width": 1224
    },
    "003297": {
      "height": 375,
      "width": 1242
    },
    "003298": {
      "height": 375,
      "width": 1242
    },
    "003299": {
      "height": 375,
      "width": 1242
    },
    "003300": {
      "height": 375,
      "width": 1242
    },
    "003301": {
      "height": 375,
      "width": 1242
    },
    "003302": {
      "height": 375,
      "width": 1242
    },
    "003303": {
      "height": 374,
      "width": 1238
    },
    "003304": {
      "height": 375,
      "width": 1242
    },
    "003305": {
      "height": 375,
      "width": 1242
    },
    "003306": {
      "height": 374,
      "width": 1238
    },
    "003307": {
      "height": 370,
      "width": 1224
    },
    "003308": {
      "height": 375,
      "width": 1242
    },
    "003309": {
      "height": 375,
      "width": 1242
    },
    "003310": {
      "height": 375,
      "width": 1242
    },
    "003311": {
      "height": 375,
      "width": 1242
    },
    "003312": {
      "height": 375,
      "width": 1242
    },
    "003313": {
      "height": 375,
      "width": 1242
    },
    "003314": {
      "height": 375,
      "width": 1242
    },
    "003315": {
      "height": 370,
      "width": 1224
    },
    "003316": {
      "height": 375,
      "width": 1242
    },
    "003317": {
      "height": 375,
      "width": 1242
    },
    "003318": {
      "height": 375,
      "width": 1242
    },
    "003319": {
      "height": 375,
      "width": 1242
    },
    "003320": {
      "height": 375,
      "width": 1242
    },
    "003321": {
      "height": 370,
      "width": 1224
    },
    "003322": {
      "height": 375,
      "width": 1242
    },
    "003323": {
      "height": 375,
      "width": 1242
    },
    "003324": {
      "height": 370,
      "width": 1224
    },
    "003325": {
      "height": 375,
      "width": 1242
    },
    "003326": {
      "height": 375,
      "width": 1242
    },
    "003327": {
      "height": 375,
      "width": 1242
    },
    "003328": {
      "height": 375,
      "width": 1242
    },
    "003329": {
      "height": 375,
      "width": 1242
    },
    "003330": {
      "height": 370,
      "width": 1224
    },
    "003331": {
      "height": 370,
      "width": 1224
    },
    "003332": {
      "height": 375,
      "width": 1242
    },
    "003333": {
      "height": 370,
      "width": 1224
    },
    "003334": {
      "height": 375,
      "width": 1242
    },
    "003335": {
      "height": 375,
      "width": 1242
    },
    "003336": {
      "height": 375,
      "width": 1242
    },
    "003337": {
      "height": 375,
      "width": 1242
    },
    "003338": {
      "height": 375,
      "width": 1242
    },
    "003339": {
      "height": 370,
      "width": 1224
    },
    "003340": {
      "height": 375,
      "width": 1242
    },
    "003341": {
      "height": 376,
      "width": 1241
    },
    "003342": {
      "height": 375,
      "width": 1242
    },
    "003343": {
      "height": 375,
      "width": 1242
    },
    "003344": {
      "height": 375,
      "width": 1242
    },
    "003345": {
      "height": 375,
      "width": 1242
    },
    "003346": {
      "height": 375,
      "width": 1242
    },
    "003347": {
      "height": 375,
      "width": 1242
    },
    "003348": {
      "height": 375,
      "width": 1242
    },
    "003349": {
      "height": 370,
      "width": 1224
    },
    "003350": {
      "height": 375,
      "width": 1242
    },
    "003351": {
      "height": 375,
      "width": 1242
    },
    "003352": {
      "height": 375,
      "width": 1242
    },
    "003353": {
      "height": 375,
      "width": 1242
    },
    "003354": {
      "height": 375,
      "width": 1242
    },
    "003355": {
      "height": 375,
      "width": 1242
    },
    "003356": {
      "height": 375,
      "width": 1242
    },
    "003357": {
      "height": 375,
      "width": 1242
    },
    "003358": {
      "height": 375,
      "width": 1242
    },
    "003359": {
      "height": 375,
      "width": 1242
    },
    "003360": {
      "height": 375,
      "width": 1242
    },
    "003361": {
      "height": 375,
      "width": 1242
    },
    "003362": {
      "height": 375,
      "width": 1242
    },
    "003363": {
      "height": 375,
      "width": 1242
    },
    "003364": {
      "height": 375,
      "width": 1242
    },
    "003365": {
      "height": 375,
      "width": 1242
    },
    "003366": {
      "height": 375,
      "width": 1242
    },
    "003367": {
      "height": 375,
      "width": 1242
    },
    "003368": {
      "height": 370,
      "width": 1224
    },
    "003369": {
      "height": 375,
      "width": 1242
    },
    "003370": {
      "height": 375,
      "width": 1242
    },
    "003371": {
      "height": 375,
      "width": 1242
    },
    "003372": {
      "height": 375,
      "width": 1242
    },
    "003373": {
      "height": 375,
      "width": 1242
    },
    "003374": {
      "height": 370,
      "width": 1224
    },
    "003375": {
      "height": 375,
      "width": 1242
    },
    "003376": {
      "height": 375,
      "width": 1242
    },
    "003377": {
      "height": 375,
      "width": 1242
    },
    "003378": {
      "height": 375,
      "width": 1242
    },
    "003379": {
      "height": 375,
      "width": 1242
    },
    "003380": {
      "height": 375,
      "width": 1242
    },
    "003381": {
      "height": 375,
      "width": 1242
    },
    "003382": {
      "height": 375,
      "width": 1242
    },
    "003383": {
      "height": 375,
      "width": 1242
    },
    "003384": {
      "height": 375,
      "width": 1242
    },
    "003385": {
      "height": 375,
      "width": 1242
    },
    "003386": {
      "height": 375,
      "width": 1242
    },
    "003387": {
      "height": 370,
      "width": 1224
    },
    "003388": {
      "height": 375,
      "width": 1242
    },
    "003389": {
      "height": 375,
      "width": 1242
    },
    "003390": {
      "height": 375,
      "width": 1242
    },
    "003391": {
      "height": 375,
      "width": 1242
    },
    "003392": {
      "height": 370,
      "width": 1224
    },
    "003393": {
      "height": 374,
      "width": 1238
    },
    "003394": {
      "height": 375,
      "width": 1242
    },
    "003395": {
      "height": 375,
      "width": 1242
    },
    "003396": {
      "height": 376,
      "width": 1241
    },
    "003397": {
      "height": 375,
      "width": 1242
    },
    "003398": {
      "height": 370,
      "width": 1224
    },
    "003399": {
      "height": 375,
      "width": 1242
    },
    "003400": {
      "height": 375,
      "width": 1242
    },
    "003401": {
      "height": 375,
      "width": 1242
    },
    "003402": {
      "height": 375,
      "width": 1242
    },
    "003403": {
      "height": 375,
      "width": 1242
    },
    "003404": {
      "height": 370,
      "width": 1224
    },
    "003405": {
      "height": 375,
      "width": 1242
    },
    "003406": {
      "height": 375,
      "width": 1242
    },
    "003407": {
      "height": 375,
      "width": 1242
    },
    "003408": {
      "height": 375,
      "width": 1242
    },
    "003409": {
      "height": 375,
      "width": 1242
    },
    "003410": {
      "height": 375,
      "width": 1242
    },
    "003411": {
      "height": 375,
      "width": 1242
    },
    "003412": {
      "height": 375,
      "width": 1242
    },
    "003413": {
      "height": 375,
      "width": 1242
    },
    "003414": {
      "height": 375,
      "width": 1242
    },
    "003415": {
      "height": 375,
      "width": 1242
    },
    "003416": {
      "height": 375,
      "width": 1242
    },
    "003417": {
      "height": 375,
      "width": 1242
    },
    "003418": {
      "height": 375,
      "width": 1242
    },
    "003419": {
      "height": 375,
      "width": 1242
    },
    "003420": {
      "height": 375,
      "width": 1242
    },
    "003421": {
      "height": 370,
      "width": 1224
    },
    "003422": {
      "height": 375,
      "width": 1242
    },
    "003423": {
      "height": 375,
      "width": 1242
    },
    "003424": {
      "height": 375,
      "width": 1242
    },
    "003425": {
      "height": 375,
      "width": 1242
    },
    "003426": {
      "height": 370,
      "width": 1224
    },
    "003427": {
      "height": 375,
      "width": 1242
    },
    "003428": {
      "height": 375,
      "width": 1242
    },
    "003429": {
      "height": 375,
      "width": 1242
    },
    "003430": {
      "height": 375,
      "width": 1242
    },
    "003431": {
      "height": 375,
      "width": 1242
    },
    "003432": {
      "height": 375,
      "width": 1242
    },
    "003433": {
      "height": 375,
      "width": 1242
    },
    "003434": {
      "height": 375,
      "width": 1242
    },
    "003435": {
      "height": 370,
      "width": 1224
    },
    "003436": {
      "height": 375,
      "width": 1242
    },
    "003437": {
      "height": 370,
      "width": 1224
    },
    "003438": {
      "height": 375,
      "width": 1242
    },
    "003439": {
      "height": 375,
      "width": 1242
    },
    "003440": {
      "height": 375,
      "width": 1242
    },
    "003441": {
      "height": 375,
      "width": 1242
    },
    "003442": {
      "height": 375,
      "width": 1242
    },
    "003443": {
      "height": 375,
      "width": 1242
    },
    "003444": {
      "height": 375,
      "width": 1242
    },
    "003445": {
      "height": 375,
      "width": 1242
    },
    "003446": {
      "height": 375,
      "width": 1242
    },
    "003447": {
      "height": 375,
      "width": 1242
    },
    "003448": {
      "height": 375,
      "width": 1242
    },
    "003449": {
      "height": 375,
      "width": 1242
    },
    "003450": {
      "height": 375,
      "width": 1242
    },
    "003451": {
      "height": 375,
      "width": 1242
    },
    "003452": {
      "height": 375,
      "width": 1242
    },
    "003453": {
      "height": 375,
      "width": 1242
    },
    "003454": {
      "height": 370,
      "width": 1224
    },
    "003455": {
      "height": 375,
      "width": 1242
    },
    "003456": {
      "height": 375,
      "width": 1242
    },
    "003457": {
      "height": 375,
      "width": 1242
    },
    "003458": {
      "height": 375,
      "width": 1242
    },
    "003459": {
      "height": 375,
      "width": 1242
    },
    "003460": {
      "height": 375,
      "width": 1242
    },
    "003461": {
      "height": 374,
      "width": 1238
    },
    "003462": {
      "height": 375,
      "width": 1242
    },
    "003463": {
      "height": 375,
      "width": 1242
    },
    "003464": {
      "height": 370,
      "width": 1224
    },
    "003465": {
      "height": 375,
      "width": 1242
    },
    "003466": {
      "height": 374,
      "width": 1238
    },
    "003467": {
      "height": 375,
      "width": 1242
    },
    "003468": {
      "height": 375,
      "width": 1242
    },
    "003469": {
      "height": 375,
      "width": 1242
    },
    "003470": {
      "height": 375,
      "width": 1242
    },
    "003471": {
      "height": 376,
      "width": 1241
    },
    "003472": {
      "height": 370,
      "width": 1224
    },
    "003473": {
      "height": 375,
      "width": 1242
    },
    "003474": {
      "height": 375,
      "width": 1242
    },
    "003475": {
      "height": 375,
      "width": 1242
    },
    "003476": {
      "height": 370,
      "width": 1224
    },
    "003477": {
      "height": 375,
      "width": 1242
    },
    "003478": {
      "height": 375,
      "width": 1242
    },
    "003479": {
      "height": 375,
      "width": 1242
    },
    "003480": {
      "height": 374,
      "width": 1238
    },
    "003481": {
      "height": 370,
      "width": 1224
    },
    "003482": {
      "height": 375,
      "width": 1242
    },
    "003483": {
      "height": 375,
      "width": 1242
    },
    "003484": {
      "height": 375,
      "width": 1242
    },
    "003485": {
      "height": 375,
      "width": 1242
    },
    "003486": {
      "height": 375,
      "width": 1242
    },
    "003487": {
      "height": 374,
      "width": 1238
    },
    "003488": {
      "height": 376,
      "width": 1241
    },
    "003489": {
      "height": 370,
      "width": 1224
    },
    "003490": {
      "height": 375,
      "width": 1242
    },
    "003491": {
      "height": 370,
      "width": 1224
    },
    "003492": {
      "height": 374,
      "width": 1238
    },
    "003493": {
      "height": 375,
      "width": 1242
    },
    "003494": {
      "height": 375,
      "width": 1242
    },
    "003495": {
      "height": 375,
      "width": 1242
    },
    "003496": {
      "height": 375,
      "width": 1242
    },
    "003497": {
      "height": 375,
      "width": 1242
    },
    "003498": {
      "height": 375,
      "width": 1242
    },
    "003499": {
      "height": 375,
      "width": 1242
    },
    "003500": {
      "height": 375,
      "width": 1242
    },
    "003501": {
      "height": 375,
      "width": 1242
    },
    "003502": {
      "height": 370,
      "width": 1224
    },
    "003503": {
      "height": 375,
      "width": 1242
    },
    "003504": {
      "height": 375,
      "width": 1242
    },
    "003505": {
      "height": 375,
      "width": 1242
    },
    "003506": {
      "height": 375,
      "width": 1242
    },
    "003507": {
      "height": 375,
      "width": 1242
    },
    "003508": {
      "height": 375,
      "width": 1242
    },
    "003509": {
      "height": 375,
      "width": 1242
    },
    "003510": {
      "height": 375,
      "width": 1242
    },
    "003511": {
      "height": 374,
      "width": 1238
    },
    "003512": {
      "height": 375,
      "width": 1242
    },
    "003513": {
      "height": 375,
      "width": 1242
    },
    "003514": {
      "height": 375,
      "width": 1242
    },
    "003515": {
      "height": 375,
      "width": 1242
    },
    "003516": {
      "height": 375,
      "width": 1242
    },
    "003517": {
      "height": 375,
      "width": 1242
    },
    "003518": {
      "height": 375,
      "width": 1242
    },
    "003519": {
      "height": 375,
      "width": 1242
    },
    "003520": {
      "height": 375,
      "width": 1242
    },
    "003521": {
      "height": 375,
      "width": 1242
    },
    "003522": {
      "height": 375,
      "width": 1242
    },
    "003523": {
      "height": 375,
      "width": 1242
    },
    "003524": {
      "height": 375,
      "width": 1242
    },
    "003525": {
      "height": 375,
      "width": 1242
    },
    "003526": {
      "height": 370,
      "width": 1224
    },
    "003527": {
      "height": 370,
      "width": 1224
    },
    "003528": {
      "height": 376,
      "width": 1241
    },
    "003529": {
      "height": 375,
      "width": 1242
    },
    "003530": {
      "height": 375,
      "width": 1242
    },
    "003531": {
      "height": 375,
      "width": 1242
    },
    "003532": {
      "height": 375,
      "width": 1242
    },
    "003533": {
      "height": 375,
      "width": 1242
    },
    "003534": {
      "height": 375,
      "width": 1242
    },
    "003535": {
      "height": 375,
      "width": 1242
    },
    "003536": {
      "height": 375,
      "width": 1242
    },
    "003537": {
      "height": 375,
      "width": 1242
    },
    "003538": {
      "height": 375,
      "width": 1242
    },
    "003539": {
      "height": 375,
      "width": 1242
    },
    "003540": {
      "height": 375,
      "width": 1242
    },
    "003541": {
      "height": 375,
      "width": 1242
    },
    "003542": {
      "height": 375,
      "width": 1242
    },
    "003543": {
      "height": 374,
      "width": 1238
    },
    "003544": {
      "height": 375,
      "width": 1242
    },
    "003545": {
      "height": 375,
      "width": 1242
    },
    "003546": {
      "height": 375,
      "width": 1242
    },
    "003547": {
      "height": 375,
      "width": 1242
    },
    "003548": {
      "height": 375,
      "width": 1242
    },
    "003549": {
      "height": 375,
      "width": 1242
    },
    "003550": {
      "height": 374,
      "width": 1238
    },
    "003551": {
      "height": 375,
      "width": 1242
    },
    "003552": {
      "height": 374,
      "width": 1238
    },
    "003553": {
      "height": 375,
      "width": 1242
    },
    "003554": {
      "height": 375,
      "width": 1242
    },
    "003555": {
      "height": 375,
      "width": 1242
    },
    "003556": {
      "height": 375,
      "width": 1242
    },
    "003557": {
      "height": 375,
      "width": 1242
    },
    "003558": {
      "height": 375,
      "width": 1242
    },
    "003559": {
      "height": 375,
      "width": 1242
    },
    "003560": {
      "height": 370,
      "width": 1224
    },
    "003561": {
      "height": 375,
      "width": 1242
    },
    "003562": {
      "height": 375,
      "width": 1242
    },
    "003563": {
      "height": 376,
      "width": 1241
    },
    "003564": {
      "height": 375,
      "width": 1242
    },
    "003565": {
      "height": 375,
      "width": 1242
    },
    "003566": {
      "height": 375,
      "width": 1242
    },
    "003567": {
      "height": 375,
      "width": 1242
    },
    "003568": {
      "height": 376,
      "width": 1241
    },
    "003569": {
      "height": 375,
      "width": 1242
    },
    "003570": {
      "height": 374,
      "width": 1238
    },
    "003571": {
      "height": 370,
      "width": 1224
    },
    "003572": {
      "height": 375,
      "width": 1242
    },
    "003573": {
      "height": 375,
      "width": 1242
    },
    "003574": {
      "height": 375,
      "width": 1242
    },
    "003575": {
      "height": 375,
      "width": 1242
    },
    "003576": {
      "height": 370,
      "width": 1224
    },
    "003577": {
      "height": 375,
      "width": 1242
    },
    "003578": {
      "height": 375,
      "width": 1242
    },
    "003579": {
      "height": 375,
      "width": 1242
    },
    "003580": {
      "height": 375,
      "width": 1242
    },
    "003581": {
      "height": 375,
      "width": 1242
    },
    "003582": {
      "height": 375,
      "width": 1242
    },
    "003583": {
      "height": 375,
      "width": 1242
    },
    "003584": {
      "height": 375,
      "width": 1242
    },
    "003585": {
      "height": 375,
      "width": 1242
    },
    "003586": {
      "height": 375,
      "width": 1242
    },
    "003587": {
      "height": 370,
      "width": 1224
    },
    "003588": {
      "height": 375,
      "width": 1242
    },
    "003589": {
      "height": 375,
      "width": 1242
    },
    "003590": {
      "height": 375,
      "width": 1242
    },
    "003591": {
      "height": 370,
      "width": 1224
    },
    "003592": {
      "height": 375,
      "width": 1242
    },
    "003593": {
      "height": 375,
      "width": 1242
    },
    "003594": {
      "height": 375,
      "width": 1242
    },
    "003595": {
      "height": 375,
      "width": 1242
    },
    "003596": {
      "height": 375,
      "width": 1242
    },
    "003597": {
      "height": 375,
      "width": 1242
    },
    "003598": {
      "height": 375,
      "width": 1242
    },
    "003599": {
      "height": 375,
      "width": 1242
    },
    "003600": {
      "height": 375,
      "width": 1242
    },
    "003601": {
      "height": 375,
      "width": 1242
    },
    "003602": {
      "height": 375,
      "width": 1242
    },
    "003603": {
      "height": 375,
      "width": 1242
    },
    "003604": {
      "height": 375,
      "width": 1242
    },
    "003605": {
      "height": 375,
      "width": 1242
    },
    "003606": {
      "height": 375,
      "width": 1242
    },
    "003607": {
      "height": 375,
      "width": 1242
    },
    "003608": {
      "height": 375,
      "width": 1242
    },
    "003609": {
      "height": 375,
      "width": 1242
    },
    "003610": {
      "height": 375,
      "width": 1242
    },
    "003611": {
      "height": 375,
      "width": 1242
    },
    "003612": {
      "height": 375,
      "width": 1242
    },
    "003613": {
      "height": 375,
      "width": 1242
    },
    "003614": {
      "height": 376,
      "width": 1241
    },
    "003615": {
      "height": 375,
      "width": 1242
    },
    "003616": {
      "height": 375,
      "width": 1242
    },
    "003617": {
      "height": 370,
      "width": 1224
    },
    "003618": {
      "height": 375,
      "width": 1242
    },
    "003619": {
      "height": 375,
      "width": 1242
    },
    "003620": {
      "height": 375,
      "width": 1242
    },
    "003621": {
      "height": 375,
      "width": 1242
    },
    "003622": {
      "height": 376,
      "width": 1241
    },
    "003623": {
      "height": 375,
      "width": 1242
    },
    "003624": {
      "height": 375,
      "width": 1242
    },
    "003625": {
      "height": 370,
      "width": 1224
    },
    "003626": {
      "height": 375,
      "width": 1242
    },
    "003627": {
      "height": 375,
      "width": 1242
    },
    "003628": {
      "height": 370,
      "width": 1224
    },
    "003629": {
      "height": 375,
      "width": 1242
    },
    "003630": {
      "height": 375,
      "width": 1242
    },
    "003631": {
      "height": 375,
      "width": 1242
    },
    "003632": {
      "height": 374,
      "width": 1238
    },
    "003633": {
      "height": 375,
      "width": 1242
    },
    "003634": {
      "height": 370,
      "width": 1224
    },
    "003635": {
      "height": 375,
      "width": 1242
    },
    "003636": {
      "height": 375,
      "width": 1242
    },
    "003637": {
      "height": 375,
      "width": 1242
    },
    "003638": {
      "height": 375,
      "width": 1242
    },
    "003639": {
      "height": 375,
      "width": 1242
    },
    "003640": {
      "height": 375,
      "width": 1242
    },
    "003641": {
      "height": 375,
      "width": 1242
    },
    "003642": {
      "height": 375,
      "width": 1242
    },
    "003643": {
      "height": 375,
      "width": 1242
    },
    "003644": {
      "height": 375,
      "width": 1242
    },
    "003645": {
      "height": 375,
      "width": 1242
    },
    "003646": {
      "height": 375,
      "width": 1242
    },
    "003647": {
      "height": 375,
      "width": 1242
    },
    "003648": {
      "height": 375,
      "width": 1242
    },
    "003649": {
      "height": 370,
      "width": 1224
    },
    "003650": {
      "height": 375,
      "width": 1242
    },
    "003651": {
      "height": 375,
      "width": 1242
    },
    "003652": {
      "height": 375,
      "width": 1242
    },
    "003653": {
      "height": 375,
      "width": 1242
    },
    "003654": {
      "height": 375,
      "width": 1242
    },
    "003655": {
      "height": 375,
      "width": 1242
    },
    "003656": {
      "height": 375,
      "width": 1242
    },
    "003657": {
      "height": 375,
      "width": 1242
    },
    "003658": {
      "height": 375,
      "width": 1242
    },
    "003659": {
      "height": 376,
      "width": 1241
    },
    "003660": {
      "height": 375,
      "width": 1242
    },
    "003661": {
      "height": 375,
      "width": 1242
    },
    "003662": {
      "height": 375,
      "width": 1242
    },
    "003663": {
      "height": 375,
      "width": 1242
    },
    "003664": {
      "height": 375,
      "width": 1242
    },
    "003665": {
      "height": 375,
      "width": 1242
    },
    "003666": {
      "height": 375,
      "width": 1242
    },
    "003667": {
      "height": 370,
      "width": 1224
    },
    "003668": {
      "height": 370,
      "width": 1224
    },
    "003669": {
      "height": 374,
      "width": 1238
    },
    "003670": {
      "height": 370,
      "width": 1224
    },
    "003671": {
      "height": 375,
      "width": 1242
    },
    "003672": {
      "height": 375,
      "width": 1242
    },
    "003673": {
      "height": 375,
      "width": 1242
    },
    "003674": {
      "height": 375,
      "width": 1242
    },
    "003675": {
      "height": 370,
      "width": 1224
    },
    "003676": {
      "height": 375,
      "width": 1242
    },
    "003677": {
      "height": 375,
      "width": 1242
    },
    "003678": {
      "height": 375,
      "width": 1242
    },
    "003679": {
      "height": 375,
      "width": 1242
    },
    "003680": {
      "height": 375,
      "width": 1242
    },
    "003681": {
      "height": 375,
      "width": 1242
    },
    "003682": {
      "height": 375,
      "width": 1242
    },
    "003683": {
      "height": 374,
      "width": 1238
    },
    "003684": {
      "height": 375,
      "width": 1242
    },
    "003685": {
      "height": 375,
      "width": 1242
    },
    "003686": {
      "height": 375,
      "width": 1242
    },
    "003687": {
      "height": 375,
      "width": 1242
    },
    "003688": {
      "height": 374,
      "width": 1238
    },
    "003689": {
      "height": 375,
      "width": 1242
    },
    "003690": {
      "height": 375,
      "width": 1242
    },
    "003691": {
      "height": 375,
      "width": 1242
    },
    "003692": {
      "height": 375,
      "width": 1242
    },
    "003693": {
      "height": 375,
      "width": 1242
    },
    "003694": {
      "height": 370,
      "width": 1224
    },
    "003695": {
      "height": 375,
      "width": 1242
    },
    "003696": {
      "height": 375,
      "width": 1242
    },
    "003697": {
      "height": 375,
      "width": 1242
    },
    "003698": {
      "height": 375,
      "width": 1242
    },
    "003699": {
      "height": 375,
      "width": 1242
    },
    "003700": {
      "height": 370,
      "width": 1224
    },
    "003701": {
      "height": 375,
      "width": 1242
    },
    "003702": {
      "height": 375,
      "width": 1242
    },
    "003703": {
      "height": 370,
      "width": 1224
    },
    "003704": {
      "height": 375,
      "width": 1242
    },
    "003705": {
      "height": 375,
      "width": 1242
    },
    "003706": {
      "height": 375,
      "width": 1242
    },
    "003707": {
      "height": 375,
      "width": 1242
    },
    "003708": {
      "height": 375,
      "width": 1242
    },
    "003709": {
      "height": 370,
      "width": 1224
    },
    "003710": {
      "height": 375,
      "width": 1242
    },
    "003711": {
      "height": 375,
      "width": 1242
    },
    "003712": {
      "height": 375,
      "width": 1242
    },
    "003713": {
      "height": 375,
      "width": 1242
    },
    "003714": {
      "height": 375,
      "width": 1242
    },
    "003715": {
      "height": 375,
      "width": 1242
    },
    "003716": {
      "height": 375,
      "width": 1242
    },
    "003717": {
      "height": 370,
      "width": 1224
    },
    "003718": {
      "height": 374,
      "width": 1238
    },
    "003719": {
      "height": 375,
      "width": 1242
    },
    "003720": {
      "height": 375,
      "width": 1242
    },
    "003721": {
      "height": 375,
      "width": 1242
    },
    "003722": {
      "height": 375,
      "width": 1242
    },
    "003723": {
      "height": 374,
      "width": 1238
    },
    "003724": {
      "height": 375,
      "width": 1242
    },
    "003725": {
      "height": 375,
      "width": 1242
    },
    "003726": {
      "height": 375,
      "width": 1242
    },
    "003727": {
      "height": 375,
      "width": 1242
    },
    "003728": {
      "height": 374,
      "width": 1238
    },
    "003729": {
      "height": 375,
      "width": 1242
    },
    "003730": {
      "height": 375,
      "width": 1242
    },
    "003731": {
      "height": 375,
      "width": 1242
    },
    "003732": {
      "height": 374,
      "width": 1238
    },
    "003733": {
      "height": 375,
      "width": 1242
    },
    "003734": {
      "height": 375,
      "width": 1242
    },
    "003735": {
      "height": 376,
      "width": 1241
    },
    "003736": {
      "height": 375,
      "width": 1242
    },
    "003737": {
      "height": 370,
      "width": 1224
    },
    "003738": {
      "height": 375,
      "width": 1242
    },
    "003739": {
      "height": 374,
      "width": 1238
    },
    "003740": {
      "height": 375,
      "width": 1242
    },
    "003741": {
      "height": 375,
      "width": 1242
    },
    "003742": {
      "height": 374,
      "width": 1238
    },
    "003743": {
      "height": 375,
      "width": 1242
    },
    "003744": {
      "height": 375,
      "width": 1242
    },
    "003745": {
      "height": 375,
      "width": 1242
    },
    "003746": {
      "height": 375,
      "width": 1242
    },
    "003747": {
      "height": 375,
      "width": 1242
    },
    "003748": {
      "height": 375,
      "width": 1242
    },
    "003749": {
      "height": 375,
      "width": 1242
    },
    "003750": {
      "height": 375,
      "width": 1242
    },
    "003751": {
      "height": 375,
      "width": 1242
    },
    "003752": {
      "height": 375,
      "width": 1242
    },
    "003753": {
      "height": 374,
      "width": 1238
    },
    "003754": {
      "height": 375,
      "width": 1242
    },
    "003755": {
      "height": 375,
      "width": 1242
    },
    "003756": {
      "height": 375,
      "width": 1242
    },
    "003757": {
      "height": 375,
      "width": 1242
    },
    "003758": {
      "height": 375,
      "width": 1242
    },
    "003759": {
      "height": 375,
      "width": 1242
    },
    "003760": {
      "height": 375,
      "width": 1242
    },
    "003761": {
      "height": 375,
      "width": 1242
    },
    "003762": {
      "height": 375,
      "width": 1242
    },
    "003763": {
      "height": 374,
      "width": 1238
    },
    "003764": {
      "height": 375,
      "width": 1242
    },
    "003765": {
      "height": 375,
      "width": 1242
    },
    "003766": {
      "height": 370,
      "width": 1224
    },
    "003767": {
      "height": 375,
      "width": 1242
    },
    "003768": {
      "height": 375,
      "width": 1242
    },
    "003769": {
      "height": 375,
      "width": 1242
    },
    "003770": {
      "height": 370,
      "width": 1224
    },
    "003771": {
      "height": 375,
      "width": 1242
    },
    "003772": {
      "height": 375,
      "width": 1242
    },
    "003773": {
      "height": 375,
      "width": 1242
    },
    "003774": {
      "height": 370,
      "width": 1224
    },
    "003775": {
      "height": 375,
      "width": 1242
    },
    "003776": {
      "height": 375,
      "width": 1242
    },
    "003777": {
      "height": 375,
      "width": 1242
    },
    "003778": {
      "height": 375,
      "width": 1242
    },
    "003779": {
      "height": 375,
      "width": 1242
    },
    "003780": {
      "height": 374,
      "width": 1238
    },
    "003781": {
      "height": 376,
      "width": 1241
    },
    "003782": {
      "height": 375,
      "width": 1242
    },
    "003783": {
      "height": 375,
      "width": 1242
    },
    "003784": {
      "height": 375,
      "width": 1242
    },
    "003785": {
      "height": 375,
      "width": 1242
    },
    "003786": {
      "height": 370,
      "width": 1224
    },
    "003787": {
      "height": 370,
      "width": 1224
    },
    "003788": {
      "height": 375,
      "width": 1242
    },
    "003789": {
      "height": 375,
      "width": 1242
    },
    "003790": {
      "height": 375,
      "width": 1242
    },
    "003791": {
      "height": 375,
      "width": 1242
    },
    "003792": {
      "height": 375,
      "width": 1242
    },
    "003793": {
      "height": 370,
      "width": 1224
    },
    "003794": {
      "height": 375,
      "width": 1242
    },
    "003795": {
      "height": 375,
      "width": 1242
    },
    "003796": {
      "height": 375,
      "width": 1242
    },
    "003797": {
      "height": 370,
      "width": 1224
    },
    "003798": {
      "height": 375,
      "width": 1242
    },
    "003799": {
      "height": 375,
      "width": 1242
    },
    "003800": {
      "height": 375,
      "width": 1242
    },
    "003801": {
      "height": 375,
      "width": 1242
    },
    "003802": {
      "height": 375,
      "width": 1242
    },
    "003803": {
      "height": 375,
      "width": 1242
    },
    "003804": {
      "height": 376,
      "width": 1241
    },
    "003805": {
      "height": 374,
      "width": 1238
    },
    "003806": {
      "height": 375,
      "width": 1242
    },
    "003807": {
      "height": 376,
      "width": 1241
    },
    "003808": {
      "height": 375,
      "width": 1242
    },
    "003809": {
      "height": 375,
      "width": 1242
    },
    "003810": {
      "height": 375,
      "width": 1242
    },
    "003811": {
      "height": 375,
      "width": 1242
    },
    "003812": {
      "height": 376,
      "width": 1241
    },
    "003813": {
      "height": 375,
      "width": 1242
    },
    "003814": {
      "height": 375,
      "width": 1242
    },
    "003815": {
      "height": 375,
      "width": 1242
    },
    "003816": {
      "height": 375,
      "width": 1242
    },
    "003817": {
      "height": 375,
      "width": 1242
    },
    "003818": {
      "height": 375,
      "width": 1242
    },
    "003819": {
      "height": 370,
      "width": 1224
    },
    "003820": {
      "height": 375,
      "width": 1242
    },
    "003821": {
      "height": 375,
      "width": 1242
    },
    "003822": {
      "height": 375,
      "width": 1242
    },
    "003823": {
      "height": 370,
      "width": 1224
    },
    "003824": {
      "height": 375,
      "width": 1242
    },
    "003825": {
      "height": 375,
      "width": 1242
    },
    "003826": {
      "height": 374,
      "width": 1238
    },
    "003827": {
      "height": 375,
      "width": 1242
    },
    "003828": {
      "height": 375,
      "width": 1242
    },
    "003829": {
      "height": 374,
      "width": 1238
    },
    "003830": {
      "height": 375,
      "width": 1242
    },
    "003831": {
      "height": 375,
      "width": 1242
    },
    "003832": {
      "height": 375,
      "width": 1242
    },
    "003833": {
      "height": 375,
      "width": 1242
    },
    "003834": {
      "height": 375,
      "width": 1242
    },
    "003835": {
      "height": 375,
      "width": 1242
    },
    "003836": {
      "height": 375,
      "width": 1242
    },
    "003837": {
      "height": 375,
      "width": 1242
    },
    "003838": {
      "height": 370,
      "width": 1224
    },
    "003839": {
      "height": 375,
      "width": 1242
    },
    "003840": {
      "height": 375,
      "width": 1242
    },
    "003841": {
      "height": 375,
      "width": 1242
    },
    "003842": {
      "height": 370,
      "width": 1224
    },
    "003843": {
      "height": 375,
      "width": 1242
    },
    "003844": {
      "height": 375,
      "width": 1242
    },
    "003845": {
      "height": 375,
      "width": 1242
    },
    "003846": {
      "height": 375,
      "width": 1242
    },
    "003847": {
      "height": 375,
      "width": 1242
    },
    "003848": {
      "height": 375,
      "width": 1242
    },
    "003849": {
      "height": 370,
      "width": 1224
    },
    "003850": {
      "height": 375,
      "width": 1242
    },
    "003851": {
      "height": 375,
      "width": 1242
    },
    "003852": {
      "height": 375,
      "width": 1242
    },
    "003853": {
      "height": 374,
      "width": 1238
    },
    "003854": {
      "height": 375,
      "width": 1242
    },
    "003855": {
      "height": 375,
      "width": 1242
    },
    "003856": {
      "height": 370,
      "width": 1224
    },
    "003857": {
      "height": 375,
      "width": 1242
    },
    "003858": {
      "height": 370,
      "width": 1224
    },
    "003859": {
      "height": 374,
      "width": 1238
    },
    "003860": {
      "height": 375,
      "width": 1242
    },
    "003861": {
      "height": 375,
      "width": 1242
    },
    "003862": {
      "height": 375,
      "width": 1242
    },
    "003863": {
      "height": 375,
      "width": 1242
    },
    "003864": {
      "height": 375,
      "width": 1242
    },
    "003865": {
      "height": 375,
      "width": 1242
    },
    "003866": {
      "height": 375,
      "width": 1242
    },
    "003867": {
      "height": 375,
      "width": 1242
    },
    "003868": {
      "height": 375,
      "width": 1242
    },
    "003869": {
      "height": 375,
      "width": 1242
    },
    "003870": {
      "height": 375,
      "width": 1242
    },
    "003871": {
      "height": 375,
      "width": 1242
    },
    "003872": {
      "height": 376,
      "width": 1241
    },
    "003873": {
      "height": 375,
      "width": 1242
    },
    "003874": {
      "height": 375,
      "width": 1242
    },
    "003875": {
      "height": 375,
      "width": 1242
    },
    "003876": {
      "height": 375,
      "width": 1242
    },
    "003877": {
      "height": 375,
      "width": 1242
    },
    "003878": {
      "height": 375,
      "width": 1242
    },
    "003879": {
      "height": 375,
      "width": 1242
    },
    "003880": {
      "height": 374,
      "width": 1238
    },
    "003881": {
      "height": 374,
      "width": 1238
    },
    "003882": {
      "height": 370,
      "width": 1224
    },
    "003883": {
      "height": 374,
      "width": 1238
    },
    "003884": {
      "height": 375,
      "width": 1242
    },
    "003885": {
      "height": 375,
      "width": 1242
    },
    "003886": {
      "height": 375,
      "width": 1242
    },
    "003887": {
      "height": 375,
      "width": 1242
    },
    "003888": {
      "height": 375,
      "width": 1242
    },
    "003889": {
      "height": 375,
      "width": 1242
    },
    "003890": {
      "height": 375,
      "width": 1242
    },
    "003891": {
      "height": 375,
      "width": 1242
    },
    "003892": {
      "height": 375,
      "width": 1242
    },
    "003893": {
      "height": 375,
      "width": 1242
    },
    "003894": {
      "height": 375,
      "width": 1242
    },
    "003895": {
      "height": 375,
      "width": 1242
    },
    "003896": {
      "height": 375,
      "width": 1242
    },
    "003897": {
      "height": 375,
      "width": 1242
    },
    "003898": {
      "height": 375,
      "width": 1242
    },
    "003899": {
      "height": 375,
      "width": 1242
    },
    "003900": {
      "height": 375,
      "width": 1242
    },
    "003901": {
      "height": 375,
      "width": 1242
    },
    "003902": {
      "height": 375,
      "width": 1242
    },
    "003903": {
      "height": 375,
      "width": 1242
    },
    "003904": {
      "height": 375,
      "width": 1242
    },
    "003905": {
      "height": 370,
      "width": 1224
    },
    "003906": {
      "height": 375,
      "width": 1242
    },
    "003907": {
      "height": 375,
      "width": 1242
    },
    "003908": {
      "height": 375,
      "width": 1242
    },
    "003909": {
      "height": 375,
      "width": 1242
    },
    "003910": {
      "height": 375,
      "width": 1242
    },
    "003911": {
      "height": 375,
      "width": 1242
    },
    "003912": {
      "height": 375,
      "width": 1242
    },
    "003913": {
      "height": 375,
      "width": 1242
    },
    "003914": {
      "height": 375,
      "width": 1242
    },
    "003915": {
      "height": 375,
      "width": 1242
    },
    "003916": {
      "height": 375,
      "width": 1242
    },
    "003917": {
      "height": 375,
      "width": 1242
    },
    "003918": {
      "height": 375,
      "width": 1242
    },
    "003919": {
      "height": 375,
      "width": 1242
    },
    "003920": {
      "height": 374,
      "width": 1238
    },
    "003921": {
      "height": 375,
      "width": 1242
    },
    "003922": {
      "height": 370,
      "width": 1224
    },
    "003923": {
      "height": 375,
      "width": 1242
    },
    "003924": {
      "height": 374,
      "width": 1238
    },
    "003925": {
      "height": 375,
      "width": 1242
    },
    "003926": {
      "height": 375,
      "width": 1242
    },
    "003927": {
      "height": 370,
      "width": 1224
    },
    "003928": {
      "height": 375,
      "width": 1242
    },
    "003929": {
      "height": 375,
      "width": 1242
    },
    "003930": {
      "height": 375,
      "width": 1242
    },
    "003931": {
      "height": 375,
      "width": 1242
    },
    "003932": {
      "height": 375,
      "width": 1242
    },
    "003933": {
      "height": 375,
      "width": 1242
    },
    "003934": {
      "height": 375,
      "width": 1242
    },
    "003935": {
      "height": 375,
      "width": 1242
    },
    "003936": {
      "height": 375,
      "width": 1242
    },
    "003937": {
      "height": 375,
      "width": 1242
    },
    "003938": {
      "height": 375,
      "width": 1242
    },
    "003939": {
      "height": 375,
      "width": 1242
    },
    "003940": {
      "height": 375,
      "width": 1242
    },
    "003941": {
      "height": 375,
      "width": 1242
    },
    "003942": {
      "height": 375,
      "width": 1242
    },
    "003943": {
      "height": 375,
      "width": 1242
    },
    "003944": {
      "height": 375,
      "width": 1242
    },
    "003945": {
      "height": 375,
      "width": 1242
    },
    "003946": {
      "height": 375,
      "width": 1242
    },
    "003947": {
      "height": 370,
      "width": 1224
    },
    "003948": {
      "height": 375,
      "width": 1242
    },
    "003949": {
      "height": 375,
      "width": 1242
    },
    "003950": {
      "height": 375,
      "width": 1242
    },
    "003951": {
      "height": 370,
      "width": 1224
    },
    "003952": {
      "height": 375,
      "width": 1242
    },
    "003953": {
      "height": 375,
      "width": 1242
    },
    "003954": {
      "height": 370,
      "width": 1224
    },
    "003955": {
      "height": 375,
      "width": 1242
    },
    "003956": {
      "height": 375,
      "width": 1242
    },
    "003957": {
      "height": 375,
      "width": 1242
    },
    "003958": {
      "height": 375,
      "width": 1242
    },
    "003959": {
      "height": 375,
      "width": 1242
    },
    "003960": {
      "height": 375,
      "width": 1242
    },
    "003961": {
      "height": 375,
      "width": 1242
    },
    "003962": {
      "height": 375,
      "width": 1242
    },
    "003963": {
      "height": 375,
      "width": 1242
    },
    "003964": {
      "height": 375,
      "width": 1242
    },
    "003965": {
      "height": 375,
      "width": 1242
    },
    "003966": {
      "height": 375,
      "width": 1242
    },
    "003967": {
      "height": 370,
      "width": 1224
    },
    "003968": {
      "height": 375,
      "width": 1242
    },
    "003969": {
      "height": 370,
      "width": 1224
    },
    "003970": {
      "height": 375,
      "width": 1242
    },
    "003971": {
      "height": 370,
      "width": 1224
    },
    "003972": {
      "height": 375,
      "width": 1242
    },
    "003973": {
      "height": 375,
      "width": 1242
    },
    "003974": {
      "height": 370,
      "width": 1224
    },
    "003975": {
      "height": 375,
      "width": 1242
    },
    "003976": {
      "height": 375,
      "width": 1242
    },
    "003977": {
      "height": 375,
      "width": 1242
    },
    "003978": {
      "height": 375,
      "width": 1242
    },
    "003979": {
      "height": 375,
      "width": 1242
    },
    "003980": {
      "height": 375,
      "width": 1242
    },
    "003981": {
      "height": 370,
      "width": 1224
    },
    "003982": {
      "height": 375,
      "width": 1242
    },
    "003983": {
      "height": 370,
      "width": 1224
    },
    "003984": {
      "height": 375,
      "width": 1242
    },
    "003985": {
      "height": 375,
      "width": 1242
    },
    "003986": {
      "height": 375,
      "width": 1242
    },
    "003987": {
      "height": 375,
      "width": 1242
    },
    "003988": {
      "height": 375,
      "width": 1242
    },
    "003989": {
      "height": 370,
      "width": 1224
    },
    "003990": {
      "height": 375,
      "width": 1242
    },
    "003991": {
      "height": 375,
      "width": 1242
    },
    "003992": {
      "height": 375,
      "width": 1242
    },
    "003993": {
      "height": 375,
      "width": 1242
    },
    "003994": {
      "height": 375,
      "width": 1242
    },
    "003995": {
      "height": 375,
      "width": 1242
    },
    "003996": {
      "height": 376,
      "width": 1241
    },
    "003997": {
      "height": 370,
      "width": 1224
    },
    "003998": {
      "height": 375,
      "width": 1242
    },
    "003999": {
      "height": 375,
      "width": 1242
    },
    "004000": {
      "height": 375,
      "width": 1242
    },
    "004001": {
      "height": 375,
      "width": 1242
    },
    "004002": {
      "height": 374,
      "width": 1238
    },
    "004003": {
      "height": 375,
      "width": 1242
    },
    "004004": {
      "height": 375,
      "width": 1242
    },
    "004005": {
      "height": 375,
      "width": 1242
    },
    "004006": {
      "height": 375,
      "width": 1242
    },
    "004007": {
      "height": 375,
      "width": 1242
    },
    "004008": {
      "height": 375,
      "width": 1242
    },
    "004009": {
      "height": 375,
      "width": 1242
    },
    "004010": {
      "height": 376,
      "width": 1241
    },
    "004011": {
      "height": 375,
      "width": 1242
    },
    "004012": {
      "height": 370,
      "width": 1224
    },
    "004013": {
      "height": 375,
      "width": 1242
    },
    "004014": {
      "height": 375,
      "width": 1242
    },
    "004015": {
      "height": 375,
      "width": 1242
    },
    "004016": {
      "height": 376,
      "width": 1241
    },
    "004017": {
      "height": 375,
      "width": 1242
    },
    "004018": {
      "height": 375,
      "width": 1242
    },
    "004019": {
      "height": 375,
      "width": 1242
    },
    "004020": {
      "height": 375,
      "width": 1242
    },
    "004021": {
      "height": 370,
      "width": 1224
    },
    "004022": {
      "height": 375,
      "width": 1242
    },
    "004023": {
      "height": 370,
      "width": 1224
    },
    "004024": {
      "height": 375,
      "width": 1242
    },
    "004025": {
      "height": 374,
      "width": 1238
    },
    "004026": {
      "height": 375,
      "width": 1242
    },
    "004027": {
      "height": 374,
      "width": 1238
    },
    "004028": {
      "height": 374,
      "width": 1238
    },
    "004029": {
      "height": 375,
      "width": 1242
    },
    "004030": {
      "height": 375,
      "width": 1242
    },
    "004031": {
      "height": 375,
      "width": 1242
    },
    "004032": {
      "height": 375,
      "width": 1242
    },
    "004033": {
      "height": 375,
      "width": 1242
    },
    "004034": {
      "height": 375,
      "width": 1242
    },
    "004035": {
      "height": 375,
      "width": 1242
    },
    "004036": {
      "height": 375,
      "width": 1242
    },
    "004037": {
      "height": 375,
      "width": 1242
    },
    "004038": {
      "height": 375,
      "width": 1242
    },
    "004039": {
      "height": 375,
      "width": 1242
    },
    "004040": {
      "height": 375,
      "width": 1242
    },
    "004041": {
      "height": 375,
      "width": 1242
    },
    "004042": {
      "height": 375,
      "width": 1242
    },
    "004043": {
      "height": 375,
      "width": 1242
    },
    "004044": {
      "height": 375,
      "width": 1242
    },
    "004045": {
      "height": 375,
      "width": 1242
    },
    "004046": {
      "height": 375,
      "width": 1242
    },
    "004047": {
      "height": 375,
      "width": 1242
    },
    "004048": {
      "height": 375,
      "width": 1242
    },
    "004049": {
      "height": 370,
      "width": 1224
    },
    "004050": {
      "height": 375,
      "width": 1242
    },
    "004051": {
      "height": 375,
      "width": 1242
    },
    "004052": {
      "height": 375,
      "width": 1242
    },
    "004053": {
      "height": 370,
      "width": 1224
    },
    "004054": {
      "height": 375,
      "width": 1242
    },
    "004055": {
      "height": 374,
      "width": 1238
    },
    "004056": {
      "height": 375,
      "width": 1242
    },
    "004057": {
      "height": 375,
      "width": 1242
    },
    "004058": {
      "height": 375,
      "width": 1242
    },
    "004059": {
      "height": 375,
      "width": 1242
    },
    "004060": {
      "height": 375,
      "width": 1242
    },
    "004061": {
      "height": 376,
      "width": 1241
    },
    "004062": {
      "height": 375,
      "width": 1242
    },
    "004063": {
      "height": 375,
      "width": 1242
    },
    "004064": {
      "height": 375,
      "width": 1242
    },
    "004065": {
      "height": 370,
      "width": 1224
    },
    "004066": {
      "height": 375,
      "width": 1242
    },
    "004067": {
      "height": 375,
      "width": 1242
    },
    "004068": {
      "height": 375,
      "width": 1242
    },
    "004069": {
      "height": 375,
      "width": 1242
    },
    "004070": {
      "height": 375,
      "width": 1242
    },
    "004071": {
      "height": 375,
      "width": 1242
    },
    "004072": {
      "height": 375,
      "width": 1242
    },
    "004073": {
      "height": 370,
      "width": 1224
    },
    "004074": {
      "height": 370,
      "width": 1224
    },
    "004075": {
      "height": 375,
      "width": 1242
    },
    "004076": {
      "height": 375,
      "width": 1242
    },
    "004077": {
      "height": 375,
      "width": 1242
    },
    "004078": {
      "height": 370,
      "width": 1224
    },
    "004079": {
      "height": 375,
      "width": 1242
    },
    "004080": {
      "height": 375,
      "width": 1242
    },
    "004081": {
      "height": 375,
      "width": 1242
    },
    "004082": {
      "height": 376,
      "width": 1241
    },
    "004083": {
      "height": 375,
      "width": 1242
    },
    "004084": {
      "height": 375,
      "width": 1242
    },
    "004085": {
      "height": 370,
      "width": 1224
    },
    "004086": {
      "height": 375,
      "width": 1242
    },
    "004087": {
      "height": 375,
      "width": 1242
    },
    "004088": {
      "height": 375,
      "width": 1242
    },
    "004089": {
      "height": 375,
      "width": 1242
    },
    "004090": {
      "height": 375,
      "width": 1242
    },
    "004091": {
      "height": 375,
      "width": 1242
    },
    "004092": {
      "height": 375,
      "width": 1242
    },
    "004093": {
      "height": 370,
      "width": 1224
    },
    "004094": {
      "height": 375,
      "width": 1242
    },
    "004095": {
      "height": 375,
      "width": 1242
    },
    "004096": {
      "height": 375,
      "width": 1242
    },
    "004097": {
      "height": 375,
      "width": 1242
    },
    "004098": {
      "height": 375,
      "width": 1242
    },
    "004099": {
      "height": 375,
      "width": 1242
    },
    "004100": {
      "height": 376,
      "width": 1241
    },
    "004101": {
      "height": 375,
      "width": 1242
    },
    "004102": {
      "height": 375,
      "width": 1242
    },
    "004103": {
      "height": 370,
      "width": 1224
    },
    "004104": {
      "height": 375,
      "width": 1242
    },
    "004105": {
      "height": 375,
      "width": 1242
    },
    "004106": {
      "height": 375,
      "width": 1242
    },
    "004107": {
      "height": 376,
      "width": 1241
    },
    "004108": {
      "height": 375,
      "width": 1242
    },
    "004109": {
      "height": 370,
      "width": 1224
    },
    "004110": {
      "height": 375,
      "width": 1242
    },
    "004111": {
      "height": 375,
      "width": 1242
    },
    "004112": {
      "height": 375,
      "width": 1242
    },
    "004113": {
      "height": 375,
      "width": 1242
    },
    "004114": {
      "height": 375,
      "width": 1242
    },
    "004115": {
      "height": 375,
      "width": 1242
    },
    "004116": {
      "height": 370,
      "width": 1224
    },
    "004117": {
      "height": 375,
      "width": 1242
    },
    "004118": {
      "height": 375,
      "width": 1242
    },
    "004119": {
      "height": 375,
      "width": 1242
    },
    "004120": {
      "height": 375,
      "width": 1242
    },
    "004121": {
      "height": 375,
      "width": 1242
    },
    "004122": {
      "height": 375,
      "width": 1242
    },
    "004123": {
      "height": 374,
      "width": 1238
    },
    "004124": {
      "height": 375,
      "width": 1242
    },
    "004125": {
      "height": 370,
      "width": 1224
    },
    "004126": {
      "height": 375,
      "width": 1242
    },
    "004127": {
      "height": 375,
      "width": 1242
    },
    "004128": {
      "height": 375,
      "width": 1242
    },
    "004129": {
      "height": 375,
      "width": 1242
    },
    "004130": {
      "height": 375,
      "width": 1242
    },
    "004131": {
      "height": 375,
      "width": 1242
    },
    "004132": {
      "height": 375,
      "width": 1242
    },
    "004133": {
      "height": 375,
      "width": 1242
    },
    "004134": {
      "height": 375,
      "width": 1242
    },
    "004135": {
      "height": 375,
      "width": 1242
    },
    "004136": {
      "height": 370,
      "width": 1224
    },
    "004137": {
      "height": 375,
      "width": 1242
    },
    "004138": {
      "height": 376,
      "width": 1241
    },
    "004139": {
      "height": 370,
      "width": 1224
    },
    "004140": {
      "height": 375,
      "width": 1242
    },
    "004141": {
      "height": 375,
      "width": 1242
    },
    "004142": {
      "height": 370,
      "width": 1224
    },
    "004143": {
      "height": 375,
      "width": 1242
    },
    "004144": {
      "height": 375,
      "width": 1242
    },
    "004145": {
      "height": 375,
      "width": 1242
    },
    "004146": {
      "height": 375,
      "width": 1242
    },
    "004147": {
      "height": 370,
      "width": 1224
    },
    "004148": {
      "height": 375,
      "width": 1242
    },
    "004149": {
      "height": 374,
      "width": 1238
    },
    "004150": {
      "height": 376,
      "width": 1241
    },
    "004151": {
      "height": 375,
      "width": 1242
    },
    "004152": {
      "height": 375,
      "width": 1242
    },
    "004153": {
      "height": 375,
      "width": 1242
    },
    "004154": {
      "height": 375,
      "width": 1242
    },
    "004155": {
      "height": 375,
      "width": 1242
    },
    "004156": {
      "height": 375,
      "width": 1242
    },
    "004157": {
      "height": 375,
      "width": 1242
    },
    "004158": {
      "height": 375,
      "width": 1242
    },
    "004159": {
      "height": 375,
      "width": 1242
    },
    "004160": {
      "height": 375,
      "width": 1242
    },
    "004161": {
      "height": 375,
      "width": 1242
    },
    "004162": {
      "height": 375,
      "width": 1242
    },
    "004163": {
      "height": 375,
      "width": 1242
    },
    "004164": {
      "height": 375,
      "width": 1242
    },
    "004165": {
      "height": 370,
      "width": 1224
    },
    "004166": {
      "height": 375,
      "width": 1242
    },
    "004167": {
      "height": 375,
      "width": 1242
    },
    "004168": {
      "height": 375,
      "width": 1242
    },
    "004169": {
      "height": 375,
      "width": 1242
    },
    "004170": {
      "height": 370,
      "width": 1224
    },
    "004171": {
      "height": 375,
      "width": 1242
    },
    "004172": {
      "height": 375,
      "width": 1242
    },
    "004173": {
      "height": 375,
      "width": 1242
    },
    "004174": {
      "height": 375,
      "width": 1242
    },
    "004175": {
      "height": 375,
      "width": 1242
    },
    "004176": {
      "height": 375,
      "width": 1242
    },
    "004177": {
      "height": 375,
      "width": 1242
    },
    "004178": {
      "height": 375,
      "width": 1242
    },
    "004179": {
      "height": 375,
      "width": 1242
    },
    "004180": {
      "height": 375,
      "width": 1242
    },
    "004181": {
      "height": 375,
      "width": 1242
    },
    "004182": {
      "height": 375,
      "width": 1242
    },
    "004183": {
      "height": 375,
      "width": 1242
    },
    "004184": {
      "height": 375,
      "width": 1242
    },
    "004185": {
      "height": 370,
      "width": 1224
    },
    "004186": {
      "height": 375,
      "width": 1242
    },
    "004187": {
      "height": 375,
      "width": 1242
    },
    "004188": {
      "height": 375,
      "width": 1242
    },
    "004189": {
      "height": 375,
      "width": 1242
    },
    "004190": {
      "height": 376,
      "width": 1241
    },
    "004191": {
      "height": 375,
      "width": 1242
    },
    "004192": {
      "height": 370,
      "width": 1224
    },
    "004193": {
      "height": 375,
      "width": 1242
    },
    "004194": {
      "height": 375,
      "width": 1242
    },
    "004195": {
      "height": 376,
      "width": 1241
    },
    "004196": {
      "height": 375,
      "width": 1242
    },
    "004197": {
      "height": 375,
      "width": 1242
    },
    "004198": {
      "height": 375,
      "width": 1242
    },
    "004199": {
      "height": 375,
      "width": 1242
    },
    "004200": {
      "height": 375,
      "width": 1242
    },
    "004201": {
      "height": 375,
      "width": 1242
    },
    "004202": {
      "height": 375,
      "width": 1242
    },
    "004203": {
      "height": 375,
      "width": 1242
    },
    "004204": {
      "height": 375,
      "width": 1242
    },
    "004205": {
      "height": 375,
      "width": 1242
    },
    "004206": {
      "height": 375,
      "width": 1242
    },
    "004207": {
      "height": 375,
      "width": 1242
    },
    "004208": {
      "height": 375,
      "width": 1242
    },
    "004209": {
      "height": 376,
      "width": 1241
    },
    "004210": {
      "height": 375,
      "width": 1242
    },
    "004211": {
      "height": 375,
      "width": 1242
    },
    "004212": {
      "height": 375,
      "width": 1242
    },
    "004213": {
      "height": 375,
      "width": 1242
    },
    "004214": {
      "height": 376,
      "width": 1241
    },
    "004215": {
      "height": 375,
      "width": 1242
    },
    "004216": {
      "height": 375,
      "width": 1242
    },
    "004217": {
      "height": 375,
      "width": 1242
    },
    "004218": {
      "height": 375,
      "width": 1242
    },
    "004219": {
      "height": 370,
      "width": 1224
    },
    "004220": {
      "height": 375,
      "width": 1242
    },
    "004221": {
      "height": 375,
      "width": 1242
    },
    "004222": {
      "height": 375,
      "width": 1242
    },
    "004223": {
      "height": 374,
      "width": 1238
    },
    "004224": {
      "height": 375,
      "width": 1242
    },
    "004225": {
      "height": 375,
      "width": 1242
    },
    "004226": {
      "height": 375,
      "width": 1242
    },
    "004227": {
      "height": 375,
      "width": 1242
    },
    "004228": {
      "height": 375,
      "width": 1242
    },
    "004229": {
      "height": 370,
      "width": 1224
    },
    "004230": {
      "height": 375,
      "width": 1242
    },
    "004231": {
      "height": 375,
      "width": 1242
    },
    "004232": {
      "height": 375,
      "width": 1242
    },
    "004233": {
      "height": 375,
      "width": 1242
    },
    "004234": {
      "height": 375,
      "width": 1242
    },
    "004235": {
      "height": 375,
      "width": 1242
    },
    "004236": {
      "height": 375,
      "width": 1242
    },
    "004237": {
      "height": 370,
      "width": 1224
    },
    "004238": {
      "height": 375,
      "width": 1242
    },
    "004239": {
      "height": 375,
      "width": 1242
    },
    "004240": {
      "height": 375,
      "width": 1242
    },
    "004241": {
      "height": 375,
      "width": 1242
    },
    "004242": {
      "height": 375,
      "width": 1242
    },
    "004243": {
      "height": 376,
      "width": 1241
    },
    "004244": {
      "height": 375,
      "width": 1242
    },
    "004245": {
      "height": 375,
      "width": 1242
    },
    "004246": {
      "height": 375,
      "width": 1242
    },
    "004247": {
      "height": 370,
      "width": 1224
    },
    "004248": {
      "height": 376,
      "width": 1241
    },
    "004249": {
      "height": 375,
      "width": 1242
    },
    "004250": {
      "height": 370,
      "width": 1224
    },
    "004251": {
      "height": 375,
      "width": 1242
    },
    "004252": {
      "height": 370,
      "width": 1224
    },
    "004253": {
      "height": 375,
      "width": 1242
    },
    "004254": {
      "height": 376,
      "width": 1241
    },
    "004255": {
      "height": 375,
      "width": 1242
    },
    "004256": {
      "height": 375,
      "width": 1242
    },
    "004257": {
      "height": 375,
      "width": 1242
    },
    "004258": {
      "height": 375,
      "width": 1242
    },
    "004259": {
      "height": 375,
      "width": 1242
    },
    "004260": {
      "height": 375,
      "width": 1242
    },
    "004261": {
      "height": 370,
      "width": 1224
    },
    "004262": {
      "height": 370,
      "width": 1224
    },
    "004263": {
      "height": 375,
      "width": 1242
    },
    "004264": {
      "height": 375,
      "width": 1242
    },
    "004265": {
      "height": 375,
      "width": 1242
    },
    "004266": {
      "height": 375,
      "width": 1242
    },
    "004267": {
      "height": 375,
      "width": 1242
    },
    "004268": {
      "height": 370,
      "width": 1224
    },
    "004269": {
      "height": 375,
      "width": 1242
    },
    "004270": {
      "height": 376,
      "width": 1241
    },
    "004271": {
      "height": 374,
      "width": 1238
    },
    "004272": {
      "height": 375,
      "width": 1242
    },
    "004273": {
      "height": 375,
      "width": 1242
    },
    "004274": {
      "height": 375,
      "width": 1242
    },
    "004275": {
      "height": 376,
      "width": 1241
    },
    "004276": {
      "height": 374,
      "width": 1238
    },
    "004277": {
      "height": 376,
      "width": 1241
    },
    "004278": {
      "height": 375,
      "width": 1242
    },
    "004279": {
      "height": 375,
      "width": 1242
    },
    "004280": {
      "height": 375,
      "width": 1242
    },
    "004281": {
      "height": 375,
      "width": 1242
    },
    "004282": {
      "height": 375,
      "width": 1242
    },
    "004283": {
      "height": 370,
      "width": 1224
    },
    "004284": {
      "height": 375,
      "width": 1242
    },
    "004285": {
      "height": 375,
      "width": 1242
    },
    "004286": {
      "height": 375,
      "width": 1242
    },
    "004287": {
      "height": 375,
      "width": 1242
    },
    "004288": {
      "height": 376,
      "width": 1241
    },
    "004289": {
      "height": 375,
      "width": 1242
    },
    "004290": {
      "height": 375,
      "width": 1242
    },
    "004291": {
      "height": 370,
      "width": 1224
    },
    "004292": {
      "height": 375,
      "width": 1242
    },
    "004293": {
      "height": 375,
      "width": 1242
    },
    "004294": {
      "height": 375,
      "width": 1242
    },
    "004295": {
      "height": 376,
      "width": 1241
    },
    "004296": {
      "height": 375,
      "width": 1242
    },
    "004297": {
      "height": 375,
      "width": 1242
    },
    "004298": {
      "height": 374,
      "width": 1238
    },
    "004299": {
      "height": 375,
      "width": 1242
    },
    "004300": {
      "height": 375,
      "width": 1242
    },
    "004301": {
      "height": 375,
      "width": 1242
    },
    "004302": {
      "height": 375,
      "width": 1242
    },
    "004303": {
      "height": 375,
      "width": 1242
    },
    "004304": {
      "height": 375,
      "width": 1242
    },
    "004305": {
      "height": 375,
      "width": 1242
    },
    "004306": {
      "height": 375,
      "width": 1242
    },
    "004307": {
      "height": 375,
      "width": 1242
    },
    "004308": {
      "height": 370,
      "width": 1224
    },
    "004309": {
      "height": 375,
      "width": 1242
    },
    "004310": {
      "height": 374,
      "width": 1238
    },
    "004311": {
      "height": 375,
      "width": 1242
    },
    "004312": {
      "height": 375,
      "width": 1242
    },
    "004313": {
      "height": 375,
      "width": 1242
    },
    "004314": {
      "height": 375,
      "width": 1242
    },
    "004315": {
      "height": 375,
      "width": 1242
    },
    "004316": {
      "height": 375,
      "width": 1242
    },
    "004317": {
      "height": 375,
      "width": 1242
    },
    "004318": {
      "height": 376,
      "width": 1241
    },
    "004319": {
      "height": 375,
      "width": 1242
    },
    "004320": {
      "height": 374,
      "width": 1238
    },
    "004321": {
      "height": 374,
      "width": 1238
    },
    "004322": {
      "height": 375,
      "width": 1242
    },
    "004323": {
      "height": 370,
      "width": 1224
    },
    "004324": {
      "height": 375,
      "width": 1242
    },
    "004325": {
      "height": 375,
      "width": 1242
    },
    "004326": {
      "height": 375,
      "width": 1242
    },
    "004327": {
      "height": 374,
      "width": 1238
    },
    "004328": {
      "height": 375,
      "width": 1242
    },
    "004329": {
      "height": 375,
      "width": 1242
    },
    "004330": {
      "height": 370,
      "width": 1224
    },
    "004331": {
      "height": 375,
      "width": 1242
    },
    "004332": {
      "height": 375,
      "width": 1242
    },
    "004333": {
      "height": 375,
      "width": 1242
    },
    "004334": {
      "height": 375,
      "width": 1242
    },
    "004335": {
      "height": 375,
      "width": 1242
    },
    "004336": {
      "height": 375,
      "width": 1242
    },
    "004337": {
      "height": 375,
      "width": 1242
    },
    "004338": {
      "height": 370,
      "width": 1224
    },
    "004339": {
      "height": 375,
      "width": 1242
    },
    "004340": {
      "height": 375,
      "width": 1242
    },
    "004341": {
      "height": 375,
      "width": 1242
    },
    "004342": {
      "height": 376,
      "width": 1241
    },
    "004343": {
      "height": 374,
      "width": 1238
    },
    "004344": {
      "height": 375,
      "width": 1242
    },
    "004345": {
      "height": 375,
      "width": 1242
    },
    "004346": {
      "height": 370,
      "width": 1224
    },
    "004347": {
      "height": 375,
      "width": 1242
    },
    "004348": {
      "height": 375,
      "width": 1242
    },
    "004349": {
      "height": 374,
      "width": 1238
    },
    "004350": {
      "height": 375,
      "width": 1242
    },
    "004351": {
      "height": 375,
      "width": 1242
    },
    "004352": {
      "height": 375,
      "width": 1242
    },
    "004353": {
      "height": 375,
      "width": 1242
    },
    "004354": {
      "height": 375,
      "width": 1242
    },
    "004355": {
      "height": 375,
      "width": 1242
    },
    "004356": {
      "height": 375,
      "width": 1242
    },
    "004357": {
      "height": 375,
      "width": 1242
    },
    "004358": {
      "height": 375,
      "width": 1242
    },
    "004359": {
      "height": 375,
      "width": 1242
    },
    "004360": {
      "height": 375,
      "width": 1242
    },
    "004361": {
      "height": 370,
      "width": 1224
    },
    "004362": {
      "height": 370,
      "width": 1224
    },
    "004363": {
      "height": 375,
      "width": 1242
    },
    "004364": {
      "height": 375,
      "width": 1242
    },
    "004365": {
      "height": 375,
      "width": 1242
    },
    "004366": {
      "height": 375,
      "width": 1242
    },
    "004367": {
      "height": 370,
      "width": 1224
    },
    "004368": {
      "height": 375,
      "width": 1242
    },
    "004369": {
      "height": 375,
      "width": 1242
    },
    "004370": {
      "height": 375,
      "width": 1242
    },
    "004371": {
      "height": 375,
      "width": 1242
    },
    "004372": {
      "height": 375,
      "width": 1242
    },
    "004373": {
      "height": 375,
      "width": 1242
    },
    "004374": {
      "height": 375,
      "width": 1242
    },
    "004375": {
      "height": 374,
      "width": 1238
    },
    "004376": {
      "height": 375,
      "width": 1242
    },
    "004377": {
      "height": 375,
      "width": 1242
    },
    "004378": {
      "height": 375,
      "width": 1242
    },
    "004379": {
      "height": 375,
      "width": 1242
    },
    "004380": {
      "height": 375,
      "width": 1242
    },
    "004381": {
      "height": 375,
      "width": 1242
    },
    "004382": {
      "height": 375,
      "width": 1242
    },
    "004383": {
      "height": 375,
      "width": 1242
    },
    "004384": {
      "height": 370,
      "width": 1224
    },
    "004385": {
      "height": 374,
      "width": 1238
    },
    "004386": {
      "height": 375,
      "width": 1242
    },
    "004387": {
      "height": 375,
      "width": 1242
    },
    "004388": {
      "height": 375,
      "width": 1242
    },
    "004389": {
      "height": 370,
      "width": 1224
    },
    "004390": {
      "height": 375,
      "width": 1242
    },
    "004391": {
      "height": 375,
      "width": 1242
    },
    "004392": {
      "height": 375,
      "width": 1242
    },
    "004393": {
      "height": 375,
      "width": 1242
    },
    "004394": {
      "height": 375,
      "width": 1242
    },
    "004395": {
      "height": 375,
      "width": 1242
    },
    "004396": {
      "height": 375,
      "width": 1242
    },
    "004397": {
      "height": 375,
      "width": 1242
    },
    "004398": {
      "height": 375,
      "width": 1242
    },
    "004399": {
      "height": 375,
      "width": 1242
    },
    "004400": {
      "height": 370,
      "width": 1224
    },
    "004401": {
      "height": 375,
      "width": 1242
    },
    "004402": {
      "height": 375,
      "width": 1242
    },
    "004403": {
      "height": 375,
      "width": 1242
    },
    "004404": {
      "height": 370,
      "width": 1224
    },
    "004405": {
      "height": 375,
      "width": 1242
    },
    "004406": {
      "height": 375,
      "width": 1242
    },
    "004407": {
      "height": 375,
      "width": 1242
    },
    "004408": {
      "height": 375,
      "width": 1242
    },
    "004409": {
      "height": 375,
      "width": 1242
    },
    "004410": {
      "height": 370,
      "width": 1224
    },
    "004411": {
      "height": 375,
      "width": 1242
    },
    "004412": {
      "height": 374,
      "width": 1238
    },
    "004413": {
      "height": 375,
      "width": 1242
    },
    "004414": {
      "height": 370,
      "width": 1224
    },
    "004415": {
      "height": 375,
      "width": 1242
    },
    "004416": {
      "height": 375,
      "width": 1242
    },
    "004417": {
      "height": 370,
      "width": 1224
    },
    "004418": {
      "height": 375,
      "width": 1242
    },
    "004419": {
      "height": 375,
      "width": 1242
    },
    "004420": {
      "height": 375,
      "width": 1242
    },
    "004421": {
      "height": 375,
      "width": 1242
    },
    "004422": {
      "height": 370,
      "width": 1224
    },
    "004423": {
      "height": 374,
      "width": 1238
    },
    "004424": {
      "height": 375,
      "width": 1242
    },
    "004425": {
      "height": 374,
      "width": 1238
    },
    "004426": {
      "height": 375,
      "width": 1242
    },
    "004427": {
      "height": 375,
      "width": 1242
    },
    "004428": {
      "height": 375,
      "width": 1242
    },
    "004429": {
      "height": 375,
      "width": 1242
    },
    "004430": {
      "height": 375,
      "width": 1242
    },
    "004431": {
      "height": 375,
      "width": 1242
    },
    "004432": {
      "height": 375,
      "width": 1242
    },
    "004433": {
      "height": 375,
      "width": 1242
    },
    "004434": {
      "height": 375,
      "width": 1242
    },
    "004435": {
      "height": 375,
      "width": 1242
    },
    "004436": {
      "height": 375,
      "width": 1242
    },
    "004437": {
      "height": 375,
      "width": 1242
    },
    "004438": {
      "height": 375,
      "width": 1242
    },
    "004439": {
      "height": 375,
      "width": 1242
    },
    "004440": {
      "height": 375,
      "width": 1242
    },
    "004441": {
      "height": 375,
      "width": 1242
    },
    "004442": {
      "height": 375,
      "width": 1242
    },
    "004443": {
      "height": 374,
      "width": 1238
    },
    "004444": {
      "height": 375,
      "width": 1242
    },
    "004445": {
      "height": 375,
      "width": 1242
    },
    "004446": {
      "height": 375,
      "width": 1242
    },
    "004447": {
      "height": 376,
      "width": 1241
    },
    "004448": {
      "height": 375,
      "width": 1242
    },
    "004449": {
      "height": 370,
      "width": 1224
    },
    "004450": {
      "height": 375,
      "width": 1242
    },
    "004451": {
      "height": 375,
      "width": 1242
    },
    "004452": {
      "height": 375,
      "width": 1242
    },
    "004453": {
      "height": 375,
      "width": 1242
    },
    "004454": {
      "height": 376,
      "width": 1241
    },
    "004455": {
      "height": 375,
      "width": 1242
    },
    "004456": {
      "height": 375,
      "width": 1242
    },
    "004457": {
      "height": 375,
      "width": 1242
    },
    "004458": {
      "height": 375,
      "width": 1242
    },
    "004459": {
      "height": 375,
      "width": 1242
    },
    "004460": {
      "height": 375,
      "width": 1242
    },
    "004461": {
      "height": 375,
      "width": 1242
    },
    "004462": {
      "height": 375,
      "width": 1242
    },
    "004463": {
      "height": 374,
      "width": 1238
    },
    "004464": {
      "height": 375,
      "width": 1242
    },
    "004465": {
      "height": 375,
      "width": 1242
    },
    "004466": {
      "height": 375,
      "width": 1242
    },
    "004467": {
      "height": 375,
      "width": 1242
    },
    "004468": {
      "height": 375,
      "width": 1242
    },
    "004469": {
      "height": 370,
      "width": 1224
    },
    "004470": {
      "height": 370,
      "width": 1224
    },
    "004471": {
      "height": 375,
      "width": 1242
    },
    "004472": {
      "height": 375,
      "width": 1242
    },
    "004473": {
      "height": 375,
      "width": 1242
    },
    "004474": {
      "height": 374,
      "width": 1238
    },
    "004475": {
      "height": 376,
      "width": 1241
    },
    "004476": {
      "height": 375,
      "width": 1242
    },
    "004477": {
      "height": 375,
      "width": 1242
    },
    "004478": {
      "height": 375,
      "width": 1242
    },
    "004479": {
      "height": 375,
      "width": 1242
    },
    "004480": {
      "height": 375,
      "width": 1242
    },
    "004481": {
      "height": 376,
      "width": 1241
    },
    "004482": {
      "height": 370,
      "width": 1224
    },
    "004483": {
      "height": 375,
      "width": 1242
    },
    "004484": {
      "height": 375,
      "width": 1242
    },
    "004485": {
      "height": 375,
      "width": 1242
    },
    "004486": {
      "height": 375,
      "width": 1242
    },
    "004487": {
      "height": 374,
      "width": 1238
    },
    "004488": {
      "height": 375,
      "width": 1242
    },
    "004489": {
      "height": 375,
      "width": 1242
    },
    "004490": {
      "height": 375,
      "width": 1242
    },
    "004491": {
      "height": 375,
      "width": 1242
    },
    "004492": {
      "height": 375,
      "width": 1242
    },
    "004493": {
      "height": 375,
      "width": 1242
    },
    "004494": {
      "height": 374,
      "width": 1238
    },
    "004495": {
      "height": 375,
      "width": 1242
    },
    "004496": {
      "height": 375,
      "width": 1242
    },
    "004497": {
      "height": 375,
      "width": 1242
    },
    "004498": {
      "height": 375,
      "width": 1242
    },
    "004499": {
      "height": 375,
      "width": 1242
    },
    "004500": {
      "height": 375,
      "width": 1242
    },
    "004501": {
      "height": 375,
      "width": 1242
    },
    "004502": {
      "height": 375,
      "width": 1242
    },
    "004503": {
      "height": 370,
      "width": 1224
    },
    "004504": {
      "height": 375,
      "width": 1242
    },
    "004505": {
      "height": 375,
      "width": 1242
    },
    "004506": {
      "height": 375,
      "width": 1242
    },
    "004507": {
      "height": 375,
      "width": 1242
    },
    "004508": {
      "height": 370,
      "width": 1224
    },
    "004509": {
      "height": 370,
      "width": 1224
    },
    "004510": {
      "height": 375,
      "width": 1242
    },
    "004511": {
      "height": 376,
      "width": 1241
    },
    "004512": {
      "height": 375,
      "width": 1242
    },
    "004513": {
      "height": 375,
      "width": 1242
    },
    "004514": {
      "height": 370,
      "width": 1224
    },
    "004515": {
      "height": 375,
      "width": 1242
    },
    "004516": {
      "height": 375,
      "width": 1242
    },
    "004517": {
      "height": 376,
      "width": 1241
    },
    "004518": {
      "height": 375,
      "width": 1242
    },
    "004519": {
      "height": 376,
      "width": 1241
    },
    "004520": {
      "height": 375,
      "width": 1242
    },
    "004521": {
      "height": 370,
      "width": 1224
    },
    "004522": {
      "height": 375,
      "width": 1242
    },
    "004523": {
      "height": 375,
      "width": 1242
    },
    "004524": {
      "height": 375,
      "width": 1242
    },
    "004525": {
      "height": 375,
      "width": 1242
    },
    "004526": {
      "height": 375,
      "width": 1242
    },
    "004527": {
      "height": 370,
      "width": 1224
    },
    "004528": {
      "height": 375,
      "width": 1242
    },
    "004529": {
      "height": 375,
      "width": 1242
    },
    "004530": {
      "height": 375,
      "width": 1242
    },
    "004531": {
      "height": 375,
      "width": 1242
    },
    "004532": {
      "height": 374,
      "width": 1238
    },
    "004533": {
      "height": 375,
      "width": 1242
    },
    "004534": {
      "height": 375,
      "width": 1242
    },
    "004535": {
      "height": 375,
      "width": 1242
    },
    "004536": {
      "height": 370,
      "width": 1224
    },
    "004537": {
      "height": 375,
      "width": 1242
    },
    "004538": {
      "height": 375,
      "width": 1242
    },
    "004539": {
      "height": 375,
      "width": 1242
    },
    "004540": {
      "height": 375,
      "width": 1242
    },
    "004541": {
      "height": 375,
      "width": 1242
    },
    "004542": {
      "height": 375,
      "width": 1242
    },
    "004543": {
      "height": 370,
      "width": 1224
    },
    "004544": {
      "height": 374,
      "width": 1238
    },
    "004545": {
      "height": 375,
      "width": 1242
    },
    "004546": {
      "height": 370,
      "width": 1224
    },
    "004547": {
      "height": 375,
      "width": 1242
    },
    "004548": {
      "height": 375,
      "width": 1242
    },
    "004549": {
      "height": 375,
      "width": 1242
    },
    "004550": {
      "height": 375,
      "width": 1242
    },
    "004551": {
      "height": 375,
      "width": 1242
    },
    "004552": {
      "height": 375,
      "width": 1242
    },
    "004553": {
      "height": 375,
      "width": 1242
    },
    "004554": {
      "height": 375,
      "width": 1242
    },
    "004555": {
      "height": 375,
      "width": 1242
    },
    "004556": {
      "height": 375,
      "width": 1242
    },
    "004557": {
      "height": 375,
      "width": 1242
    },
    "004558": {
      "height": 375,
      "width": 1242
    },
    "004559": {
      "height": 375,
      "width": 1242
    },
    "004560": {
      "height": 375,
      "width": 1242
    },
    "004561": {
      "height": 375,
      "width": 1242
    },
    "004562": {
      "height": 374,
      "width": 1238
    },
    "004563": {
      "height": 375,
      "width": 1242
    },
    "004564": {
      "height": 370,
      "width": 1224
    },
    "004565": {
      "height": 375,
      "width": 1242
    },
    "004566": {
      "height": 375,
      "width": 1242
    },
    "004567": {
      "height": 370,
      "width": 1224
    },
    "004568": {
      "height": 375,
      "width": 1242
    },
    "004569": {
      "height": 370,
      "width": 1224
    },
    "004570": {
      "height": 375,
      "width": 1242
    },
    "004571": {
      "height": 374,
      "width": 1238
    },
    "004572": {
      "height": 375,
      "width": 1242
    },
    "004573": {
      "height": 375,
      "width": 1242
    },
    "004574": {
      "height": 375,
      "width": 1242
    },
    "004575": {
      "height": 375,
      "width": 1242
    },
    "004576": {
      "height": 375,
      "width": 1242
    },
    "004577": {
      "height": 375,
      "width": 1242
    },
    "004578": {
      "height": 375,
      "width": 1242
    },
    "004579": {
      "height": 375,
      "width": 1242
    },
    "004580": {
      "height": 370,
      "width": 1224
    },
    "004581": {
      "height": 375,
      "width": 1242
    },
    "004582": {
      "height": 375,
      "width": 1242
    },
    "004583": {
      "height": 370,
      "width": 1224
    },
    "004584": {
      "height": 370,
      "width": 1224
    },
    "004585": {
      "height": 375,
      "width": 1242
    },
    "004586": {
      "height": 375,
      "width": 1242
    },
    "004587": {
      "height": 375,
      "width": 1242
    },
    "004588": {
      "height": 374,
      "width": 1238
    },
    "004589": {
      "height": 375,
      "width": 1242
    },
    "004590": {
      "height": 375,
      "width": 1242
    },
    "004591": {
      "height": 375,
      "width": 1242
    },
    "004592": {
      "height": 375,
      "width": 1242
    },
    "004593": {
      "height": 375,
      "width": 1242
    },
    "004594": {
      "height": 370,
      "width": 1224
    },
    "004595": {
      "height": 370,
      "width": 1224
    },
    "004596": {
      "height": 375,
      "width": 1242
    },
    "004597": {
      "height": 370,
      "width": 1224
    },
    "004598": {
      "height": 375,
      "width": 1242
    },
    "004599": {
      "height": 375,
      "width": 1242
    },
    "004600": {
      "height": 375,
      "width": 1242
    },
    "004601": {
      "height": 375,
      "width": 1242
    },
    "004602": {
      "height": 375,
      "width": 1242
    },
    "004603": {
      "height": 375,
      "width": 1242
    },
    "004604": {
      "height": 375,
      "width": 1242
    },
    "004605": {
      "height": 370,
      "width": 1224
    },
    "004606": {
      "height": 375,
      "width": 1242
    },
    "004607": {
      "height": 375,
      "width": 1242
    },
    "004608": {
      "height": 375,
      "width": 1242
    },
    "004609": {
      "height": 375,
      "width": 1242
    },
    "004610": {
      "height": 375,
      "width": 1242
    },
    "004611": {
      "height": 375,
      "width": 1242
    },
    "004612": {
      "height": 375,
      "width": 1242
    },
    "004613": {
      "height": 375,
      "width": 1242
    },
    "004614": {
      "height": 375,
      "width": 1242
    },
    "004615": {
      "height": 375,
      "width": 1242
    },
    "004616": {
      "height": 375,
      "width": 1242
    },
    "004617": {
      "height": 375,
      "width": 1242
    },
    "004618": {
      "height": 375,
      "width": 1242
    },
    "004619": {
      "height": 370,
      "width": 1224
    },
    "004620": {
      "height": 375,
      "width": 1242
    },
    "004621": {
      "height": 375,
      "width": 1242
    },
    "004622": {
      "height": 375,
      "width": 1242
    },
    "004623": {
      "height": 375,
      "width": 1242
    },
    "004624": {
      "height": 375,
      "width": 1242
    },
    "004625": {
      "height": 375,
      "width": 1242
    },
    "004626": {
      "height": 375,
      "width": 1242
    },
    "004627": {
      "height": 370,
      "width": 1224
    },
    "004628": {
      "height": 375,
      "width": 1242
    },
    "004629": {
      "height": 375,
      "width": 1242
    },
    "004630": {
      "height": 375,
      "width": 1242
    },
    "004631": {
      "height": 375,
      "width": 1242
    },
    "004632": {
      "height": 375,
      "width": 1242
    },
    "004633": {
      "height": 375,
      "width": 1242
    },
    "004634": {
      "height": 370,
      "width": 1224
    },
    "004635": {
      "height": 375,
      "width": 1242
    },
    "004636": {
      "height": 375,
      "width": 1242
    },
    "004637": {
      "height": 375,
      "width": 1242
    },
    "004638": {
      "height": 375,
      "width": 1242
    },
    "004639": {
      "height": 375,
      "width": 1242
    },
    "004640": {
      "height": 375,
      "width": 1242
    },
    "004641": {
      "height": 375,
      "width": 1242
    },
    "004642": {
      "height": 375,
      "width": 1242
    },
    "004643": {
      "height": 375,
      "width": 1242
    },
    "004644": {
      "height": 375,
      "width": 1242
    },
    "004645": {
      "height": 375,
      "width": 1242
    },
    "004646": {
      "height": 370,
      "width": 1224
    },
    "004647": {
      "height": 375,
      "width": 1242
    },
    "004648": {
      "height": 376,
      "width": 1241
    },
    "004649": {
      "height": 375,
      "width": 1242
    },
    "004650": {
      "height": 375,
      "width": 1242
    },
    "004651": {
      "height": 375,
      "width": 1242
    },
    "004652": {
      "height": 375,
      "width": 1242
    },
    "004653": {
      "height": 375,
      "width": 1242
    },
    "004654": {
      "height": 375,
      "width": 1242
    },
    "004655": {
      "height": 375,
      "width": 1242
    },
    "004656": {
      "height": 375,
      "width": 1242
    },
    "004657": {
      "height": 374,
      "width": 1238
    },
    "004658": {
      "height": 375,
      "width": 1242
    },
    "004659": {
      "height": 375,
      "width": 1242
    },
    "004660": {
      "height": 375,
      "width": 1242
    },
    "004661": {
      "height": 375,
      "width": 1242
    },
    "004662": {
      "height": 375,
      "width": 1242
    },
    "004663": {
      "height": 375,
      "width": 1242
    },
    "004664": {
      "height": 375,
      "width": 1242
    },
    "004665": {
      "height": 374,
      "width": 1238
    },
    "004666": {
      "height": 375,
      "width": 1242
    },
    "004667": {
      "height": 375,
      "width": 1242
    },
    "004668": {
      "height": 375,
      "width": 1242
    },
    "004669": {
      "height": 375,
      "width": 1242
    },
    "004670": {
      "height": 375,
      "width": 1242
    },
    "004671": {
      "height": 375,
      "width": 1242
    },
    "004672": {
      "height": 375,
      "width": 1242
    },
    "004673": {
      "height": 375,
      "width": 1242
    },
    "004674": {
      "height": 370,
      "width": 1224
    },
    "004675": {
      "height": 375,
      "width": 1242
    },
    "004676": {
      "height": 375,
      "width": 1242
    },
    "004677": {
      "height": 375,
      "width": 1242
    },
    "004678": {
      "height": 375,
      "width": 1242
    },
    "004679": {
      "height": 370,
      "width": 1224
    },
    "004680": {
      "height": 375,
      "width": 1242
    },
    "004681": {
      "height": 375,
      "width": 1242
    },
    "004682": {
      "height": 376,
      "width": 1241
    },
    "004683": {
      "height": 375,
      "width": 1242
    },
    "004684": {
      "height": 375,
      "width": 1242
    },
    "004685": {
      "height": 375,
      "width": 1242
    },
    "004686": {
      "height": 376,
      "width": 1241
    },
    "004687": {
      "height": 375,
      "width": 1242
    },
    "004688": {
      "height": 375,
      "width": 1242
    },
    "004689": {
      "height": 375,
      "width": 1242
    },
    "004690": {
      "height": 375,
      "width": 1242
    },
    "004691": {
      "height": 375,
      "width": 1242
    },
    "004692": {
      "height": 370,
      "width": 1224
    },
    "004693": {
      "height": 375,
      "width": 1242
    },
    "004694": {
      "height": 376,
      "width": 1241
    },
    "004695": {
      "height": 375,
      "width": 1242
    },
    "004696": {
      "height": 370,
      "width": 1224
    },
    "004697": {
      "height": 375,
      "width": 1242
    },
    "004698": {
      "height": 375,
      "width": 1242
    },
    "004699": {
      "height": 375,
      "width": 1242
    },
    "004700": {
      "height": 375,
      "width": 1242
    },
    "004701": {
      "height": 375,
      "width": 1242
    },
    "004702": {
      "height": 375,
      "width": 1242
    },
    "004703": {
      "height": 375,
      "width": 1242
    },
    "004704": {
      "height": 370,
      "width": 1224
    },
    "004705": {
      "height": 375,
      "width": 1242
    },
    "004706": {
      "height": 375,
      "width": 1242
    },
    "004707": {
      "height": 375,
      "width": 1242
    },
    "004708": {
      "height": 375,
      "width": 1242
    },
    "004709": {
      "height": 375,
      "width": 1242
    },
    "004710": {
      "height": 375,
      "width": 1242
    },
    "004711": {
      "height": 375,
      "width": 1242
    },
    "004712": {
      "height": 375,
      "width": 1242
    },
    "004713": {
      "height": 375,
      "width": 1242
    },
    "004714": {
      "height": 375,
      "width": 1242
    },
    "004715": {
      "height": 376,
      "width": 1241
    },
    "004716": {
      "height": 375,
      "width": 1242
    },
    "004717": {
      "height": 375,
      "width": 1242
    },
    "004718": {
      "height": 375,
      "width": 1242
    },
    "004719": {
      "height": 374,
      "width": 1238
    },
    "004720": {
      "height": 370,
      "width": 1224
    },
    "004721": {
      "height": 375,
      "width": 1242
    },
    "004722": {
      "height": 374,
      "width": 1238
    },
    "004723": {
      "height": 370,
      "width": 1224
    },
    "004724": {
      "height": 370,
      "width": 1224
    },
    "004725": {
      "height": 375,
      "width": 1242
    },
    "004726": {
      "height": 375,
      "width": 1242
    },
    "004727": {
      "height": 375,
      "width": 1242
    },
    "004728": {
      "height": 375,
      "width": 1242
    },
    "004729": {
      "height": 375,
      "width": 1242
    },
    "004730": {
      "height": 375,
      "width": 1242
    },
    "004731": {
      "height": 370,
      "width": 1224
    },
    "004732": {
      "height": 375,
      "width": 1242
    },
    "004733": {
      "height": 370,
      "width": 1224
    },
    "004734": {
      "height": 375,
      "width": 1242
    },
    "004735": {
      "height": 375,
      "width": 1242
    },
    "004736": {
      "height": 375,
      "width": 1242
    },
    "004737": {
      "height": 375,
      "width": 1242
    },
    "004738": {
      "height": 375,
      "width": 1242
    },
    "004739": {
      "height": 375,
      "width": 1242
    },
    "004740": {
      "height": 374,
      "width": 1238
    },
    "004741": {
      "height": 375,
      "width": 1242
    },
    "004742": {
      "height": 376,
      "width": 1241
    },
    "004743": {
      "height": 376,
      "width": 1241
    },
    "004744": {
      "height": 375,
      "width": 1242
    },
    "004745": {
      "height": 374,
      "width": 1238
    },
    "004746": {
      "height": 375,
      "width": 1242
    },
    "004747": {
      "height": 370,
      "width": 1224
    },
    "004748": {
      "height": 375,
      "width": 1242
    },
    "004749": {
      "height": 375,
      "width": 1242
    },
    "004750": {
      "height": 375,
      "width": 1242
    },
    "004751": {
      "height": 375,
      "width": 1242
    },
    "004752": {
      "height": 375,
      "width": 1242
    },
    "004753": {
      "height": 375,
      "width": 1242
    },
    "004754": {
      "height": 375,
      "width": 1242
    },
    "004755": {
      "height": 375,
      "width": 1242
    },
    "004756": {
      "height": 376,
      "width": 1241
    },
    "004757": {
      "height": 375,
      "width": 1242
    },
    "004758": {
      "height": 375,
      "width": 1242
    },
    "004759": {
      "height": 375,
      "width": 1242
    },
    "004760": {
      "height": 375,
      "width": 1242
    },
    "004761": {
      "height": 375,
      "width": 1242
    },
    "004762": {
      "height": 370,
      "width": 1224
    },
    "004763": {
      "height": 375,
      "width": 1242
    },
    "004764": {
      "height": 375,
      "width": 1242
    },
    "004765": {
      "height": 375,
      "width": 1242
    },
    "004766": {
      "height": 375,
      "width": 1242
    },
    "004767": {
      "height": 375,
      "width": 1242
    },
    "004768": {
      "height": 375,
      "width": 1242
    },
    "004769": {
      "height": 376,
      "width": 1241
    },
    "004770": {
      "height": 375,
      "width": 1242
    },
    "004771": {
      "height": 375,
      "width": 1242
    },
    "004772": {
      "height": 375,
      "width": 1242
    },
    "004773": {
      "height": 375,
      "width": 1242
    },
    "004774": {
      "height": 375,
      "width": 1242
    },
    "004775": {
      "height": 375,
      "width": 1242
    },
    "004776": {
      "height": 375,
      "width": 1242
    },
    "004777": {
      "height": 374,
      "width": 1238
    },
    "004778": {
      "height": 375,
      "width": 1242
    },
    "004779": {
      "height": 375,
      "width": 1242
    },
    "004780": {
      "height": 370,
      "width": 1224
    },
    "004781": {
      "height": 375,
      "width": 1242
    },
    "004782": {
      "height": 375,
      "width": 1242
    },
    "004783": {
      "height": 375,
      "width": 1242
    },
    "004784": {
      "height": 375,
      "width": 1242
    },
    "004785": {
      "height": 375,
      "width": 1242
    },
    "004786": {
      "height": 370,
      "width": 1224
    },
    "004787": {
      "height": 375,
      "width": 1242
    },
    "004788": {
      "height": 370,
      "width": 1224
    },
    "004789": {
      "height": 374,
      "width": 1238
    },
    "004790": {
      "height": 375,
      "width": 1242
    },
    "004791": {
      "height": 375,
      "width": 1242
    },
    "004792": {
      "height": 375,
      "width": 1242
    },
    "004793": {
      "height": 375,
      "width": 1242
    },
    "004794": {
      "height": 374,
      "width": 1238
    },
    "004795": {
      "height": 375,
      "width": 1242
    },
    "004796": {
      "height": 375,
      "width": 1242
    },
    "004797": {
      "height": 370,
      "width": 1224
    },
    "004798": {
      "height": 375,
      "width": 1242
    },
    "004799": {
      "height": 375,
      "width": 1242
    },
    "004800": {
      "height": 375,
      "width": 1242
    },
    "004801": {
      "height": 375,
      "width": 1242
    },
    "004802": {
      "height": 375,
      "width": 1242
    },
    "004803": {
      "height": 375,
      "width": 1242
    },
    "004804": {
      "height": 370,
      "width": 1224
    },
    "004805": {
      "height": 374,
      "width": 1238
    },
    "004806": {
      "height": 375,
      "width": 1242
    },
    "004807": {
      "height": 375,
      "width": 1242
    },
    "004808": {
      "height": 375,
      "width": 1242
    },
    "004809": {
      "height": 375,
      "width": 1242
    },
    "004810": {
      "height": 375,
      "width": 1242
    },
    "004811": {
      "height": 375,
      "width": 1242
    },
    "004812": {
      "height": 375,
      "width": 1242
    },
    "004813": {
      "height": 375,
      "width": 1242
    },
    "004814": {
      "height": 375,
      "width": 1242
    },
    "004815": {
      "height": 375,
      "width": 1242
    },
    "004816": {
      "height": 374,
      "width": 1238
    },
    "004817": {
      "height": 375,
      "width": 1242
    },
    "004818": {
      "height": 375,
      "width": 1242
    },
    "004819": {
      "height": 375,
      "width": 1242
    },
    "004820": {
      "height": 375,
      "width": 1242
    },
    "004821": {
      "height": 375,
      "width": 1242
    },
    "004822": {
      "height": 375,
      "width": 1242
    },
    "004823": {
      "height": 375,
      "width": 1242
    },
    "004824": {
      "height": 375,
      "width": 1242
    },
    "004825": {
      "height": 374,
      "width": 1238
    },
    "004826": {
      "height": 375,
      "width": 1242
    },
    "004827": {
      "height": 375,
      "width": 1242
    },
    "004828": {
      "height": 375,
      "width": 1242
    },
    "004829": {
      "height": 375,
      "width": 1242
    },
    "004830": {
      "height": 375,
      "width": 1242
    },
    "004831": {
      "height": 375,
      "width": 1242
    },
    "004832": {
      "height": 376,
      "width": 1241
    },
    "004833": {
      "height": 375,
      "width": 1242
    },
    "004834": {
      "height": 375,
      "width": 1242
    },
    "004835": {
      "height": 375,
      "width": 1242
    },
    "004836": {
      "height": 375,
      "width": 1242
    },
    "004837": {
      "height": 375,
      "width": 1242
    },
    "004838": {
      "height": 375,
      "width": 1242
    },
    "004839": {
      "height": 375,
      "width": 1242
    },
    "004840": {
      "height": 375,
      "width": 1242
    },
    "004841": {
      "height": 375,
      "width": 1242
    },
    "004842": {
      "height": 375,
      "width": 1242
    },
    "004843": {
      "height": 376,
      "width": 1241
    },
    "004844": {
      "height": 370,
      "width": 1224
    },
    "004845": {
      "height": 375,
      "width": 1242
    },
    "004846": {
      "height": 375,
      "width": 1242
    },
    "004847": {
      "height": 375,
      "width": 1242
    },
    "004848": {
      "height": 375,
      "width": 1242
    },
    "004849": {
      "height": 375,
      "width": 1242
    },
    "004850": {
      "height": 375,
      "width": 1242
    },
    "004851": {
      "height": 375,
      "width": 1242
    },
    "004852": {
      "height": 375,
      "width": 1242
    },
    "004853": {
      "height": 375,
      "width": 1242
    },
    "004854": {
      "height": 370,
      "width": 1224
    },
    "004855": {
      "height": 375,
      "width": 1242
    },
    "004856": {
      "height": 375,
      "width": 1242
    },
    "004857": {
      "height": 375,
      "width": 1242
    },
    "004858": {
      "height": 375,
      "width": 1242
    },
    "004859": {
      "height": 375,
      "width": 1242
    },
    "004860": {
      "height": 375,
      "width": 1242
    },
    "004861": {
      "height": 375,
      "width": 1242
    },
    "004862": {
      "height": 375,
      "width": 1242
    },
    "004863": {
      "height": 375,
      "width": 1242
    },
    "004864": {
      "height": 375,
      "width": 1242
    },
    "004865": {
      "height": 375,
      "width": 1242
    },
    "004866": {
      "height": 375,
      "width": 1242
    },
    "004867": {
      "height": 375,
      "width": 1242
    },
    "004868": {
      "height": 375,
      "width": 1242
    },
    "004869": {
      "height": 370,
      "width": 1224
    },
    "004870": {
      "height": 375,
      "width": 1242
    },
    "004871": {
      "height": 375,
      "width": 1242
    },
    "004872": {
      "height": 375,
      "width": 1242
    },
    "004873": {
      "height": 375,
      "width": 1242
    },
    "004874": {
      "height": 375,
      "width": 1242
    },
    "004875": {
      "height": 375,
      "width": 1242
    },
    "004876": {
      "height": 375,
      "width": 1242
    },
    "004877": {
      "height": 375,
      "width": 1242
    },
    "004878": {
      "height": 375,
      "width": 1242
    },
    "004879": {
      "height": 375,
      "width": 1242
    },
    "004880": {
      "height": 375,
      "width": 1242
    },
    "004881": {
      "height": 375,
      "width": 1242
    },
    "004882": {
      "height": 375,
      "width": 1242
    },
    "004883": {
      "height": 375,
      "width": 1242
    },
    "004884": {
      "height": 375,
      "width": 1242
    },
    "004885": {
      "height": 376,
      "width": 1241
    },
    "004886": {
      "height": 375,
      "width": 1242
    },
    "004887": {
      "height": 375,
      "width": 1242
    },
    "004888": {
      "height": 375,
      "width": 1242
    },
    "004889": {
      "height": 375,
      "width": 1242
    },
    "004890": {
      "height": 375,
      "width": 1242
    },
    "004891": {
      "height": 375,
      "width": 1242
    },
    "004892": {
      "height": 375,
      "width": 1242
    },
    "004893": {
      "height": 375,
      "width": 1242
    },
    "004894": {
      "height": 375,
      "width": 1242
    },
    "004895": {
      "height": 375,
      "width": 1242
    },
    "004896": {
      "height": 375,
      "width": 1242
    },
    "004897": {
      "height": 375,
      "width": 1242
    },
    "004898": {
      "height": 375,
      "width": 1242
    },
    "004899": {
      "height": 375,
      "width": 1242
    },
    "004900": {
      "height": 375,
      "width": 1242
    },
    "004901": {
      "height": 370,
      "width": 1224
    },
    "004902": {
      "height": 375,
      "width": 1242
    },
    "004903": {
      "height": 370,
      "width": 1224
    },
    "004904": {
      "height": 375,
      "width": 1242
    },
    "004905": {
      "height": 375,
      "width": 1242
    },
    "004906": {
      "height": 375,
      "width": 1242
    },
    "004907": {
      "height": 375,
      "width": 1242
    },
    "004908": {
      "height": 375,
      "width": 1242
    },
    "004909": {
      "height": 375,
      "width": 1242
    },
    "004910": {
      "height": 375,
      "width": 1242
    },
    "004911": {
      "height": 375,
      "width": 1242
    },
    "004912": {
      "height": 370,
      "width": 1224
    },
    "004913": {
      "height": 375,
      "width": 1242
    },
    "004914": {
      "height": 375,
      "width": 1242
    },
    "004915": {
      "height": 370,
      "width": 1224
    },
    "004916": {
      "height": 375,
      "width": 1242
    },
    "004917": {
      "height": 375,
      "width": 1242
    },
    "004918": {
      "height": 375,
      "width": 1242
    },
    "004919": {
      "height": 375,
      "width": 1242
    },
    "004920": {
      "height": 375,
      "width": 1242
    },
    "004921": {
      "height": 375,
      "width": 1242
    },
    "004922": {
      "height": 375,
      "width": 1242
    },
    "004923": {
      "height": 375,
      "width": 1242
    },
    "004924": {
      "height": 370,
      "width": 1224
    },
    "004925": {
      "height": 375,
      "width": 1242
    },
    "004926": {
      "height": 375,
      "width": 1242
    },
    "004927": {
      "height": 375,
      "width": 1242
    },
    "004928": {
      "height": 370,
      "width": 1224
    },
    "004929": {
      "height": 376,
      "width": 1241
    },
    "004930": {
      "height": 375,
      "width": 1242
    },
    "004931": {
      "height": 375,
      "width": 1242
    },
    "004932": {
      "height": 375,
      "width": 1242
    },
    "004933": {
      "height": 374,
      "width": 1238
    },
    "004934": {
      "height": 375,
      "width": 1242
    },
    "004935": {
      "height": 375,
      "width": 1242
    },
    "004936": {
      "height": 375,
      "width": 1242
    },
    "004937": {
      "height": 375,
      "width": 1242
    },
    "004938": {
      "height": 375,
      "width": 1242
    },
    "004939": {
      "height": 375,
      "width": 1242
    },
    "004940": {
      "height": 370,
      "width": 1224
    },
    "004941": {
      "height": 375,
      "width": 1242
    },
    "004942": {
      "height": 375,
      "width": 1242
    },
    "004943": {
      "height": 375,
      "width": 1242
    },
    "004944": {
      "height": 370,
      "width": 1224
    },
    "004945": {
      "height": 375,
      "width": 1242
    },
    "004946": {
      "height": 375,
      "width": 1242
    },
    "004947": {
      "height": 375,
      "width": 1242
    },
    "004948": {
      "height": 375,
      "width": 1242
    },
    "004949": {
      "height": 370,
      "width": 1224
    },
    "004950": {
      "height": 375,
      "width": 1242
    },
    "004951": {
      "height": 375,
      "width": 1242
    },
    "004952": {
      "height": 375,
      "width": 1242
    },
    "004953": {
      "height": 375,
      "width": 1242
    },
    "004954": {
      "height": 374,
      "width": 1238
    },
    "004955": {
      "height": 375,
      "width": 1242
    },
    "004956": {
      "height": 375,
      "width": 1242
    },
    "004957": {
      "height": 375,
      "width": 1242
    },
    "004958": {
      "height": 374,
      "width": 1238
    },
    "004959": {
      "height": 375,
      "width": 1242
    },
    "004960": {
      "height": 370,
      "width": 1224
    },
    "004961": {
      "height": 375,
      "width": 1242
    },
    "004962": {
      "height": 375,
      "width": 1242
    },
    "004963": {
      "height": 374,
      "width": 1238
    },
    "004964": {
      "height": 375,
      "width": 1242
    },
    "004965": {
      "height": 375,
      "width": 1242
    },
    "004966": {
      "height": 375,
      "width": 1242
    },
    "004967": {
      "height": 370,
      "width": 1224
    },
    "004968": {
      "height": 375,
      "width": 1242
    },
    "004969": {
      "height": 375,
      "width": 1242
    },
    "004970": {
      "height": 375,
      "width": 1242
    },
    "004971": {
      "height": 375,
      "width": 1242
    },
    "004972": {
      "height": 370,
      "width": 1224
    },
    "004973": {
      "height": 370,
      "width": 1224
    },
    "004974": {
      "height": 375,
      "width": 1242
    },
    "004975": {
      "height": 375,
      "width": 1242
    },
    "004976": {
      "height": 370,
      "width": 1224
    },
    "004977": {
      "height": 375,
      "width": 1242
    },
    "004978": {
      "height": 375,
      "width": 1242
    },
    "004979": {
      "height": 375,
      "width": 1242
    },
    "004980": {
      "height": 375,
      "width": 1242
    },
    "004981": {
      "height": 376,
      "width": 1241
    },
    "004982": {
      "height": 375,
      "width": 1242
    },
    "004983": {
      "height": 374,
      "width": 1238
    },
    "004984": {
      "height": 374,
      "width": 1238
    },
    "004985": {
      "height": 375,
      "width": 1242
    },
    "004986": {
      "height": 370,
      "width": 1224
    },
    "004987": {
      "height": 375,
      "width": 1242
    },
    "004988": {
      "height": 375,
      "width": 1242
    },
    "004989": {
      "height": 374,
      "width": 1238
    },
    "004990": {
      "height": 375,
      "width": 1242
    },
    "004991": {
      "height": 375,
      "width": 1242
    },
    "004992": {
      "height": 375,
      "width": 1242
    },
    "004993": {
      "height": 376,
      "width": 1241
    },
    "004994": {
      "height": 375,
      "width": 1242
    },
    "004995": {
      "height": 375,
      "width": 1242
    },
    "004996": {
      "height": 370,
      "width": 1224
    },
    "004997": {
      "height": 375,
      "width": 1242
    },
    "004998": {
      "height": 375,
      "width": 1242
    },
    "004999": {
      "height": 375,
      "width": 1242
    },
    "005000": {
      "height": 370,
      "width": 1224
    },
    "005001": {
      "height": 375,
      "width": 1242
    },
    "005002": {
      "height": 375,
      "width": 1242
    },
    "005003": {
      "height": 375,
      "width": 1242
    },
    "005004": {
      "height": 375,
      "width": 1242
    },
    "005005": {
      "height": 370,
      "width": 1224
    },
    "005006": {
      "height": 375,
      "width": 1242
    },
    "005007": {
      "height": 370,
      "width": 1224
    },
    "005008": {
      "height": 376,
      "width": 1241
    },
    "005009": {
      "height": 375,
      "width": 1242
    },
    "005010": {
      "height": 375,
      "width": 1242
    },
    "005011": {
      "height": 375,
      "width": 1242
    },
    "005012": {
      "height": 375,
      "width": 1242
    },
    "005013": {
      "height": 375,
      "width": 1242
    },
    "005014": {
      "height": 375,
      "width": 1242
    },
    "005015": {
      "height": 375,
      "width": 1242
    },
    "005016": {
      "height": 375,
      "width": 1242
    },
    "005017": {
      "height": 370,
      "width": 1224
    },
    "005018": {
      "height": 375,
      "width": 1242
    },
    "005019": {
      "height": 376,
      "width": 1241
    },
    "005020": {
      "height": 375,
      "width": 1242
    },
    "005021": {
      "height": 375,
      "width": 1242
    },
    "005022": {
      "height": 375,
      "width": 1242
    },
    "005023": {
      "height": 375,
      "width": 1242
    },
    "005024": {
      "height": 375,
      "width": 1242
    },
    "005025": {
      "height": 375,
      "width": 1242
    },
    "005026": {
      "height": 375,
      "width": 1242
    },
    "005027": {
      "height": 370,
      "width": 1224
    },
    "005028": {
      "height": 375,
      "width": 1242
    },
    "005029": {
      "height": 375,
      "width": 1242
    },
    "005030": {
      "height": 370,
      "width": 1224
    },
    "005031": {
      "height": 370,
      "width": 1224
    },
    "005032": {
      "height": 375,
      "width": 1242
    },
    "005033": {
      "height": 375,
      "width": 1242
    },
    "005034": {
      "height": 375,
      "width": 1242
    },
    "005035": {
      "height": 375,
      "width": 1242
    },
    "005036": {
      "height": 375,
      "width": 1242
    },
    "005037": {
      "height": 370,
      "width": 1224
    },
    "005038": {
      "height": 375,
      "width": 1242
    },
    "005039": {
      "height": 375,
      "width": 1242
    },
    "005040": {
      "height": 375,
      "width": 1242
    },
    "005041": {
      "height": 375,
      "width": 1242
    },
    "005042": {
      "height": 375,
      "width": 1242
    },
    "005043": {
      "height": 375,
      "width": 1242
    },
    "005044": {
      "height": 375,
      "width": 1242
    },
    "005045": {
      "height": 375,
      "width": 1242
    },
    "005046": {
      "height": 375,
      "width": 1242
    },
    "005047": {
      "height": 375,
      "width": 1242
    },
    "005048": {
      "height": 375,
      "width": 1242
    },
    "005049": {
      "height": 375,
      "width": 1242
    },
    "005050": {
      "height": 370,
      "width": 1224
    },
    "005051": {
      "height": 375,
      "width": 1242
    },
    "005052": {
      "height": 375,
      "width": 1242
    },
    "005053": {
      "height": 376,
      "width": 1241
    },
    "005054": {
      "height": 375,
      "width": 1242
    },
    "005055": {
      "height": 375,
      "width": 1242
    },
    "005056": {
      "height": 375,
      "width": 1242
    },
    "005057": {
      "height": 375,
      "width": 1242
    },
    "005058": {
      "height": 376,
      "width": 1241
    },
    "005059": {
      "height": 370,
      "width": 1224
    },
    "005060": {
      "height": 375,
      "width": 1242
    },
    "005061": {
      "height": 375,
      "width": 1242
    },
    "005062": {
      "height": 375,
      "width": 1242
    },
    "005063": {
      "height": 375,
      "width": 1242
    },
    "005064": {
      "height": 370,
      "width": 1224
    },
    "005065": {
      "height": 374,
      "width": 1238
    },
    "005066": {
      "height": 375,
      "width": 1242
    },
    "005067": {
      "height": 376,
      "width": 1241
    },
    "005068": {
      "height": 375,
      "width": 1242
    },
    "005069": {
      "height": 375,
      "width": 1242
    },
    "005070": {
      "height": 370,
      "width": 1224
    },
    "005071": {
      "height": 375,
      "width": 1242
    },
    "005072": {
      "height": 375,
      "width": 1242
    },
    "005073": {
      "height": 376,
      "width": 1241
    },
    "005074": {
      "height": 375,
      "width": 1242
    },
    "005075": {
      "height": 370,
      "width": 1224
    },
    "005076": {
      "height": 375,
      "width": 1242
    },
    "005077": {
      "height": 375,
      "width": 1242
    },
    "005078": {
      "height": 370,
      "width": 1224
    },
    "005079": {
      "height": 375,
      "width": 1242
    },
    "005080": {
      "height": 375,
      "width": 1242
    },
    "005081": {
      "height": 375,
      "width": 1242
    },
    "005082": {
      "height": 374,
      "width": 1238
    },
    "005083": {
      "height": 375,
      "width": 1242
    },
    "005084": {
      "height": 375,
      "width": 1242
    },
    "005085": {
      "height": 375,
      "width": 1242
    },
    "005086": {
      "height": 374,
      "width": 1238
    },
    "005087": {
      "height": 375,
      "width": 1242
    },
    "005088": {
      "height": 375,
      "width": 1242
    },
    "005089": {
      "height": 374,
      "width": 1238
    },
    "005090": {
      "height": 375,
      "width": 1242
    },
    "005091": {
      "height": 375,
      "width": 1242
    },
    "005092": {
      "height": 375,
      "width": 1242
    },
    "005093": {
      "height": 375,
      "width": 1242
    },
    "005094": {
      "height": 376,
      "width": 1241
    },
    "005095": {
      "height": 375,
      "width": 1242
    },
    "005096": {
      "height": 375,
      "width": 1242
    },
    "005097": {
      "height": 375,
      "width": 1242
    },
    "005098": {
      "height": 375,
      "width": 1242
    },
    "005099": {
      "height": 375,
      "width": 1242
    },
    "005100": {
      "height": 375,
      "width": 1242
    },
    "005101": {
      "height": 375,
      "width": 1242
    },
    "005102": {
      "height": 375,
      "width": 1242
    },
    "005103": {
      "height": 375,
      "width": 1242
    },
    "005104": {
      "height": 375,
      "width": 1242
    },
    "005105": {
      "height": 375,
      "width": 1242
    },
    "005106": {
      "height": 370,
      "width": 1224
    },
    "005107": {
      "height": 370,
      "width": 1224
    },
    "005108": {
      "height": 376,
      "width": 1241
    },
    "005109": {
      "height": 375,
      "width": 1242
    },
    "005110": {
      "height": 370,
      "width": 1224
    },
    "005111": {
      "height": 375,
      "width": 1242
    },
    "005112": {
      "height": 370,
      "width": 1224
    },
    "005113": {
      "height": 375,
      "width": 1242
    },
    "005114": {
      "height": 375,
      "width": 1242
    },
    "005115": {
      "height": 370,
      "width": 1224
    },
    "005116": {
      "height": 375,
      "width": 1242
    },
    "005117": {
      "height": 375,
      "width": 1242
    },
    "005118": {
      "height": 375,
      "width": 1242
    },
    "005119": {
      "height": 375,
      "width": 1242
    },
    "005120": {
      "height": 375,
      "width": 1242
    },
    "005121": {
      "height": 375,
      "width": 1242
    },
    "005122": {
      "height": 375,
      "width": 1242
    },
    "005123": {
      "height": 375,
      "width": 1242
    },
    "005124": {
      "height": 370,
      "width": 1224
    },
    "005125": {
      "height": 375,
      "width": 1242
    },
    "005126": {
      "height": 375,
      "width": 1242
    },
    "005127": {
      "height": 375,
      "width": 1242
    },
    "005128": {
      "height": 375,
      "width": 1242
    },
    "005129": {
      "height": 375,
      "width": 1242
    },
    "005130": {
      "height": 375,
      "width": 1242
    },
    "005131": {
      "height": 370,
      "width": 1224
    },
    "005132": {
      "height": 375,
      "width": 1242
    },
    "005133": {
      "height": 375,
      "width": 1242
    },
    "005134": {
      "height": 375,
      "width": 1242
    },
    "005135": {
      "height": 375,
      "width": 1242
    },
    "005136": {
      "height": 375,
      "width": 1242
    },
    "005137": {
      "height": 375,
      "width": 1242
    },
    "005138": {
      "height": 376,
      "width": 1241
    },
    "005139": {
      "height": 375,
      "width": 1242
    },
    "005140": {
      "height": 375,
      "width": 1242
    },
    "005141": {
      "height": 375,
      "width": 1242
    },
    "005142": {
      "height": 370,
      "width": 1224
    },
    "005143": {
      "height": 375,
      "width": 1242
    },
    "005144": {
      "height": 375,
      "width": 1242
    },
    "005145": {
      "height": 375,
      "width": 1242
    },
    "005146": {
      "height": 375,
      "width": 1242
    },
    "005147": {
      "height": 375,
      "width": 1242
    },
    "005148": {
      "height": 375,
      "width": 1242
    },
    "005149": {
      "height": 375,
      "width": 1242
    },
    "005150": {
      "height": 370,
      "width": 1224
    },
    "005151": {
      "height": 375,
      "width": 1242
    },
    "005152": {
      "height": 375,
      "width": 1242
    },
    "005153": {
      "height": 375,
      "width": 1242
    },
    "005154": {
      "height": 375,
      "width": 1242
    },
    "005155": {
      "height": 375,
      "width": 1242
    },
    "005156": {
      "height": 375,
      "width": 1242
    },
    "005157": {
      "height": 375,
      "width": 1242
    },
    "005158": {
      "height": 375,
      "width": 1242
    },
    "005159": {
      "height": 375,
      "width": 1242
    },
    "005160": {
      "height": 375,
      "width": 1242
    },
    "005161": {
      "height": 375,
      "width": 1242
    },
    "005162": {
      "height": 375,
      "width": 1242
    },
    "005163": {
      "height": 375,
      "width": 1242
    },
    "005164": {
      "height": 376,
      "width": 1241
    },
    "005165": {
      "height": 370,
      "width": 1224
    },
    "005166": {
      "height": 375,
      "width": 1242
    },
    "005167": {
      "height": 375,
      "width": 1242
    },
    "005168": {
      "height": 370,
      "width": 1224
    },
    "005169": {
      "height": 375,
      "width": 1242
    },
    "005170": {
      "height": 374,
      "width": 1238
    },
    "005171": {
      "height": 375,
      "width": 1242
    },
    "005172": {
      "height": 375,
      "width": 1242
    },
    "005173": {
      "height": 375,
      "width": 1242
    },
    "005174": {
      "height": 375,
      "width": 1242
    },
    "005175": {
      "height": 374,
      "width": 1238
    },
    "005176": {
      "height": 375,
      "width": 1242
    },
    "005177": {
      "height": 375,
      "width": 1242
    },
    "005178": {
      "height": 375,
      "width": 1242
    },
    "005179": {
      "height": 375,
      "width": 1242
    },
    "005180": {
      "height": 375,
      "width": 1242
    },
    "005181": {
      "height": 375,
      "width": 1242
    },
    "005182": {
      "height": 375,
      "width": 1242
    },
    "005183": {
      "height": 375,
      "width": 1242
    },
    "005184": {
      "height": 375,
      "width": 1242
    },
    "005185": {
      "height": 375,
      "width": 1242
    },
    "005186": {
      "height": 375,
      "width": 1242
    },
    "005187": {
      "height": 375,
      "width": 1242
    },
    "005188": {
      "height": 375,
      "width": 1242
    },
    "005189": {
      "height": 375,
      "width": 1242
    },
    "005190": {
      "height": 375,
      "width": 1242
    },
    "005191": {
      "height": 376,
      "width": 1241
    },
    "005192": {
      "height": 375,
      "width": 1242
    },
    "005193": {
      "height": 375,
      "width": 1242
    },
    "005194": {
      "height": 375,
      "width": 1242
    },
    "005195": {
      "height": 375,
      "width": 1242
    },
    "005196": {
      "height": 375,
      "width": 1242
    },
    "005197": {
      "height": 375,
      "width": 1242
    },
    "005198": {
      "height": 375,
      "width": 1242
    },
    "005199": {
      "height": 375,
      "width": 1242
    },
    "005200": {
      "height": 375,
      "width": 1242
    },
    "005201": {
      "height": 375,
      "width": 1242
    },
    "005202": {
      "height": 375,
      "width": 1242
    },
    "005203": {
      "height": 375,
      "width": 1242
    },
    "005204": {
      "height": 375,
      "width": 1242
    },
    "005205": {
      "height": 375,
      "width": 1242
    },
    "005206": {
      "height": 375,
      "width": 1242
    },
    "005207": {
      "height": 375,
      "width": 1242
    },
    "005208": {
      "height": 375,
      "width": 1242
    },
    "005209": {
      "height": 375,
      "width": 1242
    },
    "005210": {
      "height": 375,
      "width": 1242
    },
    "005211": {
      "height": 375,
      "width": 1242
    },
    "005212": {
      "height": 375,
      "width": 1242
    },
    "005213": {
      "height": 375,
      "width": 1242
    },
    "005214": {
      "height": 376,
      "width": 1241
    },
    "005215": {
      "height": 375,
      "width": 1242
    },
    "005216": {
      "height": 375,
      "width": 1242
    },
    "005217": {
      "height": 375,
      "width": 1242
    },
    "005218": {
      "height": 370,
      "width": 1224
    },
    "005219": {
      "height": 375,
      "width": 1242
    },
    "005220": {
      "height": 375,
      "width": 1242
    },
    "005221": {
      "height": 375,
      "width": 1242
    },
    "005222": {
      "height": 375,
      "width": 1242
    },
    "005223": {
      "height": 375,
      "width": 1242
    },
    "005224": {
      "height": 375,
      "width": 1242
    },
    "005225": {
      "height": 375,
      "width": 1242
    },
    "005226": {
      "height": 370,
      "width": 1224
    },
    "005227": {
      "height": 375,
      "width": 1242
    },
    "005228": {
      "height": 375,
      "width": 1242
    },
    "005229": {
      "height": 376,
      "width": 1241
    },
    "005230": {
      "height": 375,
      "width": 1242
    },
    "005231": {
      "height": 370,
      "width": 1224
    },
    "005232": {
      "height": 375,
      "width": 1242
    },
    "005233": {
      "height": 375,
      "width": 1242
    },
    "005234": {
      "height": 375,
      "width": 1242
    },
    "005235": {
      "height": 375,
      "width": 1242
    },
    "005236": {
      "height": 375,
      "width": 1242
    },
    "005237": {
      "height": 375,
      "width": 1242
    },
    "005238": {
      "height": 375,
      "width": 1242
    },
    "005239": {
      "height": 375,
      "width": 1242
    },
    "005240": {
      "height": 370,
      "width": 1224
    },
    "005241": {
      "height": 375,
      "width": 1242
    },
    "005242": {
      "height": 375,
      "width": 1242
    },
    "005243": {
      "height": 375,
      "width": 1242
    },
    "005244": {
      "height": 375,
      "width": 1242
    },
    "005245": {
      "height": 375,
      "width": 1242
    },
    "005246": {
      "height": 375,
      "width": 1242
    },
    "005247": {
      "height": 370,
      "width": 1224
    },
    "005248": {
      "height": 375,
      "width": 1242
    },
    "005249": {
      "height": 375,
      "width": 1242
    },
    "005250": {
      "height": 375,
      "width": 1242
    },
    "005251": {
      "height": 375,
      "width": 1242
    },
    "005252": {
      "height": 375,
      "width": 1242
    },
    "005253": {
      "height": 375,
      "width": 1242
    },
    "005254": {
      "height": 375,
      "width": 1242
    },
    "005255": {
      "height": 375,
      "width": 1242
    },
    "005256": {
      "height": 375,
      "width": 1242
    },
    "005257": {
      "height": 375,
      "width": 1242
    },
    "005258": {
      "height": 375,
      "width": 1242
    },
    "005259": {
      "height": 370,
      "width": 1224
    },
    "005260": {
      "height": 376,
      "width": 1241
    },
    "005261": {
      "height": 375,
      "width": 1242
    },
    "005262": {
      "height": 375,
      "width": 1242
    },
    "005263": {
      "height": 375,
      "width": 1242
    },
    "005264": {
      "height": 375,
      "width": 1242
    },
    "005265": {
      "height": 375,
      "width": 1242
    },
    "005266": {
      "height": 375,
      "width": 1242
    },
    "005267": {
      "height": 375,
      "width": 1242
    },
    "005268": {
      "height": 375,
      "width": 1242
    },
    "005269": {
      "height": 375,
      "width": 1242
    },
    "005270": {
      "height": 375,
      "width": 1242
    },
    "005271": {
      "height": 375,
      "width": 1242
    },
    "005272": {
      "height": 375,
      "width": 1242
    },
    "005273": {
      "height": 370,
      "width": 1224
    },
    "005274": {
      "height": 375,
      "width": 1242
    },
    "005275": {
      "height": 375,
      "width": 1242
    },
    "005276": {
      "height": 375,
      "width": 1242
    },
    "005277": {
      "height": 375,
      "width": 1242
    },
    "005278": {
      "height": 375,
      "width": 1242
    },
    "005279": {
      "height": 375,
      "width": 1242
    },
    "005280": {
      "height": 375,
      "width": 1242
    },
    "005281": {
      "height": 375,
      "width": 1242
    },
    "005282": {
      "height": 375,
      "width": 1242
    },
    "005283": {
      "height": 375,
      "width": 1242
    },
    "005284": {
      "height": 375,
      "width": 1242
    },
    "005285": {
      "height": 375,
      "width": 1242
    },
    "005286": {
      "height": 375,
      "width": 1242
    },
    "005287": {
      "height": 375,
      "width": 1242
    },
    "005288": {
      "height": 375,
      "width": 1242
    },
    "005289": {
      "height": 375,
      "width": 1242
    },
    "005290": {
      "height": 375,
      "width": 1242
    },
    "005291": {
      "height": 370,
      "width": 1224
    },
    "005292": {
      "height": 375,
      "width": 1242
    },
    "005293": {
      "height": 375,
      "width": 1242
    },
    "005294": {
      "height": 375,
      "width": 1242
    },
    "005295": {
      "height": 375,
      "width": 1242
    },
    "005296": {
      "height": 375,
      "width": 1242
    },
    "005297": {
      "height": 375,
      "width": 1242
    },
    "005298": {
      "height": 375,
      "width": 1242
    },
    "005299": {
      "height": 375,
      "width": 1242
    },
    "005300": {
      "height": 375,
      "width": 1242
    },
    "005301": {
      "height": 370,
      "width": 1224
    },
    "005302": {
      "height": 375,
      "width": 1242
    },
    "005303": {
      "height": 375,
      "width": 1242
    },
    "005304": {
      "height": 375,
      "width": 1242
    },
    "005305": {
      "height": 374,
      "width": 1238
    },
    "005306": {
      "height": 375,
      "width": 1242
    },
    "005307": {
      "height": 375,
      "width": 1242
    },
    "005308": {
      "height": 375,
      "width": 1242
    },
    "005309": {
      "height": 375,
      "width": 1242
    },
    "005310": {
      "height": 375,
      "width": 1242
    },
    "005311": {
      "height": 375,
      "width": 1242
    },
    "005312": {
      "height": 375,
      "width": 1242
    },
    "005313": {
      "height": 375,
      "width": 1242
    },
    "005314": {
      "height": 375,
      "width": 1242
    },
    "005315": {
      "height": 375,
      "width": 1242
    },
    "005316": {
      "height": 375,
      "width": 1242
    },
    "005317": {
      "height": 370,
      "width": 1224
    },
    "005318": {
      "height": 376,
      "width": 1241
    },
    "005319": {
      "height": 375,
      "width": 1242
    },
    "005320": {
      "height": 375,
      "width": 1242
    },
    "005321": {
      "height": 376,
      "width": 1241
    },
    "005322": {
      "height": 375,
      "width": 1242
    },
    "005323": {
      "height": 375,
      "width": 1242
    },
    "005324": {
      "height": 370,
      "width": 1224
    },
    "005325": {
      "height": 375,
      "width": 1242
    },
    "005326": {
      "height": 375,
      "width": 1242
    },
    "005327": {
      "height": 375,
      "width": 1242
    },
    "005328": {
      "height": 375,
      "width": 1242
    },
    "005329": {
      "height": 370,
      "width": 1224
    },
    "005330": {
      "height": 375,
      "width": 1242
    },
    "005331": {
      "height": 375,
      "width": 1242
    },
    "005332": {
      "height": 375,
      "width": 1242
    },
    "005333": {
      "height": 375,
      "width": 1242
    },
    "005334": {
      "height": 375,
      "width": 1242
    },
    "005335": {
      "height": 375,
      "width": 1242
    },
    "005336": {
      "height": 374,
      "width": 1238
    },
    "005337": {
      "height": 375,
      "width": 1242
    },
    "005338": {
      "height": 375,
      "width": 1242
    },
    "005339": {
      "height": 375,
      "width": 1242
    },
    "005340": {
      "height": 375,
      "width": 1242
    },
    "005341": {
      "height": 375,
      "width": 1242
    },
    "005342": {
      "height": 375,
      "width": 1242
    },
    "005343": {
      "height": 375,
      "width": 1242
    },
    "005344": {
      "height": 375,
      "width": 1242
    },
    "005345": {
      "height": 375,
      "width": 1242
    },
    "005346": {
      "height": 375,
      "width": 1242
    },
    "005347": {
      "height": 375,
      "width": 1242
    },
    "005348": {
      "height": 375,
      "width": 1242
    },
    "005349": {
      "height": 375,
      "width": 1242
    },
    "005350": {
      "height": 375,
      "width": 1242
    },
    "005351": {
      "height": 375,
      "width": 1242
    },
    "005352": {
      "height": 375,
      "width": 1242
    },
    "005353": {
      "height": 375,
      "width": 1242
    },
    "005354": {
      "height": 375,
      "width": 1242
    },
    "005355": {
      "height": 375,
      "width": 1242
    },
    "005356": {
      "height": 375,
      "width": 1242
    },
    "005357": {
      "height": 375,
      "width": 1242
    },
    "005358": {
      "height": 375,
      "width": 1242
    },
    "005359": {
      "height": 375,
      "width": 1242
    },
    "005360": {
      "height": 375,
      "width": 1242
    },
    "005361": {
      "height": 374,
      "width": 1238
    },
    "005362": {
      "height": 375,
      "width": 1242
    },
    "005363": {
      "height": 375,
      "width": 1242
    },
    "005364": {
      "height": 375,
      "width": 1242
    },
    "005365": {
      "height": 375,
      "width": 1242
    },
    "005366": {
      "height": 376,
      "width": 1241
    },
    "005367": {
      "height": 375,
      "width": 1242
    },
    "005368": {
      "height": 375,
      "width": 1242
    },
    "005369": {
      "height": 374,
      "width": 1238
    },
    "005370": {
      "height": 375,
      "width": 1242
    },
    "005371": {
      "height": 375,
      "width": 1242
    },
    "005372": {
      "height": 375,
      "width": 1242
    },
    "005373": {
      "height": 375,
      "width": 1242
    },
    "005374": {
      "height": 375,
      "width": 1242
    },
    "005375": {
      "height": 375,
      "width": 1242
    },
    "005376": {
      "height": 375,
      "width": 1242
    },
    "005377": {
      "height": 375,
      "width": 1242
    },
    "005378": {
      "height": 375,
      "width": 1242
    },
    "005379": {
      "height": 375,
      "width": 1242
    },
    "005380": {
      "height": 375,
      "width": 1242
    },
    "005381": {
      "height": 375,
      "width": 1242
    },
    "005382": {
      "height": 375,
      "width": 1242
    },
    "005383": {
      "height": 375,
      "width": 1242
    },
    "005384": {
      "height": 375,
      "width": 1242
    },
    "005385": {
      "height": 375,
      "width": 1242
    },
    "005386": {
      "height": 375,
      "width": 1242
    },
    "005387": {
      "height": 375,
      "width": 1242
    },
    "005388": {
      "height": 375,
      "width": 1242
    },
    "005389": {
      "height": 375,
      "width": 1242
    },
    "005390": {
      "height": 375,
      "width": 1242
    },
    "005391": {
      "height": 370,
      "width": 1224
    },
    "005392": {
      "height": 370,
      "width": 1224
    },
    "005393": {
      "height": 375,
      "width": 1242
    },
    "005394": {
      "height": 375,
      "width": 1242
    },
    "005395": {
      "height": 375,
      "width": 1242
    },
    "005396": {
      "height": 375,
      "width": 1242
    },
    "005397": {
      "height": 375,
      "width": 1242
    },
    "005398": {
      "height": 375,
      "width": 1242
    },
    "005399": {
      "height": 375,
      "width": 1242
    },
    "005400": {
      "height": 375,
      "width": 1242
    },
    "005401": {
      "height": 374,
      "width": 1238
    },
    "005402": {
      "height": 375,
      "width": 1242
    },
    "005403": {
      "height": 375,
      "width": 1242
    },
    "005404": {
      "height": 375,
      "width": 1242
    },
    "005405": {
      "height": 375,
      "width": 1242
    },
    "005406": {
      "height": 375,
      "width": 1242
    },
    "005407": {
      "height": 375,
      "width": 1242
    },
    "005408": {
      "height": 370,
      "width": 1224
    },
    "005409": {
      "height": 375,
      "width": 1242
    },
    "005410": {
      "height": 375,
      "width": 1242
    },
    "005411": {
      "height": 375,
      "width": 1242
    },
    "005412": {
      "height": 375,
      "width": 1242
    },
    "005413": {
      "height": 375,
      "width": 1242
    },
    "005414": {
      "height": 375,
      "width": 1242
    },
    "005415": {
      "height": 375,
      "width": 1242
    },
    "005416": {
      "height": 375,
      "width": 1242
    },
    "005417": {
      "height": 375,
      "width": 1242
    },
    "005418": {
      "height": 370,
      "width": 1224
    },
    "005419": {
      "height": 375,
      "width": 1242
    },
    "005420": {
      "height": 374,
      "width": 1238
    },
    "005421": {
      "height": 375,
      "width": 1242
    },
    "005422": {
      "height": 370,
      "width": 1224
    },
    "005423": {
      "height": 375,
      "width": 1242
    },
    "005424": {
      "height": 375,
      "width": 1242
    },
    "005425": {
      "height": 375,
      "width": 1242
    },
    "005426": {
      "height": 375,
      "width": 1242
    },
    "005427": {
      "height": 375,
      "width": 1242
    },
    "005428": {
      "height": 375,
      "width": 1242
    },
    "005429": {
      "height": 370,
      "width": 1224
    },
    "005430": {
      "height": 375,
      "width": 1242
    },
    "005431": {
      "height": 376,
      "width": 1241
    },
    "005432": {
      "height": 375,
      "width": 1242
    },
    "005433": {
      "height": 370,
      "width": 1224
    },
    "005434": {
      "height": 374,
      "width": 1238
    },
    "005435": {
      "height": 375,
      "width": 1242
    },
    "005436": {
      "height": 375,
      "width": 1242
    },
    "005437": {
      "height": 375,
      "width": 1242
    },
    "005438": {
      "height": 375,
      "width": 1242
    },
    "005439": {
      "height": 370,
      "width": 1224
    },
    "005440": {
      "height": 375,
      "width": 1242
    },
    "005441": {
      "height": 375,
      "width": 1242
    },
    "005442": {
      "height": 375,
      "width": 1242
    },
    "005443": {
      "height": 375,
      "width": 1242
    },
    "005444": {
      "height": 376,
      "width": 1241
    },
    "005445": {
      "height": 375,
      "width": 1242
    },
    "005446": {
      "height": 375,
      "width": 1242
    },
    "005447": {
      "height": 375,
      "width": 1242
    },
    "005448": {
      "height": 375,
      "width": 1242
    },
    "005449": {
      "height": 375,
      "width": 1242
    },
    "005450": {
      "height": 375,
      "width": 1242
    },
    "005451": {
      "height": 375,
      "width": 1242
    },
    "005452": {
      "height": 375,
      "width": 1242
    },
    "005453": {
      "height": 375,
      "width": 1242
    },
    "005454": {
      "height": 375,
      "width": 1242
    },
    "005455": {
      "height": 375,
      "width": 1242
    },
    "005456": {
      "height": 375,
      "width": 1242
    },
    "005457": {
      "height": 375,
      "width": 1242
    },
    "005458": {
      "height": 375,
      "width": 1242
    },
    "005459": {
      "height": 374,
      "width": 1238
    },
    "005460": {
      "height": 375,
      "width": 1242
    },
    "005461": {
      "height": 375,
      "width": 1242
    },
    "005462": {
      "height": 375,
      "width": 1242
    },
    "005463": {
      "height": 375,
      "width": 1242
    },
    "005464": {
      "height": 375,
      "width": 1242
    },
    "005465": {
      "height": 375,
      "width": 1242
    },
    "005466": {
      "height": 375,
      "width": 1242
    },
    "005467": {
      "height": 374,
      "width": 1238
    },
    "005468": {
      "height": 375,
      "width": 1242
    },
    "005469": {
      "height": 375,
      "width": 1242
    },
    "005470": {
      "height": 375,
      "width": 1242
    },
    "005471": {
      "height": 375,
      "width": 1242
    },
    "005472": {
      "height": 374,
      "width": 1238
    },
    "005473": {
      "height": 375,
      "width": 1242
    },
    "005474": {
      "height": 375,
      "width": 1242
    },
    "005475": {
      "height": 375,
      "width": 1242
    },
    "005476": {
      "height": 375,
      "width": 1242
    },
    "005477": {
      "height": 375,
      "width": 1242
    },
    "005478": {
      "height": 375,
      "width": 1242
    },
    "005479": {
      "height": 375,
      "width": 1242
    },
    "005480": {
      "height": 370,
      "width": 1224
    },
    "005481": {
      "height": 375,
      "width": 1242
    },
    "005482": {
      "height": 370,
      "width": 1224
    },
    "005483": {
      "height": 375,
      "width": 1242
    },
    "005484": {
      "height": 375,
      "width": 1242
    },
    "005485": {
      "height": 375,
      "width": 1242
    },
    "005486": {
      "height": 375,
      "width": 1242
    },
    "005487": {
      "height": 375,
      "width": 1242
    },
    "005488": {
      "height": 375,
      "width": 1242
    },
    "005489": {
      "height": 375,
      "width": 1242
    },
    "005490": {
      "height": 375,
      "width": 1242
    },
    "005491": {
      "height": 375,
      "width": 1242
    },
    "005492": {
      "height": 370,
      "width": 1224
    },
    "005493": {
      "height": 375,
      "width": 1242
    },
    "005494": {
      "height": 375,
      "width": 1242
    },
    "005495": {
      "height": 370,
      "width": 1224
    },
    "005496": {
      "height": 375,
      "width": 1242
    },
    "005497": {
      "height": 375,
      "width": 1242
    },
    "005498": {
      "height": 376,
      "width": 1241
    },
    "005499": {
      "height": 370,
      "width": 1224
    },
    "005500": {
      "height": 375,
      "width": 1242
    },
    "005501": {
      "height": 375,
      "width": 1242
    },
    "005502": {
      "height": 375,
      "width": 1242
    },
    "005503": {
      "height": 375,
      "width": 1242
    },
    "005504": {
      "height": 370,
      "width": 1224
    },
    "005505": {
      "height": 375,
      "width": 1242
    },
    "005506": {
      "height": 375,
      "width": 1242
    },
    "005507": {
      "height": 375,
      "width": 1242
    },
    "005508": {
      "height": 375,
      "width": 1242
    },
    "005509": {
      "height": 375,
      "width": 1242
    },
    "005510": {
      "height": 374,
      "width": 1238
    },
    "005511": {
      "height": 370,
      "width": 1224
    },
    "005512": {
      "height": 375,
      "width": 1242
    },
    "005513": {
      "height": 375,
      "width": 1242
    },
    "005514": {
      "height": 375,
      "width": 1242
    },
    "005515": {
      "height": 375,
      "width": 1242
    },
    "005516": {
      "height": 375,
      "width": 1242
    },
    "005517": {
      "height": 375,
      "width": 1242
    },
    "005518": {
      "height": 375,
      "width": 1242
    },
    "005519": {
      "height": 375,
      "width": 1242
    },
    "005520": {
      "height": 375,
      "width": 1242
    },
    "005521": {
      "height": 375,
      "width": 1242
    },
    "005522": {
      "height": 375,
      "width": 1242
    },
    "005523": {
      "height": 375,
      "width": 1242
    },
    "005524": {
      "height": 370,
      "width": 1224
    },
    "005525": {
      "height": 375,
      "width": 1242
    },
    "005526": {
      "height": 370,
      "width": 1224
    },
    "005527": {
      "height": 375,
      "width": 1242
    },
    "005528": {
      "height": 375,
      "width": 1242
    },
    "005529": {
      "height": 375,
      "width": 1242
    },
    "005530": {
      "height": 375,
      "width": 1242
    },
    "005531": {
      "height": 375,
      "width": 1242
    },
    "005532": {
      "height": 370,
      "width": 1224
    },
    "005533": {
      "height": 375,
      "width": 1242
    },
    "005534": {
      "height": 375,
      "width": 1242
    },
    "005535": {
      "height": 375,
      "width": 1242
    },
    "005536": {
      "height": 375,
      "width": 1242
    },
    "005537": {
      "height": 375,
      "width": 1242
    },
    "005538": {
      "height": 375,
      "width": 1242
    },
    "005539": {
      "height": 375,
      "width": 1242
    },
    "005540": {
      "height": 370,
      "width": 1224
    },
    "005541": {
      "height": 375,
      "width": 1242
    },
    "005542": {
      "height": 376,
      "width": 1241
    },
    "005543": {
      "height": 375,
      "width": 1242
    },
    "005544": {
      "height": 375,
      "width": 1242
    },
    "005545": {
      "height": 374,
      "width": 1238
    },
    "005546": {
      "height": 375,
      "width": 1242
    },
    "005547": {
      "height": 375,
      "width": 1242
    },
    "005548": {
      "height": 375,
      "width": 1242
    },
    "005549": {
      "height": 375,
      "width": 1242
    },
    "005550": {
      "height": 375,
      "width": 1242
    },
    "005551": {
      "height": 375,
      "width": 1242
    },
    "005552": {
      "height": 375,
      "width": 1242
    },
    "005553": {
      "height": 375,
      "width": 1242
    },
    "005554": {
      "height": 375,
      "width": 1242
    },
    "005555": {
      "height": 375,
      "width": 1242
    },
    "005556": {
      "height": 375,
      "width": 1242
    },
    "005557": {
      "height": 376,
      "width": 1241
    },
    "005558": {
      "height": 376,
      "width": 1241
    },
    "005559": {
      "height": 370,
      "width": 1224
    },
    "005560": {
      "height": 375,
      "width": 1242
    },
    "005561": {
      "height": 375,
      "width": 1242
    },
    "005562": {
      "height": 375,
      "width": 1242
    },
    "005563": {
      "height": 375,
      "width": 1242
    },
    "005564": {
      "height": 370,
      "width": 1224
    },
    "005565": {
      "height": 375,
      "width": 1242
    },
    "005566": {
      "height": 375,
      "width": 1242
    },
    "005567": {
      "height": 370,
      "width": 1224
    },
    "005568": {
      "height": 370,
      "width": 1224
    },
    "005569": {
      "height": 375,
      "width": 1242
    },
    "005570": {
      "height": 375,
      "width": 1242
    },
    "005571": {
      "height": 375,
      "width": 1242
    },
    "005572": {
      "height": 375,
      "width": 1242
    },
    "005573": {
      "height": 376,
      "width": 1241
    },
    "005574": {
      "height": 374,
      "width": 1238
    },
    "005575": {
      "height": 375,
      "width": 1242
    },
    "005576": {
      "height": 375,
      "width": 1242
    },
    "005577": {
      "height": 375,
      "width": 1242
    },
    "005578": {
      "height": 375,
      "width": 1242
    },
    "005579": {
      "height": 375,
      "width": 1242
    },
    "005580": {
      "height": 375,
      "width": 1242
    },
    "005581": {
      "height": 375,
      "width": 1242
    },
    "005582": {
      "height": 375,
      "width": 1242
    },
    "005583": {
      "height": 375,
      "width": 1242
    },
    "005584": {
      "height": 375,
      "width": 1242
    },
    "005585": {
      "height": 375,
      "width": 1242
    },
    "005586": {
      "height": 375,
      "width": 1242
    },
    "005587": {
      "height": 375,
      "width": 1242
    },
    "005588": {
      "height": 375,
      "width": 1242
    },
    "005589": {
      "height": 375,
      "width": 1242
    },
    "005590": {
      "height": 375,
      "width": 1242
    },
    "005591": {
      "height": 375,
      "width": 1242
    },
    "005592": {
      "height": 375,
      "width": 1242
    },
    "005593": {
      "height": 375,
      "width": 1242
    },
    "005594": {
      "height": 375,
      "width": 1242
    },
    "005595": {
      "height": 375,
      "width": 1242
    },
    "005596": {
      "height": 374,
      "width": 1238
    },
    "005597": {
      "height": 375,
      "width": 1242
    },
    "005598": {
      "height": 375,
      "width": 1242
    },
    "005599": {
      "height": 374,
      "width": 1238
    },
    "005600": {
      "height": 375,
      "width": 1242
    },
    "005601": {
      "height": 375,
      "width": 1242
    },
    "005602": {
      "height": 375,
      "width": 1242
    },
    "005603": {
      "height": 375,
      "width": 1242
    },
    "005604": {
      "height": 375,
      "width": 1242
    },
    "005605": {
      "height": 375,
      "width": 1242
    },
    "005606": {
      "height": 375,
      "width": 1242
    },
    "005607": {
      "height": 375,
      "width": 1242
    },
    "005608": {
      "height": 370,
      "width": 1224
    },
    "005609": {
      "height": 375,
      "width": 1242
    },
    "005610": {
      "height": 375,
      "width": 1242
    },
    "005611": {
      "height": 375,
      "width": 1242
    },
    "005612": {
      "height": 375,
      "width": 1242
    },
    "005613": {
      "height": 375,
      "width": 1242
    },
    "005614": {
      "height": 375,
      "width": 1242
    },
    "005615": {
      "height": 375,
      "width": 1242
    },
    "005616": {
      "height": 375,
      "width": 1242
    },
    "005617": {
      "height": 374,
      "width": 1238
    },
    "005618": {
      "height": 375,
      "width": 1242
    },
    "005619": {
      "height": 375,
      "width": 1242
    },
    "005620": {
      "height": 375,
      "width": 1242
    },
    "005621": {
      "height": 375,
      "width": 1242
    },
    "005622": {
      "height": 375,
      "width": 1242
    },
    "005623": {
      "height": 375,
      "width": 1242
    },
    "005624": {
      "height": 374,
      "width": 1238
    },
    "005625": {
      "height": 375,
      "width": 1242
    },
    "005626": {
      "height": 375,
      "width": 1242
    },
    "005627": {
      "height": 375,
      "width": 1242
    },
    "005628": {
      "height": 375,
      "width": 1242
    },
    "005629": {
      "height": 375,
      "width": 1242
    },
    "005630": {
      "height": 375,
      "width": 1242
    },
    "005631": {
      "height": 375,
      "width": 1242
    },
    "005632": {
      "height": 375,
      "width": 1242
    },
    "005633": {
      "height": 370,
      "width": 1224
    },
    "005634": {
      "height": 375,
      "width": 1242
    },
    "005635": {
      "height": 376,
      "width": 1241
    },
    "005636": {
      "height": 375,
      "width": 1242
    },
    "005637": {
      "height": 370,
      "width": 1224
    },
    "005638": {
      "height": 375,
      "width": 1242
    },
    "005639": {
      "height": 375,
      "width": 1242
    },
    "005640": {
      "height": 375,
      "width": 1242
    },
    "005641": {
      "height": 375,
      "width": 1242
    },
    "005642": {
      "height": 375,
      "width": 1242
    },
    "005643": {
      "height": 375,
      "width": 1242
    },
    "005644": {
      "height": 375,
      "width": 1242
    },
    "005645": {
      "height": 375,
      "width": 1242
    },
    "005646": {
      "height": 375,
      "width": 1242
    },
    "005647": {
      "height": 370,
      "width": 1224
    },
    "005648": {
      "height": 375,
      "width": 1242
    },
    "005649": {
      "height": 375,
      "width": 1242
    },
    "005650": {
      "height": 375,
      "width": 1242
    },
    "005651": {
      "height": 375,
      "width": 1242
    },
    "005652": {
      "height": 376,
      "width": 1241
    },
    "005653": {
      "height": 375,
      "width": 1242
    },
    "005654": {
      "height": 375,
      "width": 1242
    },
    "005655": {
      "height": 375,
      "width": 1242
    },
    "005656": {
      "height": 375,
      "width": 1242
    },
    "005657": {
      "height": 375,
      "width": 1242
    },
    "005658": {
      "height": 375,
      "width": 1242
    },
    "005659": {
      "height": 375,
      "width": 1242
    },
    "005660": {
      "height": 375,
      "width": 1242
    },
    "005661": {
      "height": 375,
      "width": 1242
    },
    "005662": {
      "height": 375,
      "width": 1242
    },
    "005663": {
      "height": 370,
      "width": 1224
    },
    "005664": {
      "height": 375,
      "width": 1242
    },
    "005665": {
      "height": 370,
      "width": 1224
    },
    "005666": {
      "height": 375,
      "width": 1242
    },
    "005667": {
      "height": 375,
      "width": 1242
    },
    "005668": {
      "height": 375,
      "width": 1242
    },
    "005669": {
      "height": 375,
      "width": 1242
    },
    "005670": {
      "height": 370,
      "width": 1224
    },
    "005671": {
      "height": 375,
      "width": 1242
    },
    "005672": {
      "height": 375,
      "width": 1242
    },
    "005673": {
      "height": 375,
      "width": 1242
    },
    "005674": {
      "height": 375,
      "width": 1242
    },
    "005675": {
      "height": 375,
      "width": 1242
    },
    "005676": {
      "height": 375,
      "width": 1242
    },
    "005677": {
      "height": 375,
      "width": 1242
    },
    "005678": {
      "height": 375,
      "width": 1242
    },
    "005679": {
      "height": 375,
      "width": 1242
    },
    "005680": {
      "height": 376,
      "width": 1241
    },
    "005681": {
      "height": 375,
      "width": 1242
    },
    "005682": {
      "height": 375,
      "width": 1242
    },
    "005683": {
      "height": 370,
      "width": 1224
    },
    "005684": {
      "height": 375,
      "width": 1242
    },
    "005685": {
      "height": 376,
      "width": 1241
    },
    "005686": {
      "height": 370,
      "width": 1224
    },
    "005687": {
      "height": 375,
      "width": 1242
    },
    "005688": {
      "height": 375,
      "width": 1242
    },
    "005689": {
      "height": 375,
      "width": 1242
    },
    "005690": {
      "height": 375,
      "width": 1242
    },
    "005691": {
      "height": 375,
      "width": 1242
    },
    "005692": {
      "height": 375,
      "width": 1242
    },
    "005693": {
      "height": 375,
      "width": 1242
    },
    "005694": {
      "height": 375,
      "width": 1242
    },
    "005695": {
      "height": 375,
      "width": 1242
    },
    "005696": {
      "height": 370,
      "width": 1224
    },
    "005697": {
      "height": 375,
      "width": 1242
    },
    "005698": {
      "height": 375,
      "width": 1242
    },
    "005699": {
      "height": 375,
      "width": 1242
    },
    "005700": {
      "height": 376,
      "width": 1241
    },
    "005701": {
      "height": 375,
      "width": 1242
    },
    "005702": {
      "height": 375,
      "width": 1242
    },
    "005703": {
      "height": 375,
      "width": 1242
    },
    "005704": {
      "height": 375,
      "width": 1242
    },
    "005705": {
      "height": 375,
      "width": 1242
    },
    "005706": {
      "height": 376,
      "width": 1241
    },
    "005707": {
      "height": 375,
      "width": 1242
    },
    "005708": {
      "height": 375,
      "width": 1242
    },
    "005709": {
      "height": 375,
      "width": 1242
    },
    "005710": {
      "height": 375,
      "width": 1242
    },
    "005711": {
      "height": 370,
      "width": 1224
    },
    "005712": {
      "height": 375,
      "width": 1242
    },
    "005713": {
      "height": 375,
      "width": 1242
    },
    "005714": {
      "height": 374,
      "width": 1238
    },
    "005715": {
      "height": 375,
      "width": 1242
    },
    "005716": {
      "height": 375,
      "width": 1242
    },
    "005717": {
      "height": 375,
      "width": 1242
    },
    "005718": {
      "height": 375,
      "width": 1242
    },
    "005719": {
      "height": 375,
      "width": 1242
    },
    "005720": {
      "height": 375,
      "width": 1242
    },
    "005721": {
      "height": 375,
      "width": 1242
    },
    "005722": {
      "height": 374,
      "width": 1238
    },
    "005723": {
      "height": 375,
      "width": 1242
    },
    "005724": {
      "height": 370,
      "width": 1224
    },
    "005725": {
      "height": 375,
      "width": 1242
    },
    "005726": {
      "height": 375,
      "width": 1242
    },
    "005727": {
      "height": 375,
      "width": 1242
    },
    "005728": {
      "height": 376,
      "width": 1241
    },
    "005729": {
      "height": 375,
      "width": 1242
    },
    "005730": {
      "height": 375,
      "width": 1242
    },
    "005731": {
      "height": 375,
      "width": 1242
    },
    "005732": {
      "height": 375,
      "width": 1242
    },
    "005733": {
      "height": 375,
      "width": 1242
    },
    "005734": {
      "height": 375,
      "width": 1242
    },
    "005735": {
      "height": 370,
      "width": 1224
    },
    "005736": {
      "height": 375,
      "width": 1242
    },
    "005737": {
      "height": 375,
      "width": 1242
    },
    "005738": {
      "height": 375,
      "width": 1242
    },
    "005739": {
      "height": 375,
      "width": 1242
    },
    "005740": {
      "height": 375,
      "width": 1242
    },
    "005741": {
      "height": 375,
      "width": 1242
    },
    "005742": {
      "height": 375,
      "width": 1242
    },
    "005743": {
      "height": 374,
      "width": 1238
    },
    "005744": {
      "height": 370,
      "width": 1224
    },
    "005745": {
      "height": 375,
      "width": 1242
    },
    "005746": {
      "height": 370,
      "width": 1224
    },
    "005747": {
      "height": 375,
      "width": 1242
    },
    "005748": {
      "height": 375,
      "width": 1242
    },
    "005749": {
      "height": 375,
      "width": 1242
    },
    "005750": {
      "height": 375,
      "width": 1242
    },
    "005751": {
      "height": 375,
      "width": 1242
    },
    "005752": {
      "height": 375,
      "width": 1242
    },
    "005753": {
      "height": 375,
      "width": 1242
    },
    "005754": {
      "height": 376,
      "width": 1241
    },
    "005755": {
      "height": 374,
      "width": 1238
    },
    "005756": {
      "height": 375,
      "width": 1242
    },
    "005757": {
      "height": 375,
      "width": 1242
    },
    "005758": {
      "height": 375,
      "width": 1242
    },
    "005759": {
      "height": 375,
      "width": 1242
    },
    "005760": {
      "height": 370,
      "width": 1224
    },
    "005761": {
      "height": 375,
      "width": 1242
    },
    "005762": {
      "height": 375,
      "width": 1242
    },
    "005763": {
      "height": 370,
      "width": 1224
    },
    "005764": {
      "height": 375,
      "width": 1242
    },
    "005765": {
      "height": 374,
      "width": 1238
    },
    "005766": {
      "height": 375,
      "width": 1242
    },
    "005767": {
      "height": 375,
      "width": 1242
    },
    "005768": {
      "height": 375,
      "width": 1242
    },
    "005769": {
      "height": 375,
      "width": 1242
    },
    "005770": {
      "height": 375,
      "width": 1242
    },
    "005771": {
      "height": 375,
      "width": 1242
    },
    "005772": {
      "height": 375,
      "width": 1242
    },
    "005773": {
      "height": 375,
      "width": 1242
    },
    "005774": {
      "height": 375,
      "width": 1242
    },
    "005775": {
      "height": 375,
      "width": 1242
    },
    "005776": {
      "height": 375,
      "width": 1242
    },
    "005777": {
      "height": 375,
      "width": 1242
    },
    "005778": {
      "height": 375,
      "width": 1242
    },
    "005779": {
      "height": 370,
      "width": 1224
    },
    "005780": {
      "height": 375,
      "width": 1242
    },
    "005781": {
      "height": 375,
      "width": 1242
    },
    "005782": {
      "height": 375,
      "width": 1242
    },
    "005783": {
      "height": 375,
      "width": 1242
    },
    "005784": {
      "height": 375,
      "width": 1242
    },
    "005785": {
      "height": 375,
      "width": 1242
    },
    "005786": {
      "height": 375,
      "width": 1242
    },
    "005787": {
      "height": 375,
      "width": 1242
    },
    "005788": {
      "height": 375,
      "width": 1242
    },
    "005789": {
      "height": 375,
      "width": 1242
    },
    "005790": {
      "height": 370,
      "width": 1224
    },
    "005791": {
      "height": 375,
      "width": 1242
    },
    "005792": {
      "height": 375,
      "width": 1242
    },
    "005793": {
      "height": 375,
      "width": 1242
    },
    "005794": {
      "height": 375,
      "width": 1242
    },
    "005795": {
      "height": 375,
      "width": 1242
    },
    "005796": {
      "height": 370,
      "width": 1224
    },
    "005797": {
      "height": 375,
      "width": 1242
    },
    "005798": {
      "height": 375,
      "width": 1242
    },
    "005799": {
      "height": 375,
      "width": 1242
    },
    "005800": {
      "height": 375,
      "width": 1242
    },
    "005801": {
      "height": 370,
      "width": 1224
    },
    "005802": {
      "height": 375,
      "width": 1242
    },
    "005803": {
      "height": 375,
      "width": 1242
    },
    "005804": {
      "height": 370,
      "width": 1224
    },
    "005805": {
      "height": 375,
      "width": 1242
    },
    "005806": {
      "height": 370,
      "width": 1224
    },
    "005807": {
      "height": 375,
      "width": 1242
    },
    "005808": {
      "height": 370,
      "width": 1224
    },
    "005809": {
      "height": 375,
      "width": 1242
    },
    "005810": {
      "height": 375,
      "width": 1242
    },
    "005811": {
      "height": 376,
      "width": 1241
    },
    "005812": {
      "height": 374,
      "width": 1238
    },
    "005813": {
      "height": 375,
      "width": 1242
    },
    "005814": {
      "height": 375,
      "width": 1242
    },
    "005815": {
      "height": 375,
      "width": 1242
    },
    "005816": {
      "height": 375,
      "width": 1242
    },
    "005817": {
      "height": 370,
      "width": 1224
    },
    "005818": {
      "height": 375,
      "width": 1242
    },
    "005819": {
      "height": 375,
      "width": 1242
    },
    "005820": {
      "height": 375,
      "width": 1242
    },
    "005821": {
      "height": 375,
      "width": 1242
    },
    "005822": {
      "height": 375,
      "width": 1242
    },
    "005823": {
      "height": 375,
      "width": 1242
    },
    "005824": {
      "height": 375,
      "width": 1242
    },
    "005825": {
      "height": 370,
      "width": 1224
    },
    "005826": {
      "height": 375,
      "width": 1242
    },
    "005827": {
      "height": 375,
      "width": 1242
    },
    "005828": {
      "height": 375,
      "width": 1242
    },
    "005829": {
      "height": 375,
      "width": 1242
    },
    "005830": {
      "height": 375,
      "width": 1242
    },
    "005831": {
      "height": 375,
      "width": 1242
    },
    "005832": {
      "height": 375,
      "width": 1242
    },
    "005833": {
      "height": 375,
      "width": 1242
    },
    "005834": {
      "height": 376,
      "width": 1241
    },
    "005835": {
      "height": 370,
      "width": 1224
    },
    "005836": {
      "height": 370,
      "width": 1224
    },
    "005837": {
      "height": 375,
      "width": 1242
    },
    "005838": {
      "height": 370,
      "width": 1224
    },
    "005839": {
      "height": 376,
      "width": 1241
    },
    "005840": {
      "height": 370,
      "width": 1224
    },
    "005841": {
      "height": 375,
      "width": 1242
    },
    "005842": {
      "height": 375,
      "width": 1242
    },
    "005843": {
      "height": 375,
      "width": 1242
    },
    "005844": {
      "height": 375,
      "width": 1242
    },
    "005845": {
      "height": 375,
      "width": 1242
    },
    "005846": {
      "height": 375,
      "width": 1242
    },
    "005847": {
      "height": 375,
      "width": 1242
    },
    "005848": {
      "height": 375,
      "width": 1242
    },
    "005849": {
      "height": 375,
      "width": 1242
    },
    "005850": {
      "height": 375,
      "width": 1242
    },
    "005851": {
      "height": 375,
      "width": 1242
    },
    "005852": {
      "height": 375,
      "width": 1242
    },
    "005853": {
      "height": 375,
      "width": 1242
    },
    "005854": {
      "height": 375,
      "width": 1242
    },
    "005855": {
      "height": 376,
      "width": 1241
    },
    "005856": {
      "height": 375,
      "width": 1242
    },
    "005857": {
      "height": 375,
      "width": 1242
    },
    "005858": {
      "height": 375,
      "width": 1242
    },
    "005859": {
      "height": 375,
      "width": 1242
    },
    "005860": {
      "height": 375,
      "width": 1242
    },
    "005861": {
      "height": 375,
      "width": 1242
    },
    "005862": {
      "height": 375,
      "width": 1242
    },
    "005863": {
      "height": 375,
      "width": 1242
    },
    "005864": {
      "height": 375,
      "width": 1242
    },
    "005865": {
      "height": 375,
      "width": 1242
    },
    "005866": {
      "height": 375,
      "width": 1242
    },
    "005867": {
      "height": 375,
      "width": 1242
    },
    "005868": {
      "height": 375,
      "width": 1242
    },
    "005869": {
      "height": 375,
      "width": 1242
    },
    "005870": {
      "height": 375,
      "width": 1242
    },
    "005871": {
      "height": 375,
      "width": 1242
    },
    "005872": {
      "height": 375,
      "width": 1242
    },
    "005873": {
      "height": 375,
      "width": 1242
    },
    "005874": {
      "height": 375,
      "width": 1242
    },
    "005875": {
      "height": 374,
      "width": 1238
    },
    "005876": {
      "height": 375,
      "width": 1242
    },
    "005877": {
      "height": 374,
      "width": 1238
    },
    "005878": {
      "height": 375,
      "width": 1242
    },
    "005879": {
      "height": 375,
      "width": 1242
    },
    "005880": {
      "height": 375,
      "width": 1242
    },
    "005881": {
      "height": 375,
      "width": 1242
    },
    "005882": {
      "height": 375,
      "width": 1242
    },
    "005883": {
      "height": 375,
      "width": 1242
    },
    "005884": {
      "height": 375,
      "width": 1242
    },
    "005885": {
      "height": 375,
      "width": 1242
    },
    "005886": {
      "height": 375,
      "width": 1242
    },
    "005887": {
      "height": 375,
      "width": 1242
    },
    "005888": {
      "height": 370,
      "width": 1224
    },
    "005889": {
      "height": 375,
      "width": 1242
    },
    "005890": {
      "height": 375,
      "width": 1242
    },
    "005891": {
      "height": 375,
      "width": 1242
    },
    "005892": {
      "height": 375,
      "width": 1242
    },
    "005893": {
      "height": 376,
      "width": 1241
    },
    "005894": {
      "height": 375,
      "width": 1242
    },
    "005895": {
      "height": 375,
      "width": 1242
    },
    "005896": {
      "height": 375,
      "width": 1242
    },
    "005897": {
      "height": 375,
      "width": 1242
    },
    "005898": {
      "height": 375,
      "width": 1242
    },
    "005899": {
      "height": 370,
      "width": 1224
    },
    "005900": {
      "height": 375,
      "width": 1242
    },
    "005901": {
      "height": 375,
      "width": 1242
    },
    "005902": {
      "height": 370,
      "width": 1224
    },
    "005903": {
      "height": 375,
      "width": 1242
    },
    "005904": {
      "height": 375,
      "width": 1242
    },
    "005905": {
      "height": 374,
      "width": 1238
    },
    "005906": {
      "height": 375,
      "width": 1242
    },
    "005907": {
      "height": 375,
      "width": 1242
    },
    "005908": {
      "height": 375,
      "width": 1242
    },
    "005909": {
      "height": 375,
      "width": 1242
    },
    "005910": {
      "height": 374,
      "width": 1238
    },
    "005911": {
      "height": 375,
      "width": 1242
    },
    "005912": {
      "height": 375,
      "width": 1242
    },
    "005913": {
      "height": 376,
      "width": 1241
    },
    "005914": {
      "height": 375,
      "width": 1242
    },
    "005915": {
      "height": 375,
      "width": 1242
    },
    "005916": {
      "height": 375,
      "width": 1242
    },
    "005917": {
      "height": 375,
      "width": 1242
    },
    "005918": {
      "height": 375,
      "width": 1242
    },
    "005919": {
      "height": 375,
      "width": 1242
    },
    "005920": {
      "height": 375,
      "width": 1242
    },
    "005921": {
      "height": 375,
      "width": 1242
    },
    "005922": {
      "height": 375,
      "width": 1242
    },
    "005923": {
      "height": 375,
      "width": 1242
    },
    "005924": {
      "height": 370,
      "width": 1224
    },
    "005925": {
      "height": 375,
      "width": 1242
    },
    "005926": {
      "height": 375,
      "width": 1242
    },
    "005927": {
      "height": 375,
      "width": 1242
    },
    "005928": {
      "height": 375,
      "width": 1242
    },
    "005929": {
      "height": 375,
      "width": 1242
    },
    "005930": {
      "height": 375,
      "width": 1242
    },
    "005931": {
      "height": 370,
      "width": 1224
    },
    "005932": {
      "height": 375,
      "width": 1242
    },
    "005933": {
      "height": 375,
      "width": 1242
    },
    "005934": {
      "height": 370,
      "width": 1224
    },
    "005935": {
      "height": 376,
      "width": 1241
    },
    "005936": {
      "height": 375,
      "width": 1242
    },
    "005937": {
      "height": 370,
      "width": 1224
    },
    "005938": {
      "height": 375,
      "width": 1242
    },
    "005939": {
      "height": 375,
      "width": 1242
    },
    "005940": {
      "height": 375,
      "width": 1242
    },
    "005941": {
      "height": 375,
      "width": 1242
    },
    "005942": {
      "height": 374,
      "width": 1238
    },
    "005943": {
      "height": 375,
      "width": 1242
    },
    "005944": {
      "height": 375,
      "width": 1242
    },
    "005945": {
      "height": 375,
      "width": 1242
    },
    "005946": {
      "height": 375,
      "width": 1242
    },
    "005947": {
      "height": 375,
      "width": 1242
    },
    "005948": {
      "height": 375,
      "width": 1242
    },
    "005949": {
      "height": 375,
      "width": 1242
    },
    "005950": {
      "height": 375,
      "width": 1242
    },
    "005951": {
      "height": 375,
      "width": 1242
    },
    "005952": {
      "height": 376,
      "width": 1241
    },
    "005953": {
      "height": 375,
      "width": 1242
    },
    "005954": {
      "height": 370,
      "width": 1224
    },
    "005955": {
      "height": 375,
      "width": 1242
    },
    "005956": {
      "height": 375,
      "width": 1242
    },
    "005957": {
      "height": 375,
      "width": 1242
    },
    "005958": {
      "height": 370,
      "width": 1224
    },
    "005959": {
      "height": 375,
      "width": 1242
    },
    "005960": {
      "height": 375,
      "width": 1242
    },
    "005961": {
      "height": 375,
      "width": 1242
    },
    "005962": {
      "height": 375,
      "width": 1242
    },
    "005963": {
      "height": 375,
      "width": 1242
    },
    "005964": {
      "height": 375,
      "width": 1242
    },
    "005965": {
      "height": 370,
      "width": 1224
    },
    "005966": {
      "height": 375,
      "width": 1242
    },
    "005967": {
      "height": 375,
      "width": 1242
    },
    "005968": {
      "height": 375,
      "width": 1242
    },
    "005969": {
      "height": 375,
      "width": 1242
    },
    "005970": {
      "height": 375,
      "width": 1242
    },
    "005971": {
      "height": 375,
      "width": 1242
    },
    "005972": {
      "height": 374,
      "width": 1238
    },
    "005973": {
      "height": 375,
      "width": 1242
    },
    "005974": {
      "height": 375,
      "width": 1242
    },
    "005975": {
      "height": 375,
      "width": 1242
    },
    "005976": {
      "height": 375,
      "width": 1242
    },
    "005977": {
      "height": 375,
      "width": 1242
    },
    "005978": {
      "height": 375,
      "width": 1242
    },
    "005979": {
      "height": 375,
      "width": 1242
    },
    "005980": {
      "height": 375,
      "width": 1242
    },
    "005981": {
      "height": 376,
      "width": 1241
    },
    "005982": {
      "height": 375,
      "width": 1242
    },
    "005983": {
      "height": 375,
      "width": 1242
    },
    "005984": {
      "height": 375,
      "width": 1242
    },
    "005985": {
      "height": 375,
      "width": 1242
    },
    "005986": {
      "height": 376,
      "width": 1241
    },
    "005987": {
      "height": 375,
      "width": 1242
    },
    "005988": {
      "height": 370,
      "width": 1224
    },
    "005989": {
      "height": 375,
      "width": 1242
    },
    "005990": {
      "height": 375,
      "width": 1242
    },
    "005991": {
      "height": 375,
      "width": 1242
    },
    "005992": {
      "height": 375,
      "width": 1242
    },
    "005993": {
      "height": 375,
      "width": 1242
    },
    "005994": {
      "height": 370,
      "width": 1224
    },
    "005995": {
      "height": 375,
      "width": 1242
    },
    "005996": {
      "height": 375,
      "width": 1242
    },
    "005997": {
      "height": 375,
      "width": 1242
    },
    "005998": {
      "height": 375,
      "width": 1242
    },
    "005999": {
      "height": 375,
      "width": 1242
    },
    "006000": {
      "height": 375,
      "width": 1242
    },
    "006001": {
      "height": 375,
      "width": 1242
    },
    "006002": {
      "height": 375,
      "width": 1242
    },
    "006003": {
      "height": 375,
      "width": 1242
    },
    "006004": {
      "height": 375,
      "width": 1242
    },
    "006005": {
      "height": 375,
      "width": 1242
    },
    "006006": {
      "height": 370,
      "width": 1224
    },
    "006007": {
      "height": 375,
      "width": 1242
    },
    "006008": {
      "height": 376,
      "width": 1241
    },
    "006009": {
      "height": 375,
      "width": 1242
    },
    "006010": {
      "height": 375,
      "width": 1242
    },
    "006011": {
      "height": 375,
      "width": 1242
    },
    "006012": {
      "height": 375,
      "width": 1242
    },
    "006013": {
      "height": 375,
      "width": 1242
    },
    "006014": {
      "height": 375,
      "width": 1242
    },
    "006015": {
      "height": 374,
      "width": 1238
    },
    "006016": {
      "height": 375,
      "width": 1242
    },
    "006017": {
      "height": 375,
      "width": 1242
    },
    "006018": {
      "height": 375,
      "width": 1242
    },
    "006019": {
      "height": 375,
      "width": 1242
    },
    "006020": {
      "height": 375,
      "width": 1242
    },
    "006021": {
      "height": 375,
      "width": 1242
    },
    "006022": {
      "height": 375,
      "width": 1242
    },
    "006023": {
      "height": 375,
      "width": 1242
    },
    "006024": {
      "height": 370,
      "width": 1224
    },
    "006025": {
      "height": 375,
      "width": 1242
    },
    "006026": {
      "height": 375,
      "width": 1242
    },
    "006027": {
      "height": 375,
      "width": 1242
    },
    "006028": {
      "height": 375,
      "width": 1242
    },
    "006029": {
      "height": 370,
      "width": 1224
    },
    "006030": {
      "height": 375,
      "width": 1242
    },
    "006031": {
      "height": 375,
      "width": 1242
    },
    "006032": {
      "height": 375,
      "width": 1242
    },
    "006033": {
      "height": 375,
      "width": 1242
    },
    "006034": {
      "height": 375,
      "width": 1242
    },
    "006035": {
      "height": 370,
      "width": 1224
    },
    "006036": {
      "height": 375,
      "width": 1242
    },
    "006037": {
      "height": 375,
      "width": 1242
    },
    "006038": {
      "height": 375,
      "width": 1242
    },
    "006039": {
      "height": 375,
      "width": 1242
    },
    "006040": {
      "height": 375,
      "width": 1242
    },
    "006041": {
      "height": 376,
      "width": 1241
    },
    "006042": {
      "height": 375,
      "width": 1242
    },
    "006043": {
      "height": 375,
      "width": 1242
    },
    "006044": {
      "height": 375,
      "width": 1242
    },
    "006045": {
      "height": 376,
      "width": 1241
    },
    "006046": {
      "height": 375,
      "width": 1242
    },
    "006047": {
      "height": 375,
      "width": 1242
    },
    "006048": {
      "height": 376,
      "width": 1241
    },
    "006049": {
      "height": 375,
      "width": 1242
    },
    "006050": {
      "height": 375,
      "width": 1242
    },
    "006051": {
      "height": 370,
      "width": 1224
    },
    "006052": {
      "height": 375,
      "width": 1242
    },
    "006053": {
      "height": 370,
      "width": 1224
    },
    "006054": {
      "height": 375,
      "width": 1242
    },
    "006055": {
      "height": 375,
      "width": 1242
    },
    "006056": {
      "height": 375,
      "width": 1242
    },
    "006057": {
      "height": 375,
      "width": 1242
    },
    "006058": {
      "height": 375,
      "width": 1242
    },
    "006059": {
      "height": 375,
      "width": 1242
    },
    "006060": {
      "height": 375,
      "width": 1242
    },
    "006061": {
      "height": 375,
      "width": 1242
    },
    "006062": {
      "height": 375,
      "width": 1242
    },
    "006063": {
      "height": 376,
      "width": 1241
    },
    "006064": {
      "height": 375,
      "width": 1242
    },
    "006065": {
      "height": 370,
      "width": 1224
    },
    "006066": {
      "height": 375,
      "width": 1242
    },
    "006067": {
      "height": 375,
      "width": 1242
    },
    "006068": {
      "height": 375,
      "width": 1242
    },
    "006069": {
      "height": 375,
      "width": 1242
    },
    "006070": {
      "height": 374,
      "width": 1238
    },
    "006071": {
      "height": 374,
      "width": 1238
    },
    "006072": {
      "height": 375,
      "width": 1242
    },
    "006073": {
      "height": 370,
      "width": 1224
    },
    "006074": {
      "height": 375,
      "width": 1242
    },
    "006075": {
      "height": 370,
      "width": 1224
    },
    "006076": {
      "height": 370,
      "width": 1224
    },
    "006077": {
      "height": 375,
      "width": 1242
    },
    "006078": {
      "height": 375,
      "width": 1242
    },
    "006079": {
      "height": 375,
      "width": 1242
    },
    "006080": {
      "height": 375,
      "width": 1242
    },
    "006081": {
      "height": 375,
      "width": 1242
    },
    "006082": {
      "height": 375,
      "width": 1242
    },
    "006083": {
      "height": 374,
      "width": 1238
    },
    "006084": {
      "height": 375,
      "width": 1242
    },
    "006085": {
      "height": 375,
      "width": 1242
    },
    "006086": {
      "height": 375,
      "width": 1242
    },
    "006087": {
      "height": 370,
      "width": 1224
    },
    "006088": {
      "height": 374,
      "width": 1238
    },
    "006089": {
      "height": 370,
      "width": 1224
    },
    "006090": {
      "height": 375,
      "width": 1242
    },
    "006091": {
      "height": 375,
      "width": 1242
    },
    "006092": {
      "height": 375,
      "width": 1242
    },
    "006093": {
      "height": 375,
      "width": 1242
    },
    "006094": {
      "height": 375,
      "width": 1242
    },
    "006095": {
      "height": 375,
      "width": 1242
    },
    "006096": {
      "height": 375,
      "width": 1242
    },
    "006097": {
      "height": 375,
      "width": 1242
    },
    "006098": {
      "height": 375,
      "width": 1242
    },
    "006099": {
      "height": 375,
      "width": 1242
    },
    "006100": {
      "height": 375,
      "width": 1242
    },
    "006101": {
      "height": 375,
      "width": 1242
    },
    "006102": {
      "height": 375,
      "width": 1242
    },
    "006103": {
      "height": 375,
      "width": 1242
    },
    "006104": {
      "height": 375,
      "width": 1242
    },
    "006105": {
      "height": 375,
      "width": 1242
    },
    "006106": {
      "height": 375,
      "width": 1242
    },
    "006107": {
      "height": 375,
      "width": 1242
    },
    "006108": {
      "height": 375,
      "width": 1242
    },
    "006109": {
      "height": 375,
      "width": 1242
    },
    "006110": {
      "height": 375,
      "width": 1242
    },
    "006111": {
      "height": 375,
      "width": 1242
    },
    "006112": {
      "height": 375,
      "width": 1242
    },
    "006113": {
      "height": 375,
      "width": 1242
    },
    "006114": {
      "height": 375,
      "width": 1242
    },
    "006115": {
      "height": 375,
      "width": 1242
    },
    "006116": {
      "height": 375,
      "width": 1242
    },
    "006117": {
      "height": 375,
      "width": 1242
    },
    "006118": {
      "height": 370,
      "width": 1224
    },
    "006119": {
      "height": 375,
      "width": 1242
    },
    "006120": {
      "height": 375,
      "width": 1242
    },
    "006121": {
      "height": 370,
      "width": 1224
    },
    "006122": {
      "height": 374,
      "width": 1238
    },
    "006123": {
      "height": 375,
      "width": 1242
    },
    "006124": {
      "height": 375,
      "width": 1242
    },
    "006125": {
      "height": 375,
      "width": 1242
    },
    "006126": {
      "height": 375,
      "width": 1242
    },
    "006127": {
      "height": 376,
      "width": 1241
    },
    "006128": {
      "height": 375,
      "width": 1242
    },
    "006129": {
      "height": 370,
      "width": 1224
    },
    "006130": {
      "height": 374,
      "width": 1238
    },
    "006131": {
      "height": 375,
      "width": 1242
    },
    "006132": {
      "height": 375,
      "width": 1242
    },
    "006133": {
      "height": 375,
      "width": 1242
    },
    "006134": {
      "height": 375,
      "width": 1242
    },
    "006135": {
      "height": 375,
      "width": 1242
    },
    "006136": {
      "height": 375,
      "width": 1242
    },
    "006137": {
      "height": 375,
      "width": 1242
    },
    "006138": {
      "height": 375,
      "width": 1242
    },
    "006139": {
      "height": 370,
      "width": 1224
    },
    "006140": {
      "height": 375,
      "width": 1242
    },
    "006141": {
      "height": 375,
      "width": 1242
    },
    "006142": {
      "height": 375,
      "width": 1242
    },
    "006143": {
      "height": 375,
      "width": 1242
    },
    "006144": {
      "height": 376,
      "width": 1241
    },
    "006145": {
      "height": 375,
      "width": 1242
    },
    "006146": {
      "height": 375,
      "width": 1242
    },
    "006147": {
      "height": 375,
      "width": 1242
    },
    "006148": {
      "height": 370,
      "width": 1224
    },
    "006149": {
      "height": 375,
      "width": 1242
    },
    "006150": {
      "height": 375,
      "width": 1242
    },
    "006151": {
      "height": 370,
      "width": 1224
    },
    "006152": {
      "height": 370,
      "width": 1224
    },
    "006153": {
      "height": 375,
      "width": 1242
    },
    "006154": {
      "height": 376,
      "width": 1241
    },
    "006155": {
      "height": 375,
      "width": 1242
    },
    "006156": {
      "height": 375,
      "width": 1242
    },
    "006157": {
      "height": 375,
      "width": 1242
    },
    "006158": {
      "height": 370,
      "width": 1224
    },
    "006159": {
      "height": 375,
      "width": 1242
    },
    "006160": {
      "height": 375,
      "width": 1242
    },
    "006161": {
      "height": 375,
      "width": 1242
    },
    "006162": {
      "height": 375,
      "width": 1242
    },
    "006163": {
      "height": 375,
      "width": 1242
    },
    "006164": {
      "height": 370,
      "width": 1224
    },
    "006165": {
      "height": 375,
      "width": 1242
    },
    "006166": {
      "height": 375,
      "width": 1242
    },
    "006167": {
      "height": 375,
      "width": 1242
    },
    "006168": {
      "height": 375,
      "width": 1242
    },
    "006169": {
      "height": 375,
      "width": 1242
    },
    "006170": {
      "height": 375,
      "width": 1242
    },
    "006171": {
      "height": 375,
      "width": 1242
    },
    "006172": {
      "height": 375,
      "width": 1242
    },
    "006173": {
      "height": 375,
      "width": 1242
    },
    "006174": {
      "height": 375,
      "width": 1242
    },
    "006175": {
      "height": 375,
      "width": 1242
    },
    "006176": {
      "height": 374,
      "width": 1238
    },
    "006177": {
      "height": 375,
      "width": 1242
    },
    "006178": {
      "height": 375,
      "width": 1242
    },
    "006179": {
      "height": 375,
      "width": 1242
    },
    "006180": {
      "height": 375,
      "width": 1242
    },
    "006181": {
      "height": 375,
      "width": 1242
    },
    "006182": {
      "height": 374,
      "width": 1238
    },
    "006183": {
      "height": 375,
      "width": 1242
    },
    "006184": {
      "height": 375,
      "width": 1242
    },
    "006185": {
      "height": 375,
      "width": 1242
    },
    "006186": {
      "height": 375,
      "width": 1242
    },
    "006187": {
      "height": 375,
      "width": 1242
    },
    "006188": {
      "height": 375,
      "width": 1242
    },
    "006189": {
      "height": 375,
      "width": 1242
    },
    "006190": {
      "height": 375,
      "width": 1242
    },
    "006191": {
      "height": 375,
      "width": 1242
    },
    "006192": {
      "height": 375,
      "width": 1242
    },
    "006193": {
      "height": 375,
      "width": 1242
    },
    "006194": {
      "height": 370,
      "width": 1224
    },
    "006195": {
      "height": 375,
      "width": 1242
    },
    "006196": {
      "height": 370,
      "width": 1224
    },
    "006197": {
      "height": 375,
      "width": 1242
    },
    "006198": {
      "height": 375,
      "width": 1242
    },
    "006199": {
      "height": 375,
      "width": 1242
    },
    "006200": {
      "height": 375,
      "width": 1242
    },
    "006201": {
      "height": 370,
      "width": 1224
    },
    "006202": {
      "height": 376,
      "width": 1241
    },
    "006203": {
      "height": 375,
      "width": 1242
    },
    "006204": {
      "height": 375,
      "width": 1242
    },
    "006205": {
      "height": 375,
      "width": 1242
    },
    "006206": {
      "height": 375,
      "width": 1242
    },
    "006207": {
      "height": 375,
      "width": 1242
    },
    "006208": {
      "height": 375,
      "width": 1242
    },
    "006209": {
      "height": 375,
      "width": 1242
    },
    "006210": {
      "height": 374,
      "width": 1238
    },
    "006211": {
      "height": 375,
      "width": 1242
    },
    "006212": {
      "height": 375,
      "width": 1242
    },
    "006213": {
      "height": 375,
      "width": 1242
    },
    "006214": {
      "height": 375,
      "width": 1242
    },
    "006215": {
      "height": 370,
      "width": 1224
    },
    "006216": {
      "height": 375,
      "width": 1242
    },
    "006217": {
      "height": 375,
      "width": 1242
    },
    "006218": {
      "height": 374,
      "width": 1238
    },
    "006219": {
      "height": 375,
      "width": 1242
    },
    "006220": {
      "height": 375,
      "width": 1242
    },
    "006221": {
      "height": 375,
      "width": 1242
    },
    "006222": {
      "height": 370,
      "width": 1224
    },
    "006223": {
      "height": 375,
      "width": 1242
    },
    "006224": {
      "height": 375,
      "width": 1242
    },
    "006225": {
      "height": 375,
      "width": 1242
    },
    "006226": {
      "height": 375,
      "width": 1242
    },
    "006227": {
      "height": 375,
      "width": 1242
    },
    "006228": {
      "height": 370,
      "width": 1224
    },
    "006229": {
      "height": 375,
      "width": 1242
    },
    "006230": {
      "height": 375,
      "width": 1242
    },
    "006231": {
      "height": 375,
      "width": 1242
    },
    "006232": {
      "height": 375,
      "width": 1242
    },
    "006233": {
      "height": 375,
      "width": 1242
    },
    "006234": {
      "height": 375,
      "width": 1242
    },
    "006235": {
      "height": 375,
      "width": 1242
    },
    "006236": {
      "height": 375,
      "width": 1242
    },
    "006237": {
      "height": 375,
      "width": 1242
    },
    "006238": {
      "height": 376,
      "width": 1241
    },
    "006239": {
      "height": 375,
      "width": 1242
    },
    "006240": {
      "height": 370,
      "width": 1224
    },
    "006241": {
      "height": 375,
      "width": 1242
    },
    "006242": {
      "height": 375,
      "width": 1242
    },
    "006243": {
      "height": 375,
      "width": 1242
    },
    "006244": {
      "height": 375,
      "width": 1242
    },
    "006245": {
      "height": 375,
      "width": 1242
    },
    "006246": {
      "height": 375,
      "width": 1242
    },
    "006247": {
      "height": 375,
      "width": 1242
    },
    "006248": {
      "height": 375,
      "width": 1242
    },
    "006249": {
      "height": 375,
      "width": 1242
    },
    "006250": {
      "height": 375,
      "width": 1242
    },
    "006251": {
      "height": 375,
      "width": 1242
    },
    "006252": {
      "height": 375,
      "width": 1242
    },
    "006253": {
      "height": 375,
      "width": 1242
    },
    "006254": {
      "height": 375,
      "width": 1242
    },
    "006255": {
      "height": 375,
      "width": 1242
    },
    "006256": {
      "height": 375,
      "width": 1242
    },
    "006257": {
      "height": 375,
      "width": 1242
    },
    "006258": {
      "height": 370,
      "width": 1224
    },
    "006259": {
      "height": 375,
      "width": 1242
    },
    "006260": {
      "height": 375,
      "width": 1242
    },
    "006261": {
      "height": 375,
      "width": 1242
    },
    "006262": {
      "height": 375,
      "width": 1242
    },
    "006263": {
      "height": 375,
      "width": 1242
    },
    "006264": {
      "height": 375,
      "width": 1242
    },
    "006265": {
      "height": 375,
      "width": 1242
    },
    "006266": {
      "height": 375,
      "width": 1242
    },
    "006267": {
      "height": 375,
      "width": 1242
    },
    "006268": {
      "height": 375,
      "width": 1242
    },
    "006269": {
      "height": 375,
      "width": 1242
    },
    "006270": {
      "height": 375,
      "width": 1242
    },
    "006271": {
      "height": 375,
      "width": 1242
    },
    "006272": {
      "height": 375,
      "width": 1242
    },
    "006273": {
      "height": 375,
      "width": 1242
    },
    "006274": {
      "height": 375,
      "width": 1242
    },
    "006275": {
      "height": 375,
      "width": 1242
    },
    "006276": {
      "height": 370,
      "width": 1224
    },
    "006277": {
      "height": 375,
      "width": 1242
    },
    "006278": {
      "height": 375,
      "width": 1242
    },
    "006279": {
      "height": 375,
      "width": 1242
    },
    "006280": {
      "height": 375,
      "width": 1242
    },
    "006281": {
      "height": 375,
      "width": 1242
    },
    "006282": {
      "height": 375,
      "width": 1242
    },
    "006283": {
      "height": 375,
      "width": 1242
    },
    "006284": {
      "height": 370,
      "width": 1224
    },
    "006285": {
      "height": 375,
      "width": 1242
    },
    "006286": {
      "height": 375,
      "width": 1242
    },
    "006287": {
      "height": 375,
      "width": 1242
    },
    "006288": {
      "height": 375,
      "width": 1242
    },
    "006289": {
      "height": 375,
      "width": 1242
    },
    "006290": {
      "height": 375,
      "width": 1242
    },
    "006291": {
      "height": 375,
      "width": 1242
    },
    "006292": {
      "height": 370,
      "width": 1224
    },
    "006293": {
      "height": 375,
      "width": 1242
    },
    "006294": {
      "height": 370,
      "width": 1224
    },
    "006295": {
      "height": 375,
      "width": 1242
    },
    "006296": {
      "height": 375,
      "width": 1242
    },
    "006297": {
      "height": 375,
      "width": 1242
    },
    "006298": {
      "height": 375,
      "width": 1242
    },
    "006299": {
      "height": 370,
      "width": 1224
    },
    "006300": {
      "height": 375,
      "width": 1242
    },
    "006301": {
      "height": 375,
      "width": 1242
    },
    "006302": {
      "height": 375,
      "width": 1242
    },
    "006303": {
      "height": 375,
      "width": 1242
    },
    "006304": {
      "height": 375,
      "width": 1242
    },
    "006305": {
      "height": 375,
      "width": 1242
    },
    "006306": {
      "height": 375,
      "width": 1242
    },
    "006307": {
      "height": 375,
      "width": 1242
    },
    "006308": {
      "height": 370,
      "width": 1224
    },
    "006309": {
      "height": 375,
      "width": 1242
    },
    "006310": {
      "height": 375,
      "width": 1242
    },
    "006311": {
      "height": 375,
      "width": 1242
    },
    "006312": {
      "height": 376,
      "width": 1241
    },
    "006313": {
      "height": 375,
      "width": 1242
    },
    "006314": {
      "height": 375,
      "width": 1242
    },
    "006315": {
      "height": 375,
      "width": 1242
    },
    "006316": {
      "height": 375,
      "width": 1242
    },
    "006317": {
      "height": 375,
      "width": 1242
    },
    "006318": {
      "height": 375,
      "width": 1242
    },
    "006319": {
      "height": 375,
      "width": 1242
    },
    "006320": {
      "height": 375,
      "width": 1242
    },
    "006321": {
      "height": 375,
      "width": 1242
    },
    "006322": {
      "height": 375,
      "width": 1242
    },
    "006323": {
      "height": 375,
      "width": 1242
    },
    "006324": {
      "height": 375,
      "width": 1242
    },
    "006325": {
      "height": 375,
      "width": 1242
    },
    "006326": {
      "height": 375,
      "width": 1242
    },
    "006327": {
      "height": 375,
      "width": 1242
    },
    "006328": {
      "height": 375,
      "width": 1242
    },
    "006329": {
      "height": 375,
      "width": 1242
    },
    "006330": {
      "height": 375,
      "width": 1242
    },
    "006331": {
      "height": 375,
      "width": 1242
    },
    "006332": {
      "height": 375,
      "width": 1242
    },
    "006333": {
      "height": 375,
      "width": 1242
    },
    "006334": {
      "height": 375,
      "width": 1242
    },
    "006335": {
      "height": 375,
      "width": 1242
    },
    "006336": {
      "height": 375,
      "width": 1242
    },
    "006337": {
      "height": 370,
      "width": 1224
    },
    "006338": {
      "height": 375,
      "width": 1242
    },
    "006339": {
      "height": 375,
      "width": 1242
    },
    "006340": {
      "height": 375,
      "width": 1242
    },
    "006341": {
      "height": 370,
      "width": 1224
    },
    "006342": {
      "height": 375,
      "width": 1242
    },
    "006343": {
      "height": 375,
      "width": 1242
    },
    "006344": {
      "height": 376,
      "width": 1241
    },
    "006345": {
      "height": 376,
      "width": 1241
    },
    "006346": {
      "height": 375,
      "width": 1242
    },
    "006347": {
      "height": 375,
      "width": 1242
    },
    "006348": {
      "height": 375,
      "width": 1242
    },
    "006349": {
      "height": 375,
      "width": 1242
    },
    "006350": {
      "height": 375,
      "width": 1242
    },
    "006351": {
      "height": 375,
      "width": 1242
    },
    "006352": {
      "height": 375,
      "width": 1242
    },
    "006353": {
      "height": 375,
      "width": 1242
    },
    "006354": {
      "height": 370,
      "width": 1224
    },
    "006355": {
      "height": 375,
      "width": 1242
    },
    "006356": {
      "height": 375,
      "width": 1242
    },
    "006357": {
      "height": 375,
      "width": 1242
    },
    "006358": {
      "height": 375,
      "width": 1242
    },
    "006359": {
      "height": 375,
      "width": 1242
    },
    "006360": {
      "height": 375,
      "width": 1242
    },
    "006361": {
      "height": 375,
      "width": 1242
    },
    "006362": {
      "height": 375,
      "width": 1242
    },
    "006363": {
      "height": 375,
      "width": 1242
    },
    "006364": {
      "height": 375,
      "width": 1242
    },
    "006365": {
      "height": 375,
      "width": 1242
    },
    "006366": {
      "height": 375,
      "width": 1242
    },
    "006367": {
      "height": 375,
      "width": 1242
    },
    "006368": {
      "height": 375,
      "width": 1242
    },
    "006369": {
      "height": 375,
      "width": 1242
    },
    "006370": {
      "height": 375,
      "width": 1242
    },
    "006371": {
      "height": 375,
      "width": 1242
    },
    "006372": {
      "height": 375,
      "width": 1242
    },
    "006373": {
      "height": 375,
      "width": 1242
    },
    "006374": {
      "height": 375,
      "width": 1242
    },
    "006375": {
      "height": 375,
      "width": 1242
    },
    "006376": {
      "height": 375,
      "width": 1242
    },
    "006377": {
      "height": 375,
      "width": 1242
    },
    "006378": {
      "height": 375,
      "width": 1242
    },
    "006379": {
      "height": 375,
      "width": 1242
    },
    "006380": {
      "height": 375,
      "width": 1242
    },
    "006381": {
      "height": 375,
      "width": 1242
    },
    "006382": {
      "height": 375,
      "width": 1242
    },
    "006383": {
      "height": 375,
      "width": 1242
    },
    "006384": {
      "height": 375,
      "width": 1242
    },
    "006385": {
      "height": 375,
      "width": 1242
    },
    "006386": {
      "height": 375,
      "width": 1242
    },
    "006387": {
      "height": 375,
      "width": 1242
    },
    "006388": {
      "height": 375,
      "width": 1242
    },
    "006389": {
      "height": 375,
      "width": 1242
    },
    "006390": {
      "height": 375,
      "width": 1242
    },
    "006391": {
      "height": 375,
      "width": 1242
    },
    "006392": {
      "height": 375,
      "width": 1242
    },
    "006393": {
      "height": 375,
      "width": 1242
    },
    "006394": {
      "height": 376,
      "width": 1241
    },
    "006395": {
      "height": 374,
      "width": 1238
    },
    "006396": {
      "height": 370,
      "width": 1224
    },
    "006397": {
      "height": 375,
      "width": 1242
    },
    "006398": {
      "height": 375,
      "width": 1242
    },
    "006399": {
      "height": 375,
      "width": 1242
    },
    "006400": {
      "height": 375,
      "width": 1242
    },
    "006401": {
      "height": 375,
      "width": 1242
    },
    "006402": {
      "height": 375,
      "width": 1242
    },
    "006403": {
      "height": 375,
      "width": 1242
    },
    "006404": {
      "height": 375,
      "width": 1242
    },
    "006405": {
      "height": 375,
      "width": 1242
    },
    "006406": {
      "height": 375,
      "width": 1242
    },
    "006407": {
      "height": 370,
      "width": 1224
    },
    "006408": {
      "height": 375,
      "width": 1242
    },
    "006409": {
      "height": 375,
      "width": 1242
    },
    "006410": {
      "height": 375,
      "width": 1242
    },
    "006411": {
      "height": 370,
      "width": 1224
    },
    "006412": {
      "height": 375,
      "width": 1242
    },
    "006413": {
      "height": 375,
      "width": 1242
    },
    "006414": {
      "height": 375,
      "width": 1242
    },
    "006415": {
      "height": 375,
      "width": 1242
    },
    "006416": {
      "height": 375,
      "width": 1242
    },
    "006417": {
      "height": 375,
      "width": 1242
    },
    "006418": {
      "height": 375,
      "width": 1242
    },
    "006419": {
      "height": 375,
      "width": 1242
    },
    "006420": {
      "height": 376,
      "width": 1241
    },
    "006421": {
      "height": 375,
      "width": 1242
    },
    "006422": {
      "height": 370,
      "width": 1224
    },
    "006423": {
      "height": 376,
      "width": 1241
    },
    "006424": {
      "height": 374,
      "width": 1238
    },
    "006425": {
      "height": 375,
      "width": 1242
    },
    "006426": {
      "height": 375,
      "width": 1242
    },
    "006427": {
      "height": 374,
      "width": 1238
    },
    "006428": {
      "height": 375,
      "width": 1242
    },
    "006429": {
      "height": 375,
      "width": 1242
    },
    "006430": {
      "height": 374,
      "width": 1238
    },
    "006431": {
      "height": 375,
      "width": 1242
    },
    "006432": {
      "height": 375,
      "width": 1242
    },
    "006433": {
      "height": 370,
      "width": 1224
    },
    "006434": {
      "height": 370,
      "width": 1224
    },
    "006435": {
      "height": 375,
      "width": 1242
    },
    "006436": {
      "height": 375,
      "width": 1242
    },
    "006437": {
      "height": 375,
      "width": 1242
    },
    "006438": {
      "height": 375,
      "width": 1242
    },
    "006439": {
      "height": 375,
      "width": 1242
    },
    "006440": {
      "height": 375,
      "width": 1242
    },
    "006441": {
      "height": 374,
      "width": 1238
    },
    "006442": {
      "height": 375,
      "width": 1242
    },
    "006443": {
      "height": 375,
      "width": 1242
    },
    "006444": {
      "height": 374,
      "width": 1238
    },
    "006445": {
      "height": 375,
      "width": 1242
    },
    "006446": {
      "height": 375,
      "width": 1242
    },
    "006447": {
      "height": 375,
      "width": 1242
    },
    "006448": {
      "height": 375,
      "width": 1242
    },
    "006449": {
      "height": 375,
      "width": 1242
    },
    "006450": {
      "height": 375,
      "width": 1242
    },
    "006451": {
      "height": 370,
      "width": 1224
    },
    "006452": {
      "height": 375,
      "width": 1242
    },
    "006453": {
      "height": 374,
      "width": 1238
    },
    "006454": {
      "height": 375,
      "width": 1242
    },
    "006455": {
      "height": 374,
      "width": 1238
    },
    "006456": {
      "height": 375,
      "width": 1242
    },
    "006457": {
      "height": 375,
      "width": 1242
    },
    "006458": {
      "height": 370,
      "width": 1224
    },
    "006459": {
      "height": 375,
      "width": 1242
    },
    "006460": {
      "height": 375,
      "width": 1242
    },
    "006461": {
      "height": 375,
      "width": 1242
    },
    "006462": {
      "height": 375,
      "width": 1242
    },
    "006463": {
      "height": 375,
      "width": 1242
    },
    "006464": {
      "height": 375,
      "width": 1242
    },
    "006465": {
      "height": 375,
      "width": 1242
    },
    "006466": {
      "height": 375,
      "width": 1242
    },
    "006467": {
      "height": 375,
      "width": 1242
    },
    "006468": {
      "height": 375,
      "width": 1242
    },
    "006469": {
      "height": 376,
      "width": 1241
    },
    "006470": {
      "height": 375,
      "width": 1242
    },
    "006471": {
      "height": 370,
      "width": 1224
    },
    "006472": {
      "height": 370,
      "width": 1224
    },
    "006473": {
      "height": 375,
      "width": 1242
    },
    "006474": {
      "height": 375,
      "width": 1242
    },
    "006475": {
      "height": 374,
      "width": 1238
    },
    "006476": {
      "height": 375,
      "width": 1242
    },
    "006477": {
      "height": 375,
      "width": 1242
    },
    "006478": {
      "height": 374,
      "width": 1238
    },
    "006479": {
      "height": 375,
      "width": 1242
    },
    "006480": {
      "height": 370,
      "width": 1224
    },
    "006481": {
      "height": 375,
      "width": 1242
    },
    "006482": {
      "height": 375,
      "width": 1242
    },
    "006483": {
      "height": 375,
      "width": 1242
    },
    "006484": {
      "height": 375,
      "width": 1242
    },
    "006485": {
      "height": 375,
      "width": 1242
    },
    "006486": {
      "height": 370,
      "width": 1224
    },
    "006487": {
      "height": 375,
      "width": 1242
    },
    "006488": {
      "height": 374,
      "width": 1238
    },
    "006489": {
      "height": 375,
      "width": 1242
    },
    "006490": {
      "height": 375,
      "width": 1242
    },
    "006491": {
      "height": 375,
      "width": 1242
    },
    "006492": {
      "height": 375,
      "width": 1242
    },
    "006493": {
      "height": 375,
      "width": 1242
    },
    "006494": {
      "height": 375,
      "width": 1242
    },
    "006495": {
      "height": 375,
      "width": 1242
    },
    "006496": {
      "height": 370,
      "width": 1224
    },
    "006497": {
      "height": 376,
      "width": 1241
    },
    "006498": {
      "height": 375,
      "width": 1242
    },
    "006499": {
      "height": 375,
      "width": 1242
    },
    "006500": {
      "height": 375,
      "width": 1242
    },
    "006501": {
      "height": 375,
      "width": 1242
    },
    "006502": {
      "height": 370,
      "width": 1224
    },
    "006503": {
      "height": 370,
      "width": 1224
    },
    "006504": {
      "height": 375,
      "width": 1242
    },
    "006505": {
      "height": 375,
      "width": 1242
    },
    "006506": {
      "height": 374,
      "width": 1238
    },
    "006507": {
      "height": 375,
      "width": 1242
    },
    "006508": {
      "height": 375,
      "width": 1242
    },
    "006509": {
      "height": 375,
      "width": 1242
    },
    "006510": {
      "height": 374,
      "width": 1238
    },
    "006511": {
      "height": 375,
      "width": 1242
    },
    "006512": {
      "height": 375,
      "width": 1242
    },
    "006513": {
      "height": 375,
      "width": 1242
    },
    "006514": {
      "height": 375,
      "width": 1242
    },
    "006515": {
      "height": 375,
      "width": 1242
    },
    "006516": {
      "height": 375,
      "width": 1242
    },
    "006517": {
      "height": 375,
      "width": 1242
    },
    "006518": {
      "height": 375,
      "width": 1242
    },
    "006519": {
      "height": 375,
      "width": 1242
    },
    "006520": {
      "height": 375,
      "width": 1242
    },
    "006521": {
      "height": 375,
      "width": 1242
    },
    "006522": {
      "height": 370,
      "width": 1224
    },
    "006523": {
      "height": 375,
      "width": 1242
    },
    "006524": {
      "height": 375,
      "width": 1242
    },
    "006525": {
      "height": 375,
      "width": 1242
    },
    "006526": {
      "height": 370,
      "width": 1224
    },
    "006527": {
      "height": 375,
      "width": 1242
    },
    "006528": {
      "height": 375,
      "width": 1242
    },
    "006529": {
      "height": 375,
      "width": 1242
    },
    "006530": {
      "height": 375,
      "width": 1242
    },
    "006531": {
      "height": 375,
      "width": 1242
    },
    "006532": {
      "height": 375,
      "width": 1242
    },
    "006533": {
      "height": 375,
      "width": 1242
    },
    "006534": {
      "height": 375,
      "width": 1242
    },
    "006535": {
      "height": 375,
      "width": 1242
    },
    "006536": {
      "height": 375,
      "width": 1242
    },
    "006537": {
      "height": 375,
      "width": 1242
    },
    "006538": {
      "height": 375,
      "width": 1242
    },
    "006539": {
      "height": 370,
      "width": 1224
    },
    "006540": {
      "height": 375,
      "width": 1242
    },
    "006541": {
      "height": 375,
      "width": 1242
    },
    "006542": {
      "height": 375,
      "width": 1242
    },
    "006543": {
      "height": 375,
      "width": 1242
    },
    "006544": {
      "height": 375,
      "width": 1242
    },
    "006545": {
      "height": 375,
      "width": 1242
    },
    "006546": {
      "height": 375,
      "width": 1242
    },
    "006547": {
      "height": 375,
      "width": 1242
    },
    "006548": {
      "height": 375,
      "width": 1242
    },
    "006549": {
      "height": 375,
      "width": 1242
    },
    "006550": {
      "height": 375,
      "width": 1242
    },
    "006551": {
      "height": 375,
      "width": 1242
    },
    "006552": {
      "height": 370,
      "width": 1224
    },
    "006553": {
      "height": 375,
      "width": 1242
    },
    "006554": {
      "height": 375,
      "width": 1242
    },
    "006555": {
      "height": 375,
      "width": 1242
    },
    "006556": {
      "height": 375,
      "width": 1242
    },
    "006557": {
      "height": 375,
      "width": 1242
    },
    "006558": {
      "height": 375,
      "width": 1242
    },
    "006559": {
      "height": 375,
      "width": 1242
    },
    "006560": {
      "height": 375,
      "width": 1242
    },
    "006561": {
      "height": 375,
      "width": 1242
    },
    "006562": {
      "height": 375,
      "width": 1242
    },
    "006563": {
      "height": 375,
      "width": 1242
    },
    "006564": {
      "height": 375,
      "width": 1242
    },
    "006565": {
      "height": 375,
      "width": 1242
    },
    "006566": {
      "height": 375,
      "width": 1242
    },
    "006567": {
      "height": 375,
      "width": 1242
    },
    "006568": {
      "height": 375,
      "width": 1242
    },
    "006569": {
      "height": 375,
      "width": 1242
    },
    "006570": {
      "height": 375,
      "width": 1242
    },
    "006571": {
      "height": 375,
      "width": 1242
    },
    "006572": {
      "height": 375,
      "width": 1242
    },
    "006573": {
      "height": 370,
      "width": 1224
    },
    "006574": {
      "height": 375,
      "width": 1242
    },
    "006575": {
      "height": 370,
      "width": 1224
    },
    "006576": {
      "height": 375,
      "width": 1242
    },
    "006577": {
      "height": 375,
      "width": 1242
    },
    "006578": {
      "height": 374,
      "width": 1238
    },
    "006579": {
      "height": 375,
      "width": 1242
    },
    "006580": {
      "height": 375,
      "width": 1242
    },
    "006581": {
      "height": 375,
      "width": 1242
    },
    "006582": {
      "height": 375,
      "width": 1242
    },
    "006583": {
      "height": 370,
      "width": 1224
    },
    "006584": {
      "height": 375,
      "width": 1242
    },
    "006585": {
      "height": 370,
      "width": 1224
    },
    "006586": {
      "height": 375,
      "width": 1242
    },
    "006587": {
      "height": 375,
      "width": 1242
    },
    "006588": {
      "height": 375,
      "width": 1242
    },
    "006589": {
      "height": 375,
      "width": 1242
    },
    "006590": {
      "height": 375,
      "width": 1242
    },
    "006591": {
      "height": 375,
      "width": 1242
    },
    "006592": {
      "height": 375,
      "width": 1242
    },
    "006593": {
      "height": 375,
      "width": 1242
    },
    "006594": {
      "height": 375,
      "width": 1242
    },
    "006595": {
      "height": 375,
      "width": 1242
    },
    "006596": {
      "height": 375,
      "width": 1242
    },
    "006597": {
      "height": 375,
      "width": 1242
    },
    "006598": {
      "height": 375,
      "width": 1242
    },
    "006599": {
      "height": 375,
      "width": 1242
    },
    "006600": {
      "height": 375,
      "width": 1242
    },
    "006601": {
      "height": 375,
      "width": 1242
    },
    "006602": {
      "height": 375,
      "width": 1242
    },
    "006603": {
      "height": 375,
      "width": 1242
    },
    "006604": {
      "height": 375,
      "width": 1242
    },
    "006605": {
      "height": 375,
      "width": 1242
    },
    "006606": {
      "height": 374,
      "width": 1238
    },
    "006607": {
      "height": 375,
      "width": 1242
    },
    "006608": {
      "height": 375,
      "width": 1242
    },
    "006609": {
      "height": 375,
      "width": 1242
    },
    "006610": {
      "height": 375,
      "width": 1242
    },
    "006611": {
      "height": 375,
      "width": 1242
    },
    "006612": {
      "height": 375,
      "width": 1242
    },
    "006613": {
      "height": 374,
      "width": 1238
    },
    "006614": {
      "height": 375,
      "width": 1242
    },
    "006615": {
      "height": 375,
      "width": 1242
    },
    "006616": {
      "height": 375,
      "width": 1242
    },
    "006617": {
      "height": 370,
      "width": 1224
    },
    "006618": {
      "height": 375,
      "width": 1242
    },
    "006619": {
      "height": 375,
      "width": 1242
    },
    "006620": {
      "height": 375,
      "width": 1242
    },
    "006621": {
      "height": 375,
      "width": 1242
    },
    "006622": {
      "height": 375,
      "width": 1242
    },
    "006623": {
      "height": 375,
      "width": 1242
    },
    "006624": {
      "height": 376,
      "width": 1241
    },
    "006625": {
      "height": 375,
      "width": 1242
    },
    "006626": {
      "height": 375,
      "width": 1242
    },
    "006627": {
      "height": 375,
      "width": 1242
    },
    "006628": {
      "height": 375,
      "width": 1242
    },
    "006629": {
      "height": 375,
      "width": 1242
    },
    "006630": {
      "height": 375,
      "width": 1242
    },
    "006631": {
      "height": 375,
      "width": 1242
    },
    "006632": {
      "height": 375,
      "width": 1242
    },
    "006633": {
      "height": 375,
      "width": 1242
    },
    "006634": {
      "height": 375,
      "width": 1242
    },
    "006635": {
      "height": 375,
      "width": 1242
    },
    "006636": {
      "height": 375,
      "width": 1242
    },
    "006637": {
      "height": 376,
      "width": 1241
    },
    "006638": {
      "height": 375,
      "width": 1242
    },
    "006639": {
      "height": 375,
      "width": 1242
    },
    "006640": {
      "height": 375,
      "width": 1242
    },
    "006641": {
      "height": 375,
      "width": 1242
    },
    "006642": {
      "height": 375,
      "width": 1242
    },
    "006643": {
      "height": 375,
      "width": 1242
    },
    "006644": {
      "height": 375,
      "width": 1242
    },
    "006645": {
      "height": 375,
      "width": 1242
    },
    "006646": {
      "height": 375,
      "width": 1242
    },
    "006647": {
      "height": 375,
      "width": 1242
    },
    "006648": {
      "height": 375,
      "width": 1242
    },
    "006649": {
      "height": 375,
      "width": 1242
    },
    "006650": {
      "height": 375,
      "width": 1242
    },
    "006651": {
      "height": 375,
      "width": 1242
    },
    "006652": {
      "height": 370,
      "width": 1224
    },
    "006653": {
      "height": 370,
      "width": 1224
    },
    "006654": {
      "height": 375,
      "width": 1242
    },
    "006655": {
      "height": 375,
      "width": 1242
    },
    "006656": {
      "height": 370,
      "width": 1224
    },
    "006657": {
      "height": 375,
      "width": 1242
    },
    "006658": {
      "height": 376,
      "width": 1241
    },
    "006659": {
      "height": 375,
      "width": 1242
    },
    "006660": {
      "height": 375,
      "width": 1242
    },
    "006661": {
      "height": 370,
      "width": 1224
    },
    "006662": {
      "height": 375,
      "width": 1242
    },
    "006663": {
      "height": 375,
      "width": 1242
    },
    "006664": {
      "height": 374,
      "width": 1238
    },
    "006665": {
      "height": 374,
      "width": 1238
    },
    "006666": {
      "height": 370,
      "width": 1224
    },
    "006667": {
      "height": 375,
      "width": 1242
    },
    "006668": {
      "height": 375,
      "width": 1242
    },
    "006669": {
      "height": 375,
      "width": 1242
    },
    "006670": {
      "height": 376,
      "width": 1241
    },
    "006671": {
      "height": 374,
      "width": 1238
    },
    "006672": {
      "height": 370,
      "width": 1224
    },
    "006673": {
      "height": 375,
      "width": 1242
    },
    "006674": {
      "height": 375,
      "width": 1242
    },
    "006675": {
      "height": 375,
      "width": 1242
    },
    "006676": {
      "height": 370,
      "width": 1224
    },
    "006677": {
      "height": 375,
      "width": 1242
    },
    "006678": {
      "height": 375,
      "width": 1242
    },
    "006679": {
      "height": 375,
      "width": 1242
    },
    "006680": {
      "height": 375,
      "width": 1242
    },
    "006681": {
      "height": 375,
      "width": 1242
    },
    "006682": {
      "height": 370,
      "width": 1224
    },
    "006683": {
      "height": 375,
      "width": 1242
    },
    "006684": {
      "height": 375,
      "width": 1242
    },
    "006685": {
      "height": 370,
      "width": 1224
    },
    "006686": {
      "height": 375,
      "width": 1242
    },
    "006687": {
      "height": 375,
      "width": 1242
    },
    "006688": {
      "height": 375,
      "width": 1242
    },
    "006689": {
      "height": 375,
      "width": 1242
    },
    "006690": {
      "height": 375,
      "width": 1242
    },
    "006691": {
      "height": 375,
      "width": 1242
    },
    "006692": {
      "height": 375,
      "width": 1242
    },
    "006693": {
      "height": 375,
      "width": 1242
    },
    "006694": {
      "height": 375,
      "width": 1242
    },
    "006695": {
      "height": 375,
      "width": 1242
    },
    "006696": {
      "height": 375,
      "width": 1242
    },
    "006697": {
      "height": 374,
      "width": 1238
    },
    "006698": {
      "height": 375,
      "width": 1242
    },
    "006699": {
      "height": 370,
      "width": 1224
    },
    "006700": {
      "height": 375,
      "width": 1242
    },
    "006701": {
      "height": 375,
      "width": 1242
    },
    "006702": {
      "height": 375,
      "width": 1242
    },
    "006703": {
      "height": 374,
      "width": 1238
    },
    "006704": {
      "height": 375,
      "width": 1242
    },
    "006705": {
      "height": 375,
      "width": 1242
    },
    "006706": {
      "height": 375,
      "width": 1242
    },
    "006707": {
      "height": 375,
      "width": 1242
    },
    "006708": {
      "height": 375,
      "width": 1242
    },
    "006709": {
      "height": 375,
      "width": 1242
    },
    "006710": {
      "height": 375,
      "width": 1242
    },
    "006711": {
      "height": 375,
      "width": 1242
    },
    "006712": {
      "height": 375,
      "width": 1242
    },
    "006713": {
      "height": 375,
      "width": 1242
    },
    "006714": {
      "height": 375,
      "width": 1242
    },
    "006715": {
      "height": 375,
      "width": 1242
    },
    "006716": {
      "height": 375,
      "width": 1242
    },
    "006717": {
      "height": 375,
      "width": 1242
    },
    "006718": {
      "height": 375,
      "width": 1242
    },
    "006719": {
      "height": 375,
      "width": 1242
    },
    "006720": {
      "height": 375,
      "width": 1242
    },
    "006721": {
      "height": 375,
      "width": 1242
    },
    "006722": {
      "height": 370,
      "width": 1224
    },
    "006723": {
      "height": 375,
      "width": 1242
    },
    "006724": {
      "height": 375,
      "width": 1242
    },
    "006725": {
      "height": 375,
      "width": 1242
    },
    "006726": {
      "height": 376,
      "width": 1241
    },
    "006727": {
      "height": 375,
      "width": 1242
    },
    "006728": {
      "height": 375,
      "width": 1242
    },
    "006729": {
      "height": 375,
      "width": 1242
    },
    "006730": {
      "height": 375,
      "width": 1242
    },
    "006731": {
      "height": 375,
      "width": 1242
    },
    "006732": {
      "height": 375,
      "width": 1242
    },
    "006733": {
      "height": 375,
      "width": 1242
    },
    "006734": {
      "height": 375,
      "width": 1242
    },
    "006735": {
      "height": 375,
      "width": 1242
    },
    "006736": {
      "height": 375,
      "width": 1242
    },
    "006737": {
      "height": 375,
      "width": 1242
    },
    "006738": {
      "height": 375,
      "width": 1242
    },
    "006739": {
      "height": 375,
      "width": 1242
    },
    "006740": {
      "height": 375,
      "width": 1242
    },
    "006741": {
      "height": 375,
      "width": 1242
    },
    "006742": {
      "height": 375,
      "width": 1242
    },
    "006743": {
      "height": 375,
      "width": 1242
    },
    "006744": {
      "height": 370,
      "width": 1224
    },
    "006745": {
      "height": 374,
      "width": 1238
    },
    "006746": {
      "height": 375,
      "width": 1242
    },
    "006747": {
      "height": 375,
      "width": 1242
    },
    "006748": {
      "height": 375,
      "width": 1242
    },
    "006749": {
      "height": 370,
      "width": 1224
    },
    "006750": {
      "height": 370,
      "width": 1224
    },
    "006751": {
      "height": 375,
      "width": 1242
    },
    "006752": {
      "height": 375,
      "width": 1242
    },
    "006753": {
      "height": 375,
      "width": 1242
    },
    "006754": {
      "height": 375,
      "width": 1242
    },
    "006755": {
      "height": 375,
      "width": 1242
    },
    "006756": {
      "height": 375,
      "width": 1242
    },
    "006757": {
      "height": 375,
      "width": 1242
    },
    "006758": {
      "height": 375,
      "width": 1242
    },
    "006759": {
      "height": 375,
      "width": 1242
    },
    "006760": {
      "height": 375,
      "width": 1242
    },
    "006761": {
      "height": 375,
      "width": 1242
    },
    "006762": {
      "height": 375,
      "width": 1242
    },
    "006763": {
      "height": 375,
      "width": 1242
    },
    "006764": {
      "height": 375,
      "width": 1242
    },
    "006765": {
      "height": 375,
      "width": 1242
    },
    "006766": {
      "height": 375,
      "width": 1242
    },
    "006767": {
      "height": 375,
      "width": 1242
    },
    "006768": {
      "height": 374,
      "width": 1238
    },
    "006769": {
      "height": 375,
      "width": 1242
    },
    "006770": {
      "height": 375,
      "width": 1242
    },
    "006771": {
      "height": 375,
      "width": 1242
    },
    "006772": {
      "height": 375,
      "width": 1242
    },
    "006773": {
      "height": 376,
      "width": 1241
    },
    "006774": {
      "height": 375,
      "width": 1242
    },
    "006775": {
      "height": 375,
      "width": 1242
    },
    "006776": {
      "height": 375,
      "width": 1242
    },
    "006777": {
      "height": 375,
      "width": 1242
    },
    "006778": {
      "height": 375,
      "width": 1242
    },
    "006779": {
      "height": 375,
      "width": 1242
    },
    "006780": {
      "height": 376,
      "width": 1241
    },
    "006781": {
      "height": 376,
      "width": 1241
    },
    "006782": {
      "height": 375,
      "width": 1242
    },
    "006783": {
      "height": 375,
      "width": 1242
    },
    "006784": {
      "height": 375,
      "width": 1242
    },
    "006785": {
      "height": 375,
      "width": 1242
    },
    "006786": {
      "height": 375,
      "width": 1242
    },
    "006787": {
      "height": 375,
      "width": 1242
    },
    "006788": {
      "height": 375,
      "width": 1242
    },
    "006789": {
      "height": 376,
      "width": 1241
    },
    "006790": {
      "height": 375,
      "width": 1242
    },
    "006791": {
      "height": 375,
      "width": 1242
    },
    "006792": {
      "height": 375,
      "width": 1242
    },
    "006793": {
      "height": 375,
      "width": 1242
    },
    "006794": {
      "height": 375,
      "width": 1242
    },
    "006795": {
      "height": 370,
      "width": 1224
    },
    "006796": {
      "height": 375,
      "width": 1242
    },
    "006797": {
      "height": 375,
      "width": 1242
    },
    "006798": {
      "height": 375,
      "width": 1242
    },
    "006799": {
      "height": 370,
      "width": 1224
    },
    "006800": {
      "height": 374,
      "width": 1238
    },
    "006801": {
      "height": 370,
      "width": 1224
    },
    "006802": {
      "height": 375,
      "width": 1242
    },
    "006803": {
      "height": 375,
      "width": 1242
    },
    "006804": {
      "height": 370,
      "width": 1224
    },
    "006805": {
      "height": 375,
      "width": 1242
    },
    "006806": {
      "height": 375,
      "width": 1242
    },
    "006807": {
      "height": 370,
      "width": 1224
    },
    "006808": {
      "height": 375,
      "width": 1242
    },
    "006809": {
      "height": 375,
      "width": 1242
    },
    "006810": {
      "height": 375,
      "width": 1242
    },
    "006811": {
      "height": 375,
      "width": 1242
    },
    "006812": {
      "height": 375,
      "width": 1242
    },
    "006813": {
      "height": 370,
      "width": 1224
    },
    "006814": {
      "height": 375,
      "width": 1242
    },
    "006815": {
      "height": 375,
      "width": 1242
    },
    "006816": {
      "height": 375,
      "width": 1242
    },
    "006817": {
      "height": 375,
      "width": 1242
    },
    "006818": {
      "height": 374,
      "width": 1238
    },
    "006819": {
      "height": 375,
      "width": 1242
    },
    "006820": {
      "height": 375,
      "width": 1242
    },
    "006821": {
      "height": 375,
      "width": 1242
    },
    "006822": {
      "height": 375,
      "width": 1242
    },
    "006823": {
      "height": 375,
      "width": 1242
    },
    "006824": {
      "height": 375,
      "width": 1242
    },
    "006825": {
      "height": 375,
      "width": 1242
    },
    "006826": {
      "height": 375,
      "width": 1242
    },
    "006827": {
      "height": 375,
      "width": 1242
    },
    "006828": {
      "height": 374,
      "width": 1238
    },
    "006829": {
      "height": 375,
      "width": 1242
    },
    "006830": {
      "height": 370,
      "width": 1224
    },
    "006831": {
      "height": 375,
      "width": 1242
    },
    "006832": {
      "height": 374,
      "width": 1238
    },
    "006833": {
      "height": 375,
      "width": 1242
    },
    "006834": {
      "height": 375,
      "width": 1242
    },
    "006835": {
      "height": 375,
      "width": 1242
    },
    "006836": {
      "height": 375,
      "width": 1242
    },
    "006837": {
      "height": 375,
      "width": 1242
    },
    "006838": {
      "height": 375,
      "width": 1242
    },
    "006839": {
      "height": 375,
      "width": 1242
    },
    "006840": {
      "height": 375,
      "width": 1242
    },
    "006841": {
      "height": 375,
      "width": 1242
    },
    "006842": {
      "height": 375,
      "width": 1242
    },
    "006843": {
      "height": 375,
      "width": 1242
    },
    "006844": {
      "height": 375,
      "width": 1242
    },
    "006845": {
      "height": 375,
      "width": 1242
    },
    "006846": {
      "height": 375,
      "width": 1242
    },
    "006847": {
      "height": 374,
      "width": 1238
    },
    "006848": {
      "height": 375,
      "width": 1242
    },
    "006849": {
      "height": 375,
      "width": 1242
    },
    "006850": {
      "height": 375,
      "width": 1242
    },
    "006851": {
      "height": 375,
      "width": 1242
    },
    "006852": {
      "height": 376,
      "width": 1241
    },
    "006853": {
      "height": 376,
      "width": 1241
    },
    "006854": {
      "height": 375,
      "width": 1242
    },
    "006855": {
      "height": 375,
      "width": 1242
    },
    "006856": {
      "height": 376,
      "width": 1241
    },
    "006857": {
      "height": 375,
      "width": 1242
    },
    "006858": {
      "height": 375,
      "width": 1242
    },
    "006859": {
      "height": 375,
      "width": 1242
    },
    "006860": {
      "height": 375,
      "width": 1242
    },
    "006861": {
      "height": 370,
      "width": 1224
    },
    "006862": {
      "height": 376,
      "width": 1241
    },
    "006863": {
      "height": 375,
      "width": 1242
    },
    "006864": {
      "height": 375,
      "width": 1242
    },
    "006865": {
      "height": 375,
      "width": 1242
    },
    "006866": {
      "height": 375,
      "width": 1242
    },
    "006867": {
      "height": 375,
      "width": 1242
    },
    "006868": {
      "height": 375,
      "width": 1242
    },
    "006869": {
      "height": 375,
      "width": 1242
    },
    "006870": {
      "height": 375,
      "width": 1242
    },
    "006871": {
      "height": 375,
      "width": 1242
    },
    "006872": {
      "height": 374,
      "width": 1238
    },
    "006873": {
      "height": 370,
      "width": 1224
    },
    "006874": {
      "height": 375,
      "width": 1242
    },
    "006875": {
      "height": 375,
      "width": 1242
    },
    "006876": {
      "height": 375,
      "width": 1242
    },
    "006877": {
      "height": 375,
      "width": 1242
    },
    "006878": {
      "height": 370,
      "width": 1224
    },
    "006879": {
      "height": 375,
      "width": 1242
    },
    "006880": {
      "height": 375,
      "width": 1242
    },
    "006881": {
      "height": 375,
      "width": 1242
    },
    "006882": {
      "height": 375,
      "width": 1242
    },
    "006883": {
      "height": 375,
      "width": 1242
    },
    "006884": {
      "height": 375,
      "width": 1242
    },
    "006885": {
      "height": 375,
      "width": 1242
    },
    "006886": {
      "height": 375,
      "width": 1242
    },
    "006887": {
      "height": 375,
      "width": 1242
    },
    "006888": {
      "height": 375,
      "width": 1242
    },
    "006889": {
      "height": 376,
      "width": 1241
    },
    "006890": {
      "height": 375,
      "width": 1242
    },
    "006891": {
      "height": 375,
      "width": 1242
    },
    "006892": {
      "height": 370,
      "width": 1224
    },
    "006893": {
      "height": 375,
      "width": 1242
    },
    "006894": {
      "height": 370,
      "width": 1224
    },
    "006895": {
      "height": 375,
      "width": 1242
    },
    "006896": {
      "height": 375,
      "width": 1242
    },
    "006897": {
      "height": 375,
      "width": 1242
    },
    "006898": {
      "height": 375,
      "width": 1242
    },
    "006899": {
      "height": 375,
      "width": 1242
    },
    "006900": {
      "height": 375,
      "width": 1242
    },
    "006901": {
      "height": 375,
      "width": 1242
    },
    "006902": {
      "height": 375,
      "width": 1242
    },
    "006903": {
      "height": 370,
      "width": 1224
    },
    "006904": {
      "height": 375,
      "width": 1242
    },
    "006905": {
      "height": 375,
      "width": 1242
    },
    "006906": {
      "height": 375,
      "width": 1242
    },
    "006907": {
      "height": 376,
      "width": 1241
    },
    "006908": {
      "height": 374,
      "width": 1238
    },
    "006909": {
      "height": 375,
      "width": 1242
    },
    "006910": {
      "height": 375,
      "width": 1242
    },
    "006911": {
      "height": 375,
      "width": 1242
    },
    "006912": {
      "height": 375,
      "width": 1242
    },
    "006913": {
      "height": 375,
      "width": 1242
    },
    "006914": {
      "height": 375,
      "width": 1242
    },
    "006915": {
      "height": 375,
      "width": 1242
    },
    "006916": {
      "height": 375,
      "width": 1242
    },
    "006917": {
      "height": 375,
      "width": 1242
    },
    "006918": {
      "height": 375,
      "width": 1242
    },
    "006919": {
      "height": 375,
      "width": 1242
    },
    "006920": {
      "height": 375,
      "width": 1242
    },
    "006921": {
      "height": 374,
      "width": 1238
    },
    "006922": {
      "height": 375,
      "width": 1242
    },
    "006923": {
      "height": 375,
      "width": 1242
    },
    "006924": {
      "height": 375,
      "width": 1242
    },
    "006925": {
      "height": 374,
      "width": 1238
    },
    "006926": {
      "height": 375,
      "width": 1242
    },
    "006927": {
      "height": 375,
      "width": 1242
    },
    "006928": {
      "height": 376,
      "width": 1241
    },
    "006929": {
      "height": 375,
      "width": 1242
    },
    "006930": {
      "height": 375,
      "width": 1242
    },
    "006931": {
      "height": 375,
      "width": 1242
    },
    "006932": {
      "height": 375,
      "width": 1242
    },
    "006933": {
      "height": 375,
      "width": 1242
    },
    "006934": {
      "height": 375,
      "width": 1242
    },
    "006935": {
      "height": 370,
      "width": 1224
    },
    "006936": {
      "height": 375,
      "width": 1242
    },
    "006937": {
      "height": 374,
      "width": 1238
    },
    "006938": {
      "height": 375,
      "width": 1242
    },
    "006939": {
      "height": 375,
      "width": 1242
    },
    "006940": {
      "height": 375,
      "width": 1242
    },
    "006941": {
      "height": 370,
      "width": 1224
    },
    "006942": {
      "height": 375,
      "width": 1242
    },
    "006943": {
      "height": 376,
      "width": 1241
    },
    "006944": {
      "height": 375,
      "width": 1242
    },
    "006945": {
      "height": 375,
      "width": 1242
    },
    "006946": {
      "height": 375,
      "width": 1242
    },
    "006947": {
      "height": 375,
      "width": 1242
    },
    "006948": {
      "height": 375,
      "width": 1242
    },
    "006949": {
      "height": 375,
      "width": 1242
    },
    "006950": {
      "height": 375,
      "width": 1242
    },
    "006951": {
      "height": 370,
      "width": 1224
    },
    "006952": {
      "height": 375,
      "width": 1242
    },
    "006953": {
      "height": 376,
      "width": 1241
    },
    "006954": {
      "height": 375,
      "width": 1242
    },
    "006955": {
      "height": 370,
      "width": 1224
    },
    "006956": {
      "height": 375,
      "width": 1242
    },
    "006957": {
      "height": 375,
      "width": 1242
    },
    "006958": {
      "height": 375,
      "width": 1242
    },
    "006959": {
      "height": 375,
      "width": 1242
    },
    "006960": {
      "height": 375,
      "width": 1242
    },
    "006961": {
      "height": 375,
      "width": 1242
    },
    "006962": {
      "height": 375,
      "width": 1242
    },
    "006963": {
      "height": 375,
      "width": 1242
    },
    "006964": {
      "height": 375,
      "width": 1242
    },
    "006965": {
      "height": 375,
      "width": 1242
    },
    "006966": {
      "height": 374,
      "width": 1238
    },
    "006967": {
      "height": 375,
      "width": 1242
    },
    "006968": {
      "height": 375,
      "width": 1242
    },
    "006969": {
      "height": 375,
      "width": 1242
    },
    "006970": {
      "height": 375,
      "width": 1242
    },
    "006971": {
      "height": 375,
      "width": 1242
    },
    "006972": {
      "height": 375,
      "width": 1242
    },
    "006973": {
      "height": 375,
      "width": 1242
    },
    "006974": {
      "height": 375,
      "width": 1242
    },
    "006975": {
      "height": 375,
      "width": 1242
    },
    "006976": {
      "height": 375,
      "width": 1242
    },
    "006977": {
      "height": 375,
      "width": 1242
    },
    "006978": {
      "height": 376,
      "width": 1241
    },
    "006979": {
      "height": 370,
      "width": 1224
    },
    "006980": {
      "height": 374,
      "width": 1238
    },
    "006981": {
      "height": 375,
      "width": 1242
    },
    "006982": {
      "height": 375,
      "width": 1242
    },
    "006983": {
      "height": 375,
      "width": 1242
    },
    "006984": {
      "height": 375,
      "width": 1242
    },
    "006985": {
      "height": 375,
      "width": 1242
    },
    "006986": {
      "height": 375,
      "width": 1242
    },
    "006987": {
      "height": 375,
      "width": 1242
    },
    "006988": {
      "height": 375,
      "width": 1242
    },
    "006989": {
      "height": 375,
      "width": 1242
    },
    "006990": {
      "height": 375,
      "width": 1242
    },
    "006991": {
      "height": 375,
      "width": 1242
    },
    "006992": {
      "height": 375,
      "width": 1242
    },
    "006993": {
      "height": 375,
      "width": 1242
    },
    "006994": {
      "height": 370,
      "width": 1224
    },
    "006995": {
      "height": 375,
      "width": 1242
    },
    "006996": {
      "height": 375,
      "width": 1242
    },
    "006997": {
      "height": 375,
      "width": 1242
    },
    "006998": {
      "height": 375,
      "width": 1242
    },
    "006999": {
      "height": 375,
      "width": 1242
    },
    "007000": {
      "height": 375,
      "width": 1242
    },
    "007001": {
      "height": 375,
      "width": 1242
    },
    "007002": {
      "height": 375,
      "width": 1242
    },
    "007003": {
      "height": 375,
      "width": 1242
    },
    "007004": {
      "height": 370,
      "width": 1224
    },
    "007005": {
      "height": 375,
      "width": 1242
    },
    "007006": {
      "height": 375,
      "width": 1242
    },
    "007007": {
      "height": 375,
      "width": 1242
    },
    "007008": {
      "height": 376,
      "width": 1241
    },
    "007009": {
      "height": 375,
      "width": 1242
    },
    "007010": {
      "height": 375,
      "width": 1242
    },
    "007011": {
      "height": 375,
      "width": 1242
    },
    "007012": {
      "height": 375,
      "width": 1242
    },
    "007013": {
      "height": 375,
      "width": 1242
    },
    "007014": {
      "height": 375,
      "width": 1242
    },
    "007015": {
      "height": 376,
      "width": 1241
    },
    "007016": {
      "height": 375,
      "width": 1242
    },
    "007017": {
      "height": 370,
      "width": 1224
    },
    "007018": {
      "height": 375,
      "width": 1242
    },
    "007019": {
      "height": 375,
      "width": 1242
    },
    "007020": {
      "height": 375,
      "width": 1242
    },
    "007021": {
      "height": 375,
      "width": 1242
    },
    "007022": {
      "height": 375,
      "width": 1242
    },
    "007023": {
      "height": 370,
      "width": 1224
    },
    "007024": {
      "height": 375,
      "width": 1242
    },
    "007025": {
      "height": 375,
      "width": 1242
    },
    "007026": {
      "height": 375,
      "width": 1242
    },
    "007027": {
      "height": 375,
      "width": 1242
    },
    "007028": {
      "height": 375,
      "width": 1242
    },
    "007029": {
      "height": 375,
      "width": 1242
    },
    "007030": {
      "height": 375,
      "width": 1242
    },
    "007031": {
      "height": 375,
      "width": 1242
    },
    "007032": {
      "height": 375,
      "width": 1242
    },
    "007033": {
      "height": 375,
      "width": 1242
    },
    "007034": {
      "height": 375,
      "width": 1242
    },
    "007035": {
      "height": 375,
      "width": 1242
    },
    "007036": {
      "height": 370,
      "width": 1224
    },
    "007037": {
      "height": 375,
      "width": 1242
    },
    "007038": {
      "height": 375,
      "width": 1242
    },
    "007039": {
      "height": 375,
      "width": 1242
    },
    "007040": {
      "height": 375,
      "width": 1242
    },
    "007041": {
      "height": 375,
      "width": 1242
    },
    "007042": {
      "height": 375,
      "width": 1242
    },
    "007043": {
      "height": 375,
      "width": 1242
    },
    "007044": {
      "height": 375,
      "width": 1242
    },
    "007045": {
      "height": 375,
      "width": 1242
    },
    "007046": {
      "height": 375,
      "width": 1242
    },
    "007047": {
      "height": 375,
      "width": 1242
    },
    "007048": {
      "height": 375,
      "width": 1242
    },
    "007049": {
      "height": 375,
      "width": 1242
    },
    "007050": {
      "height": 375,
      "width": 1242
    },
    "007051": {
      "height": 375,
      "width": 1242
    },
    "007052": {
      "height": 375,
      "width": 1242
    },
    "007053": {
      "height": 375,
      "width": 1242
    },
    "007054": {
      "height": 375,
      "width": 1242
    },
    "007055": {
      "height": 375,
      "width": 1242
    },
    "007056": {
      "height": 375,
      "width": 1242
    },
    "007057": {
      "height": 375,
      "width": 1242
    },
    "007058": {
      "height": 375,
      "width": 1242
    },
    "007059": {
      "height": 375,
      "width": 1242
    },
    "007060": {
      "height": 375,
      "width": 1242
    },
    "007061": {
      "height": 375,
      "width": 1242
    },
    "007062": {
      "height": 375,
      "width": 1242
    },
    "007063": {
      "height": 375,
      "width": 1242
    },
    "007064": {
      "height": 375,
      "width": 1242
    },
    "007065": {
      "height": 375,
      "width": 1242
    },
    "007066": {
      "height": 375,
      "width": 1242
    },
    "007067": {
      "height": 375,
      "width": 1242
    },
    "007068": {
      "height": 375,
      "width": 1242
    },
    "007069": {
      "height": 375,
      "width": 1242
    },
    "007070": {
      "height": 375,
      "width": 1242
    },
    "007071": {
      "height": 375,
      "width": 1242
    },
    "007072": {
      "height": 375,
      "width": 1242
    },
    "007073": {
      "height": 370,
      "width": 1224
    },
    "007074": {
      "height": 375,
      "width": 1242
    },
    "007075": {
      "height": 370,
      "width": 1224
    },
    "007076": {
      "height": 375,
      "width": 1242
    },
    "007077": {
      "height": 370,
      "width": 1224
    },
    "007078": {
      "height": 375,
      "width": 1242
    },
    "007079": {
      "height": 375,
      "width": 1242
    },
    "007080": {
      "height": 375,
      "width": 1242
    },
    "007081": {
      "height": 375,
      "width": 1242
    },
    "007082": {
      "height": 375,
      "width": 1242
    },
    "007083": {
      "height": 375,
      "width": 1242
    },
    "007084": {
      "height": 370,
      "width": 1224
    },
    "007085": {
      "height": 375,
      "width": 1242
    },
    "007086": {
      "height": 375,
      "width": 1242
    },
    "007087": {
      "height": 375,
      "width": 1242
    },
    "007088": {
      "height": 375,
      "width": 1242
    },
    "007089": {
      "height": 375,
      "width": 1242
    },
    "007090": {
      "height": 375,
      "width": 1242
    },
    "007091": {
      "height": 375,
      "width": 1242
    },
    "007092": {
      "height": 375,
      "width": 1242
    },
    "007093": {
      "height": 375,
      "width": 1242
    },
    "007094": {
      "height": 375,
      "width": 1242
    },
    "007095": {
      "height": 375,
      "width": 1242
    },
    "007096": {
      "height": 375,
      "width": 1242
    },
    "007097": {
      "height": 374,
      "width": 1238
    },
    "007098": {
      "height": 375,
      "width": 1242
    },
    "007099": {
      "height": 375,
      "width": 1242
    },
    "007100": {
      "height": 375,
      "width": 1242
    },
    "007101": {
      "height": 375,
      "width": 1242
    },
    "007102": {
      "height": 375,
      "width": 1242
    },
    "007103": {
      "height": 375,
      "width": 1242
    },
    "007104": {
      "height": 375,
      "width": 1242
    },
    "007105": {
      "height": 375,
      "width": 1242
    },
    "007106": {
      "height": 375,
      "width": 1242
    },
    "007107": {
      "height": 375,
      "width": 1242
    },
    "007108": {
      "height": 375,
      "width": 1242
    },
    "007109": {
      "height": 374,
      "width": 1238
    },
    "007110": {
      "height": 370,
      "width": 1224
    },
    "007111": {
      "height": 375,
      "width": 1242
    },
    "007112": {
      "height": 375,
      "width": 1242
    },
    "007113": {
      "height": 375,
      "width": 1242
    },
    "007114": {
      "height": 375,
      "width": 1242
    },
    "007115": {
      "height": 375,
      "width": 1242
    },
    "007116": {
      "height": 375,
      "width": 1242
    },
    "007117": {
      "height": 375,
      "width": 1242
    },
    "007118": {
      "height": 375,
      "width": 1242
    },
    "007119": {
      "height": 375,
      "width": 1242
    },
    "007120": {
      "height": 375,
      "width": 1242
    },
    "007121": {
      "height": 375,
      "width": 1242
    },
    "007122": {
      "height": 375,
      "width": 1242
    },
    "007123": {
      "height": 375,
      "width": 1242
    },
    "007124": {
      "height": 375,
      "width": 1242
    },
    "007125": {
      "height": 375,
      "width": 1242
    },
    "007126": {
      "height": 370,
      "width": 1224
    },
    "007127": {
      "height": 375,
      "width": 1242
    },
    "007128": {
      "height": 375,
      "width": 1242
    },
    "007129": {
      "height": 375,
      "width": 1242
    },
    "007130": {
      "height": 375,
      "width": 1242
    },
    "007131": {
      "height": 375,
      "width": 1242
    },
    "007132": {
      "height": 375,
      "width": 1242
    },
    "007133": {
      "height": 375,
      "width": 1242
    },
    "007134": {
      "height": 375,
      "width": 1242
    },
    "007135": {
      "height": 375,
      "width": 1242
    },
    "007136": {
      "height": 374,
      "width": 1238
    },
    "007137": {
      "height": 375,
      "width": 1242
    },
    "007138": {
      "height": 375,
      "width": 1242
    },
    "007139": {
      "height": 375,
      "width": 1242
    },
    "007140": {
      "height": 375,
      "width": 1242
    },
    "007141": {
      "height": 375,
      "width": 1242
    },
    "007142": {
      "height": 375,
      "width": 1242
    },
    "007143": {
      "height": 375,
      "width": 1242
    },
    "007144": {
      "height": 375,
      "width": 1242
    },
    "007145": {
      "height": 375,
      "width": 1242
    },
    "007146": {
      "height": 375,
      "width": 1242
    },
    "007147": {
      "height": 370,
      "width": 1224
    },
    "007148": {
      "height": 374,
      "width": 1238
    },
    "007149": {
      "height": 375,
      "width": 1242
    },
    "007150": {
      "height": 370,
      "width": 1224
    },
    "007151": {
      "height": 375,
      "width": 1242
    },
    "007152": {
      "height": 375,
      "width": 1242
    },
    "007153": {
      "height": 375,
      "width": 1242
    },
    "007154": {
      "height": 370,
      "width": 1224
    },
    "007155": {
      "height": 375,
      "width": 1242
    },
    "007156": {
      "height": 375,
      "width": 1242
    },
    "007157": {
      "height": 375,
      "width": 1242
    },
    "007158": {
      "height": 375,
      "width": 1242
    },
    "007159": {
      "height": 370,
      "width": 1224
    },
    "007160": {
      "height": 375,
      "width": 1242
    },
    "007161": {
      "height": 374,
      "width": 1238
    },
    "007162": {
      "height": 376,
      "width": 1241
    },
    "007163": {
      "height": 375,
      "width": 1242
    },
    "007164": {
      "height": 375,
      "width": 1242
    },
    "007165": {
      "height": 375,
      "width": 1242
    },
    "007166": {
      "height": 375,
      "width": 1242
    },
    "007167": {
      "height": 375,
      "width": 1242
    },
    "007168": {
      "height": 375,
      "width": 1242
    },
    "007169": {
      "height": 375,
      "width": 1242
    },
    "007170": {
      "height": 374,
      "width": 1238
    },
    "007171": {
      "height": 375,
      "width": 1242
    },
    "007172": {
      "height": 374,
      "width": 1238
    },
    "007173": {
      "height": 375,
      "width": 1242
    },
    "007174": {
      "height": 375,
      "width": 1242
    },
    "007175": {
      "height": 375,
      "width": 1242
    },
    "007176": {
      "height": 375,
      "width": 1242
    },
    "007177": {
      "height": 375,
      "width": 1242
    },
    "007178": {
      "height": 375,
      "width": 1242
    },
    "007179": {
      "height": 375,
      "width": 1242
    },
    "007180": {
      "height": 375,
      "width": 1242
    },
    "007181": {
      "height": 375,
      "width": 1242
    },
    "007182": {
      "height": 375,
      "width": 1242
    },
    "007183": {
      "height": 375,
      "width": 1242
    },
    "007184": {
      "height": 375,
      "width": 1242
    },
    "007185": {
      "height": 375,
      "width": 1242
    },
    "007186": {
      "height": 375,
      "width": 1242
    },
    "007187": {
      "height": 375,
      "width": 1242
    },
    "007188": {
      "height": 370,
      "width": 1224
    },
    "007189": {
      "height": 375,
      "width": 1242
    },
    "007190": {
      "height": 375,
      "width": 1242
    },
    "007191": {
      "height": 375,
      "width": 1242
    },
    "007192": {
      "height": 375,
      "width": 1242
    },
    "007193": {
      "height": 375,
      "width": 1242
    },
    "007194": {
      "height": 375,
      "width": 1242
    },
    "007195": {
      "height": 375,
      "width": 1242
    },
    "007196": {
      "height": 375,
      "width": 1242
    },
    "007197": {
      "height": 375,
      "width": 1242
    },
    "007198": {
      "height": 375,
      "width": 1242
    },
    "007199": {
      "height": 375,
      "width": 1242
    },
    "007200": {
      "height": 375,
      "width": 1242
    },
    "007201": {
      "height": 374,
      "width": 1238
    },
    "007202": {
      "height": 374,
      "width": 1238
    },
    "007203": {
      "height": 375,
      "width": 1242
    },
    "007204": {
      "height": 375,
      "width": 1242
    },
    "007205": {
      "height": 375,
      "width": 1242
    },
    "007206": {
      "height": 375,
      "width": 1242
    },
    "007207": {
      "height": 375,
      "width": 1242
    },
    "007208": {
      "height": 375,
      "width": 1242
    },
    "007209": {
      "height": 375,
      "width": 1242
    },
    "007210": {
      "height": 375,
      "width": 1242
    },
    "007211": {
      "height": 375,
      "width": 1242
    },
    "007212": {
      "height": 375,
      "width": 1242
    },
    "007213": {
      "height": 375,
      "width": 1242
    },
    "007214": {
      "height": 375,
      "width": 1242
    },
    "007215": {
      "height": 375,
      "width": 1242
    },
    "007216": {
      "height": 370,
      "width": 1224
    },
    "007217": {
      "height": 375,
      "width": 1242
    },
    "007218": {
      "height": 375,
      "width": 1242
    },
    "007219": {
      "height": 374,
      "width": 1238
    },
    "007220": {
      "height": 375,
      "width": 1242
    },
    "007221": {
      "height": 375,
      "width": 1242
    },
    "007222": {
      "height": 375,
      "width": 1242
    },
    "007223": {
      "height": 370,
      "width": 1224
    },
    "007224": {
      "height": 375,
      "width": 1242
    },
    "007225": {
      "height": 375,
      "width": 1242
    },
    "007226": {
      "height": 375,
      "width": 1242
    },
    "007227": {
      "height": 375,
      "width": 1242
    },
    "007228": {
      "height": 375,
      "width": 1242
    },
    "007229": {
      "height": 375,
      "width": 1242
    },
    "007230": {
      "height": 375,
      "width": 1242
    },
    "007231": {
      "height": 370,
      "width": 1224
    },
    "007232": {
      "height": 375,
      "width": 1242
    },
    "007233": {
      "height": 375,
      "width": 1242
    },
    "007234": {
      "height": 375,
      "width": 1242
    },
    "007235": {
      "height": 375,
      "width": 1242
    },
    "007236": {
      "height": 375,
      "width": 1242
    },
    "007237": {
      "height": 370,
      "width": 1224
    },
    "007238": {
      "height": 375,
      "width": 1242
    },
    "007239": {
      "height": 375,
      "width": 1242
    },
    "007240": {
      "height": 375,
      "width": 1242
    },
    "007241": {
      "height": 374,
      "width": 1238
    },
    "007242": {
      "height": 375,
      "width": 1242
    },
    "007243": {
      "height": 375,
      "width": 1242
    },
    "007244": {
      "height": 374,
      "width": 1238
    },
    "007245": {
      "height": 375,
      "width": 1242
    },
    "007246": {
      "height": 375,
      "width": 1242
    },
    "007247": {
      "height": 375,
      "width": 1242
    },
    "007248": {
      "height": 375,
      "width": 1242
    },
    "007249": {
      "height": 375,
      "width": 1242
    },
    "007250": {
      "height": 375,
      "width": 1242
    },
    "007251": {
      "height": 375,
      "width": 1242
    },
    "007252": {
      "height": 375,
      "width": 1242
    },
    "007253": {
      "height": 375,
      "width": 1242
    },
    "007254": {
      "height": 375,
      "width": 1242
    },
    "007255": {
      "height": 370,
      "width": 1224
    },
    "007256": {
      "height": 375,
      "width": 1242
    },
    "007257": {
      "height": 375,
      "width": 1242
    },
    "007258": {
      "height": 375,
      "width": 1242
    },
    "007259": {
      "height": 375,
      "width": 1242
    },
    "007260": {
      "height": 375,
      "width": 1242
    },
    "007261": {
      "height": 375,
      "width": 1242
    },
    "007262": {
      "height": 375,
      "width": 1242
    },
    "007263": {
      "height": 375,
      "width": 1242
    },
    "007264": {
      "height": 375,
      "width": 1242
    },
    "007265": {
      "height": 374,
      "width": 1238
    },
    "007266": {
      "height": 375,
      "width": 1242
    },
    "007267": {
      "height": 375,
      "width": 1242
    },
    "007268": {
      "height": 375,
      "width": 1242
    },
    "007269": {
      "height": 370,
      "width": 1224
    },
    "007270": {
      "height": 375,
      "width": 1242
    },
    "007271": {
      "height": 375,
      "width": 1242
    },
    "007272": {
      "height": 370,
      "width": 1224
    },
    "007273": {
      "height": 376,
      "width": 1241
    },
    "007274": {
      "height": 376,
      "width": 1241
    },
    "007275": {
      "height": 375,
      "width": 1242
    },
    "007276": {
      "height": 375,
      "width": 1242
    },
    "007277": {
      "height": 375,
      "width": 1242
    },
    "007278": {
      "height": 370,
      "width": 1224
    },
    "007279": {
      "height": 375,
      "width": 1242
    },
    "007280": {
      "height": 375,
      "width": 1242
    },
    "007281": {
      "height": 375,
      "width": 1242
    },
    "007282": {
      "height": 375,
      "width": 1242
    },
    "007283": {
      "height": 375,
      "width": 1242
    },
    "007284": {
      "height": 375,
      "width": 1242
    },
    "007285": {
      "height": 375,
      "width": 1242
    },
    "007286": {
      "height": 375,
      "width": 1242
    },
    "007287": {
      "height": 375,
      "width": 1242
    },
    "007288": {
      "height": 370,
      "width": 1224
    },
    "007289": {
      "height": 375,
      "width": 1242
    },
    "007290": {
      "height": 370,
      "width": 1224
    },
    "007291": {
      "height": 375,
      "width": 1242
    },
    "007292": {
      "height": 375,
      "width": 1242
    },
    "007293": {
      "height": 375,
      "width": 1242
    },
    "007294": {
      "height": 375,
      "width": 1242
    },
    "007295": {
      "height": 375,
      "width": 1242
    },
    "007296": {
      "height": 375,
      "width": 1242
    },
    "007297": {
      "height": 375,
      "width": 1242
    },
    "007298": {
      "height": 375,
      "width": 1242
    },
    "007299": {
      "height": 375,
      "width": 1242
    },
    "007300": {
      "height": 375,
      "width": 1242
    },
    "007301": {
      "height": 375,
      "width": 1242
    },
    "007302": {
      "height": 375,
      "width": 1242
    },
    "007303": {
      "height": 375,
      "width": 1242
    },
    "007304": {
      "height": 375,
      "width": 1242
    },
    "007305": {
      "height": 375,
      "width": 1242
    },
    "007306": {
      "height": 375,
      "width": 1242
    },
    "007307": {
      "height": 375,
      "width": 1242
    },
    "007308": {
      "height": 370,
      "width": 1224
    },
    "007309": {
      "height": 375,
      "width": 1242
    },
    "007310": {
      "height": 370,
      "width": 1224
    },
    "007311": {
      "height": 375,
      "width": 1242
    },
    "007312": {
      "height": 375,
      "width": 1242
    },
    "007313": {
      "height": 375,
      "width": 1242
    },
    "007314": {
      "height": 375,
      "width": 1242
    },
    "007315": {
      "height": 375,
      "width": 1242
    },
    "007316": {
      "height": 375,
      "width": 1242
    },
    "007317": {
      "height": 375,
      "width": 1242
    },
    "007318": {
      "height": 374,
      "width": 1238
    },
    "007319": {
      "height": 374,
      "width": 1238
    },
    "007320": {
      "height": 370,
      "width": 1224
    },
    "007321": {
      "height": 370,
      "width": 1224
    },
    "007322": {
      "height": 375,
      "width": 1242
    },
    "007323": {
      "height": 375,
      "width": 1242
    },
    "007324": {
      "height": 375,
      "width": 1242
    },
    "007325": {
      "height": 375,
      "width": 1242
    },
    "007326": {
      "height": 370,
      "width": 1224
    },
    "007327": {
      "height": 370,
      "width": 1224
    },
    "007328": {
      "height": 375,
      "width": 1242
    },
    "007329": {
      "height": 376,
      "width": 1241
    },
    "007330": {
      "height": 375,
      "width": 1242
    },
    "007331": {
      "height": 375,
      "width": 1242
    },
    "007332": {
      "height": 375,
      "width": 1242
    },
    "007333": {
      "height": 375,
      "width": 1242
    },
    "007334": {
      "height": 375,
      "width": 1242
    },
    "007335": {
      "height": 375,
      "width": 1242
    },
    "007336": {
      "height": 375,
      "width": 1242
    },
    "007337": {
      "height": 375,
      "width": 1242
    },
    "007338": {
      "height": 375,
      "width": 1242
    },
    "007339": {
      "height": 376,
      "width": 1241
    },
    "007340": {
      "height": 375,
      "width": 1242
    },
    "007341": {
      "height": 375,
      "width": 1242
    },
    "007342": {
      "height": 375,
      "width": 1242
    },
    "007343": {
      "height": 375,
      "width": 1242
    },
    "007344": {
      "height": 370,
      "width": 1224
    },
    "007345": {
      "height": 375,
      "width": 1242
    },
    "007346": {
      "height": 374,
      "width": 1238
    },
    "007347": {
      "height": 370,
      "width": 1224
    },
    "007348": {
      "height": 375,
      "width": 1242
    },
    "007349": {
      "height": 375,
      "width": 1242
    },
    "007350": {
      "height": 375,
      "width": 1242
    },
    "007351": {
      "height": 370,
      "width": 1224
    },
    "007352": {
      "height": 376,
      "width": 1241
    },
    "007353": {
      "height": 375,
      "width": 1242
    },
    "007354": {
      "height": 375,
      "width": 1242
    },
    "007355": {
      "height": 375,
      "width": 1242
    },
    "007356": {
      "height": 375,
      "width": 1242
    },
    "007357": {
      "height": 375,
      "width": 1242
    },
    "007358": {
      "height": 375,
      "width": 1242
    },
    "007359": {
      "height": 370,
      "width": 1224
    },
    "007360": {
      "height": 376,
      "width": 1241
    },
    "007361": {
      "height": 375,
      "width": 1242
    },
    "007362": {
      "height": 370,
      "width": 1224
    },
    "007363": {
      "height": 375,
      "width": 1242
    },
    "007364": {
      "height": 376,
      "width": 1241
    },
    "007365": {
      "height": 375,
      "width": 1242
    },
    "007366": {
      "height": 375,
      "width": 1242
    },
    "007367": {
      "height": 375,
      "width": 1242
    },
    "007368": {
      "height": 375,
      "width": 1242
    },
    "007369": {
      "height": 374,
      "width": 1238
    },
    "007370": {
      "height": 375,
      "width": 1242
    },
    "007371": {
      "height": 375,
      "width": 1242
    },
    "007372": {
      "height": 370,
      "width": 1224
    },
    "007373": {
      "height": 375,
      "width": 1242
    },
    "007374": {
      "height": 375,
      "width": 1242
    },
    "007375": {
      "height": 374,
      "width": 1238
    },
    "007376": {
      "height": 375,
      "width": 1242
    },
    "007377": {
      "height": 375,
      "width": 1242
    },
    "007378": {
      "height": 375,
      "width": 1242
    },
    "007379": {
      "height": 375,
      "width": 1242
    },
    "007380": {
      "height": 375,
      "width": 1242
    },
    "007381": {
      "height": 375,
      "width": 1242
    },
    "007382": {
      "height": 374,
      "width": 1238
    },
    "007383": {
      "height": 375,
      "width": 1242
    },
    "007384": {
      "height": 375,
      "width": 1242
    },
    "007385": {
      "height": 375,
      "width": 1242
    },
    "007386": {
      "height": 375,
      "width": 1242
    },
    "007387": {
      "height": 370,
      "width": 1224
    },
    "007388": {
      "height": 375,
      "width": 1242
    },
    "007389": {
      "height": 376,
      "width": 1241
    },
    "007390": {
      "height": 375,
      "width": 1242
    },
    "007391": {
      "height": 375,
      "width": 1242
    },
    "007392": {
      "height": 375,
      "width": 1242
    },
    "007393": {
      "height": 375,
      "width": 1242
    },
    "007394": {
      "height": 375,
      "width": 1242
    },
    "007395": {
      "height": 375,
      "width": 1242
    },
    "007396": {
      "height": 376,
      "width": 1241
    },
    "007397": {
      "height": 375,
      "width": 1242
    },
    "007398": {
      "height": 375,
      "width": 1242
    },
    "007399": {
      "height": 374,
      "width": 1238
    },
    "007400": {
      "height": 375,
      "width": 1242
    },
    "007401": {
      "height": 375,
      "width": 1242
    },
    "007402": {
      "height": 375,
      "width": 1242
    },
    "007403": {
      "height": 375,
      "width": 1242
    },
    "007404": {
      "height": 375,
      "width": 1242
    },
    "007405": {
      "height": 374,
      "width": 1238
    },
    "007406": {
      "height": 375,
      "width": 1242
    },
    "007407": {
      "height": 375,
      "width": 1242
    },
    "007408": {
      "height": 370,
      "width": 1224
    },
    "007409": {
      "height": 375,
      "width": 1242
    },
    "007410": {
      "height": 374,
      "width": 1238
    },
    "007411": {
      "height": 375,
      "width": 1242
    },
    "007412": {
      "height": 374,
      "width": 1238
    },
    "007413": {
      "height": 375,
      "width": 1242
    },
    "007414": {
      "height": 375,
      "width": 1242
    },
    "007415": {
      "height": 375,
      "width": 1242
    },
    "007416": {
      "height": 375,
      "width": 1242
    },
    "007417": {
      "height": 375,
      "width": 1242
    },
    "007418": {
      "height": 375,
      "width": 1242
    },
    "007419": {
      "height": 375,
      "width": 1242
    },
    "007420": {
      "height": 374,
      "width": 1238
    },
    "007421": {
      "height": 375,
      "width": 1242
    },
    "007422": {
      "height": 375,
      "width": 1242
    },
    "007423": {
      "height": 375,
      "width": 1242
    },
    "007424": {
      "height": 375,
      "width": 1242
    },
    "007425": {
      "height": 375,
      "width": 1242
    },
    "007426": {
      "height": 374,
      "width": 1238
    },
    "007427": {
      "height": 370,
      "width": 1224
    },
    "007428": {
      "height": 375,
      "width": 1242
    },
    "007429": {
      "height": 375,
      "width": 1242
    },
    "007430": {
      "height": 375,
      "width": 1242
    },
    "007431": {
      "height": 370,
      "width": 1224
    },
    "007432": {
      "height": 375,
      "width": 1242
    },
    "007433": {
      "height": 374,
      "width": 1238
    },
    "007434": {
      "height": 375,
      "width": 1242
    },
    "007435": {
      "height": 375,
      "width": 1242
    },
    "007436": {
      "height": 370,
      "width": 1224
    },
    "007437": {
      "height": 375,
      "width": 1242
    },
    "007438": {
      "height": 375,
      "width": 1242
    },
    "007439": {
      "height": 375,
      "width": 1242
    },
    "007440": {
      "height": 375,
      "width": 1242
    },
    "007441": {
      "height": 375,
      "width": 1242
    },
    "007442": {
      "height": 375,
      "width": 1242
    },
    "007443": {
      "height": 375,
      "width": 1242
    },
    "007444": {
      "height": 374,
      "width": 1238
    },
    "007445": {
      "height": 375,
      "width": 1242
    },
    "007446": {
      "height": 375,
      "width": 1242
    },
    "007447": {
      "height": 375,
      "width": 1242
    },
    "007448": {
      "height": 370,
      "width": 1224
    },
    "007449": {
      "height": 376,
      "width": 1241
    },
    "007450": {
      "height": 375,
      "width": 1242
    },
    "007451": {
      "height": 375,
      "width": 1242
    },
    "007452": {
      "height": 370,
      "width": 1224
    },
    "007453": {
      "height": 375,
      "width": 1242
    },
    "007454": {
      "height": 374,
      "width": 1238
    },
    "007455": {
      "height": 375,
      "width": 1242
    },
    "007456": {
      "height": 375,
      "width": 1242
    },
    "007457": {
      "height": 375,
      "width": 1242
    },
    "007458": {
      "height": 375,
      "width": 1242
    },
    "007459": {
      "height": 375,
      "width": 1242
    },
    "007460": {
      "height": 375,
      "width": 1242
    },
    "007461": {
      "height": 375,
      "width": 1242
    },
    "007462": {
      "height": 375,
      "width": 1242
    },
    "007463": {
      "height": 375,
      "width": 1242
    },
    "007464": {
      "height": 374,
      "width": 1238
    },
    "007465": {
      "height": 375,
      "width": 1242
    },
    "007466": {
      "height": 374,
      "width": 1238
    },
    "007467": {
      "height": 375,
      "width": 1242
    },
    "007468": {
      "height": 376,
      "width": 1241
    },
    "007469": {
      "height": 374,
      "width": 1238
    },
    "007470": {
      "height": 375,
      "width": 1242
    },
    "007471": {
      "height": 375,
      "width": 1242
    },
    "007472": {
      "height": 375,
      "width": 1242
    },
    "007473": {
      "height": 370,
      "width": 1224
    },
    "007474": {
      "height": 375,
      "width": 1242
    },
    "007475": {
      "height": 376,
      "width": 1241
    },
    "007476": {
      "height": 375,
      "width": 1242
    },
    "007477": {
      "height": 370,
      "width": 1224
    },
    "007478": {
      "height": 375,
      "width": 1242
    },
    "007479": {
      "height": 375,
      "width": 1242
    },
    "007480": {
      "height": 375,
      "width": 1242
    }
  },
  "image_count": 7481,
  "split": "training"
}
'''[1:-1],
    'splits/train.txt': r'''
000000
000003
000007
000009
000010
000011
000012
000013
000014
000016
000017
000018
000022
000026
000029
000030
000032
000034
000036
000038
000041
000043
000044
000045
000046
000049
000051
000054
000055
000056
000057
000060
000064
000067
000068
000069
000070
000071
000072
000073
000074
000075
000079
000080
000082
000083
000084
000085
000086
000087
000088
000091
000092
000095
000096
000097
000099
000100
000101
000103
000105
000109
000110
000111
000112
000113
000114
000115
000119
000120
000121
000123
000125
000127
000129
000130
000131
000133
000136
000138
000141
000142
000144
000145
000146
000148
000149
000150
000154
000155
000157
000158
000160
000162
000163
000164
000165
000166
000171
000172
000176
000177
000178
000179
000180
000184
000185
000189
000193
000198
000200
000202
000205
000206
000208
000209
000210
000214
000215
000217
000219
000220
000221
000222
000225
000227
000228
000232
000233
000238
000240
000241
000243
000244
000245
000253
000254
000255
000256
000257
000258
000259
000261
000264
000267
000271
000274
000275
000276
000277
000280
000282
000285
000286
000287
000288
000292
000294
000295
000296
000298
000299
000300
000303
000304
000306
000310
000313
000316
000317
000318
000322
000325
000326
000330
000331
000334
000337
000338
000339
000342
000344
000348
000349
000353
000358
000363
000364
000367
000368
000371
000374
000375
000380
000384
000387
000389
000390
000400
000405
000406
000410
000411
000412
000416
000417
000418
000421
000423
000424
000425
000426
000431
000432
000433
000434
000435
000438
000439
000441
000442
000444
000445
000447
000449
000456
000458
000460
000461
000462
000464
000465
000466
000467
000470
000471
000474
000482
000483
000484
000487
000488
000490
000497
000500
000501
000502
000505
000507
000511
000513
000514
000516
000518
000520
000522
000523
000525
000526
000529
000531
000532
000534
000535
000537
000538
000539
000540
000544
000547
000549
000550
000552
000553
000556
000557
000562
000563
000565
000570
000573
000574
000575
000576
000577
000578
000579
000580
000582
000584
000585
000586
000587
000592
000593
000594
000596
000597
000598
000599
000602
000603
000605
000606
000607
000608
000609
000616
000617
000621
000622
000623
000627
000629
000631
000632
000633
000637
000638
000640
000641
000643
000646
000649
000651
000652
000653
000654
000656
000661
000662
000663
000664
000665
000666
000668
000671
000672
000673
000675
000676
000678
000680
000681
000685
000686
000687
000688
000689
000690
000693
000695
000697
000701
000703
000705
000707
000709
000710
000711
000712
000713
000714
000715
000719
000720
000723
000724
000726
000730
000732
000733
000735
000738
000739
000742
000743
000744
000747
000749
000753
000755
000757
000758
000759
000760
000762
000763
000764
000770
000775
000776
000777
000780
000781
000783
000784
000785
000786
000787
000788
000789
000791
000793
000794
000796
000797
000799
000808
000813
000814
000815
000817
000818
000820
000821
000822
000824
000825
000827
000828
000829
000830
000832
000833
000834
000835
000836
000839
000842
000845
000846
000851
000853
000855
000856
000857
000858
000860
000861
000864
000865
000866
000867
000868
000870
000871
000872
000880
000882
000883
000886
000887
000888
000890
000891
000892
000895
000896
000898
000900
000901
000902
000903
000905
000906
000908
000910
000913
000914
000918
000919
000921
000924
000925
000927
000929
000933
000934
000935
000936
000937
000941
000945
000946
000947
000950
000951
000954
000955
000957
000959
000960
000962
000965
000968
000972
000975
000977
000978
000980
000982
000987
000989
000990
000992
000993
000994
000995
000996
000997
000998
001000
001001
001003
001004
001005
001009
001016
001017
001020
001023
001024
001028
001029
001030
001031
001032
001033
001034
001036
001038
001040
001041
001044
001045
001047
001048
001049
001052
001056
001057
001059
001060
001061
001062
001064
001072
001073
001074
001079
001080
001081
001082
001085
001087
001090
001091
001092
001093
001098
001100
001103
001105
001109
001110
001112
001117
001119
001121
001122
001124
001126
001128
001130
001137
001142
001146
001151
001156
001157
001159
001160
001161
001164
001165
001166
001168
001169
001170
001171
001174
001175
001181
001184
001185
001186
001190
001196
001197
001200
001201
001202
001204
001205
001208
001209
001210
001211
001212
001215
001219
001220
001223
001227
001229
001231
001233
001238
001240
001247
001248
001250
001256
001258
001262
001264
001276
001277
001278
001279
001280
001282
001283
001285
001288
001290
001293
001297
001298
001299
001300
001301
001302
001309
001310
001311
001312
001313
001315
001316
001319
001320
001321
001322
001323
001324
001325
001326
001327
001328
001335
001338
001340
001341
001343
001348
001349
001351
001354
001357
001358
001360
001361
001362
001364
001366
001367
001368
001369
001370
001371
001373
001378
001379
001383
001385
001390
001392
001393
001394
001396
001399
001400
001401
001402
001403
001404
001405
001406
001408
001409
001413
001414
001417
001418
001420
001422
001423
001425
001426
001428
001429
001430
001433
001434
001436
001440
001444
001447
001449
001452
001453
001454
001455
001456
001457
001459
001460
001462
001464
001465
001467
001468
001470
001472
001473
001474
001475
001476
001479
001482
001483
001484
001486
001490
001491
001492
001493
001494
001496
001498
001499
001500
001503
001504
001505
001506
001509
001510
001512
001515
001518
001519
001520
001523
001529
001530
001531
001532
001534
001539
001540
001541
001543
001544
001548
001550
001551
001553
001554
001556
001558
001559
001561
001563
001566
001568
001570
001571
001572
001575
001578
001580
001581
001584
001593
001595
001598
001599
001601
001604
001607
001608
001609
001611
001612
001614
001618
001620
001622
001623
001624
001626
001628
001630
001632
001636
001637
001638
001639
001641
001642
001644
001646
001648
001649
001651
001652
001653
001655
001657
001659
001661
001663
001668
001669
001671
001672
001673
001674
001676
001677
001678
001679
001681
001685
001686
001687
001688
001690
001691
001692
001695
001696
001698
001700
001703
001708
001715
001716
001720
001723
001724
001725
001728
001730
001731
001734
001735
001736
001737
001738
001739
001743
001744
001747
001748
001753
001754
001756
001757
001759
001760
001761
001763
001766
001767
001769
001770
001773
001775
001777
001779
001784
001785
001788
001789
001790
001791
001792
001793
001796
001798
001799
001803
001805
001806
001809
001810
001811
001812
001815
001816
001819
001821
001826
001827
001829
001830
001832
001833
001834
001836
001837
001838
001839
001841
001842
001843
001845
001847
001849
001850
001857
001860
001864
001865
001866
001870
001871
001873
001874
001876
001879
001882
001883
001889
001891
001894
001895
001896
001899
001901
001902
001903
001906
001907
001908
001910
001911
001912
001913
001914
001915
001916
001917
001918
001921
001922
001930
001935
001938
001939
001944
001947
001948
001949
001950
001951
001953
001955
001956
001957
001958
001961
001962
001963
001964
001965
001968
001970
001971
001973
001974
001975
001976
001981
001987
001988
001990
001992
001993
001994
001998
002003
002005
002006
002007
002009
002015
002016
002018
002020
002023
002024
002026
002030
002031
002032
002033
002039
002040
002041
002047
002051
002053
002055
002059
002060
002061
002063
002064
002065
002066
002067
002069
002070
002072
002077
002080
002083
002084
002088
002090
002092
002095
002096
002097
002098
002099
002104
002105
002106
002109
002110
002114
002116
002117
002119
002122
002125
002126
002129
002132
002133
002134
002141
002143
002144
002145
002146
002147
002148
002149
002150
002154
002155
002156
002157
002162
002164
002167
002171
002172
002174
002175
002176
002178
002180
002181
002184
002186
002189
002190
002191
002192
002194
002195
002197
002198
002199
002203
002204
002205
002208
002210
002211
002212
002213
002214
002217
002221
002222
002223
002226
002227
002230
002231
002235
002236
002237
002238
002240
002241
002242
002244
002247
002249
002252
002253
002256
002259
002261
002263
002264
002265
002267
002268
002269
002270
002271
002273
002274
002275
002278
002281
002285
002288
002289
002296
002297
002301
002302
002305
002309
002311
002312
002313
002316
002317
002318
002321
002322
002323
002324
002326
002328
002331
002333
002335
002339
002342
002343
002349
002350
002351
002352
002354
002355
002358
002360
002361
002363
002364
002368
002371
002373
002374
002375
002377
002379
002381
002388
002389
002390
002394
002395
002396
002400
002401
002402
002403
002406
002407
002408
002409
002410
002412
002413
002416
002417
002421
002426
002427
002430
002431
002435
002436
002437
002438
002441
002443
002444
002445
002447
002448
002449
002451
002452
002453
002456
002459
002464
002465
002466
002467
002468
002469
002470
002471
002472
002475
002480
002481
002482
002484
002485
002487
002489
002491
002493
002494
002496
002498
002501
002507
002508
002510
002512
002513
002514
002515
002517
002518
002522
002523
002524
002527
002533
002535
002536
002537
002542
002544
002545
002547
002549
002550
002551
002553
002554
002555
002559
002560
002561
002566
002567
002571
002573
002576
002578
002579
002582
002587
002588
002589
002591
002592
002593
002595
002596
002597
002605
002607
002608
002609
002610
002611
002614
002616
002617
002618
002620
002622
002623
002624
002627
002629
002632
002634
002637
002639
002642
002643
002647
002648
002649
002650
002652
002654
002655
002658
002659
002660
002662
002664
002665
002667
002668
002670
002671
002672
002676
002678
002679
002682
002683
002684
002687
002688
002689
002691
002697
002698
002700
002701
002703
002704
002705
002708
002714
002716
002718
002719
002723
002731
002732
002733
002734
002736
002738
002739
002741
002743
002750
002751
002754
002756
002759
002762
002766
002768
002769
002770
002771
002774
002776
002777
002778
002779
002780
002781
002782
002784
002785
002788
002790
002791
002792
002795
002798
002799
002802
002803
002807
002808
002813
002816
002817
002819
002821
002822
002823
002824
002825
002829
002832
002834
002835
002837
002838
002842
002843
002849
002850
002851
002852
002854
002855
002857
002859
002860
002862
002864
002865
002868
002869
002870
002871
002872
002873
002874
002882
002884
002886
002887
002888
002897
002898
002899
002904
002906
002907
002909
002910
002912
002913
002915
002918
002920
002921
002922
002923
002926
002927
002929
002931
002932
002933
002936
002938
002939
002940
002941
002943
002946
002949
002950
002952
002954
002956
002965
002967
002968
002969
002970
002972
002973
002975
002980
002981
002983
002986
002987
002989
002990
002992
002996
002998
003002
003008
003009
003012
003013
003014
003015
003016
003017
003018
003020
003021
003023
003026
003028
003036
003037
003039
003040
003041
003044
003045
003049
003051
003057
003059
003060
003063
003064
003068
003069
003070
003072
003075
003077
003078
003079
003081
003083
003084
003085
003086
003089
003091
003092
003093
003095
003097
003098
003100
003104
003105
003108
003111
003113
003115
003117
003119
003120
003121
003122
003123
003125
003128
003130
003132
003138
003139
003140
003143
003147
003149
003151
003152
003154
003155
003157
003158
003160
003163
003164
003166
003168
003169
003171
003173
003176
003178
003184
003185
003186
003188
003189
003191
003193
003195
003196
003198
003200
003201
003205
003206
003208
003209
003212
003213
003215
003218
003220
003223
003227
003230
003234
003235
003237
003238
003241
003243
003244
003245
003246
003248
003249
003253
003256
003258
003260
003261
003262
003263
003264
003267
003268
003270
003271
003273
003274
003277
003278
003279
003282
003284
003285
003286
003287
003289
003290
003291
003293
003294
003297
003299
003303
003307
003309
003311
003314
003317
003320
003321
003326
003327
003328
003329
003332
003333
003334
003335
003336
003339
003340
003342
003344
003345
003348
003349
003354
003356
003359
003360
003361
003362
003363
003369
003371
003372
003374
003376
003377
003378
003380
003381
003382
003383
003384
003387
003388
003389
003390
003391
003392
003398
003400
003413
003414
003415
003416
003418
003420
003423
003424
003427
003431
003433
003436
003437
003438
003439
003440
003441
003442
003444
003445
003446
003451
003452
003454
003455
003457
003458
003459
003460
003462
003463
003468
003472
003473
003475
003476
003477
003479
003485
003486
003493
003494
003498
003499
003500
003501
003505
003507
003508
003509
003510
003512
003513
003514
003516
003518
003522
003523
003525
003526
003532
003533
003534
003536
003537
003538
003540
003541
003542
003545
003546
003548
003549
003551
003555
003556
003560
003561
003564
003565
003566
003567
003569
003570
003572
003575
003576
003577
003578
003579
003581
003585
003586
003587
003589
003590
003591
003592
003593
003594
003595
003596
003597
003598
003599
003602
003603
003606
003610
003612
003613
003615
003617
003619
003625
003626
003628
003636
003637
003638
003639
003640
003641
003642
003644
003646
003648
003650
003651
003654
003656
003657
003660
003663
003664
003665
003666
003670
003672
003673
003674
003675
003680
003681
003685
003686
003687
003693
003694
003695
003696
003697
003698
003699
003700
003701
003704
003706
003709
003710
003713
003714
003717
003720
003721
003722
003724
003725
003727
003729
003730
003731
003732
003733
003734
003740
003741
003742
003743
003744
003745
003749
003752
003754
003757
003758
003759
003760
003761
003765
003766
003767
003768
003770
003772
003773
003774
003776
003780
003783
003784
003785
003786
003789
003790
003791
003792
003795
003796
003797
003799
003801
003803
003806
003810
003813
003815
003816
003817
003818
003819
003821
003823
003824
003825
003829
003831
003832
003833
003836
003838
003839
003840
003842
003843
003844
003845
003846
003848
003849
003850
003851
003853
003855
003857
003858
003861
003862
003863
003865
003867
003868
003871
003875
003876
003877
003882
003884
003887
003888
003889
003893
003895
003896
003900
003903
003904
003906
003908
003910
003911
003912
003913
003917
003918
003919
003921
003922
003925
003927
003928
003929
003930
003933
003935
003936
003939
003940
003941
003942
003944
003947
003949
003951
003952
003953
003954
003955
003957
003959
003960
003963
003966
003967
003968
003971
003973
003974
003976
003978
003979
003983
003985
003987
003988
003989
003990
003991
003993
003994
003995
003997
003999
004005
004006
004012
004013
004014
004015
004017
004018
004019
004020
004022
004023
004024
004025
004029
004030
004031
004035
004037
004039
004043
004044
004046
004047
004050
004052
004053
004054
004056
004057
004058
004060
004062
004066
004067
004069
004070
004071
004073
004075
004076
004078
004080
004084
004086
004088
004090
004093
004094
004097
004099
004102
004103
004106
004112
004114
004115
004123
004127
004133
004134
004135
004139
004141
004144
004145
004146
004147
004151
004159
004165
004166
004167
004169
004170
004176
004177
004178
004179
004180
004181
004182
004183
004184
004186
004192
004193
004194
004197
004198
004199
004200
004201
004203
004204
004208
004211
004212
004216
004217
004218
004219
004225
004227
004229
004230
004231
004233
004234
004235
004236
004238
004240
004244
004245
004247
004252
004253
004257
004258
004261
004262
004264
004265
004266
004267
004268
004269
004272
004273
004274
004276
004279
004283
004286
004287
004292
004296
004297
004302
004304
004308
004310
004313
004315
004316
004317
004320
004322
004325
004328
004331
004332
004333
004334
004339
004341
004344
004346
004347
004351
004354
004355
004356
004357
004358
004359
004361
004365
004366
004371
004372
004375
004376
004378
004379
004380
004381
004382
004386
004387
004389
004390
004394
004395
004399
004400
004405
004408
004409
004410
004411
004412
004413
004416
004417
004427
004428
004431
004432
004436
004441
004442
004445
004446
004448
004449
004451
004453
004455
004457
004459
004461
004463
004464
004466
004467
004468
004471
004473
004476
004477
004478
004479
004484
004488
004492
004495
004497
004498
004499
004500
004503
004504
004505
004506
004507
004509
004510
004512
004514
004515
004518
004522
004523
004524
004525
004533
004535
004536
004537
004538
004539
004543
004544
004545
004546
004550
004552
004554
004555
004558
004559
004560
004561
004563
004564
004565
004571
004572
004575
004577
004579
004580
004583
004584
004586
004590
004592
004593
004594
004595
004597
004600
004601
004602
004604
004605
004606
004607
004613
004614
004616
004617
004619
004621
004623
004625
004627
004628
004631
004635
004637
004639
004641
004642
004643
004645
004646
004653
004654
004656
004659
004661
004662
004663
004664
004670
004671
004674
004675
004676
004677
004678
004681
004684
004690
004696
004701
004702
004703
004704
004707
004712
004719
004723
004727
004728
004729
004731
004733
004736
004741
004747
004749
004750
004751
004754
004755
004757
004758
004760
004761
004765
004767
004771
004772
004774
004775
004778
004779
004780
004781
004784
004785
004786
004789
004793
004794
004795
004796
004798
004801
004802
004803
004805
004808
004809
004812
004818
004819
004820
004823
004824
004826
004827
004828
004833
004834
004836
004837
004838
004840
004841
004842
004844
004845
004847
004853
004854
004855
004856
004857
004865
004866
004869
004870
004872
004876
004877
004878
004879
004880
004882
004883
004884
004886
004889
004890
004894
004897
004899
004900
004901
004906
004908
004910
004911
004912
004913
004915
004916
004919
004922
004923
004925
004930
004933
004936
004937
004939
004940
004945
004950
004951
004952
004955
004957
004961
004964
004965
004967
004968
004969
004970
004971
004972
004973
004975
004977
004978
004980
004982
004984
004987
004991
004992
004997
005000
005003
005005
005006
005007
005009
005011
005012
005016
005018
005020
005022
005023
005025
005027
005029
005030
005031
005033
005035
005039
005042
005043
005044
005046
005047
005048
005051
005059
005060
005061
005066
005069
005071
005076
005083
005084
005085
005087
005088
005089
005091
005092
005096
005097
005098
005099
005100
005102
005104
005106
005107
005111
005114
005115
005116
005117
005118
005119
005123
005126
005129
005130
005131
005132
005134
005137
005142
005146
005148
005150
005151
005152
005154
005159
005160
005165
005169
005171
005173
005177
005178
005183
005186
005187
005192
005193
005195
005196
005200
005202
005203
005204
005205
005207
005208
005209
005210
005211
005212
005215
005216
005220
005223
005224
005225
005228
005231
005232
005235
005238
005239
005243
005245
005247
005248
005250
005252
005253
005254
005257
005258
005259
005261
005263
005264
005265
005266
005269
005270
005272
005277
005278
005281
005283
005285
005286
005288
005290
005291
005293
005294
005295
005300
005301
005302
005303
005305
005306
005310
005314
005317
005320
005324
005326
005327
005331
005332
005339
005340
005344
005346
005348
005351
005352
005353
005354
005355
005356
005357
005358
005361
005362
005364
005367
005370
005373
005374
005376
005380
005382
005383
005384
005387
005388
005392
005393
005394
005395
005396
005397
005398
005399
005400
005401
005402
005403
005406
005407
005408
005409
005410
005411
005412
005414
005416
005417
005418
005419
005420
005421
005424
005425
005428
005432
005433
005435
005436
005438
005439
005440
005442
005446
005451
005454
005455
005456
005457
005462
005463
005464
005468
005469
005470
005475
005478
005480
005483
005485
005488
005490
005491
005492
005493
005496
005497
005499
005500
005501
005502
005503
005504
005506
005507
005508
005509
005512
005513
005516
005517
005518
005519
005520
005521
005522
005524
005526
005527
005529
005530
005533
005535
005537
005539
005541
005543
005547
005548
005549
005550
005553
005554
005561
005562
005563
005564
005567
005568
005569
005574
005575
005578
005579
005583
005585
005591
005592
005593
005594
005597
005598
005599
005604
005605
005606
005607
005608
005609
005611
005612
005614
005615
005620
005621
005622
005624
005626
005627
005628
005629
005632
005636
005637
005641
005644
005645
005646
005647
005648
005651
005654
005655
005657
005661
005663
005665
005666
005667
005670
005671
005674
005675
005678
005679
005681
005682
005684
005686
005688
005690
005691
005692
005693
005694
005696
005697
005701
005702
005705
005710
005711
005715
005716
005718
005719
005720
005721
005722
005723
005726
005730
005732
005733
005734
005737
005738
005742
005748
005749
005750
005752
005753
005755
005756
005758
005759
005761
005764
005766
005767
005768
005769
005770
005771
005772
005773
005774
005775
005776
005778
005779
005780
005781
005788
005789
005791
005792
005795
005797
005798
005799
005802
005804
005808
005809
005810
005813
005814
005815
005816
005817
005823
005824
005825
005828
005830
005831
005832
005833
005835
005836
005837
005838
005842
005844
005845
005846
005847
005848
005849
005850
005851
005853
005858
005860
005861
005862
005863
005865
005866
005867
005868
005870
005871
005872
005874
005875
005877
005880
005884
005886
005888
005890
005891
005895
005896
005897
005898
005902
005904
005908
005915
005920
005924
005928
005929
005930
005932
005934
005936
005937
005940
005941
005942
005943
005945
005946
005950
005951
005953
005954
005956
005957
005959
005960
005964
005966
005967
005968
005971
005973
005974
005976
005977
005979
005980
005983
005987
005989
005990
005991
005992
005993
005995
005998
006000
006004
006006
006007
006011
006015
006017
006018
006019
006020
006021
006022
006025
006032
006035
006037
006040
006049
006051
006053
006055
006056
006059
006064
006065
006069
006072
006073
006076
006079
006080
006081
006082
006084
006089
006090
006091
006092
006094
006099
006101
006104
006105
006108
006109
006111
006112
006113
006119
006120
006124
006128
006129
006131
006132
006134
006135
006137
006138
006140
006141
006142
006143
006145
006147
006149
006150
006153
006155
006157
006158
006159
006160
006162
006164
006166
006170
006171
006172
006174
006175
006178
006179
006180
006181
006183
006184
006188
006189
006191
006192
006193
006197
006199
006200
006201
006203
006205
006206
006207
006209
006211
006212
006214
006216
006217
006218
006220
006221
006223
006224
006225
006226
006230
006231
006234
006235
006236
006237
006239
006241
006242
006243
006245
006248
006251
006252
006253
006254
006255
006256
006257
006259
006260
006261
006262
006264
006268
006271
006277
006279
006281
006283
006284
006285
006289
006290
006291
006292
006293
006294
006295
006296
006298
006299
006303
006304
006307
006308
006309
006310
006311
006313
006318
006319
006320
006323
006325
006326
006327
006328
006329
006330
006335
006336
006337
006341
006346
006347
006350
006352
006358
006359
006361
006362
006363
006365
006367
006373
006374
006375
006376
006378
006382
006383
006384
006387
006389
006390
006392
006397
006398
006399
006400
006401
006402
006404
006408
006412
006413
006414
006418
006419
006421
006422
006428
006429
006430
006431
006432
006438
006443
006447
006448
006449
006450
006455
006456
006457
006458
006459
006460
006461
006463
006466
006467
006471
006476
006479
006480
006485
006487
006489
006490
006492
006494
006495
006499
006500
006501
006502
006504
006509
006510
006511
006513
006518
006522
006523
006526
006527
006528
006536
006538
006539
006541
006543
006544
006545
006546
006547
006550
006552
006554
006557
006559
006562
006564
006566
006567
006571
006572
006573
006575
006579
006580
006584
006585
006587
006589
006591
006594
006598
006599
006600
006601
006605
006606
006607
006608
006609
006610
006615
006616
006617
006619
006620
006621
006622
006627
006630
006631
006635
006639
006640
006642
006644
006645
006646
006648
006652
006653
006654
006657
006661
006662
006663
006665
006668
006671
006672
006673
006675
006680
006681
006683
006684
006687
006688
006689
006690
006691
006697
006699
006700
006702
006704
006705
006706
006707
006708
006716
006717
006718
006721
006722
006724
006727
006728
006730
006735
006736
006739
006740
006742
006743
006746
006748
006749
006750
006757
006763
006766
006769
006774
006775
006776
006779
006784
006787
006788
006790
006793
006795
006799
006801
006802
006805
006809
006810
006814
006817
006820
006821
006823
006824
006825
006826
006827
006830
006831
006834
006835
006838
006839
006840
006842
006845
006846
006848
006851
006857
006859
006861
006864
006865
006867
006869
006871
006875
006877
006878
006880
006883
006886
006888
006890
006892
006893
006894
006896
006902
006904
006905
006909
006911
006912
006915
006916
006918
006919
006920
006921
006923
006924
006926
006927
006929
006931
006932
006933
006934
006935
006939
006940
006941
006946
006947
006949
006951
006952
006957
006958
006961
006963
006965
006966
006967
006969
006970
006972
006974
006975
006976
006979
006983
006984
006985
006986
006988
006991
006993
006995
006996
006998
007001
007002
007004
007007
007009
007013
007017
007018
007020
007021
007024
007025
007035
007036
007039
007040
007041
007044
007045
007046
007050
007051
007054
007057
007058
007060
007062
007064
007066
007070
007073
007075
007077
007086
007090
007092
007093
007094
007096
007097
007099
007101
007102
007104
007105
007106
007107
007108
007111
007113
007114
007116
007118
007121
007123
007124
007126
007127
007128
007129
007134
007137
007140
007141
007142
007143
007147
007148
007150
007151
007152
007153
007155
007156
007159
007160
007167
007170
007171
007173
007175
007179
007181
007184
007185
007186
007188
007189
007190
007191
007192
007193
007195
007196
007197
007203
007206
007209
007211
007213
007216
007218
007220
007222
007223
007224
007226
007228
007231
007234
007236
007237
007239
007241
007243
007245
007248
007249
007250
007251
007254
007257
007259
007263
007264
007268
007269
007270
007276
007281
007282
007285
007286
007293
007295
007296
007297
007298
007301
007305
007306
007307
007308
007312
007313
007314
007316
007317
007320
007321
007324
007328
007332
007333
007334
007335
007338
007340
007341
007346
007348
007354
007355
007356
007357
007358
007361
007362
007363
007365
007366
007367
007368
007370
007372
007373
007378
007379
007386
007387
007388
007390
007392
007393
007394
007399
007400
007404
007406
007408
007414
007417
007418
007425
007427
007428
007429
007431
007432
007438
007441
007443
007444
007446
007451
007452
007454
007455
007457
007459
007460
007461
007465
007471
007472
007474
007476
007479

'''[1:-1],
    'splits/val.txt': r'''
000001
000002
000004
000005
000006
000008
000015
000019
000020
000021
000023
000024
000025
000027
000028
000031
000033
000035
000037
000039
000040
000042
000047
000048
000050
000052
000053
000058
000059
000061
000062
000063
000065
000066
000076
000077
000078
000081
000089
000090
000093
000094
000098
000102
000104
000106
000107
000108
000116
000117
000118
000122
000124
000126
000128
000132
000134
000135
000137
000139
000140
000143
000147
000151
000152
000153
000156
000159
000161
000167
000168
000169
000170
000173
000174
000175
000181
000182
000183
000186
000187
000188
000190
000191
000192
000194
000195
000196
000197
000199
000201
000203
000204
000207
000211
000212
000213
000216
000218
000223
000224
000226
000229
000230
000231
000234
000235
000236
000237
000239
000242
000246
000247
000248
000249
000250
000251
000252
000260
000262
000263
000265
000266
000268
000269
000270
000272
000273
000278
000279
000281
000283
000284
000289
000290
000291
000293
000297
000301
000302
000305
000307
000308
000309
000311
000312
000314
000315
000319
000320
000321
000323
000324
000327
000328
000329
000332
000333
000335
000336
000340
000341
000343
000345
000346
000347
000350
000351
000352
000354
000355
000356
000357
000359
000360
000361
000362
000365
000366
000369
000370
000372
000373
000376
000377
000378
000379
000381
000382
000383
000385
000386
000388
000391
000392
000393
000394
000395
000396
000397
000398
000399
000401
000402
000403
000404
000407
000408
000409
000413
000414
000415
000419
000420
000422
000427
000428
000429
000430
000436
000437
000440
000443
000446
000448
000450
000451
000452
000453
000454
000455
000457
000459
000463
000468
000469
000472
000473
000475
000476
000477
000478
000479
000480
000481
000485
000486
000489
000491
000492
000493
000494
000495
000496
000498
000499
000503
000504
000506
000508
000509
000510
000512
000515
000517
000519
000521
000524
000527
000528
000530
000533
000536
000541
000542
000543
000545
000546
000548
000551
000554
000555
000558
000559
000560
000561
000564
000566
000567
000568
000569
000571
000572
000581
000583
000588
000589
000590
000591
000595
000600
000601
000604
000610
000611
000612
000613
000614
000615
000618
000619
000620
000624
000625
000626
000628
000630
000634
000635
000636
000639
000642
000644
000645
000647
000648
000650
000655
000657
000658
000659
000660
000667
000669
000670
000674
000677
000679
000682
000683
000684
000691
000692
000694
000696
000698
000699
000700
000702
000704
000706
000708
000716
000717
000718
000721
000722
000725
000727
000728
000729
000731
000734
000736
000737
000740
000741
000745
000746
000748
000750
000751
000752
000754
000756
000761
000765
000766
000767
000768
000769
000771
000772
000773
000774
000778
000779
000782
000790
000792
000795
000798
000800
000801
000802
000803
000804
000805
000806
000807
000809
000810
000811
000812
000816
000819
000823
000826
000831
000837
000838
000840
000841
000843
000844
000847
000848
000849
000850
000852
000854
000859
000862
000863
000869
000873
000874
000875
000876
000877
000878
000879
000881
000884
000885
000889
000893
000894
000897
000899
000904
000907
000909
000911
000912
000915
000916
000917
000920
000922
000923
000926
000928
000930
000931
000932
000938
000939
000940
000942
000943
000944
000948
000949
000952
000953
000956
000958
000961
000963
000964
000966
000967
000969
000970
000971
000973
000974
000976
000979
000981
000983
000984
000985
000986
000988
000991
000999
001002
001006
001007
001008
001010
001011
001012
001013
001014
001015
001018
001019
001021
001022
001025
001026
001027
001035
001037
001039
001042
001043
001046
001050
001051
001053
001054
001055
001058
001063
001065
001066
001067
001068
001069
001070
001071
001075
001076
001077
001078
001083
001084
001086
001088
001089
001094
001095
001096
001097
001099
001101
001102
001104
001106
001107
001108
001111
001113
001114
001115
001116
001118
001120
001123
001125
001127
001129
001131
001132
001133
001134
001135
001136
001138
001139
001140
001141
001143
001144
001145
001147
001148
001149
001150
001152
001153
001154
001155
001158
001162
001163
001167
001172
001173
001176
001177
001178
001179
001180
001182
001183
001187
001188
001189
001191
001192
001193
001194
001195
001198
001199
001203
001206
001207
001213
001214
001216
001217
001218
001221
001222
001224
001225
001226
001228
001230
001232
001234
001235
001236
001237
001239
001241
001242
001243
001244
001245
001246
001249
001251
001252
001253
001254
001255
001257
001259
001260
001261
001263
001265
001266
001267
001268
001269
001270
001271
001272
001273
001274
001275
001281
001284
001286
001287
001289
001291
001292
001294
001295
001296
001303
001304
001305
001306
001307
001308
001314
001317
001318
001329
001330
001331
001332
001333
001334
001336
001337
001339
001342
001344
001345
001346
001347
001350
001352
001353
001355
001356
001359
001363
001365
001372
001374
001375
001376
001377
001380
001381
001382
001384
001386
001387
001388
001389
001391
001395
001397
001398
001407
001410
001411
001412
001415
001416
001419
001421
001424
001427
001431
001432
001435
001437
001438
001439
001441
001442
001443
001445
001446
001448
001450
001451
001458
001461
001463
001466
001469
001471
001477
001478
001480
001481
001485
001487
001488
001489
001495
001497
001501
001502
001507
001508
001511
001513
001514
001516
001517
001521
001522
001524
001525
001526
001527
001528
001533
001535
001536
001537
001538
001542
001545
001546
001547
001549
001552
001555
001557
001560
001562
001564
001565
001567
001569
001573
001574
001576
001577
001579
001582
001583
001585
001586
001587
001588
001589
001590
001591
001592
001594
001596
001597
001600
001602
001603
001605
001606
001610
001613
001615
001616
001617
001619
001621
001625
001627
001629
001631
001633
001634
001635
001640
001643
001645
001647
001650
001654
001656
001658
001660
001662
001664
001665
001666
001667
001670
001675
001680
001682
001683
001684
001689
001693
001694
001697
001699
001701
001702
001704
001705
001706
001707
001709
001710
001711
001712
001713
001714
001717
001718
001719
001721
001722
001726
001727
001729
001732
001733
001740
001741
001742
001745
001746
001749
001750
001751
001752
001755
001758
001762
001764
001765
001768
001771
001772
001774
001776
001778
001780
001781
001782
001783
001786
001787
001794
001795
001797
001800
001801
001802
001804
001807
001808
001813
001814
001817
001818
001820
001822
001823
001824
001825
001828
001831
001835
001840
001844
001846
001848
001851
001852
001853
001854
001855
001856
001858
001859
001861
001862
001863
001867
001868
001869
001872
001875
001877
001878
001880
001881
001884
001885
001886
001887
001888
001890
001892
001893
001897
001898
001900
001904
001905
001909
001919
001920
001923
001924
001925
001926
001927
001928
001929
001931
001932
001933
001934
001936
001937
001940
001941
001942
001943
001945
001946
001952
001954
001959
001960
001966
001967
001969
001972
001977
001978
001979
001980
001982
001983
001984
001985
001986
001989
001991
001995
001996
001997
001999
002000
002001
002002
002004
002008
002010
002011
002012
002013
002014
002017
002019
002021
002022
002025
002027
002028
002029
002034
002035
002036
002037
002038
002042
002043
002044
002045
002046
002048
002049
002050
002052
002054
002056
002057
002058
002062
002068
002071
002073
002074
002075
002076
002078
002079
002081
002082
002085
002086
002087
002089
002091
002093
002094
002100
002101
002102
002103
002107
002108
002111
002112
002113
002115
002118
002120
002121
002123
002124
002127
002128
002130
002131
002135
002136
002137
002138
002139
002140
002142
002151
002152
002153
002158
002159
002160
002161
002163
002165
002166
002168
002169
002170
002173
002177
002179
002182
002183
002185
002187
002188
002193
002196
002200
002201
002202
002206
002207
002209
002215
002216
002218
002219
002220
002224
002225
002228
002229
002232
002233
002234
002239
002243
002245
002246
002248
002250
002251
002254
002255
002257
002258
002260
002262
002266
002272
002276
002277
002279
002280
002282
002283
002284
002286
002287
002290
002291
002292
002293
002294
002295
002298
002299
002300
002303
002304
002306
002307
002308
002310
002314
002315
002319
002320
002325
002327
002329
002330
002332
002334
002336
002337
002338
002340
002341
002344
002345
002346
002347
002348
002353
002356
002357
002359
002362
002365
002366
002367
002369
002370
002372
002376
002378
002380
002382
002383
002384
002385
002386
002387
002391
002392
002393
002397
002398
002399
002404
002405
002411
002414
002415
002418
002419
002420
002422
002423
002424
002425
002428
002429
002432
002433
002434
002439
002440
002442
002446
002450
002454
002455
002457
002458
002460
002461
002462
002463
002473
002474
002476
002477
002478
002479
002483
002486
002488
002490
002492
002495
002497
002499
002500
002502
002503
002504
002505
002506
002509
002511
002516
002519
002520
002521
002525
002526
002528
002529
002530
002531
002532
002534
002538
002539
002540
002541
002543
002546
002548
002552
002556
002557
002558
002562
002563
002564
002565
002568
002569
002570
002572
002574
002575
002577
002580
002581
002583
002584
002585
002586
002590
002594
002598
002599
002600
002601
002602
002603
002604
002606
002612
002613
002615
002619
002621
002625
002626
002628
002630
002631
002633
002635
002636
002638
002640
002641
002644
002645
002646
002651
002653
002656
002657
002661
002663
002666
002669
002673
002674
002675
002677
002680
002681
002685
002686
002690
002692
002693
002694
002695
002696
002699
002702
002706
002707
002709
002710
002711
002712
002713
002715
002717
002720
002721
002722
002724
002725
002726
002727
002728
002729
002730
002735
002737
002740
002742
002744
002745
002746
002747
002748
002749
002752
002753
002755
002757
002758
002760
002761
002763
002764
002765
002767
002772
002773
002775
002783
002786
002787
002789
002793
002794
002796
002797
002800
002801
002804
002805
002806
002809
002810
002811
002812
002814
002815
002818
002820
002826
002827
002828
002830
002831
002833
002836
002839
002840
002841
002844
002845
002846
002847
002848
002853
002856
002858
002861
002863
002866
002867
002875
002876
002877
002878
002879
002880
002881
002883
002885
002889
002890
002891
002892
002893
002894
002895
002896
002900
002901
002902
002903
002905
002908
002911
002914
002916
002917
002919
002924
002925
002928
002930
002934
002935
002937
002942
002944
002945
002947
002948
002951
002953
002955
002957
002958
002959
002960
002961
002962
002963
002964
002966
002971
002974
002976
002977
002978
002979
002982
002984
002985
002988
002991
002993
002994
002995
002997
002999
003000
003001
003003
003004
003005
003006
003007
003010
003011
003019
003022
003024
003025
003027
003029
003030
003031
003032
003033
003034
003035
003038
003042
003043
003046
003047
003048
003050
003052
003053
003054
003055
003056
003058
003061
003062
003065
003066
003067
003071
003073
003074
003076
003080
003082
003087
003088
003090
003094
003096
003099
003101
003102
003103
003106
003107
003109
003110
003112
003114
003116
003118
003124
003126
003127
003129
003131
003133
003134
003135
003136
003137
003141
003142
003144
003145
003146
003148
003150
003153
003156
003159
003161
003162
003165
003167
003170
003172
003174
003175
003177
003179
003180
003181
003182
003183
003187
003190
003192
003194
003197
003199
003202
003203
003204
003207
003210
003211
003214
003216
003217
003219
003221
003222
003224
003225
003226
003228
003229
003231
003232
003233
003236
003239
003240
003242
003247
003250
003251
003252
003254
003255
003257
003259
003265
003266
003269
003272
003275
003276
003280
003281
003283
003288
003292
003295
003296
003298
003300
003301
003302
003304
003305
003306
003308
003310
003312
003313
003315
003316
003318
003319
003322
003323
003324
003325
003330
003331
003337
003338
003341
003343
003346
003347
003350
003351
003352
003353
003355
003357
003358
003364
003365
003366
003367
003368
003370
003373
003375
003379
003385
003386
003393
003394
003395
003396
003397
003399
003401
003402
003403
003404
003405
003406
003407
003408
003409
003410
003411
003412
003417
003419
003421
003422
003425
003426
003428
003429
003430
003432
003434
003435
003443
003447
003448
003449
003450
003453
003456
003461
003464
003465
003466
003467
003469
003470
003471
003474
003478
003480
003481
003482
003483
003484
003487
003488
003489
003490
003491
003492
003495
003496
003497
003502
003503
003504
003506
003511
003515
003517
003519
003520
003521
003524
003527
003528
003529
003530
003531
003535
003539
003543
003544
003547
003550
003552
003553
003554
003557
003558
003559
003562
003563
003568
003571
003573
003574
003580
003582
003583
003584
003588
003600
003601
003604
003605
003607
003608
003609
003611
003614
003616
003618
003620
003621
003622
003623
003624
003627
003629
003630
003631
003632
003633
003634
003635
003643
003645
003647
003649
003652
003653
003655
003658
003659
003661
003662
003667
003668
003669
003671
003676
003677
003678
003679
003682
003683
003684
003688
003689
003690
003691
003692
003702
003703
003705
003707
003708
003711
003712
003715
003716
003718
003719
003723
003726
003728
003735
003736
003737
003738
003739
003746
003747
003748
003750
003751
003753
003755
003756
003762
003763
003764
003769
003771
003775
003777
003778
003779
003781
003782
003787
003788
003793
003794
003798
003800
003802
003804
003805
003807
003808
003809
003811
003812
003814
003820
003822
003826
003827
003828
003830
003834
003835
003837
003841
003847
003852
003854
003856
003859
003860
003864
003866
003869
003870
003872
003873
003874
003878
003879
003880
003881
003883
003885
003886
003890
003891
003892
003894
003897
003898
003899
003901
003902
003905
003907
003909
003914
003915
003916
003920
003923
003924
003926
003931
003932
003934
003937
003938
003943
003945
003946
003948
003950
003956
003958
003961
003962
003964
003965
003969
003970
003972
003975
003977
003980
003981
003982
003984
003986
003992
003996
003998
004000
004001
004002
004003
004004
004007
004008
004009
004010
004011
004016
004021
004026
004027
004028
004032
004033
004034
004036
004038
004040
004041
004042
004045
004048
004049
004051
004055
004059
004061
004063
004064
004065
004068
004072
004074
004077
004079
004081
004082
004083
004085
004087
004089
004091
004092
004095
004096
004098
004100
004101
004104
004105
004107
004108
004109
004110
004111
004113
004116
004117
004118
004119
004120
004121
004122
004124
004125
004126
004128
004129
004130
004131
004132
004136
004137
004138
004140
004142
004143
004148
004149
004150
004152
004153
004154
004155
004156
004157
004158
004160
004161
004162
004163
004164
004168
004171
004172
004173
004174
004175
004185
004187
004188
004189
004190
004191
004195
004196
004202
004205
004206
004207
004209
004210
004213
004214
004215
004220
004221
004222
004223
004224
004226
004228
004232
004237
004239
004241
004242
004243
004246
004248
004249
004250
004251
004254
004255
004256
004259
004260
004263
004270
004271
004275
004277
004278
004280
004281
004282
004284
004285
004288
004289
004290
004291
004293
004294
004295
004298
004299
004300
004301
004303
004305
004306
004307
004309
004311
004312
004314
004318
004319
004321
004323
004324
004326
004327
004329
004330
004335
004336
004337
004338
004340
004342
004343
004345
004348
004349
004350
004352
004353
004360
004362
004363
004364
004367
004368
004369
004370
004373
004374
004377
004383
004384
004385
004388
004391
004392
004393
004396
004397
004398
004401
004402
004403
004404
004406
004407
004414
004415
004418
004419
004420
004421
004422
004423
004424
004425
004426
004429
004430
004433
004434
004435
004437
004438
004439
004440
004443
004444
004447
004450
004452
004454
004456
004458
004460
004462
004465
004469
004470
004472
004474
004475
004480
004481
004482
004483
004485
004486
004487
004489
004490
004491
004493
004494
004496
004501
004502
004508
004511
004513
004516
004517
004519
004520
004521
004526
004527
004528
004529
004530
004531
004532
004534
004540
004541
004542
004547
004548
004549
004551
004553
004556
004557
004562
004566
004567
004568
004569
004570
004573
004574
004576
004578
004581
004582
004585
004587
004588
004589
004591
004596
004598
004599
004603
004608
004609
004610
004611
004612
004615
004618
004620
004622
004624
004626
004629
004630
004632
004633
004634
004636
004638
004640
004644
004647
004648
004649
004650
004651
004652
004655
004657
004658
004660
004665
004666
004667
004668
004669
004672
004673
004679
004680
004682
004683
004685
004686
004687
004688
004689
004691
004692
004693
004694
004695
004697
004698
004699
004700
004705
004706
004708
004709
004710
004711
004713
004714
004715
004716
004717
004718
004720
004721
004722
004724
004725
004726
004730
004732
004734
004735
004737
004738
004739
004740
004742
004743
004744
004745
004746
004748
004752
004753
004756
004759
004762
004763
004764
004766
004768
004769
004770
004773
004776
004777
004782
004783
004787
004788
004790
004791
004792
004797
004799
004800
004804
004806
004807
004810
004811
004813
004814
004815
004816
004817
004821
004822
004825
004829
004830
004831
004832
004835
004839
004843
004846
004848
004849
004850
004851
004852
004858
004859
004860
004861
004862
004863
004864
004867
004868
004871
004873
004874
004875
004881
004885
004887
004888
004891
004892
004893
004895
004896
004898
004902
004903
004904
004905
004907
004909
004914
004917
004918
004920
004921
004924
004926
004927
004928
004929
004931
004932
004934
004935
004938
004941
004942
004943
004944
004946
004947
004948
004949
004953
004954
004956
004958
004959
004960
004962
004963
004966
004974
004976
004979
004981
004983
004985
004986
004988
004989
004990
004993
004994
004995
004996
004998
004999
005001
005002
005004
005008
005010
005013
005014
005015
005017
005019
005021
005024
005026
005028
005032
005034
005036
005037
005038
005040
005041
005045
005049
005050
005052
005053
005054
005055
005056
005057
005058
005062
005063
005064
005065
005067
005068
005070
005072
005073
005074
005075
005077
005078
005079
005080
005081
005082
005086
005090
005093
005094
005095
005101
005103
005105
005108
005109
005110
005112
005113
005120
005121
005122
005124
005125
005127
005128
005133
005135
005136
005138
005139
005140
005141
005143
005144
005145
005147
005149
005153
005155
005156
005157
005158
005161
005162
005163
005164
005166
005167
005168
005170
005172
005174
005175
005176
005179
005180
005181
005182
005184
005185
005188
005189
005190
005191
005194
005197
005198
005199
005201
005206
005213
005214
005217
005218
005219
005221
005222
005226
005227
005229
005230
005233
005234
005236
005237
005240
005241
005242
005244
005246
005249
005251
005255
005256
005260
005262
005267
005268
005271
005273
005274
005275
005276
005279
005280
005282
005284
005287
005289
005292
005296
005297
005298
005299
005304
005307
005308
005309
005311
005312
005313
005315
005316
005318
005319
005321
005322
005323
005325
005328
005329
005330
005333
005334
005335
005336
005337
005338
005341
005342
005343
005345
005347
005349
005350
005359
005360
005363
005365
005366
005368
005369
005371
005372
005375
005377
005378
005379
005381
005385
005386
005389
005390
005391
005404
005405
005413
005415
005422
005423
005426
005427
005429
005430
005431
005434
005437
005441
005443
005444
005445
005447
005448
005449
005450
005452
005453
005458
005459
005460
005461
005465
005466
005467
005471
005472
005473
005474
005476
005477
005479
005481
005482
005484
005486
005487
005489
005494
005495
005498
005505
005510
005511
005514
005515
005523
005525
005528
005531
005532
005534
005536
005538
005540
005542
005544
005545
005546
005551
005552
005555
005556
005557
005558
005559
005560
005565
005566
005570
005571
005572
005573
005576
005577
005580
005581
005582
005584
005586
005587
005588
005589
005590
005595
005596
005600
005601
005602
005603
005610
005613
005616
005617
005618
005619
005623
005625
005630
005631
005633
005634
005635
005638
005639
005640
005642
005643
005649
005650
005652
005653
005656
005658
005659
005660
005662
005664
005668
005669
005672
005673
005676
005677
005680
005683
005685
005687
005689
005695
005698
005699
005700
005703
005704
005706
005707
005708
005709
005712
005713
005714
005717
005724
005725
005727
005728
005729
005731
005735
005736
005739
005740
005741
005743
005744
005745
005746
005747
005751
005754
005757
005760
005762
005763
005765
005777
005782
005783
005784
005785
005786
005787
005790
005793
005794
005796
005800
005801
005803
005805
005806
005807
005811
005812
005818
005819
005820
005821
005822
005826
005827
005829
005834
005839
005840
005841
005843
005852
005854
005855
005856
005857
005859
005864
005869
005873
005876
005878
005879
005881
005882
005883
005885
005887
005889
005892
005893
005894
005899
005900
005901
005903
005905
005906
005907
005909
005910
005911
005912
005913
005914
005916
005917
005918
005919
005921
005922
005923
005925
005926
005927
005931
005933
005935
005938
005939
005944
005947
005948
005949
005952
005955
005958
005961
005962
005963
005965
005969
005970
005972
005975
005978
005981
005982
005984
005985
005986
005988
005994
005996
005997
005999
006001
006002
006003
006005
006008
006009
006010
006012
006013
006014
006016
006023
006024
006026
006027
006028
006029
006030
006031
006033
006034
006036
006038
006039
006041
006042
006043
006044
006045
006046
006047
006048
006050
006052
006054
006057
006058
006060
006061
006062
006063
006066
006067
006068
006070
006071
006074
006075
006077
006078
006083
006085
006086
006087
006088
006093
006095
006096
006097
006098
006100
006102
006103
006106
006107
006110
006114
006115
006116
006117
006118
006121
006122
006123
006125
006126
006127
006130
006133
006136
006139
006144
006146
006148
006151
006152
006154
006156
006161
006163
006165
006167
006168
006169
006173
006176
006177
006182
006185
006186
006187
006190
006194
006195
006196
006198
006202
006204
006208
006210
006213
006215
006219
006222
006227
006228
006229
006232
006233
006238
006240
006244
006246
006247
006249
006250
006258
006263
006265
006266
006267
006269
006270
006272
006273
006274
006275
006276
006278
006280
006282
006286
006287
006288
006297
006300
006301
006302
006305
006306
006312
006314
006315
006316
006317
006321
006322
006324
006331
006332
006333
006334
006338
006339
006340
006342
006343
006344
006345
006348
006349
006351
006353
006354
006355
006356
006357
006360
006364
006366
006368
006369
006370
006371
006372
006377
006379
006380
006381
006385
006386
006388
006391
006393
006394
006395
006396
006403
006405
006406
006407
006409
006410
006411
006415
006416
006417
006420
006423
006424
006425
006426
006427
006433
006434
006435
006436
006437
006439
006440
006441
006442
006444
006445
006446
006451
006452
006453
006454
006462
006464
006465
006468
006469
006470
006472
006473
006474
006475
006477
006478
006481
006482
006483
006484
006486
006488
006491
006493
006496
006497
006498
006503
006505
006506
006507
006508
006512
006514
006515
006516
006517
006519
006520
006521
006524
006525
006529
006530
006531
006532
006533
006534
006535
006537
006540
006542
006548
006549
006551
006553
006555
006556
006558
006560
006561
006563
006565
006568
006569
006570
006574
006576
006577
006578
006581
006582
006583
006586
006588
006590
006592
006593
006595
006596
006597
006602
006603
006604
006611
006612
006613
006614
006618
006623
006624
006625
006626
006628
006629
006632
006633
006634
006636
006637
006638
006641
006643
006647
006649
006650
006651
006655
006656
006658
006659
006660
006664
006666
006667
006669
006670
006674
006676
006677
006678
006679
006682
006685
006686
006692
006693
006694
006695
006696
006698
006701
006703
006709
006710
006711
006712
006713
006714
006715
006719
006720
006723
006725
006726
006729
006731
006732
006733
006734
006737
006738
006741
006744
006745
006747
006751
006752
006753
006754
006755
006756
006758
006759
006760
006761
006762
006764
006765
006767
006768
006770
006771
006772
006773
006777
006778
006780
006781
006782
006783
006785
006786
006789
006791
006792
006794
006796
006797
006798
006800
006803
006804
006806
006807
006808
006811
006812
006813
006815
006816
006818
006819
006822
006828
006829
006832
006833
006836
006837
006841
006843
006844
006847
006849
006850
006852
006853
006854
006855
006856
006858
006860
006862
006863
006866
006868
006870
006872
006873
006874
006876
006879
006881
006882
006884
006885
006887
006889
006891
006895
006897
006898
006899
006900
006901
006903
006906
006907
006908
006910
006913
006914
006917
006922
006925
006928
006930
006936
006937
006938
006942
006943
006944
006945
006948
006950
006953
006954
006955
006956
006959
006960
006962
006964
006968
006971
006973
006977
006978
006980
006981
006982
006987
006989
006990
006992
006994
006997
006999
007000
007003
007005
007006
007008
007010
007011
007012
007014
007015
007016
007019
007022
007023
007026
007027
007028
007029
007030
007031
007032
007033
007034
007037
007038
007042
007043
007047
007048
007049
007052
007053
007055
007056
007059
007061
007063
007065
007067
007068
007069
007071
007072
007074
007076
007078
007079
007080
007081
007082
007083
007084
007085
007087
007088
007089
007091
007095
007098
007100
007103
007109
007110
007112
007115
007117
007119
007120
007122
007125
007130
007131
007132
007133
007135
007136
007138
007139
007144
007145
007146
007149
007154
007157
007158
007161
007162
007163
007164
007165
007166
007168
007169
007172
007174
007176
007177
007178
007180
007182
007183
007187
007194
007198
007199
007200
007201
007202
007204
007205
007207
007208
007210
007212
007214
007215
007217
007219
007221
007225
007227
007229
007230
007232
007233
007235
007238
007240
007242
007244
007246
007247
007252
007253
007255
007256
007258
007260
007261
007262
007265
007266
007267
007271
007272
007273
007274
007275
007277
007278
007279
007280
007283
007284
007287
007288
007289
007290
007291
007292
007294
007299
007300
007302
007303
007304
007309
007310
007311
007315
007318
007319
007322
007323
007325
007326
007327
007329
007330
007331
007336
007337
007339
007342
007343
007344
007345
007347
007349
007350
007351
007352
007353
007359
007360
007364
007369
007371
007374
007375
007376
007377
007380
007381
007382
007383
007384
007385
007389
007391
007395
007396
007397
007398
007401
007402
007403
007405
007407
007409
007410
007411
007412
007413
007415
007416
007419
007420
007421
007422
007423
007424
007426
007430
007433
007434
007435
007436
007437
007439
007440
007442
007445
007447
007448
007449
007450
007453
007456
007458
007462
007463
007464
007466
007467
007468
007469
007470
007473
007475
007477
007478
007480

'''[1:-1],
    'pixor_training.py': r'''
#!/usr/bin/env python3
"""Standalone, from-scratch PIXOR Car/BEV reproduction on canonical KITTI val.

This is a documented reproduction recipe, not the original author's code/weights.
Torch is imported only by GPU/model commands so geometry can be checked on CPU.
"""
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import random
import shutil
import subprocess
import sys
import time

import numpy as np

import openpcdet_benchmark as common


GEOMETRY = dict(x_min=0., x_max=70., y_min=-40., y_max=40., z_min=-2.5,
                z_max=1., resolution=.1, stride=4)
# Raw target order: cos(yaw), sin(yaw), dx, dy, log(width), log(length).
# Fixed constants, independent of both train and validation label statistics.
REG_MEAN = np.array([0., 0., 0., 0., math.log(1.6), math.log(3.9)], dtype=np.float32)
REG_SCALE = np.array([1., 1., 3., 3., 1., 1.], dtype=np.float32)
RECIPE = 'pixor_car_bev_reproduction_v1'


def default_settings():
    return dict(epochs=50, batch_size=2, accumulation_steps=4, workers=2, seed=42,
                learning_rate=.01, momentum=.9, weight_decay=.0001, amp=True,
                augmentation=True, eval_every=5, save_every=5, warmup_frames=10,
                score_threshold=.05, nms_iou=.1, pre_nms_topk=1000, max_detections=100,
                copy_data_to_local=True, resume='auto', eval_checkpoint='best',
                kitti_root='', output_root='', work_dir='', run_name='pixor_50epochs', image_sizes_json=None)


def grid_shape(geometry):
    g = geometry
    return (round((g['y_max'] - g['y_min']) / g['resolution']),
            round((g['x_max'] - g['x_min']) / g['resolution']),
            round((g['z_max'] - g['z_min']) / g['resolution']))


def output_grid(geometry):
    height, width, _ = grid_shape(geometry)
    step = geometry['resolution'] * geometry['stride']
    yy, xx = np.meshgrid(geometry['y_min'] + (np.arange(math.ceil(height / 4)) + .5) * step,
                         geometry['x_min'] + (np.arange(math.ceil(width / 4)) + .5) * step,
                         indexing='ij')
    return xx.astype(np.float32), yy.astype(np.float32)


def encode_bev(points, geometry=GEOMETRY):
    """35 height occupancy slices and max reflectance; rows=y, columns=x."""
    g = geometry
    height, width, depth = grid_shape(g)
    bev = np.zeros((depth + 1, height, width), dtype=np.float32)
    points = np.asarray(points, dtype=np.float32).reshape(-1, 4)
    mask = np.isfinite(points).all(axis=1)
    for axis, name in enumerate(('x', 'y', 'z')):
        mask &= (points[:, axis] >= g[name + '_min']) & (points[:, axis] < g[name + '_max'])
    points = points[mask]
    if not len(points):
        return bev
    indices = np.floor((points[:, :3] - [g['x_min'], g['y_min'], g['z_min']]) / g['resolution']).astype(np.int64)
    x, y, z = indices.T
    # Guard floating-point values infinitesimally inside the upper boundary.
    x, y, z = np.minimum(x, width - 1), np.minimum(y, height - 1), np.minimum(z, depth - 1)
    bev[z, y, x] = 1.
    np.maximum.at(bev[-1], (y, x), points[:, 3].clip(0, 1))
    return bev


def read_calibration(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        key, _, value = line.partition(':')
        if key in ('P2', 'R0_rect', 'R_rect', 'Tr_velo_to_cam', 'Tr_velo_cam'):
            values[key] = np.asarray([float(x) for x in value.split()], dtype=np.float64)
    rotation = np.eye(4)
    rotation[:3, :3] = values.get('R0_rect', values.get('R_rect')).reshape(3, 3)
    velo = np.eye(4)
    velo[:3] = values.get('Tr_velo_to_cam', values.get('Tr_velo_cam')).reshape(3, 4)
    result = dict(transform=rotation @ velo, projection=values['P2'].reshape(3, 4))
    if not all(np.isfinite(x).all() for x in result.values()):
        raise ValueError(f'Nonfinite calibration: {path}')
    return result


def read_objects(path, calibration):
    annotation = common.read_annotation(path)
    inverse = np.linalg.inv(calibration['transform'])
    boxes = []
    for i, name in enumerate(annotation['name']):
        if name not in ('Car', 'Van'):
            continue
        length, height, width = annotation['dimensions'][i]
        if min(length, height, width) <= 0:
            continue
        camera = np.asarray([*annotation['location'][i], 1.])
        bottom = inverse @ camera
        # KITTI camera heading is (cos(ry), 0, -sin(ry)).
        ry = annotation['rotation_y'][i]
        direction = inverse[:3, :3] @ [math.cos(ry), 0, -math.sin(ry)]
        boxes.append(dict(name=name, x=float(bottom[0]), y=float(bottom[1]), bottom_z=float(bottom[2]),
                          height=height, width=width, length=length,
                          yaw=math.atan2(float(direction[1]), float(direction[0]))))
    return boxes


def transform_scene(points, boxes, matrix):
    points = points.copy()
    points[:, :2] = points[:, :2] @ matrix[:2, :2].T + matrix[:2, 2]
    result = copy.deepcopy(boxes)
    scale = float(np.linalg.norm(matrix[:2, 0]))
    for box in result:
        xy = matrix @ [box['x'], box['y'], 1.]
        direction = matrix[:2, :2] @ [math.cos(box['yaw']), math.sin(box['yaw'])]
        box.update(x=float(xy[0]), y=float(xy[1]), yaw=math.atan2(float(direction[1]), float(direction[0])),
                   length=box['length'] * scale, width=box['width'] * scale)
    return points, result


def augmentation_matrix():
    angle = float(np.random.uniform(-math.pi / 36, math.pi / 36))
    scale = float(np.random.uniform(.95, 1.05))
    matrix = np.eye(3)
    matrix[:2, :2] = [[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]]
    matrix[:2, :2] *= scale
    if np.random.random() < .5:
        matrix[:2, 1] *= -1
    matrix[:2, 2] = np.random.uniform(-.25, .25, size=2)
    return matrix


def projection_mask(calibration, image_shape, geometry=GEOMETRY, matrix=None, dontcare=()):
    xx, yy = output_grid(geometry)
    xy = np.stack([xx.ravel(), yy.ravel(), np.ones(xx.size)])
    if matrix is not None:
        xy = np.linalg.inv(matrix) @ xy
    points = np.column_stack([xy[0], xy[1], np.full(xx.size, -1.73), np.ones(xx.size)])
    projected = points @ calibration['transform'].T @ calibration['projection'].T
    depth = projected[:, 2]
    uv = projected[:, :2] / np.maximum(depth[:, None], 1e-6)
    height, width = image_shape
    mask = (depth > 0) & (uv[:, 0] >= 0) & (uv[:, 0] < width) & (uv[:, 1] >= 0) & (uv[:, 1] < height)
    # Ground-plane projection is an explicit approximation for FOV/ignore masks.
    for x1, y1, x2, y2 in dontcare:
        mask &= ~((uv[:, 0] >= x1) & (uv[:, 0] <= x2) & (uv[:, 1] >= y1) & (uv[:, 1] <= y2))
    return mask.reshape(xx.shape)


def make_targets(boxes, geometry=GEOMETRY, valid_mask=None):
    xx, yy = output_grid(geometry)
    cls = np.zeros((1, *xx.shape), dtype=np.float32)
    reg = np.zeros((6, *xx.shape), dtype=np.float32)
    valid = np.ones((1, *xx.shape), dtype=np.float32)
    if valid_mask is not None:
        valid[0] = valid_mask
    # Small boxes are assigned last when footprints overlap.
    for box in sorted(boxes, key=lambda b: b['length'] * b['width'], reverse=True):
        dx, dy = xx - box['x'], yy - box['y']
        c, s = math.cos(box['yaw']), math.sin(box['yaw'])
        local_x, local_y = c * dx + s * dy, -s * dx + c * dy
        outer = (np.abs(local_x) <= .6 * box['length']) & (np.abs(local_y) <= .6 * box['width'])
        valid[0, outer] = 0
        inside_roi = geometry['x_min'] <= box['x'] < geometry['x_max'] and geometry['y_min'] <= box['y'] < geometry['y_max']
        if box['name'] != 'Car' or not inside_roi:
            continue
        positive = (np.abs(local_x) <= .35 * box['length']) & (np.abs(local_y) <= .35 * box['width'])
        if valid_mask is not None:
            positive &= valid_mask
        cls[0, positive] = 1
        valid[0, positive] = 1
        raw = np.stack([np.full_like(xx, c), np.full_like(xx, s), -dx, -dy,
                        np.full_like(xx, math.log(box['width'])), np.full_like(xx, math.log(box['length']))])
        normalized = (raw - REG_MEAN[:, None, None]) / REG_SCALE[:, None, None]
        reg[:, positive] = normalized[:, positive]
    # Existing positives have priority over subsequent overlapping ignore bands.
    valid[cls > 0] = 1
    return cls, reg, valid


def decode_arrays(logits, regression, geometry=GEOMETRY, threshold=.05, topk=1000, valid_mask=None):
    logits = np.asarray(logits, dtype=np.float32)
    scores = 1 / (1 + np.exp(-np.clip(logits, -50, 50)))
    mask = (scores >= threshold) & np.isfinite(scores)
    if valid_mask is not None:
        mask &= valid_mask.astype(bool)
    rows, cols = np.where(mask)
    if not len(rows):
        return np.empty((0, 6), dtype=np.float32)
    order = np.argsort(-scores[rows, cols], kind='stable')[:topk]
    rows, cols = rows[order], cols[order]
    raw = regression[:, rows, cols].T * REG_SCALE + REG_MEAN
    xx, yy = output_grid(geometry)
    boxes = np.column_stack([xx[rows, cols] + raw[:, 2], yy[rows, cols] + raw[:, 3],
        np.exp(np.clip(raw[:, 5], -4, 4)), np.exp(np.clip(raw[:, 4], -4, 4)),
        np.arctan2(raw[:, 1], raw[:, 0]), scores[rows, cols]])
    return boxes[np.isfinite(boxes).all(axis=1)].astype(np.float32)


def rotated_nms(boxes, threshold=.1, maximum=100):
    import cv2
    if not len(boxes):
        return boxes
    rectangles = [((float(x), float(y)), (float(length), float(width)), float(yaw * 180 / math.pi))
                  for x, y, length, width, yaw, _ in boxes]
    indices = cv2.dnn.NMSBoxesRotated(rectangles, boxes[:, 5].tolist(), 0., threshold)
    return boxes[np.asarray(indices, dtype=np.int64).reshape(-1)[:maximum]]


def kitti_prediction_row(box, calibration, image_shape, prior):
    x, y, length, width, yaw, score = map(float, box)
    height, bottom_z = float(prior['height']), float(prior['bottom_z'])
    transform, projection = calibration['transform'], calibration['projection']
    location = (transform @ [x, y, bottom_z, 1.])[:3]
    direction = transform[:3, :3] @ [math.cos(yaw), math.sin(yaw), 0]
    ry = math.atan2(-float(direction[2]), float(direction[0]))
    alpha = (ry - math.atan2(float(location[0]), float(location[2])) + math.pi) % (2 * math.pi) - math.pi
    corners = []
    for z in (bottom_z, bottom_z + height):
        for a, b in ((-1, -1), (-1, 1), (1, 1), (1, -1)):
            dx, dy = a * length / 2, b * width / 2
            corners.append([x + math.cos(yaw) * dx - math.sin(yaw) * dy,
                            y + math.sin(yaw) * dx + math.cos(yaw) * dy, z, 1.])
    projected = np.asarray(corners) @ transform.T @ projection.T
    projected = projected[projected[:, 2] > .1]
    if not len(projected):
        return None
    uv = projected[:, :2] / projected[:, 2:3]
    image_height, image_width = image_shape
    bbox = np.r_[uv.min(axis=0), uv.max(axis=0)]
    bbox[[0, 2]] = bbox[[0, 2]].clip(0, image_width - 1)
    bbox[[1, 3]] = bbox[[1, 3]].clip(0, image_height - 1)
    if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        return None
    values = [alpha, *bbox, height, width, length, *location, ry, score]
    return 'Car -1 -1 ' + ' '.join(f'{value:.4f}' for value in values)


def build_model():
    import torch
    from torch import nn

    def conv(in_channels, out_channels, kernel=3, stride=1):
        return nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel, stride, kernel // 2, bias=False),
                             nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

    class Bottleneck(nn.Module):
        def __init__(self, incoming, planes, stride):
            super().__init__()
            self.layers = nn.Sequential(conv(incoming, planes, 1), conv(planes, planes, 3, stride),
                nn.Conv2d(planes, 4 * planes, 1, bias=False), nn.BatchNorm2d(4 * planes))
            self.skip = (nn.Sequential(nn.Conv2d(incoming, 4 * planes, 1, stride, bias=False),
                         nn.BatchNorm2d(4 * planes)) if incoming != 4 * planes or stride != 1 else nn.Identity())
        def forward(self, inputs):
            return torch.relu(self.layers(inputs) + self.skip(inputs))

    class PIXOR(nn.Module):
        def __init__(self):
            super().__init__()
            self.stem = nn.Sequential(conv(36, 32), conv(32, 32))
            self.blocks = nn.ModuleList()
            incoming = 32
            for planes, count in ((24, 3), (48, 6), (64, 6), (96, 3)):
                layers = [Bottleneck(incoming, planes, 2)]
                layers += [Bottleneck(4 * planes, planes, 1) for _ in range(count - 1)]
                self.blocks.append(nn.Sequential(*layers))
                incoming = 4 * planes
            self.lateral5 = nn.Conv2d(384, 196, 1)
            self.lateral4 = nn.Conv2d(256, 128, 1)
            self.lateral3 = nn.Conv2d(192, 96, 1)
            self.up5 = nn.ConvTranspose2d(196, 128, 3, 2, 1)
            self.up4 = nn.ConvTranspose2d(128, 96, 3, 2, 1)
            self.header = nn.Sequential(*[conv(96, 96) for _ in range(4)], nn.Conv2d(96, 7, 3, padding=1))
            for module in self.modules():
                if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
            nn.init.normal_(self.header[-1].weight, std=.001)
            with torch.no_grad():
                self.header[-1].bias[0] = math.log(.01 / .99)
        def forward(self, inputs):
            features = self.stem(inputs)
            levels = []
            for block in self.blocks:
                features = block(features)
                levels.append(features)
            p4 = self.lateral4(levels[2])
            p4 = p4 + self.up5(self.lateral5(levels[3]), output_size=p4.shape)
            p3 = self.lateral3(levels[1])
            return self.header(p3 + self.up4(p4, output_size=p3.shape))

    return PIXOR()


def compute_loss(prediction, classification, regression, valid):
    import torch
    from torch.nn import functional as functional
    logits, values = prediction[:, :1].float(), prediction[:, 1:].float()
    classification, regression, valid = classification.float(), regression.float(), valid.float()
    probability = logits.sigmoid()
    pt = probability * classification + (1 - probability) * (1 - classification)
    alpha = .25 * classification + .75 * (1 - classification)
    denominator = classification.sum().clamp(min=1)
    cls = (functional.binary_cross_entropy_with_logits(logits, classification, reduction='none')
           * alpha * (1 - pt).square() * valid).sum() / denominator
    reg = (functional.smooth_l1_loss(values, regression, reduction='none') * classification).sum() / denominator
    return cls + reg, cls.detach(), reg.detach()


def read_settings(path):
    settings = {**default_settings(), **json.loads(Path(path).read_text())}
    for name in ('epochs', 'batch_size', 'accumulation_steps', 'eval_every', 'save_every',
                 'warmup_frames', 'pre_nms_topk', 'max_detections'):
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} must be a positive integer')
    if type(settings['workers']) is not int or settings['workers'] < 0:
        raise ValueError('workers must be a nonnegative integer')
    for name in ('score_threshold', 'nms_iou'):
        if not 0 < settings[name] < 1:
            raise ValueError(f'{name} must be between 0 and 1')
    if not settings['run_name'] or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-. ' for c in settings['run_name']) or settings['run_name'] in ('.', '..'):
        raise ValueError('run_name must be a simple directory name')
    for name in ('kitti_root', 'output_root', 'work_dir'):
        if not settings[name]:
            raise ValueError(f'Set {name}')
        settings[name] = str(Path(settings[name]).expanduser().resolve())
    raw = Path(settings['kitti_root'])
    for name in ('output_root', 'work_dir'):
        destination = Path(settings[name])
        if destination == raw or raw in destination.parents:
            raise ValueError(f'{name} must be outside the source dataset')
    return settings


def paths(settings):
    return Path(settings['work_dir']), Path(settings['output_root']) / settings['run_name']


def audit_partition(work):
    result = {}
    for name in ('train', 'val'):
        result[name] = common.read_ids(Path(work) / 'splits' / (name + '.txt'))
        digest = hashlib.sha256(('\n'.join(result[name]) + '\n').encode()).hexdigest()
        if digest != common.SPLIT_HASHES[name]:
            raise ValueError(f'{name}: expected the canonical KITTI 3712/3769 partition')
    common.check_partition(result['train'], result['val'])
    return result


def verify_reference(work):
    for name, digest in common.EVALUATOR_HASHES.items():
        if common.sha256(Path(work) / 'kitti_reference' / name) != digest:
            raise ValueError('Reference evaluator source changed: ' + name)


def prepare(settings):
    work, output = paths(settings)
    splits = audit_partition(work)
    verify_reference(work)
    raw = common.normalize_training_root(settings['kitti_root'], settings.get('image_sizes_json'))
    identifiers = splits['train'] + splits['val']
    shapes, image_sources, image_metadata = common.load_image_dimensions(raw, identifiers, settings.get('image_sizes_json'))
    root = work / 'kitti_local/training' if settings['copy_data_to_local'] else raw
    records, pending, train_heights, train_bottoms = {}, [], [], []
    signature = hashlib.sha256()
    for path in image_sources:
        if path.suffix.lower() == '.json':
            signature.update(path.read_bytes())
        else:
            with path.open('rb') as stream:
                signature.update(stream.read(24))
    for split, identifiers in splits.items():
        for index, identifier in enumerate(identifiers):
            for folder, suffix in (('velodyne', '.bin'), ('calib', '.txt'), ('label_2', '.txt')):
                original = raw / folder / (identifier + suffix)
                stat = original.stat()
                if folder == 'velodyne' and (stat.st_size == 0 or stat.st_size % 16):
                    raise ValueError(f'Invalid XYZI point file: {original}')
                signature.update(f'{identifier}/{folder}:{stat.st_size}:{stat.st_mtime_ns}'.encode())
                if folder in ('label_2', 'calib'):
                    signature.update(original.read_bytes())
                # PNG pixels are never model input; cache only dimensions.
                if settings['copy_data_to_local'] and folder != 'image_2':
                    target = root / folder / original.name
                    if not target.exists() or (target.stat().st_size, target.stat().st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
                        pending.append((original, target, stat.st_size))
            calibration = read_calibration(raw / 'calib' / (identifier + '.txt'))
            label = raw / 'label_2' / (identifier + '.txt')
            boxes = read_objects(label, calibration)
            annotation = common.read_annotation(label)
            dontcare = [annotation['bbox'][i] for i, name in enumerate(annotation['name']) if name == 'DontCare']
            record = dict(image_shape=shapes[identifier])
            if split == 'train':
                record.update(boxes=boxes, dontcare=dontcare)
                for box in boxes:
                    if box['name'] == 'Car':
                        train_heights.append(box['height'])
                        train_bottoms.append(box['bottom_z'])
            records[identifier] = record
            if (index + 1) % 500 == 0:
                print(f'Audit {split}: {index + 1}/{len(identifiers)}', flush=True)
    if not train_heights:
        raise ValueError('No Car training labels found')
    required = sum(n for _, _, n in pending)
    if pending and shutil.disk_usage(work).free < required + 1024 ** 3:
        raise RuntimeError(f'Need {required / 1e9:.1f} GB for local staging; disable COPY_DATA_TO_LOCAL or choose a larger disk.')
    for i, (original, target, _) in enumerate(pending):
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + '.copying')
        shutil.copy2(original, temporary)
        temporary.replace(target)
        if (i + 1) % 500 == 0:
            print(f'Copy local: {i + 1}/{len(pending)}', flush=True)
    output.mkdir(parents=True, exist_ok=True)
    assets = dict(training_root=str(root), source_training_root=str(raw), splits=splits, frames=records,
                  asset_signature=signature.hexdigest(), image_sizes_json=image_metadata,
                  projection_prior=dict(height=float(np.median(train_heights)), bottom_z=float(np.median(train_bottoms))),
                  prior_scope='Median Car height/bottom-z from train IDs only, used only to project BEV boxes into KITTI format. No AP3D is computed.')
    common.write_json(work / 'assets.json', assets)
    common.write_json(output / 'data_audit.json', {key: value for key, value in assets.items() if key != 'frames'})
    common.write_json(output / 'config.resolved.json', {**settings, 'recipe': RECIPE, 'geometry': GEOMETRY,
        'regression_mean': REG_MEAN.tolist(), 'regression_scale': REG_SCALE.tolist(),
        'architecture_blocks': [3, 6, 6, 3], 'classes': ['Car'], 'metric': 'APBEV_R40_IoU0.7',
        'selection': 'Best full validation Car Moderate R40, evaluated every eval_every epochs and at the final epoch',
        'changes_from_original': '50-epoch SGD recipe; fixed regression normalization; max reflectance; BN/ReLU header; inner/outer target masks; BEV-plane augmentation and projected FOV/DontCare masks. A reproduction, not an exact replay of original author experiments.'})
    for name in splits:
        shutil.copyfile(work / 'splits' / (name + '.txt'), output / (name + '.txt'))
    print('Prepared 3712 train / 3769 val; no random repartition; no val labels used for training/prior.', flush=True)


def training_identity(settings, asset_signature):
    keys = ('epochs', 'batch_size', 'accumulation_steps', 'seed', 'learning_rate', 'momentum', 'weight_decay',
            'amp', 'augmentation', 'eval_every', 'score_threshold', 'nms_iou', 'pre_nms_topk', 'max_detections')
    return common.fingerprint(dict(recipe=RECIPE, settings={key: settings[key] for key in keys},
        geometry=GEOMETRY, regression_mean=REG_MEAN.tolist(), regression_scale=REG_SCALE.tolist(),
        splits=common.SPLIT_HASHES, asset_signature=asset_signature,
        runner_sha256=common.sha256(__file__)))


def load_assets(settings):
    work, _ = paths(settings)
    splits = audit_partition(work)
    assets = json.loads((work / 'assets.json').read_text())
    if assets['splits'] != splits or assets['source_training_root'] != str(common.normalize_training_root(settings['kitti_root'], settings.get('image_sizes_json'))):
        raise ValueError('Dataset settings changed; rerun prepare')
    return assets


def setup(settings):
    import torch
    work, output = paths(settings)
    if platform.system() != 'Linux' or not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError('Use Linux Python 3.10–3.13 with CUDA PyTorch; e.g. Colab GPU')
    if not torch.cuda.is_available():
        raise RuntimeError('Enable GPU runtime before setup')
    environment = work / 'env'
    python = common.ensure_python_environment(environment)
    common.run_command([python, '-m', 'pip', 'install', 'numpy>=1.26,<2.3',
                        'numba>=0.61.2,<0.62', 'opencv-python-headless>=4.8,<4.12'])
    output.mkdir(parents=True, exist_ok=True)
    with (output / 'pip_freeze.txt').open('w') as stream:
        subprocess.run([str(python), '-m', 'pip', 'freeze'], stdout=stream, check=True)
    print('PIXOR is pure 2D PyTorch: no spconv/OpenPCDet/C++ build required.', flush=True)


def configure_runtime(seed):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required for notebook smoke/train/evaluate')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.set_device(0)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_float32_matmul_precision('highest')


def worker_seed(worker_id):
    import torch
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed)
    np.random.seed(seed)


class TrainingDataset:
    def __init__(self, assets, settings, identifiers=None):
        self.assets, self.settings = assets, settings
        self.identifiers = identifiers if identifiers is not None else assets['splits']['train']
        self.root = Path(assets['training_root'])
    def __len__(self):
        return len(self.identifiers)
    def __getitem__(self, index):
        identifier = self.identifiers[index]
        points = np.fromfile(self.root / 'velodyne' / (identifier + '.bin'), dtype='<f4').reshape(-1, 4)
        record = self.assets['frames'][identifier]
        calibration = read_calibration(self.root / 'calib' / (identifier + '.txt'))
        matrix = augmentation_matrix() if self.settings['augmentation'] else np.eye(3)
        points, boxes = transform_scene(points, record['boxes'], matrix)
        mask = projection_mask(calibration, record['image_shape'], matrix=matrix, dontcare=record['dontcare'])
        cls, reg, valid = make_targets(boxes, valid_mask=mask)
        return encode_bev(points), cls, reg, valid


def data_loader(assets, settings, epoch, smoke=False):
    import torch
    generator = torch.Generator().manual_seed(settings['seed'] + epoch)
    # Epoch seeding makes a resumed epoch reproduce shuffle and augmentations.
    random.seed(settings['seed'] + epoch)
    np.random.seed(settings['seed'] + epoch)
    dataset = TrainingDataset(assets, settings, assets['splits']['train'][:2] if smoke else None)
    return torch.utils.data.DataLoader(dataset, batch_size=min(settings['batch_size'], len(dataset)), shuffle=True,
        num_workers=0 if smoke else settings['workers'], pin_memory=True, drop_last=False,
        worker_init_fn=worker_seed, generator=generator,
        **({'prefetch_factor': 1} if settings['workers'] and not smoke else {}))


def reference_check(settings):
    import torch
    work, output = paths(settings)
    verify_reference(work)
    sys.path.insert(0, str(work))
    from kitti_reference.rotate_iou import rotate_iou_gpu_eval
    box = np.array([[0, 0, 2, 2, 0]], dtype=np.float32)
    overlap = float(rotate_iou_gpu_eval(box, box)[0, 0])
    if abs(overlap - 1) > 1e-5:
        raise RuntimeError('Reference rotated IoU GPU kernel failed')
    common.write_json(output / 'environment.json', dict(python=platform.python_version(), torch=torch.__version__,
        torch_cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),
        gpu_capability=list(torch.cuda.get_device_capability(0)),
        nvidia_smi=subprocess.check_output(['nvidia-smi'], text=True),
        evaluator_revision=common.REVISION, evaluator_sha256=common.EVALUATOR_HASHES,
        train_precision='amp-fp16' if settings['amp'] else 'fp32', inference_precision='fp32', allow_tf32=False))


def smoke(settings):
    import torch
    configure_runtime(settings['seed'])
    reference_check(settings)
    assets = load_assets(settings)
    model = build_model().cuda().train()
    inputs, cls, reg, valid = [value.cuda(non_blocking=True) for value in next(iter(data_loader(assets, settings, 0, smoke=True)))]
    scaler = torch.amp.GradScaler('cuda', enabled=settings['amp'])
    optimizer = torch.optim.SGD(model.parameters(), lr=settings['learning_rate'])
    max_attempts = 16 if settings['amp'] else 1
    for attempt in range(1, max_attempts + 1):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=settings['amp']):
            prediction = model(inputs)
            loss, cls_loss, reg_loss = compute_loss(prediction, cls, reg, valid)
        if not torch.isfinite(loss) or prediction.shape[-2:] != cls.shape[-2:]:
            raise RuntimeError('Smoke produced invalid loss/shape')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        # FP16 backward can overflow while the initial loss scale calibrates.
        # Unscale records these gradients so GradScaler can safely skip the step.
        # Only clip finite gradients; retain the strict check for FP32 failures.
        gradients_finite = torch.stack([torch.isfinite(p.grad).all()
            for p in model.parameters() if p.grad is not None]).all()
        if not settings['amp'] or gradients_finite:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10, error_if_nonfinite=True)
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() < previous_scale:
            print(f'Smoke AMP overflow {attempt}/{max_attempts}: optimizer step skipped; '
                  f'loss_scale={previous_scale:g} -> {scaler.get_scale():g}; retrying batch.', flush=True)
            continue
        print(f'Smoke OK: input={tuple(inputs.shape)}, output={tuple(prediction.shape)}, '
              f'positive_pixels={int(cls.sum())}, loss={float(loss.detach()):.5f}, '
              f'attempts={attempt}, loss_scale={scaler.get_scale():g}. No training checkpoint saved.', flush=True)
        return
    raise FloatingPointError(f'Smoke AMP gradients remain nonfinite after {max_attempts} attempts; '
                             'set AMP=False in notebook settings to diagnose in FP32.')


def save_checkpoint(path, payload):
    import torch
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def checkpoint_payload(model, optimizer, scheduler, scaler, epoch, best_ap, identity, history):
    import torch
    state = np.random.get_state()
    return dict(model_state=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(), epoch=epoch, best_ap=best_ap, identity=identity, recipe=RECIPE, history=history,
        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(), python_rng=random.getstate(),
        numpy_rng=[state[0], state[1].tolist(), int(state[2]), int(state[3]), float(state[4])])


def restore_random_states(state):
    import torch
    torch.set_rng_state(state['torch_rng'])
    torch.cuda.set_rng_state_all(state['cuda_rng'])
    random.setstate(state['python_rng'])
    value = state['numpy_rng']
    np.random.set_state((value[0], np.asarray(value[1], dtype=np.uint32), value[2], value[3], value[4]))


def finalize_epoch(model, optimizer, scheduler, scaler, settings, assets, history, best_ap, identity):
    """Also used on resume if validation was interrupted after the epoch save."""
    _, output = paths(settings)
    row = history[-1]
    epoch = row['epoch']
    is_best = False
    if epoch % settings['eval_every'] == 0 or epoch == settings['epochs']:
        report = evaluate_model(model, settings, assets, output / 'validation' / f'epoch_{epoch:03d}')
        ap = report['accuracy']['R40']['Moderate']
        row['car_bev_r40_moderate'] = ap
        if ap > best_ap:
            best_ap, is_best = ap, True
    payload = checkpoint_payload(model, optimizer, scheduler, scaler, epoch, best_ap, identity, history)
    payload['validation_pending'] = False
    checkpoint_dir = output / 'checkpoints'
    if is_best:
        save_checkpoint(checkpoint_dir / 'best.pth', payload)
    if epoch % settings['save_every'] == 0 or epoch == settings['epochs']:
        save_checkpoint(checkpoint_dir / f'epoch_{epoch:03d}.pth', payload)
    common.write_json(output / 'history.json', history)
    with (output / 'history.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(history)
    # Commit last only after best/history/periodic files are complete. A failure
    # before this point leaves the pre-validation checkpoint pending for retry.
    save_checkpoint(checkpoint_dir / 'last.pth', payload)
    print(json.dumps(row), flush=True)
    return best_ap


def score_predictions(assets, prediction_dir, identifiers, work):
    sys.path.insert(0, str(work))
    verify_reference(work)
    from kitti_reference import eval as reference
    root = Path(assets['training_root'])
    gt = [common.array_annotation(common.read_annotation(root / 'label_2' / (identifier + '.txt'))) for identifier in identifiers]
    dt = [common.array_annotation(common.read_annotation(prediction_dir / (identifier + '.txt'), prediction=True)) for identifier in identifiers]
    curves = reference.eval_class(gt, dt, [0], [0, 1, 2], 1, np.full((1, 3, 1), .7),
                                  compute_aos=False, num_parts=50)
    result = {}
    for name, function in [('R11', reference.get_mAP), ('R40', reference.get_mAP_R40)]:
        values = function(curves['precision'])[0, :, 0]
        if not np.isfinite(values).all():
            raise FloatingPointError('Nonfinite reference AP')
        result[name] = {difficulty: float(values[i]) for i, difficulty in enumerate(common.DIFFICULTIES)}
    return result


def evaluate_model(model, settings, assets, destination, *, profile=False):
    import torch
    work, _ = paths(settings)
    destination = Path(destination)
    predictions = destination / 'kitti_predictions'
    predictions.mkdir(parents=True, exist_ok=True)
    root = Path(assets['training_root'])
    model = model.float().eval()
    identifiers = assets['splits']['val']
    timings = {stage: [] for stage in common.STAGES}
    rows = []
    raw_predictions = {}
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        for index, identifier in enumerate(identifiers):
            if profile:
                torch.cuda.synchronize()
            start = time.perf_counter()
            calibration = read_calibration(root / 'calib' / (identifier + '.txt'))
            image_shape = assets['frames'][identifier]['image_shape']
            points = np.fromfile(root / 'velodyne' / (identifier + '.bin'), dtype='<f4').reshape(-1, 4)
            inputs = torch.from_numpy(encode_bev(points)).unsqueeze(0)
            mask = projection_mask(calibration, image_shape)
            preprocessed = time.perf_counter()
            if profile:
                inputs, h2d = common.timed_cuda(lambda: inputs.cuda())
                prediction, network = common.timed_cuda(lambda: model(inputs))
            else:
                prediction = model(inputs.cuda())
            decoded_start = time.perf_counter()
            array = prediction[0].float().cpu().numpy()
            boxes = rotated_nms(decode_arrays(array[0], array[1:], threshold=settings['score_threshold'],
                topk=settings['pre_nms_topk'], valid_mask=mask), settings['nms_iou'], settings['max_detections'])
            if profile:
                torch.cuda.synchronize()
            end = time.perf_counter()
            if profile:
                row = dict(frame_id=identifier, preprocess=(preprocessed - start) * 1000,
                           host_to_device=h2d, network=network, decode_nms=(end - decoded_start) * 1000,
                           input_to_detections=(end - start) * 1000, included=index >= settings['warmup_frames'])
                rows.append(row)
                if row['included']:
                    for stage in timings:
                        timings[stage].append(row[stage])
            # Projection/export/AP and GT access are outside the latency boundary.
            lines = [kitti_prediction_row(box, calibration, image_shape, assets['projection_prior']) for box in boxes]
            lines = [line for line in lines if line is not None]
            (predictions / (identifier + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))
            raw_predictions[identifier] = boxes.tolist()
            if (index + 1) % 250 == 0:
                print(f'Validation: {index + 1}/{len(identifiers)}', flush=True)
    accuracy = score_predictions(assets, predictions, identifiers, work)
    report = dict(status='ok', recipe=RECIPE, classes=['Car'], metric='BEV', iou=.7,
        frames=len(identifiers), benchmark_complete=len(identifiers) == 3769,
        accuracy=accuracy, parameters=sum(p.numel() for p in model.parameters()),
        projection_prior=assets['projection_prior'], projection_scope=assets['prior_scope'],
        evaluator_revision=common.REVISION, evaluator_sha256=common.EVALUATOR_HASHES,
        latency={stage: common.timing_summary(values) for stage, values in timings.items()} if profile else None,
        peak_memory_bytes=torch.cuda.max_memory_allocated(), precision='fp32',
        prediction_sha256={identifier: common.sha256(predictions / (identifier + '.txt')) for identifier in identifiers})
    common.write_json(destination / 'report.json', report)
    common.write_json(destination / 'bev_predictions_lidar.json', raw_predictions)
    if profile:
        with (destination / 'latency_per_frame.csv').open('w', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    print('Car APBEV@0.7 R40:', accuracy['R40'], flush=True)
    return report


def train(settings):
    import torch
    configure_runtime(settings['seed'])
    reference_check(settings)
    work, output = paths(settings)
    assets = load_assets(settings)
    identity = training_identity(settings, assets['asset_signature'])
    model = build_model().cuda()
    optimizer = torch.optim.SGD(model.parameters(), lr=settings['learning_rate'], momentum=settings['momentum'],
                                weight_decay=settings['weight_decay'])
    milestones = sorted(set([max(1, round(settings['epochs'] * .6)), max(1, round(settings['epochs'] * .8))]))
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=.1)
    scaler = torch.amp.GradScaler('cuda', enabled=settings['amp'])
    checkpoint_dir = output / 'checkpoints'
    latest = checkpoint_dir / 'last.pth'
    resume = settings['resume']
    resume_path = latest if resume == 'auto' else (Path(resume).expanduser() if resume else None)
    start_epoch, best_ap, history = 0, -1., []
    if latest.exists() and not resume:
        raise ValueError('Existing run: enable RESUME or select another RUN_NAME')
    if resume_path and resume_path.exists():
        state = torch.load(resume_path, map_location='cpu', weights_only=True)
        if state.get('identity') != identity:
            raise ValueError('Resume checkpoint has different data, split, code or training settings. Choose a new RUN_NAME.')
        model.load_state_dict(state['model_state'], strict=True)
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best_ap, history = state['epoch'], state['best_ap'], state['history']
        restore_random_states(state)
        if state.get('validation_pending'):
            print(f'Completing pending validation/checkpoint finalization for epoch {start_epoch}', flush=True)
            best_ap = finalize_epoch(model, optimizer, scheduler, scaler, settings, assets, history, best_ap, identity)
        del state
        print(f'Resume epoch {start_epoch}; target TOTAL epochs={settings["epochs"]}', flush=True)
    elif resume not in ('auto', '', None, False):
        raise FileNotFoundError(resume_path)
    for epoch in range(start_epoch, settings['epochs']):
        model.train()
        loader = data_loader(assets, settings, epoch)
        optimizer.zero_grad(set_to_none=True)
        total, examples = np.zeros(3), 0
        started = time.perf_counter()
        lr = optimizer.param_groups[0]['lr']
        accumulation = settings['accumulation_steps']
        for index, batch in enumerate(loader):
            inputs, cls, reg, valid = [value.cuda(non_blocking=True) for value in batch]
            # The final, smaller accumulation group receives the correct divisor.
            group_size = min(accumulation, len(loader) - (index // accumulation) * accumulation)
            with torch.autocast('cuda', dtype=torch.float16, enabled=settings['amp']):
                prediction = model(inputs)
                loss, cls_loss, reg_loss = compute_loss(prediction, cls, reg, valid)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Nonfinite loss at epoch {epoch + 1}, batch {index}')
            scaler.scale(loss / group_size).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10, error_if_nonfinite=not settings['amp'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            count = len(inputs)
            total += np.array([float(loss.detach()), float(cls_loss), float(reg_loss)]) * count
            examples += count
            if (index + 1) % 100 == 0:
                print(f'Epoch {epoch + 1}/{settings["epochs"]}, batch {index + 1}/{len(loader)}, loss={total[0] / examples:.4f}', flush=True)
        scheduler.step()
        row = dict(epoch=epoch + 1, learning_rate=lr, train_loss=float(total[0] / examples),
                   classification_loss=float(total[1] / examples), regression_loss=float(total[2] / examples),
                   train_frames=examples, train_seconds=time.perf_counter() - started, car_bev_r40_moderate=None)
        # Save immediately after the training epoch, before potentially long validation.
        pending = checkpoint_payload(model, optimizer, scheduler, scaler, epoch + 1, best_ap, identity, history + [row])
        pending['validation_pending'] = True
        save_checkpoint(latest, pending)
        del pending
        history.append(row)
        best_ap = finalize_epoch(model, optimizer, scheduler, scaler, settings, assets, history, best_ap, identity)
    print(f'Training complete: {settings["epochs"]} TOTAL epochs; best validation Car Moderate R40={best_ap:.3f}', flush=True)


def evaluate(settings):
    import torch
    configure_runtime(settings['seed'])
    reference_check(settings)
    _, output = paths(settings)
    assets = load_assets(settings)
    choice = settings['eval_checkpoint']
    checkpoint = output / 'checkpoints' / (choice + '.pth') if choice in ('best', 'last') else Path(choice).expanduser()
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if state.get('identity') != training_identity(settings, assets['asset_signature']):
        raise ValueError('Evaluation checkpoint differs from this run identity')
    model = build_model().cuda()
    model.load_state_dict(state['model_state'], strict=True)
    epoch, best_ap = state['epoch'], state['best_ap']
    del state
    destination = output / 'evaluation' / checkpoint.stem
    report = evaluate_model(model, settings, assets, destination, profile=True)
    report.update(checkpoint=str(checkpoint), checkpoint_sha256=common.sha256(checkpoint), epoch=epoch,
                  selection_best_car_moderate_r40=best_ap, split_sha256=common.SPLIT_HASHES)
    common.write_json(destination / 'report.json', report)
    with (destination / 'comparison_car_bev.csv').open('w', newline='') as stream:
        row = dict(model='PIXOR reproduction', epoch=epoch, parameters=report['parameters'],
            **{'car_bev_r40_' + k.lower(): v for k, v in report['accuracy']['R40'].items()},
            network_ms=report['latency']['network']['mean_ms'], network_fps=report['latency']['network']['fps'],
            pipeline_ms=report['latency']['input_to_detections']['mean_ms'], pipeline_fps=report['latency']['input_to_detections']['fps'])
        writer = csv.DictWriter(stream, fieldnames=list(row))
        writer.writeheader()
        writer.writerow(row)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=('setup', 'prepare', 'smoke', 'train', 'evaluate'))
    parser.add_argument('--settings', type=Path, required=True)
    args = parser.parse_args()
    settings = read_settings(args.settings)
    globals()[args.command](settings)


if __name__ == '__main__':
    main()

'''[1:-1],
    'openpcdet_benchmark.py': r'''
#!/usr/bin/env python3
"""Standalone KITTI checkpoint evaluation and explicit sequential GPU profiling.

Only the GPU/setup commands import Torch, NumPy or OpenPCDet. The notebook embeds
this file so it can be used without uploading the surrounding project.
"""
from __future__ import annotations

import argparse
import ast
import copy
import csv
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import re
import shutil
import statistics
import struct
import subprocess
import sys
import time
import traceback


REVISION = '233f849829b6ac19afb8af8837a0246890908755'
CLASSES = ('Car', 'Pedestrian', 'Cyclist')
DIFFICULTIES = ('Easy', 'Moderate', 'Hard')
SPLIT_HASHES = {
    'train': 'e85ce0142be11c7e4196fd7b79a8bc8c2cefdd6fe754ac61fef8d421e37aba5c',
    'val': '6e2394d97c866c3af1ffb049f828abdf3d5b707d9575d16885fd0de87b72b0c8',
}
EVALUATOR_HASHES = {
    'eval.py': 'd9260a41f68083d685fc43fecac1c417b14a1bc53df4915a46b2924f3a6a3c64',
    'rotate_iou.py': 'd9b2820b5be5d15ee7233147643134c5d73eff129c773aef4b59ce73a8ecd7f4',
}
MODELS = {
    'pointpillars': dict(config='pointpillar.yaml', filename='pointpillar_7728.pth',
        drive_id='1wMxWTpU1qUoY3DsCH31WJmvJxcjFXKlm', epoch=80, it=9280),
    'second': dict(config='second.yaml', filename='second_7862.pth',
        drive_id='1-01zsPOsqanZQqIIyy7FpNXStL3y4jdR', epoch=80, it=9280),
    'pointrcnn': dict(config='pointrcnn.yaml', filename='pointrcnn_official.pth',
        drive_id='1BCX9wMn-GYAfSOPpyxf6Iv6fc0qKLSiU', epoch=77, it=17864),
    'parta2_anchor': dict(config='PartA2.yaml', filename='parta2_anchor_official.pth',
        drive_id='10GK1aCkLqxGNeX3lVu8cLZyE0G8002hY', epoch=80, it=12400),
    'pv_rcnn': dict(config='pv_rcnn.yaml', filename='pv_rcnn_8369.pth',
        drive_id='1lIOq4Hxr0W3qsX83ilQv0nk1Cls6KAr-', epoch=77, it=17864),
}
STAGES = ('preprocess', 'host_to_device', 'network', 'decode_nms', 'input_to_detections')


def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                    allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n')
    temporary.replace(path)


def read_ids(path):
    result = []
    for line in Path(path).read_text().splitlines():
        fields = line.strip().split(';')
        if not line.strip():
            continue
        if len(fields) > 2 or (len(fields) == 2 and fields[1].strip() != 'kitti'):
            raise ValueError(f'Invalid KITTI manifest row: {line!r}')
        identifier = fields[0].strip()
        if not re.fullmatch(r'[0-9]{6}', identifier):
            raise ValueError(f'Invalid KITTI frame ID: {identifier!r}')
        result.append(identifier)
    if not result:
        raise ValueError(f'Empty split: {path}')
    if len(result) != len(set(result)):
        raise ValueError(f'Duplicate frame IDs: {path}')
    return result


def check_partition(train, val):
    overlap = sorted(set(train) & set(val))
    if overlap:
        raise ValueError(f'Train/validation overlap: {overlap[:10]}')


def audit_splits(repo):
    lists = {}
    evidence = {}
    for name in ('train', 'val'):
        path = Path(repo) / 'data/kitti/ImageSets' / (name + '.txt')
        lists[name] = read_ids(path)
        canonical = hashlib.sha256(('\n'.join(lists[name]) + '\n').encode()).hexdigest()
        if canonical != SPLIT_HASHES[name]:
            raise ValueError(f'{name} differs from the audited KITTI split: {path}')
        evidence[name] = dict(count=len(lists[name]), canonical_ids_sha256=canonical)
    check_partition(lists['train'], lists['val'])
    return lists, evidence


def verify_evaluator(repo):
    directory = Path(repo) / 'pcdet/datasets/kitti/kitti_object_eval_python'
    for name, expected in EVALUATOR_HASHES.items():
        if sha256(directory / name) != expected:
            raise ValueError(f'Pinned evaluator modified: {directory / name}')
    return dict(revision=REVISION, source_sha256=EVALUATOR_HASHES)


def read_annotation(path, *, prediction=False):
    """Keep original full-frame GT, including Van, Person_sitting and DontCare."""
    result = {key: [] for key in ('name', 'truncated', 'occluded', 'alpha', 'bbox',
                                 'dimensions', 'location', 'rotation_y', 'score')}
    for line_number, line in enumerate(Path(path).read_text().splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != (16 if prediction else 15):
            raise ValueError(f'Invalid KITTI row at {path}:{line_number}')
        values = [float(value) for value in fields[1:]]
        if not all(math.isfinite(value) for value in values) or values[1] != int(values[1]):
            raise ValueError(f'Invalid numeric annotation at {path}:{line_number}')
        if prediction and not 0 <= values[-1] <= 1:
            raise ValueError(f'Invalid prediction score at {path}:{line_number}')
        if prediction and any(value <= 0 for value in values[7:10]):
            raise ValueError(f'Invalid prediction dimensions at {path}:{line_number}')
        row = dict(name=fields[0], truncated=values[0], occluded=int(values[1]),
                   alpha=values[2], bbox=values[3:7],
                   dimensions=[values[9], values[7], values[8]],  # KITTI h,w,l -> evaluator l,h,w
                   location=values[10:13], rotation_y=values[13],
                   score=values[14] if prediction else 0.0)
        for key, value in row.items():
            result[key].append(value)
    return result


def array_annotation(annotation):
    import numpy as np
    result = {}
    for key, value in annotation.items():
        dtype = str if key == 'name' else (np.int64 if key == 'occluded' else np.float64)
        shape = (-1, 4) if key == 'bbox' else ((-1, 3) if key in ('dimensions', 'location') else (-1,))
        result[key] = np.asarray(value, dtype=dtype).reshape(shape)
    return result


def timing_summary(values):
    if not values:
        return dict(samples=0, mean_ms=None, p50_ms=None, p95_ms=None, p99_ms=None, fps=None)
    if any(not math.isfinite(v) or v < 0 for v in values):
        raise ValueError('Latency samples must be finite and nonnegative')
    ordered = sorted(values)
    def percentile(q):
        index = (len(ordered) - 1) * q
        lower, upper = math.floor(index), math.ceil(index)
        return ordered[lower] + (ordered[upper] - ordered[lower]) * (index - lower)
    mean = statistics.fmean(values)
    return dict(samples=len(values), mean_ms=mean, p50_ms=percentile(.5),
                p95_ms=percentile(.95), p99_ms=percentile(.99),
                fps=1000 / mean if mean > 0 else None)


def summary_row(report):
    complete = report.get('status') == 'ok' and report.get('benchmark_complete') is True
    row = dict(model=report.get('name'), status=report.get('status'), phase=report.get('phase'),
               benchmark_complete=complete, frames=report.get('frames'),
               parameters=report.get('model', {}).get('parameters'), error=report.get('error'))
    for mode in ('3d', 'bev'):
        for recall in ('R11', 'R40'):
            metrics = report.get('accuracy', {}).get(mode, {}).get(recall, {}) if complete else {}
            row[f'ap{mode}_{recall.lower()}_moderate'] = metrics.get('map_moderate_percent')
            for name in CLASSES:
                row[f'ap{mode}_{recall.lower()}_{name.lower()}_moderate'] = metrics.get('per_class', {}).get(name, {}).get('Moderate')
    for stage, short in (('network', 'network'), ('decode_nms', 'decode_nms'),
                         ('input_to_detections', 'pipeline')):
        timing = report.get('latency', {}).get(stage, {}) if complete else {}
        for statistic in ('mean_ms', 'p50_ms', 'p95_ms', 'p99_ms', 'fps'):
            row[f'{short}_{statistic}'] = timing.get(statistic)
    return row


def run_command(command, *, cwd=None, env=None):
    import shlex
    print('+ ' + shlex.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)


def ensure_python_environment(environment):
    """Create a pip-enabled overlay, including runtimes without ensurepip."""
    environment = Path(environment)
    python = environment / 'bin/python'
    def usable():
        return python.exists() and subprocess.run([str(python), '-m', 'pip', '--version'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
    if usable():
        return python
    try:
        run_command([sys.executable, '-m', 'venv', '--system-site-packages', environment])
        if not usable():
            raise RuntimeError('venv did not provide pip')
    except (subprocess.CalledProcessError, RuntimeError):
        print('Bootstrapping virtualenv because this Python has no working ensurepip.', flush=True)
        bootstrap = environment.parent / '.venv_bootstrap'
        run_command([sys.executable, '-m', 'pip', 'install', '--target', bootstrap, 'virtualenv>=20,<21'])
        env = os.environ.copy()
        env['PYTHONPATH'] = str(bootstrap) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
        run_command([sys.executable, '-m', 'virtualenv', '--system-site-packages', environment], env=env)
        if not usable():
            raise RuntimeError('Could not create a pip-enabled environment; use a runtime with venv/virtualenv support')
    return python


def load_settings(path):
    result = json.loads(Path(path).read_text())
    if not result.get('models') or any(name not in MODELS for name in result['models']):
        raise ValueError('Choose models from: ' + ', '.join(MODELS))
    if type(result.get('warmup_frames')) is not int or result['warmup_frames'] < 1:
        raise ValueError('warmup_frames must be a positive integer')
    maximum = result.get('max_frames')
    if maximum is not None and (type(maximum) is not int or maximum < 1):
        raise ValueError('max_frames must be null or a positive integer')
    if type(result.get('smoke_frames', 3)) is not int or result.get('smoke_frames', 3) < 2:
        raise ValueError('smoke_frames must be at least two')
    if type(result.get('build_jobs', 2)) is not int or result.get('build_jobs', 2) < 1:
        raise ValueError('build_jobs must be a positive integer')
    if not re.fullmatch(r'[A-Za-z0-9_.-]+', result['run_name']) or result['run_name'] in ('.', '..'):
        raise ValueError('run_name must be a simple directory name')
    for key in ('work_dir', 'output_root', 'kitti_root'):
        result[key] = str(Path(result[key]).expanduser().resolve())
    # Work and output directories must never be placed inside the source dataset.
    raw = Path(result['kitti_root'])
    for key in ('work_dir', 'output_root'):
        path_value = Path(result[key])
        if path_value == raw or raw in path_value.parents:
            raise ValueError(f'{key} must be outside the KITTI source dataset')
    return result


def paths_for(settings):
    work = Path(settings['work_dir'])
    return work, work / 'OpenPCDet', Path(settings['output_root']) / settings['run_name']


def normalize_training_root(path, image_sizes_json=None):
    path = Path(path).expanduser().resolve()
    if (path / 'training').is_dir():
        path = path / 'training'
    required = ('velodyne', 'label_2', 'calib')
    if any(not (path / name).is_dir() for name in required):
        raise FileNotFoundError(f'KITTI needs {required} under {path}')
    metadata = Path(image_sizes_json).expanduser().resolve() if image_sizes_json else next((p for p in
        (path / 'kitti_image_sizes.json', path.parent / 'kitti_image_sizes.json') if p.is_file()), None)
    if not (path / 'image_2').is_dir() and (metadata is None or not metadata.is_file()):
        raise FileNotFoundError(f'KITTI needs image_2 PNGs or per-frame kitti_image_sizes.json under {path}')
    return path


def image_shape(path):
    with Path(path).open('rb') as stream:
        header = stream.read(24)
    if len(header) < 24 or header[:8] != b'\x89PNG\r\n\x1a\n' or header[12:16] != b'IHDR':
        raise ValueError(f'Expected original KITTI PNG: {path}')
    width, height = struct.unpack('>II', header[16:24])
    if not width or not height:
        raise ValueError(f'Invalid image dimensions: {path}')
    return [height, width]


def load_image_dimensions(root, identifiers, metadata_path=None):
    from kitti_training_pipeline.image_metadata import load_image_sizes
    dimensions, sources = load_image_sizes(root, identifiers, metadata_path=metadata_path)
    shapes = {identifier: [height, width] for identifier, (width, height) in dimensions.items()}
    metadata = str(sources[0]) if len(sources) == 1 and sources[0].suffix.lower() == '.json' else None
    return shapes, sources, metadata


def prepare_data(settings):
    work, repo, output = paths_for(settings)
    lists, split_evidence = audit_splits(repo)
    verify_evaluator(repo)
    source_root = normalize_training_root(settings['kitti_root'], settings.get('image_sizes_json'))
    shapes, image_sources, metadata = load_image_dimensions(source_root, lists['val'], settings.get('image_sizes_json'))
    file_specs = [('velodyne', '.bin'), ('calib', '.txt'), ('label_2', '.txt')]
    if metadata is None:
        file_specs.append(('image_2', '.png'))
    root = source_root
    if settings.get('copy_val_to_local', False):
        root = work / 'kitti_validation/training'
        if root.resolve() == source_root:
            raise ValueError('Validation staging destination must differ from the source dataset.')
        pending = []
        for identifier in lists['val']:
            for directory, extension in file_specs:
                original = source_root / directory / (identifier + extension)
                target = root / directory / original.name
                stat = original.stat()
                if not target.exists() or (target.stat().st_size, target.stat().st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
                    pending.append((original, target, stat.st_size))
        required = sum(size for _, _, size in pending)
        if shutil.disk_usage(work).free < required + 512 * 1024 * 1024:
            raise RuntimeError(f'Not enough local disk for validation staging ({required / 1e9:.2f} GB).')
        for index, (original, target, _) in enumerate(pending):
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + '.copying')
            shutil.copy2(original, temporary)
            temporary.replace(target)
            if (index + 1) % 1000 == 0:
                print(f'Validation staging: {index + 1}/{len(pending)} files', flush=True)
    infos = []
    assets = hashlib.sha256()
    for path in image_sources:
        assets.update(str(path).encode())
        if path.suffix.lower() == '.json':
            assets.update(path.read_bytes())
        else:
            with path.open('rb') as stream:
                assets.update(stream.read(24))
    for identifier in lists['val']:
        for directory, extension in file_specs:
            path = root / directory / (identifier + extension)
            if not path.is_file():
                raise FileNotFoundError(path)
            if directory == 'velodyne' and (path.stat().st_size == 0 or path.stat().st_size % 16):
                raise ValueError(f'Invalid float32 XYZI point cloud: {path}')
            assets.update((str(path) + ':' + str(path.stat().st_size) + ':' + str(path.stat().st_mtime_ns)).encode())
            if directory in ('calib', 'label_2'):
                assets.update(path.read_bytes())
            if directory == 'label_2':
                read_annotation(path)
        infos.append(dict(point_cloud=dict(num_features=4, lidar_idx=identifier),
                          image=dict(image_idx=identifier, image_shape=shapes[identifier])))
    result = dict(training_root=str(root), source_training_root=str(source_root), splits=split_evidence, frame_ids=lists['val'], infos=infos,
                  asset_signature=assets.hexdigest(), image_sizes_json=metadata,
                  asset_signature_scope='Paths, sizes and mtimes of validation inputs, complete label/calibration bytes, and image metadata JSON or PNG headers; not a full point-cloud content hash.')
    write_json(work / 'validation_assets.json', result)
    write_json(output / 'validation_assets.json', {k: v for k, v in result.items() if k != 'infos'})
    print(f'Validated {len(infos)} frames; train=3712, val=3769, no overlap. No GT database or train preprocessing needed.', flush=True)


def patch_extension_apis(repo):
    """Mechanical C++ compatibility only; evaluator and Python models untouched."""
    records = []
    for path in sorted((Path(repo) / 'pcdet/ops').rglob('*')):
        if path.suffix not in ('.cpp', '.cu', '.h'):
            continue
        original = path.read_text()
        updated = original.replace('.type().is_cuda()', '.is_cuda()')
        updated = re.sub(r'\bAT_CHECK\b', 'TORCH_CHECK', updated)
        updated = re.sub(r'\.data<', '.data_ptr<', updated)
        updated = re.sub(r'(AT_DISPATCH_\w+\s*\(\s*\w+)\.type\(\)', r'\1.scalar_type()', updated)
        if updated != original:
            records.append(dict(path=str(path.relative_to(repo)),
                                before_sha256=hashlib.sha256(original.encode()).hexdigest(),
                                after_sha256=hashlib.sha256(updated.encode()).hexdigest()))
            path.write_text(updated)
    manifest = Path(repo).parent / 'extension_compatibility.json'
    if records or not manifest.exists():
        write_json(manifest, records)
    return records


def patch_kitti_dataset_imports(repo):
    """Avoid optional non-KITTI SDKs in this KITTI-only benchmark checkout."""
    repo = Path(repo)
    path = repo / 'pcdet/datasets/__init__.py'
    manifest_path = repo.parent / 'dataset_import_compatibility.json'
    original = path.read_text()
    before = hashlib.sha256(original.encode()).hexdigest()
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if before == manifest['after_sha256']:
            return manifest
        if before != manifest['before_sha256']:
            raise RuntimeError('Dataset registry changed since the recorded compatibility patch.')
    optional = {
        'nuscenes.nuscenes_dataset': 'NuScenesDataset', 'waymo.waymo_dataset': 'WaymoDataset',
        'pandaset.pandaset_dataset': 'PandasetDataset', 'lyft.lyft_dataset': 'LyftDataset',
        'once.once_dataset': 'ONCEDataset', 'argo2.argo2_dataset': 'Argo2Dataset',
        'custom.custom_dataset': 'CustomDataset',
    }
    tree = ast.parse(original)
    removed, imports, entries = set(), set(), set()
    for node in tree.body:
        if isinstance(node, ast.ImportFrom) and node.level == 1 and node.module in optional:
            name = optional[node.module]
            if len(node.names) != 1 or node.names[0].name != name or node.names[0].asname:
                raise RuntimeError('Unexpected optional dataset import; inspect the pinned checkout.')
            imports.add(name)
            removed.update(range(node.lineno, node.end_lineno + 1))
        if (isinstance(node, ast.Assign)
                and any(isinstance(target, ast.Name) and target.id == '__all__' for target in node.targets)
                and isinstance(node.value, ast.Dict)):
            for key, value in zip(node.value.keys, node.value.values):
                if isinstance(key, ast.Constant) and key.value in optional.values():
                    if not isinstance(value, ast.Name) or value.id != key.value or key.lineno != value.end_lineno:
                        raise RuntimeError('Unexpected optional dataset registry entry.')
                    # The pinned registry has one entry per line; retain all other bytes.
                    if original.splitlines()[key.lineno - 1].strip().startswith(('"', "'")):
                        entries.add(key.value)
                        removed.add(key.lineno)
    if imports != entries:
        raise RuntimeError('Optional dataset imports and registry entries do not match.')
    updated = ''.join(line for number, line in enumerate(original.splitlines(keepends=True), 1)
                      if number not in removed)
    ast.parse(updated)  # Validate before changing an existing checkout.
    manifest = dict(path='pcdet/datasets/__init__.py', scope='KITTI-only dataset registry',
                    disabled_datasets=sorted(imports), before_sha256=before,
                    after_sha256=hashlib.sha256(updated.encode()).hexdigest())
    if updated != original:
        path.write_text(updated)
    write_json(manifest_path, manifest)
    return manifest


def setup(settings):
    if platform.system() != 'Linux' or not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError('Use a Linux GPU runtime with Python 3.10–3.13 and CUDA PyTorch (e.g. Colab).')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU runtime first. CUDA PyTorch is required.')
    nvcc = shutil.which('nvcc')
    if not nvcc:
        raise RuntimeError('nvcc is missing. Install the CUDA toolkit matching your PyTorch CUDA major version, then rerun setup.')
    nvcc_text = subprocess.check_output([nvcc, '--version'], text=True)
    match = re.search(r'release (\d+)\.(\d+)', nvcc_text)
    if not match or not torch.version.cuda or int(match[1]) != int(torch.version.cuda.split('.')[0]):
        raise RuntimeError(f'CUDA toolkit / PyTorch major mismatch: nvcc={nvcc_text}, torch CUDA={torch.version.cuda}')
    work, repo, output = paths_for(settings)
    work.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    if not repo.exists():
        run_command(['git', 'clone', '--filter=blob:none', 'https://github.com/open-mmlab/OpenPCDet.git', repo])
        run_command(['git', 'checkout', '--detach', REVISION], cwd=repo)
    revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    if revision != REVISION:
        raise RuntimeError(f'{repo} must be at {REVISION}; use another work_dir for an existing checkout.')
    verify_evaluator(repo)
    audit_splits(repo)
    patch_kitti_dataset_imports(repo)
    environment = work / 'env'
    python = ensure_python_environment(environment)
    constraints = work / 'constraints.txt'
    constraints.write_text('torch==' + torch.__version__ + '\n')
    # The overlay inherits the user's CUDA Torch rather than replacing it.
    cuda = tuple(map(int, torch.version.cuda.split('.')[:2]))
    default_spconv = 'spconv-cu126==2.3.8' if cuda >= (12, 6) else (
        'spconv-cu124==2.3.8' if cuda >= (12, 4) else (
        'spconv-cu120==2.3.8' if cuda >= (12, 0) else 'spconv-cu118==2.3.8'))
    packages = ['numpy>=1.26,<2.3', 'numba>=0.61.2,<0.62', 'scipy', 'scikit-image>=0.24',
                'setuptools<81', 'wheel', 'ninja', 'easydict', 'pyyaml', 'tqdm',
                'tensorboardX', 'SharedArray', 'pyquaternion', 'gdown>=5.2,<6',
                'kornia', 'einops', 'timm', 'torchvision',
                settings.get('spconv_package') or default_spconv]
    run_command([python, '-m', 'pip', 'install', '--constraint', constraints, *packages])
    patch_extension_apis(repo)
    env = os.environ.copy()
    env['MAX_JOBS'] = str(settings.get('build_jobs', 2))
    major, minor = torch.cuda.get_device_capability()
    env['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
    run_command([python, 'setup.py', 'build_ext', '--inplace'], cwd=repo, env=env)
    verify_evaluator(repo)
    write_json(work / 'setup.json', dict(python=str(python), revision=REVISION,
               torch=torch.__version__, torch_cuda=torch.version.cuda, nvcc=nvcc_text,
               spconv_package=packages[-1], extensions_patch_manifest=str(work / 'extension_compatibility.json')))
    print(f'Environment ready: {python}', flush=True)


def activate_repo(repo):
    repo = Path(repo).resolve()
    if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip() != REVISION:
        raise RuntimeError('OpenPCDet revision changed; rerun setup with the pinned checkout.')
    sys.path.insert(0, str(repo))
    os.chdir(repo / 'tools')  # cfg_from_yaml_file resolves its _BASE_CONFIG_ here.
    verify_evaluator(repo)
    patch_kitti_dataset_imports(repo)


def runtime_info():
    import torch
    versions = {}
    for name in ('torch', 'torchvision', 'numpy', 'numba', 'numba-cuda', 'spconv-cu126',
                 'spconv-cu124', 'spconv-cu120', 'spconv-cu118', 'cumm-cu126', 'gdown'):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return dict(python=platform.python_version(), executable=sys.executable,
                packages=versions, torch_cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)),
                cpu=platform.processor(), cpu_count=os.cpu_count(),
                cuda_home=os.environ.get('CUDA_HOME'),
                nvidia_smi=subprocess.check_output(['nvidia-smi'], text=True),
                driver_version=subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).strip(),
                cuda_launch_blocking=os.environ.get('CUDA_LAUNCH_BLOCKING'),
                allow_tf32=False, precision='fp32', batch_size=1)


def configure_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is required for this benchmark.')
    torch.cuda.set_device(0)
    torch.manual_seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision('highest')
    if os.environ.get('CUDA_LAUNCH_BLOCKING', '0') not in ('', '0'):
        raise RuntimeError('Unset CUDA_LAUNCH_BLOCKING for benchmark timing; this runner synchronizes explicitly.')
    try:
        import spconv.constants
        spconv.constants.SPCONV_ALLOW_TF32 = False
    except ImportError:
        pass


def gpu_check(settings):
    import numpy as np
    import torch
    work, repo, output = paths_for(settings)
    activate_repo(repo)
    output.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(work / 'dataset_import_compatibility.json', output / 'dataset_import_compatibility.json')
    configure_gpu()
    from pcdet.ops.iou3d_nms import iou3d_nms_utils
    from pcdet.datasets.kitti.kitti_object_eval_python.rotate_iou import rotate_iou_gpu_eval
    boxes = torch.tensor([[0, 0, 0, 2, 2, 2, 0]], device='cuda', dtype=torch.float32)
    overlap = iou3d_nms_utils.boxes_iou3d_gpu(boxes, boxes).item()
    reference = rotate_iou_gpu_eval(np.array([[0, 0, 2, 2, 0]], dtype=np.float32),
                                    np.array([[0, 0, 2, 2, 0]], dtype=np.float32))[0, 0]
    if abs(overlap - 1) > 1e-5 or abs(float(reference) - 1) > 1e-5:
        raise RuntimeError('CUDA/reference overlap smoke failed.')
    info = runtime_info()
    write_json(output / 'environment.json', info)
    print(json.dumps(info, indent=2), flush=True)


def checkpoint_path(settings, name):
    override = settings.get('checkpoint_overrides', {}).get(name)
    return Path(override).expanduser().resolve() if override else paths_for(settings)[0] / 'checkpoints' / MODELS[name]['filename']


def validate_checkpoint_metadata(checkpoint, name):
    spec = MODELS[name]
    if (not isinstance(checkpoint, dict) or checkpoint.get('epoch') != spec['epoch']
            or checkpoint.get('it') != spec['it'] or not isinstance(checkpoint.get('model_state'), dict)):
        raise ValueError(f'{name}: checkpoint metadata differs from the audited official file.')


def download_checkpoints(settings):
    import gdown
    import torch
    work, repo, output = paths_for(settings)
    for name in settings['models']:
        spec = MODELS[name]
        path = checkpoint_path(settings, name)
        path.parent.mkdir(parents=True, exist_ok=True)
        candidate = path
        if not path.exists():
            if settings.get('checkpoint_overrides', {}).get(name):
                raise FileNotFoundError(path)
            temporary = path.with_suffix('.download')
            if gdown.download(id=spec['drive_id'], output=str(temporary), quiet=False) is None:
                raise RuntimeError(f'Download failed for {name}. Download the official model manually and set CHECKPOINT_OVERRIDES.')
            candidate = temporary
        checkpoint = torch.load(candidate, map_location='cpu', weights_only=True)
        validate_checkpoint_metadata(checkpoint, name)
        if candidate != path:
            candidate.replace(path)
        write_json(output / 'checkpoints' / (name + '.json'), dict(
            path=str(path), sha256=sha256(path), bytes=path.stat().st_size,
            epoch=checkpoint['epoch'], iterations=checkpoint['it'],
            source=f"https://drive.google.com/file/d/{spec['drive_id']}/view",
            provenance='Official KITTI validation model-zoo checkpoint, with audited epoch/iteration header. No training-ID manifest is embedded; metadata is supporting evidence, not proof of exact training membership.'))
        print(f'{name}: epoch={checkpoint["epoch"]}, iterations={checkpoint["it"]}, {path.stat().st_size / 1e6:.1f} MB', flush=True)
        del checkpoint


def plain(value):
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, 'tolist'):
        return value.tolist()
    return value


def make_dataset(repo, root, infos, name):
    import numpy as np
    from easydict import EasyDict
    from pcdet.config import cfg_from_yaml_file
    from pcdet.datasets.kitti.kitti_dataset import KittiDataset
    cfg = EasyDict()
    cfg_from_yaml_file(str(repo / 'tools/cfgs/kitti_models' / MODELS[name]['config']), cfg)
    if tuple(cfg.CLASS_NAMES) != CLASSES:
        raise ValueError('Checkpoint class mapping changed.')
    original = copy.deepcopy(cfg)
    cfg.DATA_CONFIG.INFO_PATH = {'train': [], 'test': []}
    dataset = KittiDataset(cfg.DATA_CONFIG, cfg.CLASS_NAMES, training=False,
                          root_path=repo / 'data/kitti', logger=None)
    dataset.root_split_path = root
    # No GT in inference samples: avoid target preparation and recall work in timing.
    dataset.kitti_infos = copy.deepcopy(infos)
    for info in dataset.kitti_infos:
        info['image']['image_shape'] = np.asarray(info['image']['image_shape'], dtype=np.int32)
    return cfg, original, dataset


def timed_cuda(function):
    import torch
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record()
    value = function()
    end.record()
    end.synchronize()
    return value, float(start.elapsed_time(end))


def evaluate_predictions(root, prediction_dir, frame_ids):
    import numpy as np
    from pcdet.datasets.kitti.kitti_object_eval_python import eval as reference
    gt = [array_annotation(read_annotation(root / 'label_2' / (identifier + '.txt'))) for identifier in frame_ids]
    predictions = [array_annotation(read_annotation(prediction_dir / (identifier + '.txt'), prediction=True)) for identifier in frame_ids]
    result = {}
    for name, metric in (('bev', 1), ('3d', 2)):
        curves = reference.eval_class(gt, predictions, [0, 1, 2], [0, 1, 2], metric,
            np.tile([.7, .5, .5], (1, 3, 1)), compute_aos=False, num_parts=50)
        result[name] = {}
        for recall, function in (('R11', reference.get_mAP), ('R40', reference.get_mAP_R40)):
            values = function(curves['precision'])[..., 0]
            if not np.isfinite(values).all():
                raise FloatingPointError('Non-finite reference AP.')
            result[name][recall] = dict(
                per_class={c: {d: float(values[i, j]) for j, d in enumerate(DIFFICULTIES)} for i, c in enumerate(CLASSES)},
                map_moderate_percent=float(values[:, 1].mean()), mean_ap_9_percent=float(values.mean()),
                valid_gt_counts={c: {d: sum(reference.clean_data(g, p, i, j)[0] for g, p in zip(gt, predictions))
                                      for j, d in enumerate(DIFFICULTIES)} for i, c in enumerate(CLASSES)})
    return result


def run_model(settings, name, phase):
    import torch
    work, repo, output = paths_for(settings)
    activate_repo(repo)
    configure_gpu()
    print(f'{name}: preparing {phase} inference on the selected GPU', flush=True)
    assets = json.loads((work / 'validation_assets.json').read_text())
    _, splits = audit_splits(repo)
    source_root = normalize_training_root(settings['kitti_root'], settings.get('image_sizes_json'))
    root = normalize_training_root(assets['training_root'], assets.get('image_sizes_json'))
    if assets['source_training_root'] != str(source_root) or assets['splits'] != splits:
        raise ValueError('Validation asset cache changed; rerun the data audit.')
    infos = assets['infos']
    maximum = settings.get('max_frames') if phase == 'full' else settings.get('smoke_frames', 3)
    if maximum is not None:
        infos = infos[:maximum]
    frame_ids = [info['point_cloud']['lidar_idx'] for info in infos]
    warmup = settings['warmup_frames'] if phase == 'full' else min(1, len(infos) - 1)
    if len(infos) <= warmup:
        raise ValueError('Need more frames than warmup_frames.')
    checkpoint = checkpoint_path(settings, name)
    cfg, original_cfg, dataset = make_dataset(repo, root, infos, name)
    environment = runtime_info()
    identity = dict(name=name, phase=phase, frame_ids=frame_ids, warmup_frames=warmup,
                    checkpoint_sha256=sha256(checkpoint), native_config=plain(original_cfg),
                    runner_sha256=sha256(__file__), revision=REVISION, asset_signature=assets['asset_signature'],
                    environment=environment['packages'], gpu=environment['gpu'], capability=environment['capability'],
                    driver_version=environment['driver_version'], torch_cuda=environment['torch_cuda'])
    run_fingerprint = fingerprint(identity)
    destination = output / phase / name
    report_path = destination / 'report.json'
    if report_path.exists():
        existing = json.loads(report_path.read_text())
        if existing.get('status') == 'ok' and existing.get('fingerprint') == run_fingerprint:
            predictions_path = destination / 'kitti_predictions'
            recorded = existing.get('prediction_sha256', {})
            if set(recorded) != set(frame_ids) or any(
                not (predictions_path / (identifier + '.txt')).is_file()
                or sha256(predictions_path / (identifier + '.txt')) != recorded[identifier]
                for identifier in frame_ids
            ):
                raise ValueError('Cached prediction files changed or are missing; choose a new RUN_NAME.')
            print(f'{name}: reusing completed identical run {report_path}', flush=True)
            return
        if existing.get('fingerprint') not in (None, run_fingerprint):
            raise ValueError('Existing run has different settings. Choose a new RUN_NAME.')
    destination.mkdir(parents=True, exist_ok=True)
    write_json(destination / 'config.resolved.json', plain(original_cfg))
    write_json(destination / 'identity.json', identity)
    from pcdet.models import build_network, load_data_to_gpu
    model = build_network(cfg.MODEL, len(CLASSES), dataset)
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    validate_checkpoint_metadata(state, name)
    model_state, updated = model._load_state_dict(state['model_state'], strict=False)
    missing = [key for key in model_state if key not in updated and not key.endswith('num_batches_tracked')]
    if missing:
        raise ValueError(f'Incomplete checkpoint loading: {missing[:20]}')
    model = model.cuda().float().eval()
    parameters = sum(parameter.numel() for parameter in model.parameters())
    del state, model_state, updated
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    timings = {stage: [] for stage in STAGES}
    prediction_dir = destination / 'kitti_predictions'
    prediction_dir.mkdir(exist_ok=True)
    per_frame_path = destination / 'latency_per_frame.csv'
    fields = ['frame_id', 'warmup'] + [stage + '_ms' for stage in STAGES]
    with per_frame_path.open('w', newline='') as stream, torch.inference_mode():
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for index, identifier in enumerate(frame_ids):
            torch.cuda.synchronize()
            total_start = time.perf_counter()
            sample = dataset[index]
            batch = dataset.collate_batch([sample])
            if 'gt_boxes' in batch:
                raise RuntimeError('Ground truth must not enter timed inference.')
            preprocess_ms = (time.perf_counter() - total_start) * 1000
            _, transfer_ms = timed_cuda(lambda: load_data_to_gpu(batch))
            def forward_network():
                value = batch
                for module in model.module_list:
                    value = module(value)
                return value
            batch, network_ms = timed_cuda(forward_network)
            decode_start = time.perf_counter()
            predictions, _ = model.post_processing(batch)
            torch.cuda.synchronize()
            decode_ms = (time.perf_counter() - decode_start) * 1000
            total_ms = (time.perf_counter() - total_start) * 1000
            measured = dict(zip(STAGES, (preprocess_ms, transfer_ms, network_ms, decode_ms, total_ms)))
            writer.writerow(dict(frame_id=identifier, warmup=index < warmup,
                                 **{stage + '_ms': value for stage, value in measured.items()}))
            if index >= warmup:
                for stage, value in measured.items():
                    timings[stage].append(value)
            # KITTI calibration/projection and text writing are outside latency.
            dataset.generate_prediction_dicts(batch, predictions, list(CLASSES), output_path=prediction_dir)
            del predictions, batch, sample
            if (index + 1) % 100 == 0 or index + 1 == len(infos):
                print(f'{name} [{phase}]: {index + 1}/{len(infos)} frames', flush=True)
    peak_memory = torch.cuda.max_memory_allocated() / 1048576
    del model
    torch.cuda.empty_cache()
    accuracy = evaluate_predictions(root, prediction_dir, frame_ids)
    complete = phase == 'full' and len(frame_ids) == 3769 and frame_ids == assets['frame_ids']
    report = dict(name=name, status='ok', phase=phase, benchmark_complete=complete,
        fingerprint=run_fingerprint, frames=len(frame_ids), timed_frames=len(frame_ids) - warmup,
        model=dict(parameters=parameters, checkpoint=str(checkpoint), checkpoint_sha256=identity['checkpoint_sha256'],
                   checkpoint_epoch=MODELS[name]['epoch'], checkpoint_iterations=MODELS[name]['it']),
        protocol=dict(evaluator=verify_evaluator(repo), classes=CLASSES, difficulties=DIFFICULTIES,
            iou_thresholds=dict(zip(CLASSES, (.7, .5, .5))), domain='full-frame original GT; metric-specific pinned ignore rules',
            macro='unweighted three-class Moderate mean; missing classes retained', split=splits['val'],
            native_preprocessing=True, point_cloud_range=plain(original_cfg.DATA_CONFIG.POINT_CLOUD_RANGE),
            fov_points_only=original_cfg.DATA_CONFIG.FOV_POINTS_ONLY,
            postprocessing=plain(original_cfg.MODEL.POST_PROCESSING), warmup_frames_excluded=warmup,
            latency_boundary='Sequential raw bin/calibration read + native FOV/voxelization + collation + H2D + module_list + final post_processing; excludes GT recall, annotation projection/text export and AP.',
            network_boundary='module_list, including architecture-specific internal box decoding and ROI proposal NMS; final post_processing is separate.',
            timing_method='H2D/network: CUDA events with end synchronization; preprocessing/final NMS/total: perf_counter with explicit CUDA synchronization.',
            fps_definition='1000 / mean_ms at batch 1; sequential offline latency, not a pipelined throughput claim',
            ap_source='Exported KITTI text predictions (upstream 4-decimal formatting); no conversion from README R11'),
        accuracy=accuracy, latency={stage: timing_summary(values) for stage, values in timings.items()},
        prediction_sha256={identifier: sha256(prediction_dir / (identifier + '.txt')) for identifier in frame_ids},
        runtime={**environment, 'torch_peak_memory_mb': peak_memory},
        outputs=dict(predictions=str(prediction_dir), per_frame_latency=str(per_frame_path)))
    write_json(report_path, report)
    print(json.dumps(summary_row(report), ensure_ascii=False, indent=2), flush=True)


def summarize(settings):
    output = paths_for(settings)[2]
    rows = []
    for name in settings['models']:
        path = output / 'full' / name / 'report.json'
        fallback = (output / 'smoke' / name / 'failure.json', output / 'download' / name / 'failure.json')
        if not path.exists():
            path = next((candidate for candidate in fallback if candidate.exists()), path)
        report = json.loads(path.read_text()) if path.exists() else dict(name=name, status='not_run', phase='full')
        rows.append(summary_row(report))
    destination = output / 'comparison.csv'
    fields = list(rows[0])
    with destination.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    write_json(output / 'comparison.json', rows)
    columns = ('model', 'status', 'benchmark_complete', 'parameters', 'ap3d_r40_moderate',
               'apbev_r40_moderate', 'network_mean_ms', 'pipeline_mean_ms', 'pipeline_fps')
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in rows:
        lines.append('| ' + ' | '.join('—' if row.get(key) is None else str(round(row[key], 3))
                      if isinstance(row[key], float) else str(row[key]) for key in columns) + ' |')
    (output / 'comparison.md').write_text('\n'.join(lines) + '\n')
    print('\n'.join(lines), flush=True)
    print(f'Results: {output}', flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=('setup', 'gpu-check', 'prepare', 'download', 'run', 'summarize'))
    parser.add_argument('--settings', required=True, type=Path)
    parser.add_argument('--model', choices=tuple(MODELS))
    parser.add_argument('--phase', choices=('smoke', 'full'), default='full')
    args = parser.parse_args(argv)
    settings = load_settings(args.settings)
    if args.command == 'run' and args.model is None:
        parser.error('--model is required for run')
    try:
        if args.command == 'run':
            run_model(settings, args.model, args.phase)
        elif args.command == 'download' and args.model:
            download_checkpoints({**settings, 'models': [args.model]})
        else:
            {'setup': setup, 'gpu-check': gpu_check, 'prepare': prepare_data,
             'download': download_checkpoints, 'summarize': summarize}[args.command](settings)
    except Exception as exc:
        if args.command == 'run' or (args.command == 'download' and args.model):
            phase = args.phase if args.command == 'run' else 'download'
            destination = paths_for(settings)[2] / phase / args.model
            error_path = destination / 'failure.json'
            failure = dict(name=args.model, status='error', phase=phase, benchmark_complete=False,
                           error=f'{type(exc).__name__}: {exc}', traceback=traceback.format_exc())
            write_json(error_path, failure)
            # A stale successful report must never hide a failed rerun.
            if args.command == 'run':
                write_json(destination / 'report.json', failure)
        raise


if __name__ == '__main__':
    main()

'''[1:-1],
    'kitti_reference/eval.py': r'''
import io as sysio

import numba
import numpy as np

from .rotate_iou import rotate_iou_gpu_eval


@numba.jit
def get_thresholds(scores: np.ndarray, num_gt, num_sample_pts=41):
    scores.sort()
    scores = scores[::-1]
    current_recall = 0
    thresholds = []
    for i, score in enumerate(scores):
        l_recall = (i + 1) / num_gt
        if i < (len(scores) - 1):
            r_recall = (i + 2) / num_gt
        else:
            r_recall = l_recall
        if (((r_recall - current_recall) < (current_recall - l_recall))
                and (i < (len(scores) - 1))):
            continue
        # recall = l_recall
        thresholds.append(score)
        current_recall += 1 / (num_sample_pts - 1.0)
    return thresholds


def clean_data(gt_anno, dt_anno, current_class, difficulty):
    CLASS_NAMES = ['car', 'pedestrian', 'cyclist', 'van', 'person_sitting', 'truck']
    MIN_HEIGHT = [40, 25, 25]
    MAX_OCCLUSION = [0, 1, 2]
    MAX_TRUNCATION = [0.15, 0.3, 0.5]
    dc_bboxes, ignored_gt, ignored_dt = [], [], []
    current_cls_name = CLASS_NAMES[current_class].lower()
    num_gt = len(gt_anno["name"])
    num_dt = len(dt_anno["name"])
    num_valid_gt = 0
    for i in range(num_gt):
        bbox = gt_anno["bbox"][i]
        gt_name = gt_anno["name"][i].lower()
        height = bbox[3] - bbox[1]
        valid_class = -1
        if (gt_name == current_cls_name):
            valid_class = 1
        elif (current_cls_name == "Pedestrian".lower()
              and "Person_sitting".lower() == gt_name):
            valid_class = 0
        elif (current_cls_name == "Car".lower() and "Van".lower() == gt_name):
            valid_class = 0
        else:
            valid_class = -1
        ignore = False
        if ((gt_anno["occluded"][i] > MAX_OCCLUSION[difficulty])
                or (gt_anno["truncated"][i] > MAX_TRUNCATION[difficulty])
                or (height <= MIN_HEIGHT[difficulty])):
            # if gt_anno["difficulty"][i] > difficulty or gt_anno["difficulty"][i] == -1:
            ignore = True
        if valid_class == 1 and not ignore:
            ignored_gt.append(0)
            num_valid_gt += 1
        elif (valid_class == 0 or (ignore and (valid_class == 1))):
            ignored_gt.append(1)
        else:
            ignored_gt.append(-1)
    # for i in range(num_gt):
        if gt_anno["name"][i] == "DontCare":
            dc_bboxes.append(gt_anno["bbox"][i])
    for i in range(num_dt):
        if (dt_anno["name"][i].lower() == current_cls_name):
            valid_class = 1
        else:
            valid_class = -1
        height = abs(dt_anno["bbox"][i, 3] - dt_anno["bbox"][i, 1])
        if height < MIN_HEIGHT[difficulty]:
            ignored_dt.append(1)
        elif valid_class == 1:
            ignored_dt.append(0)
        else:
            ignored_dt.append(-1)

    return num_valid_gt, ignored_gt, ignored_dt, dc_bboxes


@numba.jit(nopython=True)
def image_box_overlap(boxes, query_boxes, criterion=-1):
    N = boxes.shape[0]
    K = query_boxes.shape[0]
    overlaps = np.zeros((N, K), dtype=boxes.dtype)
    for k in range(K):
        qbox_area = ((query_boxes[k, 2] - query_boxes[k, 0]) *
                     (query_boxes[k, 3] - query_boxes[k, 1]))
        for n in range(N):
            iw = (min(boxes[n, 2], query_boxes[k, 2]) -
                  max(boxes[n, 0], query_boxes[k, 0]))
            if iw > 0:
                ih = (min(boxes[n, 3], query_boxes[k, 3]) -
                      max(boxes[n, 1], query_boxes[k, 1]))
                if ih > 0:
                    if criterion == -1:
                        ua = (
                            (boxes[n, 2] - boxes[n, 0]) *
                            (boxes[n, 3] - boxes[n, 1]) + qbox_area - iw * ih)
                    elif criterion == 0:
                        ua = ((boxes[n, 2] - boxes[n, 0]) *
                              (boxes[n, 3] - boxes[n, 1]))
                    elif criterion == 1:
                        ua = qbox_area
                    else:
                        ua = 1.0
                    overlaps[n, k] = iw * ih / ua
    return overlaps


def bev_box_overlap(boxes, qboxes, criterion=-1):
    riou = rotate_iou_gpu_eval(boxes, qboxes, criterion)
    return riou


@numba.jit(nopython=True, parallel=True)
def d3_box_overlap_kernel(boxes, qboxes, rinc, criterion=-1):
    # ONLY support overlap in CAMERA, not lider.
    N, K = boxes.shape[0], qboxes.shape[0]
    for i in range(N):
        for j in range(K):
            if rinc[i, j] > 0:
                # iw = (min(boxes[i, 1] + boxes[i, 4], qboxes[j, 1] +
                #         qboxes[j, 4]) - max(boxes[i, 1], qboxes[j, 1]))
                iw = (min(boxes[i, 1], qboxes[j, 1]) - max(
                    boxes[i, 1] - boxes[i, 4], qboxes[j, 1] - qboxes[j, 4]))

                if iw > 0:
                    area1 = boxes[i, 3] * boxes[i, 4] * boxes[i, 5]
                    area2 = qboxes[j, 3] * qboxes[j, 4] * qboxes[j, 5]
                    inc = iw * rinc[i, j]
                    if criterion == -1:
                        ua = (area1 + area2 - inc)
                    elif criterion == 0:
                        ua = area1
                    elif criterion == 1:
                        ua = area2
                    else:
                        ua = inc
                    rinc[i, j] = inc / ua
                else:
                    rinc[i, j] = 0.0


def d3_box_overlap(boxes, qboxes, criterion=-1):
    rinc = rotate_iou_gpu_eval(boxes[:, [0, 2, 3, 5, 6]],
                               qboxes[:, [0, 2, 3, 5, 6]], 2)
    d3_box_overlap_kernel(boxes, qboxes, rinc, criterion)
    return rinc


@numba.jit(nopython=True)
def compute_statistics_jit(overlaps,
                           gt_datas,
                           dt_datas,
                           ignored_gt,
                           ignored_det,
                           dc_bboxes,
                           metric,
                           min_overlap,
                           thresh=0,
                           compute_fp=False,
                           compute_aos=False):

    det_size = dt_datas.shape[0]
    gt_size = gt_datas.shape[0]
    dt_scores = dt_datas[:, -1]
    dt_alphas = dt_datas[:, 4]
    gt_alphas = gt_datas[:, 4]
    dt_bboxes = dt_datas[:, :4]
    gt_bboxes = gt_datas[:, :4]

    assigned_detection = [False] * det_size
    ignored_threshold = [False] * det_size
    if compute_fp:
        for i in range(det_size):
            if (dt_scores[i] < thresh):
                ignored_threshold[i] = True
    NO_DETECTION = -10000000
    tp, fp, fn, similarity = 0, 0, 0, 0
    # thresholds = [0.0]
    # delta = [0.0]
    thresholds = np.zeros((gt_size, ))
    thresh_idx = 0
    delta = np.zeros((gt_size, ))
    delta_idx = 0
    for i in range(gt_size):
        if ignored_gt[i] == -1:
            continue
        det_idx = -1
        valid_detection = NO_DETECTION
        max_overlap = 0
        assigned_ignored_det = False

        for j in range(det_size):
            if (ignored_det[j] == -1):
                continue
            if (assigned_detection[j]):
                continue
            if (ignored_threshold[j]):
                continue
            overlap = overlaps[j, i]
            dt_score = dt_scores[j]
            if (not compute_fp and (overlap > min_overlap)
                    and dt_score > valid_detection):
                det_idx = j
                valid_detection = dt_score
            elif (compute_fp and (overlap > min_overlap)
                  and (overlap > max_overlap or assigned_ignored_det)
                  and ignored_det[j] == 0):
                max_overlap = overlap
                det_idx = j
                valid_detection = 1
                assigned_ignored_det = False
            elif (compute_fp and (overlap > min_overlap)
                  and (valid_detection == NO_DETECTION)
                  and ignored_det[j] == 1):
                det_idx = j
                valid_detection = 1
                assigned_ignored_det = True

        if (valid_detection == NO_DETECTION) and ignored_gt[i] == 0:
            fn += 1
        elif ((valid_detection != NO_DETECTION)
              and (ignored_gt[i] == 1 or ignored_det[det_idx] == 1)):
            assigned_detection[det_idx] = True
        elif valid_detection != NO_DETECTION:
            tp += 1
            # thresholds.append(dt_scores[det_idx])
            thresholds[thresh_idx] = dt_scores[det_idx]
            thresh_idx += 1
            if compute_aos:
                # delta.append(gt_alphas[i] - dt_alphas[det_idx])
                delta[delta_idx] = gt_alphas[i] - dt_alphas[det_idx]
                delta_idx += 1

            assigned_detection[det_idx] = True
    if compute_fp:
        for i in range(det_size):
            if (not (assigned_detection[i] or ignored_det[i] == -1
                     or ignored_det[i] == 1 or ignored_threshold[i])):
                fp += 1
        nstuff = 0
        if metric == 0:
            overlaps_dt_dc = image_box_overlap(dt_bboxes, dc_bboxes, 0)
            for i in range(dc_bboxes.shape[0]):
                for j in range(det_size):
                    if (assigned_detection[j]):
                        continue
                    if (ignored_det[j] == -1 or ignored_det[j] == 1):
                        continue
                    if (ignored_threshold[j]):
                        continue
                    if overlaps_dt_dc[j, i] > min_overlap:
                        assigned_detection[j] = True
                        nstuff += 1
        fp -= nstuff
        if compute_aos:
            tmp = np.zeros((fp + delta_idx, ))
            # tmp = [0] * fp
            for i in range(delta_idx):
                tmp[i + fp] = (1.0 + np.cos(delta[i])) / 2.0
                # tmp.append((1.0 + np.cos(delta[i])) / 2.0)
            # assert len(tmp) == fp + tp
            # assert len(delta) == tp
            if tp > 0 or fp > 0:
                similarity = np.sum(tmp)
            else:
                similarity = -1
    return tp, fp, fn, similarity, thresholds[:thresh_idx]


def get_split_parts(num, num_part):
    same_part = num // num_part
    remain_num = num % num_part
    if same_part == 0:
        return [num]

    if remain_num == 0:
        return [same_part] * num_part
    else:
        return [same_part] * num_part + [remain_num]


@numba.jit(nopython=True)
def fused_compute_statistics(overlaps,
                             pr,
                             gt_nums,
                             dt_nums,
                             dc_nums,
                             gt_datas,
                             dt_datas,
                             dontcares,
                             ignored_gts,
                             ignored_dets,
                             metric,
                             min_overlap,
                             thresholds,
                             compute_aos=False):
    gt_num = 0
    dt_num = 0
    dc_num = 0
    for i in range(gt_nums.shape[0]):
        for t, thresh in enumerate(thresholds):
            overlap = overlaps[dt_num:dt_num + dt_nums[i], gt_num:
                               gt_num + gt_nums[i]]

            gt_data = gt_datas[gt_num:gt_num + gt_nums[i]]
            dt_data = dt_datas[dt_num:dt_num + dt_nums[i]]
            ignored_gt = ignored_gts[gt_num:gt_num + gt_nums[i]]
            ignored_det = ignored_dets[dt_num:dt_num + dt_nums[i]]
            dontcare = dontcares[dc_num:dc_num + dc_nums[i]]
            tp, fp, fn, similarity, _ = compute_statistics_jit(
                overlap,
                gt_data,
                dt_data,
                ignored_gt,
                ignored_det,
                dontcare,
                metric,
                min_overlap=min_overlap,
                thresh=thresh,
                compute_fp=True,
                compute_aos=compute_aos)
            pr[t, 0] += tp
            pr[t, 1] += fp
            pr[t, 2] += fn
            if similarity != -1:
                pr[t, 3] += similarity
        gt_num += gt_nums[i]
        dt_num += dt_nums[i]
        dc_num += dc_nums[i]


def calculate_iou_partly(gt_annos, dt_annos, metric, num_parts=50):
    """fast iou algorithm. this function can be used independently to
    do result analysis. Must be used in CAMERA coordinate system.
    Args:
        gt_annos: dict, must from get_label_annos() in kitti_common.py
        dt_annos: dict, must from get_label_annos() in kitti_common.py
        metric: eval type. 0: bbox, 1: bev, 2: 3d
        num_parts: int. a parameter for fast calculate algorithm
    """
    assert len(gt_annos) == len(dt_annos)
    total_dt_num = np.stack([len(a["name"]) for a in dt_annos], 0)
    total_gt_num = np.stack([len(a["name"]) for a in gt_annos], 0)
    num_examples = len(gt_annos)
    split_parts = get_split_parts(num_examples, num_parts)
    parted_overlaps = []
    example_idx = 0

    for num_part in split_parts:
        gt_annos_part = gt_annos[example_idx:example_idx + num_part]
        dt_annos_part = dt_annos[example_idx:example_idx + num_part]
        if metric == 0:
            gt_boxes = np.concatenate([a["bbox"] for a in gt_annos_part], 0)
            dt_boxes = np.concatenate([a["bbox"] for a in dt_annos_part], 0)
            overlap_part = image_box_overlap(gt_boxes, dt_boxes)
        elif metric == 1:
            loc = np.concatenate(
                [a["location"][:, [0, 2]] for a in gt_annos_part], 0)
            dims = np.concatenate(
                [a["dimensions"][:, [0, 2]] for a in gt_annos_part], 0)
            rots = np.concatenate([a["rotation_y"] for a in gt_annos_part], 0)
            gt_boxes = np.concatenate(
                [loc, dims, rots[..., np.newaxis]], axis=1)
            loc = np.concatenate(
                [a["location"][:, [0, 2]] for a in dt_annos_part], 0)
            dims = np.concatenate(
                [a["dimensions"][:, [0, 2]] for a in dt_annos_part], 0)
            rots = np.concatenate([a["rotation_y"] for a in dt_annos_part], 0)
            dt_boxes = np.concatenate(
                [loc, dims, rots[..., np.newaxis]], axis=1)
            overlap_part = bev_box_overlap(gt_boxes, dt_boxes).astype(
                np.float64)
        elif metric == 2:
            loc = np.concatenate([a["location"] for a in gt_annos_part], 0)
            dims = np.concatenate([a["dimensions"] for a in gt_annos_part], 0)
            rots = np.concatenate([a["rotation_y"] for a in gt_annos_part], 0)
            gt_boxes = np.concatenate(
                [loc, dims, rots[..., np.newaxis]], axis=1)
            loc = np.concatenate([a["location"] for a in dt_annos_part], 0)
            dims = np.concatenate([a["dimensions"] for a in dt_annos_part], 0)
            rots = np.concatenate([a["rotation_y"] for a in dt_annos_part], 0)
            dt_boxes = np.concatenate(
                [loc, dims, rots[..., np.newaxis]], axis=1)
            overlap_part = d3_box_overlap(gt_boxes, dt_boxes).astype(
                np.float64)
        else:
            raise ValueError("unknown metric")
        parted_overlaps.append(overlap_part)
        example_idx += num_part
    overlaps = []
    example_idx = 0
    for j, num_part in enumerate(split_parts):
        gt_annos_part = gt_annos[example_idx:example_idx + num_part]
        dt_annos_part = dt_annos[example_idx:example_idx + num_part]
        gt_num_idx, dt_num_idx = 0, 0
        for i in range(num_part):
            gt_box_num = total_gt_num[example_idx + i]
            dt_box_num = total_dt_num[example_idx + i]
            overlaps.append(
                parted_overlaps[j][gt_num_idx:gt_num_idx + gt_box_num,
                                   dt_num_idx:dt_num_idx + dt_box_num])
            gt_num_idx += gt_box_num
            dt_num_idx += dt_box_num
        example_idx += num_part

    return overlaps, parted_overlaps, total_gt_num, total_dt_num


def _prepare_data(gt_annos, dt_annos, current_class, difficulty):
    gt_datas_list = []
    dt_datas_list = []
    total_dc_num = []
    ignored_gts, ignored_dets, dontcares = [], [], []
    total_num_valid_gt = 0
    for i in range(len(gt_annos)):
        rets = clean_data(gt_annos[i], dt_annos[i], current_class, difficulty)
        num_valid_gt, ignored_gt, ignored_det, dc_bboxes = rets
        ignored_gts.append(np.array(ignored_gt, dtype=np.int64))
        ignored_dets.append(np.array(ignored_det, dtype=np.int64))
        if len(dc_bboxes) == 0:
            dc_bboxes = np.zeros((0, 4)).astype(np.float64)
        else:
            dc_bboxes = np.stack(dc_bboxes, 0).astype(np.float64)
        total_dc_num.append(dc_bboxes.shape[0])
        dontcares.append(dc_bboxes)
        total_num_valid_gt += num_valid_gt
        gt_datas = np.concatenate(
            [gt_annos[i]["bbox"], gt_annos[i]["alpha"][..., np.newaxis]], 1)
        dt_datas = np.concatenate([
            dt_annos[i]["bbox"], dt_annos[i]["alpha"][..., np.newaxis],
            dt_annos[i]["score"][..., np.newaxis]
        ], 1)
        gt_datas_list.append(gt_datas)
        dt_datas_list.append(dt_datas)
    total_dc_num = np.stack(total_dc_num, axis=0)
    return (gt_datas_list, dt_datas_list, ignored_gts, ignored_dets, dontcares,
            total_dc_num, total_num_valid_gt)


def eval_class(gt_annos,
               dt_annos,
               current_classes,
               difficultys,
               metric,
               min_overlaps,
               compute_aos=False,
               num_parts=100):
    """Kitti eval. support 2d/bev/3d/aos eval. support 0.5:0.05:0.95 coco AP.
    Args:
        gt_annos: dict, must from get_label_annos() in kitti_common.py
        dt_annos: dict, must from get_label_annos() in kitti_common.py
        current_classes: list of int, 0: car, 1: pedestrian, 2: cyclist
        difficultys: list of int. eval difficulty, 0: easy, 1: normal, 2: hard
        metric: eval type. 0: bbox, 1: bev, 2: 3d
        min_overlaps: float, min overlap. format: [num_overlap, metric, class].
        num_parts: int. a parameter for fast calculate algorithm

    Returns:
        dict of recall, precision and aos
    """
    assert len(gt_annos) == len(dt_annos)
    num_examples = len(gt_annos)
    split_parts = get_split_parts(num_examples, num_parts)

    rets = calculate_iou_partly(dt_annos, gt_annos, metric, num_parts)
    overlaps, parted_overlaps, total_dt_num, total_gt_num = rets
    N_SAMPLE_PTS = 41
    num_minoverlap = len(min_overlaps)
    num_class = len(current_classes)
    num_difficulty = len(difficultys)
    precision = np.zeros(
        [num_class, num_difficulty, num_minoverlap, N_SAMPLE_PTS])
    recall = np.zeros(
        [num_class, num_difficulty, num_minoverlap, N_SAMPLE_PTS])
    aos = np.zeros([num_class, num_difficulty, num_minoverlap, N_SAMPLE_PTS])
    for m, current_class in enumerate(current_classes):
        for l, difficulty in enumerate(difficultys):
            rets = _prepare_data(gt_annos, dt_annos, current_class, difficulty)
            (gt_datas_list, dt_datas_list, ignored_gts, ignored_dets,
             dontcares, total_dc_num, total_num_valid_gt) = rets
            for k, min_overlap in enumerate(min_overlaps[:, metric, m]):
                thresholdss = []
                for i in range(len(gt_annos)):
                    rets = compute_statistics_jit(
                        overlaps[i],
                        gt_datas_list[i],
                        dt_datas_list[i],
                        ignored_gts[i],
                        ignored_dets[i],
                        dontcares[i],
                        metric,
                        min_overlap=min_overlap,
                        thresh=0.0,
                        compute_fp=False)
                    tp, fp, fn, similarity, thresholds = rets
                    thresholdss += thresholds.tolist()
                thresholdss = np.array(thresholdss)
                thresholds = get_thresholds(thresholdss, total_num_valid_gt)
                thresholds = np.array(thresholds)
                pr = np.zeros([len(thresholds), 4])
                idx = 0
                for j, num_part in enumerate(split_parts):
                    gt_datas_part = np.concatenate(
                        gt_datas_list[idx:idx + num_part], 0)
                    dt_datas_part = np.concatenate(
                        dt_datas_list[idx:idx + num_part], 0)
                    dc_datas_part = np.concatenate(
                        dontcares[idx:idx + num_part], 0)
                    ignored_dets_part = np.concatenate(
                        ignored_dets[idx:idx + num_part], 0)
                    ignored_gts_part = np.concatenate(
                        ignored_gts[idx:idx + num_part], 0)
                    fused_compute_statistics(
                        parted_overlaps[j],
                        pr,
                        total_gt_num[idx:idx + num_part],
                        total_dt_num[idx:idx + num_part],
                        total_dc_num[idx:idx + num_part],
                        gt_datas_part,
                        dt_datas_part,
                        dc_datas_part,
                        ignored_gts_part,
                        ignored_dets_part,
                        metric,
                        min_overlap=min_overlap,
                        thresholds=thresholds,
                        compute_aos=compute_aos)
                    idx += num_part
                for i in range(len(thresholds)):
                    recall[m, l, k, i] = pr[i, 0] / (pr[i, 0] + pr[i, 2])
                    precision[m, l, k, i] = pr[i, 0] / (pr[i, 0] + pr[i, 1])
                    if compute_aos:
                        aos[m, l, k, i] = pr[i, 3] / (pr[i, 0] + pr[i, 1])
                for i in range(len(thresholds)):
                    precision[m, l, k, i] = np.max(
                        precision[m, l, k, i:], axis=-1)
                    recall[m, l, k, i] = np.max(recall[m, l, k, i:], axis=-1)
                    if compute_aos:
                        aos[m, l, k, i] = np.max(aos[m, l, k, i:], axis=-1)
    ret_dict = {
        "recall": recall,
        "precision": precision,
        "orientation": aos,
    }
    return ret_dict


def get_mAP(prec):
    sums = 0
    for i in range(0, prec.shape[-1], 4):
        sums = sums + prec[..., i]
    return sums / 11 * 100


def get_mAP_R40(prec):
    sums = 0
    for i in range(1, prec.shape[-1]):
        sums = sums + prec[..., i]
    return sums / 40 * 100


def print_str(value, *arg, sstream=None):
    if sstream is None:
        sstream = sysio.StringIO()
    sstream.truncate(0)
    sstream.seek(0)
    print(value, *arg, file=sstream)
    return sstream.getvalue()


def do_eval(gt_annos,
            dt_annos,
            current_classes,
            min_overlaps,
            compute_aos=False,
            PR_detail_dict=None):
    # min_overlaps: [num_minoverlap, metric, num_class]
    difficultys = [0, 1, 2]
    ret = eval_class(gt_annos, dt_annos, current_classes, difficultys, 0,
                     min_overlaps, compute_aos)
    # ret: [num_class, num_diff, num_minoverlap, num_sample_points]
    mAP_bbox = get_mAP(ret["precision"])
    mAP_bbox_R40 = get_mAP_R40(ret["precision"])

    if PR_detail_dict is not None:
        PR_detail_dict['bbox'] = ret['precision']

    mAP_aos = mAP_aos_R40 = None
    if compute_aos:
        mAP_aos = get_mAP(ret["orientation"])
        mAP_aos_R40 = get_mAP_R40(ret["orientation"])

        if PR_detail_dict is not None:
            PR_detail_dict['aos'] = ret['orientation']

    ret = eval_class(gt_annos, dt_annos, current_classes, difficultys, 1,
                     min_overlaps)
    mAP_bev = get_mAP(ret["precision"])
    mAP_bev_R40 = get_mAP_R40(ret["precision"])

    if PR_detail_dict is not None:
        PR_detail_dict['bev'] = ret['precision']

    ret = eval_class(gt_annos, dt_annos, current_classes, difficultys, 2,
                     min_overlaps)
    mAP_3d = get_mAP(ret["precision"])
    mAP_3d_R40 = get_mAP_R40(ret["precision"])
    if PR_detail_dict is not None:
        PR_detail_dict['3d'] = ret['precision']
    return mAP_bbox, mAP_bev, mAP_3d, mAP_aos, mAP_bbox_R40, mAP_bev_R40, mAP_3d_R40, mAP_aos_R40


def do_coco_style_eval(gt_annos, dt_annos, current_classes, overlap_ranges,
                       compute_aos):
    # overlap_ranges: [range, metric, num_class]
    min_overlaps = np.zeros([10, *overlap_ranges.shape[1:]])
    for i in range(overlap_ranges.shape[1]):
        for j in range(overlap_ranges.shape[2]):
            min_overlaps[:, i, j] = np.linspace(*overlap_ranges[:, i, j])
    mAP_bbox, mAP_bev, mAP_3d, mAP_aos = do_eval(
        gt_annos, dt_annos, current_classes, min_overlaps, compute_aos)
    # ret: [num_class, num_diff, num_minoverlap]
    mAP_bbox = mAP_bbox.mean(-1)
    mAP_bev = mAP_bev.mean(-1)
    mAP_3d = mAP_3d.mean(-1)
    if mAP_aos is not None:
        mAP_aos = mAP_aos.mean(-1)
    return mAP_bbox, mAP_bev, mAP_3d, mAP_aos


def get_official_eval_result(gt_annos, dt_annos, current_classes, PR_detail_dict=None):
    overlap_0_7 = np.array([[0.7, 0.5, 0.5, 0.7,
                             0.5, 0.7], [0.7, 0.5, 0.5, 0.7, 0.5, 0.7],
                            [0.7, 0.5, 0.5, 0.7, 0.5, 0.7]])
    overlap_0_5 = np.array([[0.7, 0.5, 0.5, 0.7,
                             0.5, 0.5], [0.5, 0.25, 0.25, 0.5, 0.25, 0.5],
                            [0.5, 0.25, 0.25, 0.5, 0.25, 0.5]])
    min_overlaps = np.stack([overlap_0_7, overlap_0_5], axis=0)  # [2, 3, 5]
    class_to_name = {
        0: 'Car',
        1: 'Pedestrian',
        2: 'Cyclist',
        3: 'Van',
        4: 'Person_sitting',
        5: 'Truck'
    }
    name_to_class = {v: n for n, v in class_to_name.items()}
    if not isinstance(current_classes, (list, tuple)):
        current_classes = [current_classes]
    current_classes_int = []
    for curcls in current_classes:
        if isinstance(curcls, str):
            current_classes_int.append(name_to_class[curcls])
        else:
            current_classes_int.append(curcls)
    current_classes = current_classes_int
    min_overlaps = min_overlaps[:, :, current_classes]
    result = ''
    # check whether alpha is valid
    compute_aos = False
    for anno in dt_annos:
        if anno['alpha'].shape[0] != 0:
            if anno['alpha'][0] != -10:
                compute_aos = True
            break
    mAPbbox, mAPbev, mAP3d, mAPaos, mAPbbox_R40, mAPbev_R40, mAP3d_R40, mAPaos_R40 = do_eval(
        gt_annos, dt_annos, current_classes, min_overlaps, compute_aos, PR_detail_dict=PR_detail_dict)

    ret_dict = {}
    for j, curcls in enumerate(current_classes):
        # mAP threshold array: [num_minoverlap, metric, class]
        # mAP result: [num_class, num_diff, num_minoverlap]
        for i in range(min_overlaps.shape[0]):
            result += print_str(
                (f"{class_to_name[curcls]} "
                 "AP@{:.2f}, {:.2f}, {:.2f}:".format(*min_overlaps[i, :, j])))
            result += print_str((f"bbox AP:{mAPbbox[j, 0, i]:.4f}, "
                                 f"{mAPbbox[j, 1, i]:.4f}, "
                                 f"{mAPbbox[j, 2, i]:.4f}"))
            result += print_str((f"bev  AP:{mAPbev[j, 0, i]:.4f}, "
                                 f"{mAPbev[j, 1, i]:.4f}, "
                                 f"{mAPbev[j, 2, i]:.4f}"))
            result += print_str((f"3d   AP:{mAP3d[j, 0, i]:.4f}, "
                                 f"{mAP3d[j, 1, i]:.4f}, "
                                 f"{mAP3d[j, 2, i]:.4f}"))

            if compute_aos:
                result += print_str((f"aos  AP:{mAPaos[j, 0, i]:.2f}, "
                                     f"{mAPaos[j, 1, i]:.2f}, "
                                     f"{mAPaos[j, 2, i]:.2f}"))
                # if i == 0:
                   # ret_dict['%s_aos/easy' % class_to_name[curcls]] = mAPaos[j, 0, 0]
                   # ret_dict['%s_aos/moderate' % class_to_name[curcls]] = mAPaos[j, 1, 0]
                   # ret_dict['%s_aos/hard' % class_to_name[curcls]] = mAPaos[j, 2, 0]

            result += print_str(
                (f"{class_to_name[curcls]} "
                 "AP_R40@{:.2f}, {:.2f}, {:.2f}:".format(*min_overlaps[i, :, j])))
            result += print_str((f"bbox AP:{mAPbbox_R40[j, 0, i]:.4f}, "
                                 f"{mAPbbox_R40[j, 1, i]:.4f}, "
                                 f"{mAPbbox_R40[j, 2, i]:.4f}"))
            result += print_str((f"bev  AP:{mAPbev_R40[j, 0, i]:.4f}, "
                                 f"{mAPbev_R40[j, 1, i]:.4f}, "
                                 f"{mAPbev_R40[j, 2, i]:.4f}"))
            result += print_str((f"3d   AP:{mAP3d_R40[j, 0, i]:.4f}, "
                                 f"{mAP3d_R40[j, 1, i]:.4f}, "
                                 f"{mAP3d_R40[j, 2, i]:.4f}"))
            if compute_aos:
                result += print_str((f"aos  AP:{mAPaos_R40[j, 0, i]:.2f}, "
                                     f"{mAPaos_R40[j, 1, i]:.2f}, "
                                     f"{mAPaos_R40[j, 2, i]:.2f}"))
                if i == 0:
                   ret_dict['%s_aos/easy_R40' % class_to_name[curcls]] = mAPaos_R40[j, 0, 0]
                   ret_dict['%s_aos/moderate_R40' % class_to_name[curcls]] = mAPaos_R40[j, 1, 0]
                   ret_dict['%s_aos/hard_R40' % class_to_name[curcls]] = mAPaos_R40[j, 2, 0]

            if i == 0:
                # ret_dict['%s_3d/easy' % class_to_name[curcls]] = mAP3d[j, 0, 0]
                # ret_dict['%s_3d/moderate' % class_to_name[curcls]] = mAP3d[j, 1, 0]
                # ret_dict['%s_3d/hard' % class_to_name[curcls]] = mAP3d[j, 2, 0]
                # ret_dict['%s_bev/easy' % class_to_name[curcls]] = mAPbev[j, 0, 0]
                # ret_dict['%s_bev/moderate' % class_to_name[curcls]] = mAPbev[j, 1, 0]
                # ret_dict['%s_bev/hard' % class_to_name[curcls]] = mAPbev[j, 2, 0]
                # ret_dict['%s_image/easy' % class_to_name[curcls]] = mAPbbox[j, 0, 0]
                # ret_dict['%s_image/moderate' % class_to_name[curcls]] = mAPbbox[j, 1, 0]
                # ret_dict['%s_image/hard' % class_to_name[curcls]] = mAPbbox[j, 2, 0]

                ret_dict['%s_3d/easy_R40' % class_to_name[curcls]] = mAP3d_R40[j, 0, 0]
                ret_dict['%s_3d/moderate_R40' % class_to_name[curcls]] = mAP3d_R40[j, 1, 0]
                ret_dict['%s_3d/hard_R40' % class_to_name[curcls]] = mAP3d_R40[j, 2, 0]
                ret_dict['%s_bev/easy_R40' % class_to_name[curcls]] = mAPbev_R40[j, 0, 0]
                ret_dict['%s_bev/moderate_R40' % class_to_name[curcls]] = mAPbev_R40[j, 1, 0]
                ret_dict['%s_bev/hard_R40' % class_to_name[curcls]] = mAPbev_R40[j, 2, 0]
                ret_dict['%s_image/easy_R40' % class_to_name[curcls]] = mAPbbox_R40[j, 0, 0]
                ret_dict['%s_image/moderate_R40' % class_to_name[curcls]] = mAPbbox_R40[j, 1, 0]
                ret_dict['%s_image/hard_R40' % class_to_name[curcls]] = mAPbbox_R40[j, 2, 0]

    return result, ret_dict


def get_coco_eval_result(gt_annos, dt_annos, current_classes):
    class_to_name = {
        0: 'Car',
        1: 'Pedestrian',
        2: 'Cyclist',
        3: 'Van',
        4: 'Person_sitting',
    }
    class_to_range = {
        0: [0.5, 0.95, 10],
        1: [0.25, 0.7, 10],
        2: [0.25, 0.7, 10],
        3: [0.5, 0.95, 10],
        4: [0.25, 0.7, 10],
    }
    name_to_class = {v: n for n, v in class_to_name.items()}
    if not isinstance(current_classes, (list, tuple)):
        current_classes = [current_classes]
    current_classes_int = []
    for curcls in current_classes:
        if isinstance(curcls, str):
            current_classes_int.append(name_to_class[curcls])
        else:
            current_classes_int.append(curcls)
    current_classes = current_classes_int
    overlap_ranges = np.zeros([3, 3, len(current_classes)])
    for i, curcls in enumerate(current_classes):
        overlap_ranges[:, :, i] = np.array(
            class_to_range[curcls])[:, np.newaxis]
    result = ''
    # check whether alpha is valid
    compute_aos = False
    for anno in dt_annos:
        if anno['alpha'].shape[0] != 0:
            if anno['alpha'][0] != -10:
                compute_aos = True
            break
    mAPbbox, mAPbev, mAP3d, mAPaos = do_coco_style_eval(
        gt_annos, dt_annos, current_classes, overlap_ranges, compute_aos)
    for j, curcls in enumerate(current_classes):
        # mAP threshold array: [num_minoverlap, metric, class]
        # mAP result: [num_class, num_diff, num_minoverlap]
        o_range = np.array(class_to_range[curcls])[[0, 2, 1]]
        o_range[1] = (o_range[2] - o_range[0]) / (o_range[1] - 1)
        result += print_str((f"{class_to_name[curcls]} "
                             "coco AP@{:.2f}:{:.2f}:{:.2f}:".format(*o_range)))
        result += print_str((f"bbox AP:{mAPbbox[j, 0]:.2f}, "
                             f"{mAPbbox[j, 1]:.2f}, "
                             f"{mAPbbox[j, 2]:.2f}"))
        result += print_str((f"bev  AP:{mAPbev[j, 0]:.2f}, "
                             f"{mAPbev[j, 1]:.2f}, "
                             f"{mAPbev[j, 2]:.2f}"))
        result += print_str((f"3d   AP:{mAP3d[j, 0]:.2f}, "
                             f"{mAP3d[j, 1]:.2f}, "
                             f"{mAP3d[j, 2]:.2f}"))
        if compute_aos:
            result += print_str((f"aos  AP:{mAPaos[j, 0]:.2f}, "
                                 f"{mAPaos[j, 1]:.2f}, "
                                 f"{mAPaos[j, 2]:.2f}"))
    return result

'''[1:-1],
    'kitti_reference/rotate_iou.py': r'''
#####################
# Based on https://github.com/hongzhenwang/RRPN-revise
# Licensed under The MIT License
# Author: yanyan, scrin@foxmail.com
#####################
import math

import numba
import numpy as np
from numba import cuda


@numba.jit(nopython=True)
def div_up(m, n):
    return m // n + (m % n > 0)

@cuda.jit('(float32[:], float32[:], float32[:])', device=True, inline=True)
def trangle_area(a, b, c):
    return ((a[0] - c[0]) * (b[1] - c[1]) - (a[1] - c[1]) *
            (b[0] - c[0])) / 2.0


@cuda.jit('(float32[:], int32)', device=True, inline=True)
def area(int_pts, num_of_inter):
    area_val = 0.0
    for i in range(num_of_inter - 2):
        area_val += abs(
            trangle_area(int_pts[:2], int_pts[2 * i + 2:2 * i + 4],
                         int_pts[2 * i + 4:2 * i + 6]))
    return area_val


@cuda.jit('(float32[:], int32)', device=True, inline=True)
def sort_vertex_in_convex_polygon(int_pts, num_of_inter):
    if num_of_inter > 0:
        center = cuda.local.array((2, ), dtype=numba.float32)
        center[:] = 0.0
        for i in range(num_of_inter):
            center[0] += int_pts[2 * i]
            center[1] += int_pts[2 * i + 1]
        center[0] /= num_of_inter
        center[1] /= num_of_inter
        v = cuda.local.array((2, ), dtype=numba.float32)
        vs = cuda.local.array((16, ), dtype=numba.float32)
        for i in range(num_of_inter):
            v[0] = int_pts[2 * i] - center[0]
            v[1] = int_pts[2 * i + 1] - center[1]
            d = math.sqrt(v[0] * v[0] + v[1] * v[1])
            v[0] = v[0] / d
            v[1] = v[1] / d
            if v[1] < 0:
                v[0] = -2 - v[0]
            vs[i] = v[0]
        j = 0
        temp = 0
        for i in range(1, num_of_inter):
            if vs[i - 1] > vs[i]:
                temp = vs[i]
                tx = int_pts[2 * i]
                ty = int_pts[2 * i + 1]
                j = i
                while j > 0 and vs[j - 1] > temp:
                    vs[j] = vs[j - 1]
                    int_pts[j * 2] = int_pts[j * 2 - 2]
                    int_pts[j * 2 + 1] = int_pts[j * 2 - 1]
                    j -= 1

                vs[j] = temp
                int_pts[j * 2] = tx
                int_pts[j * 2 + 1] = ty


@cuda.jit(
    '(float32[:], float32[:], int32, int32, float32[:])',
    device=True,
    inline=True)
def line_segment_intersection(pts1, pts2, i, j, temp_pts):
    A = cuda.local.array((2, ), dtype=numba.float32)
    B = cuda.local.array((2, ), dtype=numba.float32)
    C = cuda.local.array((2, ), dtype=numba.float32)
    D = cuda.local.array((2, ), dtype=numba.float32)

    A[0] = pts1[2 * i]
    A[1] = pts1[2 * i + 1]

    B[0] = pts1[2 * ((i + 1) % 4)]
    B[1] = pts1[2 * ((i + 1) % 4) + 1]

    C[0] = pts2[2 * j]
    C[1] = pts2[2 * j + 1]

    D[0] = pts2[2 * ((j + 1) % 4)]
    D[1] = pts2[2 * ((j + 1) % 4) + 1]
    BA0 = B[0] - A[0]
    BA1 = B[1] - A[1]
    DA0 = D[0] - A[0]
    CA0 = C[0] - A[0]
    DA1 = D[1] - A[1]
    CA1 = C[1] - A[1]
    acd = DA1 * CA0 > CA1 * DA0
    bcd = (D[1] - B[1]) * (C[0] - B[0]) > (C[1] - B[1]) * (D[0] - B[0])
    if acd != bcd:
        abc = CA1 * BA0 > BA1 * CA0
        abd = DA1 * BA0 > BA1 * DA0
        if abc != abd:
            DC0 = D[0] - C[0]
            DC1 = D[1] - C[1]
            ABBA = A[0] * B[1] - B[0] * A[1]
            CDDC = C[0] * D[1] - D[0] * C[1]
            DH = BA1 * DC0 - BA0 * DC1
            Dx = ABBA * DC0 - BA0 * CDDC
            Dy = ABBA * DC1 - BA1 * CDDC
            temp_pts[0] = Dx / DH
            temp_pts[1] = Dy / DH
            return True
    return False


@cuda.jit(
    '(float32[:], float32[:], int32, int32, float32[:])',
    device=True,
    inline=True)
def line_segment_intersection_v1(pts1, pts2, i, j, temp_pts):
    a = cuda.local.array((2, ), dtype=numba.float32)
    b = cuda.local.array((2, ), dtype=numba.float32)
    c = cuda.local.array((2, ), dtype=numba.float32)
    d = cuda.local.array((2, ), dtype=numba.float32)

    a[0] = pts1[2 * i]
    a[1] = pts1[2 * i + 1]

    b[0] = pts1[2 * ((i + 1) % 4)]
    b[1] = pts1[2 * ((i + 1) % 4) + 1]

    c[0] = pts2[2 * j]
    c[1] = pts2[2 * j + 1]

    d[0] = pts2[2 * ((j + 1) % 4)]
    d[1] = pts2[2 * ((j + 1) % 4) + 1]

    area_abc = trangle_area(a, b, c)
    area_abd = trangle_area(a, b, d)

    if area_abc * area_abd >= 0:
        return False

    area_cda = trangle_area(c, d, a)
    area_cdb = area_cda + area_abc - area_abd

    if area_cda * area_cdb >= 0:
        return False
    t = area_cda / (area_abd - area_abc)

    dx = t * (b[0] - a[0])
    dy = t * (b[1] - a[1])
    temp_pts[0] = a[0] + dx
    temp_pts[1] = a[1] + dy
    return True


@cuda.jit('(float32, float32, float32[:])', device=True, inline=True)
def point_in_quadrilateral(pt_x, pt_y, corners):
    ab0 = corners[2] - corners[0]
    ab1 = corners[3] - corners[1]

    ad0 = corners[6] - corners[0]
    ad1 = corners[7] - corners[1]

    ap0 = pt_x - corners[0]
    ap1 = pt_y - corners[1]

    abab = ab0 * ab0 + ab1 * ab1
    abap = ab0 * ap0 + ab1 * ap1
    adad = ad0 * ad0 + ad1 * ad1
    adap = ad0 * ap0 + ad1 * ap1

    return abab >= abap and abap >= 0 and adad >= adap and adap >= 0


@cuda.jit('(float32[:], float32[:], float32[:])', device=True, inline=True)
def quadrilateral_intersection(pts1, pts2, int_pts):
    num_of_inter = 0
    for i in range(4):
        if point_in_quadrilateral(pts1[2 * i], pts1[2 * i + 1], pts2):
            int_pts[num_of_inter * 2] = pts1[2 * i]
            int_pts[num_of_inter * 2 + 1] = pts1[2 * i + 1]
            num_of_inter += 1
        if point_in_quadrilateral(pts2[2 * i], pts2[2 * i + 1], pts1):
            int_pts[num_of_inter * 2] = pts2[2 * i]
            int_pts[num_of_inter * 2 + 1] = pts2[2 * i + 1]
            num_of_inter += 1
    temp_pts = cuda.local.array((2, ), dtype=numba.float32)
    for i in range(4):
        for j in range(4):
            has_pts = line_segment_intersection(pts1, pts2, i, j, temp_pts)
            if has_pts:
                int_pts[num_of_inter * 2] = temp_pts[0]
                int_pts[num_of_inter * 2 + 1] = temp_pts[1]
                num_of_inter += 1

    return num_of_inter


@cuda.jit('(float32[:], float32[:])', device=True, inline=True)
def rbbox_to_corners(corners, rbbox):
    # generate clockwise corners and rotate it clockwise
    angle = rbbox[4]
    a_cos = math.cos(angle)
    a_sin = math.sin(angle)
    center_x = rbbox[0]
    center_y = rbbox[1]
    x_d = rbbox[2]
    y_d = rbbox[3]
    corners_x = cuda.local.array((4, ), dtype=numba.float32)
    corners_y = cuda.local.array((4, ), dtype=numba.float32)
    corners_x[0] = -x_d / 2
    corners_x[1] = -x_d / 2
    corners_x[2] = x_d / 2
    corners_x[3] = x_d / 2
    corners_y[0] = -y_d / 2
    corners_y[1] = y_d / 2
    corners_y[2] = y_d / 2
    corners_y[3] = -y_d / 2
    for i in range(4):
        corners[2 *
                i] = a_cos * corners_x[i] + a_sin * corners_y[i] + center_x
        corners[2 * i
                + 1] = -a_sin * corners_x[i] + a_cos * corners_y[i] + center_y


@cuda.jit('(float32[:], float32[:])', device=True, inline=True)
def inter(rbbox1, rbbox2):
    corners1 = cuda.local.array((8, ), dtype=numba.float32)
    corners2 = cuda.local.array((8, ), dtype=numba.float32)
    intersection_corners = cuda.local.array((16, ), dtype=numba.float32)

    rbbox_to_corners(corners1, rbbox1)
    rbbox_to_corners(corners2, rbbox2)

    num_intersection = quadrilateral_intersection(corners1, corners2,
                                                  intersection_corners)
    sort_vertex_in_convex_polygon(intersection_corners, num_intersection)
    # print(intersection_corners.reshape([-1, 2])[:num_intersection])

    return area(intersection_corners, num_intersection)


@cuda.jit('(float32[:], float32[:], int32)', device=True, inline=True)
def devRotateIoUEval(rbox1, rbox2, criterion=-1):
    area1 = rbox1[2] * rbox1[3]
    area2 = rbox2[2] * rbox2[3]
    area_inter = inter(rbox1, rbox2)
    if criterion == -1:
        return area_inter / (area1 + area2 - area_inter)
    elif criterion == 0:
        return area_inter / area1
    elif criterion == 1:
        return area_inter / area2
    else:
        return area_inter

@cuda.jit('(int64, int64, float32[:], float32[:], float32[:], int32)', fastmath=False)
def rotate_iou_kernel_eval(N, K, dev_boxes, dev_query_boxes, dev_iou, criterion=-1):
    threadsPerBlock = 8 * 8
    row_start = cuda.blockIdx.x
    col_start = cuda.blockIdx.y
    tx = cuda.threadIdx.x
    row_size = min(N - row_start * threadsPerBlock, threadsPerBlock)
    col_size = min(K - col_start * threadsPerBlock, threadsPerBlock)
    block_boxes = cuda.shared.array(shape=(64 * 5, ), dtype=numba.float32)
    block_qboxes = cuda.shared.array(shape=(64 * 5, ), dtype=numba.float32)

    dev_query_box_idx = threadsPerBlock * col_start + tx
    dev_box_idx = threadsPerBlock * row_start + tx
    if (tx < col_size):
        block_qboxes[tx * 5 + 0] = dev_query_boxes[dev_query_box_idx * 5 + 0]
        block_qboxes[tx * 5 + 1] = dev_query_boxes[dev_query_box_idx * 5 + 1]
        block_qboxes[tx * 5 + 2] = dev_query_boxes[dev_query_box_idx * 5 + 2]
        block_qboxes[tx * 5 + 3] = dev_query_boxes[dev_query_box_idx * 5 + 3]
        block_qboxes[tx * 5 + 4] = dev_query_boxes[dev_query_box_idx * 5 + 4]
    if (tx < row_size):
        block_boxes[tx * 5 + 0] = dev_boxes[dev_box_idx * 5 + 0]
        block_boxes[tx * 5 + 1] = dev_boxes[dev_box_idx * 5 + 1]
        block_boxes[tx * 5 + 2] = dev_boxes[dev_box_idx * 5 + 2]
        block_boxes[tx * 5 + 3] = dev_boxes[dev_box_idx * 5 + 3]
        block_boxes[tx * 5 + 4] = dev_boxes[dev_box_idx * 5 + 4]
    cuda.syncthreads()
    if tx < row_size:
        for i in range(col_size):
            offset = row_start * threadsPerBlock * K + col_start * threadsPerBlock + tx * K + i
            dev_iou[offset] = devRotateIoUEval(block_qboxes[i * 5:i * 5 + 5],
                                           block_boxes[tx * 5:tx * 5 + 5], criterion)


def rotate_iou_gpu_eval(boxes, query_boxes, criterion=-1, device_id=0):
    """rotated box iou running in gpu. 500x faster than cpu version
    (take 5ms in one example with numba.cuda code).
    convert from [this project](
        https://github.com/hongzhenwang/RRPN-revise/tree/master/pcdet/rotation).
    
    Args:
        boxes (float tensor: [N, 5]): rbboxes. format: centers, dims, 
            angles(clockwise when positive)
        query_boxes (float tensor: [K, 5]): [description]
        device_id (int, optional): Defaults to 0. [description]
    
    Returns:
        [type]: [description]
    """
    box_dtype = boxes.dtype
    boxes = boxes.astype(np.float32)
    query_boxes = query_boxes.astype(np.float32)
    N = boxes.shape[0]
    K = query_boxes.shape[0]
    iou = np.zeros((N, K), dtype=np.float32)
    if N == 0 or K == 0:
        return iou
    threadsPerBlock = 8 * 8
    cuda.select_device(device_id)
    blockspergrid = (div_up(N, threadsPerBlock), div_up(K, threadsPerBlock))
    
    stream = cuda.stream()
    with stream.auto_synchronize():
        boxes_dev = cuda.to_device(boxes.reshape([-1]), stream)
        query_boxes_dev = cuda.to_device(query_boxes.reshape([-1]), stream)
        iou_dev = cuda.to_device(iou.reshape([-1]), stream)
        rotate_iou_kernel_eval[blockspergrid, threadsPerBlock, stream](
            N, K, boxes_dev, query_boxes_dev, iou_dev, criterion)
        iou_dev.copy_to_host(iou.reshape([-1]), stream=stream)
    return iou.astype(boxes.dtype)

'''[1:-1],
    'kitti_reference/__init__.py': r'''

'''[1:-1],
    'kitti_reference/LICENSE.OpenPCDet': r'''
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

'''[1:-1],
    'kitti_reference/NOTICE.txt': r'''
Reference evaluator from OpenPCDet,
revision 233f849829b6ac19afb8af8837a0246890908755.
https://github.com/open-mmlab/OpenPCDet
OpenPCDet is distributed under Apache License 2.0:
https://github.com/open-mmlab/OpenPCDet/blob/233f849829b6ac19afb8af8837a0246890908755/LICENSE
The two evaluator files are embedded unchanged; their hashes are checked before scoring.

'''[1:-1],
}

for relative, source in BUNDLE.items():
    destination = WORK_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source)
RUNNER = WORK_DIR / 'pixor_training.py'
print('Đã chuẩn bị runner và split:', WORK_DIR)

In [ ]:
# Nhận thư mục raw hoặc RAR, ZIP, TAR như Notebook Robust.
sys.path.insert(0, str(WORK_DIR))
from kitti_notebook_data import prepare_dataset
FRAME_IDS = [line.strip() for split_name in ['train', 'val']
             for line in (WORK_DIR / 'splits' / (split_name + '.txt')).read_text().splitlines() if line.strip()]
DATA_RESULT = prepare_dataset(KITTI_ROOT, EXTRACT_DIR, FRAME_IDS,
    metadata_path=IMAGE_SIZES_JSON,
    fallback_metadata=WORK_DIR / 'metadata/kitti_image_sizes.json' if USE_BUNDLED_IMAGE_SIZES else None,
    allow_install=ON_COLAB)
KITTI_ROOT = DATA_RESULT['training_root']
SETTINGS.update(kitti_root=KITTI_ROOT, image_sizes_json=DATA_RESULT['image_sizes_json'])
if DATA_RESULT['staged'] or Path(EXTRACT_DIR).expanduser().resolve() in Path(KITTI_ROOT).parents:
    SETTINGS['copy_data_to_local'] = False  # Đã ở local SSD; tránh chép thêm một bộ.
SETTINGS_PATH.write_text(json.dumps(SETTINGS, ensure_ascii=False, indent=2))
(WORK_DIR / 'dataset_preparation.json').write_text(json.dumps(DATA_RESULT, ensure_ascii=False, indent=2))
print('KITTI sẵn sàng:', KITTI_ROOT, '| số frame:', len(FRAME_IDS))

In [ ]:
# Cài NumPy/OpenCV/Numba sau khi KITTI sẵn sàng.
execute([sys.executable, RUNNER, 'setup', '--settings', SETTINGS_PATH])
PYTHON = WORK_DIR / 'env/bin/python'
(RUN_OUTPUT / 'notebook_settings.json').write_text(SETTINGS_PATH.read_text())
for name in ('pixor_training.py', 'openpcdet_benchmark.py'):
    (RUN_OUTPUT / name).write_text(BUNDLE[name])
shutil.copyfile(WORK_DIR / 'dataset_preparation.json', RUN_OUTPUT / 'dataset_preparation.json')

## 3. Audit dữ liệu, giữ nguyên split

Kiểm tra hash danh sách ID, trùng lặp/giao nhau, file XYZI/PNG/calibration/labels.
Chỉ training IDs tạo target và thống kê. Validation labels chỉ dùng khi tính AP.
Nếu chép sang ổ local, notebook giữ nguyên dữ liệu nguồn; không cần copy pixel ảnh.

PIXOR không dự đoán chiều cao và z. Khi xuất định dạng KITTI, notebook dùng **median
height/bottom-z của Car trong train** để chiếu hộp lên camera và tạo bbox 2D.
Đây là giả định xuất BEV được ghi trong report; ảnh hưởng projection/difficulty của
prediction, không tạo ra một model 3D. GT validation được giữ nguyên, không cắt ROI.

In [ ]:
execute([PYTHON, RUNNER, 'prepare', '--settings', SETTINGS_PATH], log=RUN_OUTPUT / 'prepare.log')
audit = json.loads((RUN_OUTPUT / 'data_audit.json').read_text())
print({name: len(ids) for name, ids in audit['splits'].items()})
print('Priors chỉ từ train:', audit['projection_prior'])

## 4. Smoke forward/backward trước khi train

Chạy một batch training qua model đầy đủ, loss và optimizer step trong process riêng;
kiểm tra reference IoU CUDA. Smoke không ghi checkpoint và không tính vào 50 epoch.
AMP có thể overflow gradient ở loss scale ban đầu dù loss hữu hạn. Smoke cho
GradScaler giảm scale và thử lại cùng batch tối đa 16 lần; chỉ báo `Smoke OK` sau
một optimizer step thực sự. Loss NaN/Inf hoặc gradient FP32 không hữu hạn vẫn dừng.

In [ ]:
execute([PYTHON, RUNNER, 'smoke', '--settings', SETTINGS_PATH], log=RUN_OUTPUT / 'smoke.log')

## 5. Train tổng 50 epoch

Mặc định SGD lr 0.01, momentum 0.9, weight decay 1e-4; giảm lr tại epoch 30/40.
Focal loss trên logits + Smooth-L1 trên pixel positive; normalized geometry target
cố định. Augmentation chỉ train: phản chiếu y, xoay ±5°, scale 0.95–1.05, tịnh tiến
BEV ±0.25 m. Không dùng GT database hoặc validation để huấn luyện.

Đây là recipe tái lập 50 epoch của notebook; không khẳng định trùng toàn bộ recipe
hoặc tái tạo score công bố của PIXOR gốc. `config.resolved.json` ghi mọi khác biệt.
Mỗi 5 epoch và tại epoch cuối chấm toàn bộ val để chọn `best.pth`; `last.pth` lưu
optimizer/scheduler/scaler để resume. Model tốt nhất có thể ở trước epoch 50.

In [ ]:
if RUN_TRAIN:
    execute([PYTHON, RUNNER, 'train', '--settings', SETTINGS_PATH], log=RUN_OUTPUT / 'train.log')
else:
    print('RUN_TRAIN=False: chưa train; đánh giá cần có checkpoint tương ứng.')

## 6. Đánh giá checkpoint best hoặc last, đo latency/FPS

Chấm đủ 3.769 frame, Car IoU 0.7, Easy/Moderate/Hard, R11/R40. Đo FP32 batch 1,
tắt TF32, bỏ 10 frame đầu khỏi timing nhưng vẫn giữ trong AP. Decode/NMS dùng
CPU OpenCV và bao gồm chuyển prediction về CPU. Preprocess gồm bin/calib, occupancy,
FOV và tạo tensor; kích thước ảnh đã cache. Ghi rõ khác biệt preprocessing khi so
với runner OpenPCDet. Projection/ghi file/tính AP nằm ngoài timing.

Đối chiếu với cột **Car APBEV R40** của năm model OpenPCDet. Không đưa PIXOR vào
cột AP3D hoặc macro Car/Pedestrian/Cyclist. Đây vẫn là kết quả validation được dùng
chọn checkpoint, không phải điểm trên KITTI hidden test.

In [ ]:
if RUN_EVALUATE:
    execute([PYTHON, RUNNER, 'evaluate', '--settings', SETTINGS_PATH], log=RUN_OUTPUT / 'evaluate.log')
    report_path = RUN_OUTPUT / 'evaluation' / Path(EVAL_CHECKPOINT).stem / 'report.json'
    report = json.loads(report_path.read_text())
    print('Checkpoint epoch:', report['epoch'])
    print('Car APBEV@0.7 R40:', report['accuracy']['R40'])
    print('FP32 pipeline:', report['latency']['input_to_detections'])
    print('Report:', report_path)

## 7. Xuất kết quả gọn

Checkpoint đã nằm tại output trên Drive; file ZIP dưới đây chỉ chứa config, manifest,
history, prediction, report và log. `last.pth` / `best.pth` cần giữ riêng để resume.

In [ ]:
# Không đóng gói dataset hoặc checkpoint lớn vào ZIP báo cáo.
import zipfile
archive = RUN_OUTPUT.parent / (RUN_NAME + '_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(RUN_OUTPUT.rglob('*')):
        if path.is_file() and 'checkpoints' not in path.relative_to(RUN_OUTPUT).parts:
            zipped.write(path, arcname=path.relative_to(RUN_OUTPUT))
print('ZIP:', archive)
print('Checkpoint:', RUN_OUTPUT / 'checkpoints')

Nguồn kiến trúc: [PIXOR — CVPR 2018](https://openaccess.thecvf.com/content_cvpr_2018/html/Yang_PIXOR_Real-Time_3D_CVPR_2018_paper.html).
Tham khảo implementation: [philip-huang/PIXOR](https://github.com/philip-huang/PIXOR).
Split/evaluator: [OpenPCDet](https://github.com/open-mmlab/OpenPCDet), revision
`233f849829b6ac19afb8af8837a0246890908755`, nhúng nguyên bản và kiểm tra SHA-256.

Nếu hết bộ nhớ: giảm batch trước khi train; nếu hết ổ local, tắt copy hoặc đổi ổ.
Nếu phiên bị ngắt: mở lại notebook, giữ cùng output/run/recipe và chạy từ đầu;
`RESUME='auto'` sẽ tiếp tục checkpoint epoch gần nhất, không train lại từ epoch 1.